# IMPORTS

In [ ]:
import torch
import numpy as np
import pandas as pd
import time
from PIL import Image
import shutil
import random
import torch
from torchvision import transforms
from facenet_pytorch import InceptionResnetV1
import tensorflow as tf
from art.estimators.classification import PyTorchClassifier
import tarfile
import os
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from PIL import Image
import shutil
import random
import time
import pickle
import sys
import re
import copy
import torch.optim as optim
from sklearn.metrics import (precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay)
from importlib import reload
from PIL import Image
import pandas as pd
import numpy as np
import tensorflow as tf
import csv
from torch.utils.data import WeightedRandomSampler
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch
import torch.nn as nn
from torchvision import transforms
import torchvision.models as models
from models.cydonia_resnet import resnet50
from models.adversarial_detector import AdversarialDetector, EfficientNetDetector

import torch.nn.functional as F

# Adversarial Robustness Toolbox (ART)
import art
from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import FastGradientMethod
from art.attacks.evasion import ProjectedGradientDescentPyTorch
from art.attacks.evasion import BasicIterativeMethod
from art.attacks.evasion import CarliniLInfMethod
from art.attacks.evasion import DeepFool
from art.defences.preprocessor import JpegCompression

print(f"TensorFlow version: {tf.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"ART version: {art.__version__}")
print("All libraries successfully imported.\n")

# CONFIGURATIONS

In [ ]:
# ========================================
# DIRECTORIES AND PATHS CONFIGURATIONS
# ========================================
DATA_DIR = './dataset' 
MODELS_DIR   = './models'
RESULTS_DIR = './results'
DEFENSE_SPLITS_DIR = './dataset_splits'
ADVERSARIAL_DATA_DIR = './adversarial_data'                                    # Base directory for all adversarial examples generated in the project
DATA_DIR_100 = os.path.join(DATA_DIR, 'dataset_1_per_person')                  # Dataset dir for DF and Carlini Wagner with 100 images (1 per subject)
DATA_DIR_1000 = os.path.join(DATA_DIR, 'dataset_100_subjects')                 # Dataset dir for PGD, FGSM and BIM attacks with 1000 images (10 per subject)
CSV_PATH = os.path.join(DATA_DIR, 'selected_100_subjects.csv')                 # CSV file with metadata for the 100 subjects used in the dataset
EXTRA_GENUINE_DIR  = './dataset_extra_genuine'    
DEFENSE_DATASET_DIR = './detector_dataset'
DETECTOR_TEST_DIR = os.path.join(RESULTS_DIR, 'Detector_Defense')


# ========================================
# DEVICE CONFIGURATION
# ========================================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {DEVICE}")


# ========================================
# ATTACK CONFIGURATIONS
# ========================================
BATCH_SIZE = 64
NUM_CLASSES = 8631

# ========================================
# DEFENSE CONFIGURATIONS
# ========================================
JPEG_QUALITY_VALUES = [10, 50, 80]
BATCH_SIZE_ADV_DETECTOR = 256
SEED                 = 42
TRAIN_RATIO          = 0.70
VAL_RATIO            = 0.20         # TEST_RATIO is implicitly 1 - TRAIN_RATIO - VAL_RATIO = 0.10
N_EXTRA_PER_SUBJECT = 20
random.seed(SEED)
ALL_ATTACK_FAMILIES = ['FGSM', 'BIM', 'PGD', 'CW', 'DeepFool']
DETECTOR_TO_USE = 'EfficientNet'  # 'EfficientNet' 'MobileNetV3'
DETECTOR_PATH = os.path.join(MODELS_DIR, 'detector_A_mobilenet.pth') if DETECTOR_TO_USE == 'MobileNetV3' else os.path.join(MODELS_DIR, 'detector_B_efficientnet.pth')
DETECTOR_THRESHOLD = 0.500
FINETUNED_DETECTOR_THRESHOLD = 0.110
# DEFENSE TRAINING CONFIGURATION
CLASS_WEIGHTED_LOSS = False
RUN_TRAINING_A     = False   # Experiment A: MobileNetV3-Large
RUN_TRAINING_B     = False   # Experiment B: EfficientNet-B0
RUN_BOTH_TRAININGS = False    # If True, forces both A and B regardless of individual flags.
LR           = 1e-4
EPOCHS = 50
PATIENCE_A = 20
PATIENCE_B = 20
WEIGHT_DECAY       = 1e-4
ATTACK_CONFIGS = [
    {'folder': 'FGSM',                          'subfolder': 'eps_0_100', 'structure': 'class_subfolders', 'n_samples': 200}, #acc=0.1
    {'folder': 'FGSM',                          'subfolder': 'eps_0_010', 'structure': 'class_subfolders', 'n_samples': 200}, #acc=6.7
    {'folder': 'FGSM',                          'subfolder': 'eps_0_005', 'structure': 'class_subfolders', 'n_samples': 200}, #acc=22.8
    {'folder': 'FGSM_Targeted_Fixed',           'subfolder': 'eps_0_030', 'structure': 'class_subfolders', 'n_samples': 200}, #asr=8.0
    {'folder': 'FGSM_Targeted_Fixed',           'subfolder': 'eps_0_100', 'structure': 'class_subfolders', 'n_samples': 200}, #asr=4.9
    {'folder': 'FGSM_Targeted_Fixed',           'subfolder': 'eps_0_010', 'structure': 'class_subfolders', 'n_samples': 200}, #asr=1.7
    {'folder': 'PGD_nri3_iter5',                'subfolder': 'eps_0_100', 'structure': 'class_subfolders', 'n_samples': 200}, #acc=0.0
    {'folder': 'PGD_nri5_iter5',                'subfolder': 'eps_0_010', 'structure': 'class_subfolders', 'n_samples': 200}, #acc=4.9
    {'folder': 'PGD_nri1_iter5',                'subfolder': 'eps_0_005', 'structure': 'class_subfolders', 'n_samples': 200}, #acc=27.2
    {'folder': 'PGD_Targeted_Fixed_nri1_iter40','subfolder': 'eps_0_100', 'structure': 'class_subfolders', 'n_samples': 200}, #asr=100.0
    {'folder': 'PGD_Targeted_Fixed_nri3_iter5', 'subfolder': 'eps_0_030', 'structure': 'class_subfolders', 'n_samples': 200}, #asr=82.4
    {'folder': 'PGD_Targeted_Fixed_nri5_iter5', 'subfolder': 'eps_0_010', 'structure': 'class_subfolders', 'n_samples': 200}, #asr=6.4
    {'folder': 'BIM_iter10',                    'subfolder': 'eps_0_100', 'structure': 'class_subfolders', 'n_samples': 200}, #acc=0.0
    {'folder': 'BIM_iter5',                     'subfolder': 'eps_0_010', 'structure': 'class_subfolders', 'n_samples': 200}, #acc=2.6
    {'folder': 'BIM_iter20',                    'subfolder': 'eps_0_005', 'structure': 'class_subfolders', 'n_samples': 200}, #acc=14.8
    {'folder': 'BIM_Targeted_Fixed_iter10',     'subfolder': 'eps_0_100', 'structure': 'class_subfolders', 'n_samples': 200}, #asr=100.0
    {'folder': 'BIM_Targeted_Fixed_iter5',      'subfolder': 'eps_0_030', 'structure': 'class_subfolders', 'n_samples': 200}, #asr=90.8
    {'folder': 'BIM_Targeted_Fixed_iter10',     'subfolder': 'eps_0_010', 'structure': 'class_subfolders', 'n_samples': 200}, #asr=18.6
    {'folder': 'CW_Untargeted',                 'subfolder': 'iter_1_lr_0_01_c_0_0',  'structure': 'flat', 'n_samples': 100}, #acc=28.0
    {'folder': 'CW_Untargeted',                 'subfolder': 'iter_1_lr_0_1_c_0_0',  'structure': 'flat', 'n_samples': 100}, #acc=2.0
    {'folder': 'CW_Untargeted',                 'subfolder': 'iter_1_lr_0_05_c_0_0',  'structure': 'flat', 'n_samples': 100}, #acc=3.0
    {'folder': 'CW_Untargeted',                 'subfolder': 'iter_5_lr_0_001_c_0_0',  'structure': 'flat', 'n_samples': 100}, #acc=57.0
    {'folder': 'CW_Untargeted',                 'subfolder': 'iter_5_lr_0_01_c_0_0',  'structure': 'flat', 'n_samples': 100}, #acc=4.0
    {'folder': 'CW_Untargeted',                 'subfolder': 'iter_5_lr_0_1_c_0_0',  'structure': 'flat', 'n_samples': 100}, #acc=2.0
    {'folder': 'CW_Untargeted',                 'subfolder': 'iter_5_lr_0_05_c_0_0',  'structure': 'flat', 'n_samples': 100}, #acc=3.0
    {'folder': 'CW_Untargeted',                 'subfolder': 'iter_10_lr_0_001_c_0_0',  'structure': 'flat', 'n_samples': 100}, #acc=26.0
    {'folder': 'CW_Untargeted',                 'subfolder': 'iter_10_lr_0_01_c_0_0',  'structure': 'flat', 'n_samples': 100}, #acc=4.0
    {'folder': 'CW_Untargeted',                 'subfolder': 'iter_10_lr_0_1_c_0_0',  'structure': 'flat', 'n_samples': 100}, #acc=2.0
    {'folder': 'CW_Untargeted',                 'subfolder': 'iter_20_lr_0_01_c_0_0',  'structure': 'flat', 'n_samples': 100}, #acc=4.0
    {'folder': 'CW_Untargeted',                 'subfolder': 'iter_20_lr_0_1_c_0_0',  'structure': 'flat', 'n_samples': 100}, #acc=2.0
    {'folder': 'DeepFool',                      'subfolder': 'iter_1_eps_1e-06',     'structure': 'flat', 'n_samples': 99}, #acc=14.14
    {'folder': 'DeepFool',                      'subfolder': 'iter_5_eps_1e-06',    'structure': 'flat', 'n_samples': 97}, #acc=2.06
    {'folder': 'DeepFool',                      'subfolder': 'iter_50_eps_1e-06',     'structure': 'flat', 'n_samples': 96}, #acc=1.04
    {'folder': 'DeepFool',                      'subfolder': 'iter_100_eps_1e-06',     'structure': 'flat', 'n_samples': 96}, #acc=1.04
    {'folder': 'DeepFool',                      'subfolder': 'iter_20_eps_1e-06',    'structure': 'flat', 'n_samples': 16}, #acc=1.04
    {'folder': 'DeepFool',                      'subfolder': 'iter_200_eps_1e-06',    'structure': 'flat', 'n_samples': 98}, #acc=1.02
    {'folder': 'DeepFool',                      'subfolder': 'iter_3_eps_1e-06',    'structure': 'flat', 'n_samples': 98}, #acc=4.08
    {'folder': 'DeepFool',                      'subfolder': '1000_iter_1_eps_1e-06',     'structure': 'class_subfolders', 'n_samples': 200}, #acc=16.45
    {'folder': 'DeepFool',                      'subfolder': '1000_iter_5_eps_1e-06',     'structure': 'class_subfolders', 'n_samples': 200}, #acc=2.27
    {'folder': 'DeepFool',                      'subfolder': '1000_iter_10_eps_1e-06',    'structure': 'class_subfolders', 'n_samples': 200}, #acc=1.65
]


# ========================================
# FLAG CONFIGURATIONS
# ========================================
# Configuration flag for baseline evaluation
RUN_BASELINE_EVAL = True        # Leave True ALWAYS

# Fixed target class used for all targeted attacks 
FIXED_TARGET_CLASS = 2026

# Configuration flags for adversarial attacks
RUN_FGSM_GENERIC    = False
RUN_PGD_GENERIC     = False
RUN_BIM_GENERIC     = False
RUN_CW_GENERIC      = False
RUN_DEEPFOOL_GENERIC= False
RUN_FGSM_SPECIFIC   = False
RUN_BIM_SPECIFIC    = False
RUN_PGD_SPECIFIC    = False
RUN_CW_SPECIFIC     = False

# Configuration flags for transferability evaluation
RUN_FGSM_TRANSFERABILITY_GENERIC    = False
RUN_BIM_TRANSFERABILITY_GENERIC     = False
RUN_PGD_TRANSFERABILITY_GENERIC     = False
RUN_DEEPFOOL_TRANSFERABILITY_GENERIC = False
RUN_CW_TRANSFERABILITY_GENERIC      = False
RUN_FGSM_TRANSFERABILITY_SPECIFIC   = False
RUN_BIM_TRANSFERABILITY_SPECIFIC    = False
RUN_PGD_TRANSFERABILITY_SPECIFIC    = False
RUN_CW_TRANSFERABILITY_SPECIFIC     = False


# Configuration flag for defense
RUN_DEFENSE_DATASET_PREPARATION = False
RUN_DETECTOR_TRAINING = False
RUN_DEFENSE_DETECTOR_TESTING = True
RUN_DEFENSE_DETECTOR_TESTING_WITH_NN1 = True
RUN_JPEG_DEFENSE_EVAL = False



# ========================================
# MODELS INITIALIZATIONS
# ========================================
print("\nLoading InceptionResnetV1 (NN1)")
nn1 = InceptionResnetV1(pretrained='vggface2').eval().to(DEVICE)
nn1.classify = True

print("\nLoading ResNet-50 (NN2)")
NN2_WEIGHT_PATH = os.path.join(MODELS_DIR, 'resnet50_ft_weight.pkl')
if not os.path.exists(NN2_WEIGHT_PATH):
    raise FileNotFoundError(
        f"{NN2_WEIGHT_PATH} Not Found: Please download the ResNet-50 weights from "
        "https://drive.google.com/file/d/1A94PAAnwk6L7hXdBXLFosB_s0SzEhAFU/view "
        "and place them in the ./models/ directory.\n"
    )

# Initialize the custom ResNet-50 architecture.
nn2 = resnet50(num_classes=NUM_CLASSES)                                             

with open(NN2_WEIGHT_PATH, 'rb') as f:
    obj = f.read()
    raw_weights = pickle.loads(obj, encoding='latin1')

clean_state_dict = {}
for k, v in raw_weights.items():
    clean_key = k.replace("module.", "")
    clean_state_dict[clean_key] = torch.from_numpy(v)
nn2.load_state_dict(clean_state_dict)
nn2 = nn2.eval().to(DEVICE)
print("NN2 (ResNet-50) weights loaded successfully!")


# Load official labels
fpath = tf.keras.utils.get_file(
    fname='rcmalli_vggface_labels_v2.npy',
    origin="https://github.com/rcmalli/keras-vggface/releases/download/v2.0/rcmalli_vggface_labels_v2.npy",
    cache_dir=str(MODELS_DIR),
    cache_subdir='.'
)
LABELS = np.load(fpath, allow_pickle=True)
print(f"Found {len(LABELS)} classes.")

# 1. DATASET PREPARATION

This step verifies whether the datasets have already been prepared. If the extracted
dataset folders and the subject CSV are found locally, the step is skipped entirely.

Two datasets are prepared:
- `dataset_100_subjects`: 10 images per subject (1000 total), used for FGSM, BIM, and PGD.
- `dataset_1_per_person`: 1 image per subject (100 total), used for DeepFool and Carlini-Wagner,
  which are significantly slower and cannot be run on the full 1000-image dataset.

If neither dataset is found, the metadata CSV is read to select 100 subjects from the
VGGFace2 training set, images are extracted from the local TAR archive, and the
1-per-person subset is derived from the extracted data.



In [ ]:
# ==============================================================================
#  Dataset Preparation
# ==============================================================================

print("Starting Dataset Preparation...")

# ------------------------------------------------------------------------------
# 1.0 Check if both datasets are already prepared
# ------------------------------------------------------------------------------
if os.path.exists(CSV_PATH) and os.path.exists(DATA_DIR_1000) and os.path.exists(DATA_DIR_100):
    selected_subjects = pd.read_csv(CSV_PATH)
    print(f"All datasets already prepared. Loaded {len(selected_subjects)} subjects from existing CSV.")

else:
    # --------------------------------------------------------------------------
    # 1.1 Process Metadata CSV and select 100 subjects
    # --------------------------------------------------------------------------
    if not os.path.exists(CSV_PATH):
        print("Metadata CSV not found. Processing VGGFace2 metadata...")

        LOCAL_META_CSV = os.path.join(DATA_DIR, 'vggface2_meta.csv')
        if not os.path.exists(LOCAL_META_CSV):
            raise FileNotFoundError(
                f"Metadata CSV not found at {LOCAL_META_CSV}. "
                "Place vggface2_meta.csv in the dataset directory before running this step."
            )

        df = pd.read_csv(LOCAL_META_CSV, on_bad_lines='skip', skipinitialspace=True)
        df.columns = df.columns.str.strip()

        if 'Sample_Num' not in df.columns:
            df.columns = ['Class_ID', 'Name', 'Sample_Num', 'Flag', 'Gender']

        df['Sample_Num'] = pd.to_numeric(df['Sample_Num'], errors='coerce')
        df['Flag']       = pd.to_numeric(df['Flag'],       errors='coerce')

        df_filtered = df[(df['Sample_Num'] >= 10) & (df['Flag'] == 1)].copy()
        df_filtered.loc[:, 'Class_ID'] = df_filtered['Class_ID'].astype(str).str.strip()

        selected_subjects = df_filtered.sample(n=100, random_state=42)
        target_ids = {class_id: 0 for class_id in selected_subjects['Class_ID']}

        # Normalize commas in Name column to match the official VGGFace2 label format
        selected_subjects['Name'] = (selected_subjects['Name']
                                    .str.replace(',_', '_', regex=False)
                                    .str.replace(',',  '_', regex=False))

        selected_subjects.to_csv(CSV_PATH, index=False)
        print(f"Saved metadata for 100 selected subjects to: {CSV_PATH}")
    else:
        selected_subjects = pd.read_csv(CSV_PATH)
        target_ids = {class_id: 0 for class_id in selected_subjects['Class_ID']}
        print(f"Loaded {len(selected_subjects)} subjects from existing CSV.")

    # --------------------------------------------------------------------------
    # 1.2 Extract 10 images per subject from the TAR archive (1000 images total)
    # --------------------------------------------------------------------------
    if not os.path.exists(DATA_DIR_1000):
        os.makedirs(DATA_DIR_1000, exist_ok=True)

        LOCAL_TAR_PATH = os.path.join(DATA_DIR, 'vggface2.tar.gz')
        if not os.path.exists(LOCAL_TAR_PATH):
            raise FileNotFoundError(
                f"TAR archive not found at {LOCAL_TAR_PATH}. "
                "Place vggface2.tar.gz in the dataset directory before running this step."
            )
        print("\nStarting extraction of 10 images per subject...")
        IMAGES_PER_SUBJECT = 10
        total_needed       = len(target_ids) * IMAGES_PER_SUBJECT
        total_extracted    = 0
        try:
            with tarfile.open(LOCAL_TAR_PATH, 'r:*') as tar:
                for member in tar.getmembers():
                    if total_extracted >= total_needed:
                        print(f"\nExtraction complete: {total_extracted} images collected.")
                        break

                    if member.isfile():
                        parts = member.name.split('/')
                        if len(parts) >= 2 and parts[-1].lower().endswith(('.jpg', '.jpeg', '.png')):
                            class_id = parts[-2].strip()

                            if class_id in target_ids and target_ids[class_id] < IMAGES_PER_SUBJECT:
                                dest_folder = os.path.join(DATA_DIR_1000, class_id)
                                os.makedirs(dest_folder, exist_ok=True)

                                extracted_f = tar.extractfile(member)
                                if extracted_f is not None:
                                    with open(os.path.join(dest_folder, parts[-1]), 'wb') as out:
                                        shutil.copyfileobj(extracted_f, out)

                                    target_ids[class_id] += 1
                                    total_extracted      += 1

                                    if total_extracted % 100 == 0:
                                        print(f"  Progress: {total_extracted}/{total_needed} images extracted...")

        except Exception as e:
            print(f"\nAn error occurred during extraction: {e}")

        print("10-images-per-subject dataset preparation completed.")
    else:
        print(f"Dataset '{DATA_DIR_1000}' already exists. Skipping extraction.")

    # --------------------------------------------------------------------------
    # 1.3 Build 1-per-person subset from the extracted dataset
    # --------------------------------------------------------------------------
    if not os.path.exists(DATA_DIR_100):
        os.makedirs(DATA_DIR_100, exist_ok=True)

        print("\nBuilding 1-per-person subset for DeepFool and Carlini-Wagner attacks...")

        rng             = random.Random(42)
        processed_count = 0

        for subject_folder in os.listdir(DATA_DIR_1000):
            subject_path = os.path.join(DATA_DIR_1000, subject_folder)

            if os.path.isdir(subject_path):
                images = [
                    f for f in os.listdir(subject_path)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
                ]

                if images:
                    selected_image = rng.choice(images)
                    ext            = os.path.splitext(selected_image)[1]
                    new_filename   = f"{subject_folder}{ext}"

                    shutil.copy2(
                        os.path.join(subject_path, selected_image),
                        os.path.join(DATA_DIR_100, new_filename)
                    )
                    processed_count += 1

        print(f"1-per-person subset completed: {processed_count} images saved to {DATA_DIR_100}.")
    else:
        print(f"Dataset '{DATA_DIR_100}' already exists. Skipping subset creation.")

print("\nDataset preparation completed.")

## Adversarial Attacks Setup and Helper Functions

This step initializes all shared components required by the adversarial attack pipeline:
the ART classifier wrapper around NN1, the attack preprocessing transform, and three
helper functions used by every subsequent attack step (image saving, Security Evaluation
Curve plotting, and adversarial archive management).


In [ ]:
# ==============================================================================
#  Adversarial Attacks Setup and Helper Functions
# ==============================================================================

print("Initializing adversarial attack setup and helper functions...")

# ------------------------------------------------------------------------------
# 3.0.1 ART Classifier Wrapper
# ------------------------------------------------------------------------------
criterion = nn.CrossEntropyLoss()

# ART expects inputs in [0, 1] and handles the normalization shift to [-1, 1] internally.
art_classifier = PyTorchClassifier(
    model=nn1,
    clip_values=(0.0, 1.0),
    loss=criterion,
    optimizer=None,
    input_shape=(3, 160, 160),
    nb_classes=NUM_CLASSES,
    preprocessing=(np.array([0.5, 0.5, 0.5]), np.array([0.5, 0.5, 0.5])),
    device_type=DEVICE
)

# Preprocessing for attack generation: only resize and ToTensor.
# Normalization to [-1, 1] is delegated to the ART wrapper above.
preprocess_attack = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor()
])

# ------------------------------------------------------------------------------
# Helper 1: Label Normalization
# ------------------------------------------------------------------------------
def normalize_label(label: str) -> str:
    """
    Normalizes a subject name string for safe comparison against official VGGFace2 labels.
    Converts to lowercase, replaces spaces with underscores, and removes commas.
    """
    return label.lower().replace(" ", "_").replace(",_", "_").replace(",", "_")

# ------------------------------------------------------------------------------
# Helper 2: Image Saving
# ------------------------------------------------------------------------------
def save_adversarial_images(adv_examples, true_labels, filenames, attack_name, epsilon):
    """
    Saves adversarial examples generated by ART as PNG image files.

    ART outputs arrays in the [0.0, 1.0] range due to clip_values=(0.0, 1.0).
    Each image is scaled to [0, 255], transposed from (C, H, W) to (H, W, C),
    and saved under ADVERSARIAL_DATA_DIR//eps_//.png.
    """
    eps_folder_name = f"eps_{epsilon:.3f}".replace('.', '_')
    save_dir = os.path.join(ADVERSARIAL_DATA_DIR, attack_name, eps_folder_name)
    
    # print(f"\nSaving adversarial examples to: {save_dir}")
    
    for i in range(len(adv_examples)):
        class_id   = str(true_labels[i]).strip()
        base_name  = os.path.splitext(str(filenames[i]))[0]
        subject_dir = os.path.join(save_dir, class_id)
        os.makedirs(subject_dir, exist_ok=True)

        img_array = (adv_examples[i] * 255).astype(np.uint8)
        img_array = np.transpose(img_array, (1, 2, 0))
        Image.fromarray(img_array, mode='RGB').save(
            os.path.join(subject_dir, f"{base_name}.png")
        )

# ------------------------------------------------------------------------------
# Helper 3: Security Evaluation Curve
# ------------------------------------------------------------------------------
def plot_security_evaluation_curve(epsilons, accuracies, attack_name, save_dir, baseline_acc):
    """
    Generates and saves a Security Evaluation Curve (SEC) for a given attack.

    The curve plots model accuracy as a function of the perturbation budget epsilon,
    connecting the clean baseline point (epsilon=0) to the first attack measurement
    and continuing through all evaluated epsilon values.
    """
    os.makedirs(save_dir, exist_ok=True)

    plt.figure(figsize=(9, 6))

    plt.plot([0.0, epsilons[0]], [baseline_acc, accuracies[0]],
            linestyle='-', color='#ff7f0e', linewidth=1.5)
    plt.plot(epsilons, accuracies,
            marker='o', linestyle='-', color='#ff7f0e',
            linewidth=1.5, markersize=6, label=f'NN1 Performance ({attack_name})')
    plt.plot(0.0, baseline_acc,
            marker='o', markerfacecolor='#ff7f0e', markeredgecolor='#2ca02c',
            markeredgewidth=1, markersize=6, linestyle='None',
            label=f'Baseline Accuracy ({baseline_acc:.2f}%)')

    plt.title(f'Security Evaluation Curve - {attack_name}', fontsize=16, fontweight='bold', pad=15)
    plt.xlabel(r'Perturbation Budget ($\epsilon$ - $L_\infty$ Norm)', fontsize=14)
    plt.ylabel('Model Accuracy (%)', fontsize=14)
    plt.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
    plt.xticks(np.arange(0.0, 0.11, 0.01), fontsize=12)
    plt.yticks(np.arange(0, 101, 10), fontsize=12)
    plt.ylim(-5, 105)
    plt.xlim(-0.005, 0.105)
    plt.axvline(x=0.1, color='black', linestyle=':', linewidth=2,
                label=r'Max Allowable $\epsilon$ (0.1)')
    plt.legend(loc='upper right', fontsize=12)
    plt.tight_layout()

    plot_path = os.path.join(save_dir, f'{attack_name}_SEC.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n[INFO] SEC graph saved at: {plot_path}")
    
    
    
def plot_iteration_evaluation_curve(iterations, accuracies, attack_name, save_dir, fixed_eps):
    """
    Generates and saves a curve plotting model accuracy against the number of PGD iterations.
    """
    os.makedirs(save_dir, exist_ok=True)

    plt.figure(figsize=(9, 6))

    plt.plot(iterations, accuracies,
            marker='s', linestyle='-', color='#d62728',
            linewidth=1.5, markersize=6, label=f'NN1 Performance ({attack_name})')

    plt.title(f'Iteration Impact Curve - {attack_name} (Fixed $\epsilon$ = {fixed_eps})', fontsize=16, fontweight='bold', pad=15)
    plt.xlabel('Number of Iterations (max_iter)', fontsize=14)
    plt.ylabel('Model Accuracy (%)', fontsize=14)
    plt.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
    plt.xticks(iterations, fontsize=12)
    plt.yticks(np.arange(0, 101, 10), fontsize=12)
    plt.ylim(-5, 105)
    plt.legend(loc='upper right', fontsize=12)
    plt.tight_layout()

    plot_path = os.path.join(save_dir, f'{attack_name}_Iteration_Impact.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"\n[INFO] Iteration impact graph saved at: {plot_path}")

print("Adversarial attack setup completed successfully.")



# ------------------------------------------------------------------------------
# Helper 4: Least Likely Class Index
# ------------------------------------------------------------------------------
def get_least_likely_indices(x_batch_clean):
    """
    Returns the least likely class index for each image in the batch,
    defined as the class assigned the lowest softmax probability by the
    clean model. Used as the target for error-specific least-likely attacks.
    """
    preds = art_classifier.predict(x_batch_clean)
    return np.argmin(preds, axis=1).tolist()


# ------------------------------------------------------------------------------
# Helper 5: Targeted Security Evaluation Curve
# ------------------------------------------------------------------------------
def plot_targeted_sec(epsilons, attack_success_rates, attack_name, save_dir):
    """
    Generates and saves a Targeted Security Evaluation Curve (SEC).

    Unlike the standard SEC which plots accuracy (decreasing with epsilon),
    the targeted SEC plots Attack Success Rate (ASR), which increases with
    epsilon as the attack becomes more capable of forcing the target class.
    """
    os.makedirs(save_dir, exist_ok=True)

    fig, ax = plt.subplots(figsize=(9, 6))

    ax.plot([0.0, epsilons[0]], [0.0, attack_success_rates[0]],
            linestyle='-', color='#d62728', linewidth=1.5)
    ax.plot(epsilons, attack_success_rates,
            marker='o', linestyle='-', color='#d62728',
            linewidth=1.5, markersize=6,
            label=f'Attack Success Rate ({attack_name})')
    ax.plot(0.0, 0.0,
            marker='o', markerfacecolor='#d62728',
            markeredgecolor='#2ca02c', markeredgewidth=1,
            markersize=6, linestyle='None',
            label='Baseline ASR (~0%)')

    ax.set_title(f'Targeted Security Evaluation Curve — {attack_name}', fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
    ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
    ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
    ax.set_xticks(np.arange(0.0, 0.11, 0.01))
    ax.set_yticks(np.arange(0, 101, 10))
    ax.tick_params(axis='both', labelsize=12)
    ax.set_ylim(-5, 105)
    ax.set_xlim(-0.005, 0.105)
    ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2, label=r'Max Allowable $\epsilon$ (0.1)')
    ax.legend(loc='upper left', fontsize=12)
    plt.tight_layout()

    plot_path = os.path.join(save_dir, f'{attack_name}_SEC.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n[INFO] Targeted SEC graph saved at: {plot_path}")


# ------------------------------------------------------------------------------
# Helper 6: Save and Load Least Likely Targets
# ------------------------------------------------------------------------------
def save_least_likely_targets(targets, attack_name, filenames=None):
    """
    Saves the least likely target indices computed by NN1 during attack
    generation. If filenames are provided, saves as a dictionary mapping
    'class_id/filename_stem' to target index for unique, order-independent lookup.
    """
    save_path = os.path.join(RESULTS_DIR, attack_name,f'{attack_name}_ll_targets.npy')
    if filenames is not None:
        mapping = {}
        for f, t in zip(filenames, targets):
            # Key: class_id/filename_stem — unique across all subjects
            class_id  = os.path.basename(os.path.dirname(f))
            file_stem = os.path.splitext(os.path.basename(f))[0]
            key       = f"{class_id}/{file_stem}"
            mapping[key] = int(t)
        np.save(save_path, mapping)
        print(f"[INFO] Least likely targets saved as filename mapping at: {save_path}")
    else:
        np.save(save_path, np.array(targets))
        print(f"[INFO] Least likely targets saved as array at: {save_path}")


def load_least_likely_targets(attack_name):
    """
    Loads least likely targets. Returns a dict {'class_id/filename_stem': target_index}
    if saved as mapping, or a list if saved as plain array (legacy).
    """
    load_path = os.path.join(RESULTS_DIR, attack_name,f'{attack_name}_ll_targets.npy')
    if not os.path.exists(load_path):
        raise FileNotFoundError(
            f"Least likely targets not found at {load_path}. "
            f"Re-run the {attack_name} generation step first."
        )
    data = np.load(load_path, allow_pickle=True)
    if data.ndim == 0:
        return data.item()
    return data.tolist()


# ------------------------------------------------------------------------------
# Helper 7: Clean Predicted Label Extraction
# ------------------------------------------------------------------------------
def clean_pred_label(pred_idx):
    """
    Helper function to parse output label arrays uniformly across 
    different formats and return a clean string label.
    """
    raw_l = LABELS[pred_idx]
    if isinstance(raw_l, np.ndarray):
        raw_l = raw_l.flatten()[0]
    if isinstance(raw_l, bytes):
        raw_l = raw_l.decode('utf-8')
    return str(raw_l).replace("b'", "").replace("'", "").replace("[", "").replace("]", "").strip().lower().replace(" ", "_").replace(",_", "_").replace(",", "_")


# 2. BASELINE EVALUATION

This step loads the target model NN1 (InceptionResnetV1 pretrained on VGGFace2) and NN2 (resnet50)
and evaluates its accuracy on the 1000-image clean test set. The resulting baseline
accuracy is the reference point against which all adversarial attacks will be measured.

In [ ]:
if RUN_BASELINE_EVAL:
    # ------------------------------------------------------------------------------
    #  Step 2: Baseline Evaluation on Both Datasets
    # ------------------------------------------------------------------------------
    # ------------------------------------------------------------------------------
    # 2.1 Load Metadata Mapping (ID to Name)
    # ------------------------------------------------------------------------------
    if not os.path.exists(CSV_PATH):
        raise FileNotFoundError(f"Metadata CSV not found at {CSV_PATH}. Run Step 1 first.")

    selected_subjects = pd.read_csv(CSV_PATH)
    id_to_name = dict(zip(
        selected_subjects['Class_ID'].astype(str).str.strip(),
        selected_subjects['Name'].astype(str).str.strip()
    ))
    print("Metadata loaded for label matching.")

    # ------------------------------------------------------------------------------
    # 2.2 Define Preprocessing
    # ------------------------------------------------------------------------------
    preprocess_nn1 = transforms.Compose([
        transforms.Resize((160, 160)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])
    # Nearest Neighbor interpolation preserves the exact values ​​of the original pixels without averaging, 
    # thus better preserving the L∞ perturbation. It is the most correct approach when you want to measure 
    # pure transferability without artificial attenuation.
    preprocess_nn2 = transforms.Compose([
        transforms.Resize(288, interpolation=transforms.InterpolationMode.NEAREST),                  # Resize shortest side to 288 with Nearest Neighbor interpolation
        transforms.CenterCrop(224),              # Center crop to 224x224
        transforms.ToTensor(),                   # [0, 255] -> [0.0, 1.0], shape (C, H, W)
        transforms.Lambda(lambda x: x * 255.0), # Rescale back to [0, 255]
        transforms.Lambda(lambda x: x[[2, 1, 0], :, :]),  # RGB -> BGR
        transforms.Normalize(
            mean=[91.4953, 103.8827, 131.0912],  # BGR mean subtraction (training values)
            std=[1.0, 1.0, 1.0]                  # No std scaling — pure mean subtraction
        )
    ])

    # ------------------------------------------------------------------------------
    # 2.3 Dataset Configurations Definitions
    # ------------------------------------------------------------------------------
    # Define targets to evaluate both datasets sequentially
    datasets_to_evaluate = [
        {"name": "DATASET_1000", "path": DATA_DIR_1000, "structure": "nested"},
        {"name": "DATASET_100",  "path": DATA_DIR_100,  "structure": "flat"}
    ]
    
    baseline_results = {}

    for target_ds in datasets_to_evaluate:
        ds_name = target_ds["name"]
        ds_path = target_ds["path"]
        ds_structure = target_ds["structure"]

        if not os.path.exists(ds_path):
            print(f"\n[WARNING] Path for {ds_name} does not exist at {ds_path}. Skipping evaluation.")
            continue

        current_image_paths = []
        current_true_ids = []

        # --------------------------------------------------------------------------
        # 2.3.1 Build Image List based on directory structure
        # --------------------------------------------------------------------------
        if ds_structure == "nested":
            # For 1000 images dataset (Subject Folders -> Images)
            for class_id in os.listdir(ds_path):
                class_folder = os.path.join(ds_path, class_id)
                if not os.path.isdir(class_folder):
                    continue
                for img_name in os.listdir(class_folder):
                    if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                        current_image_paths.append(os.path.join(class_folder, img_name))
                        current_true_ids.append(class_id.strip())
        
        elif ds_structure == "flat":
            # For 100 images dataset (Flat Folder -> ClassID.jpg)
            for img_name in os.listdir(ds_path):
                if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                    current_image_paths.append(os.path.join(ds_path, img_name))
                    # Extract class target index from the filename (e.g., 'n001234.jpg' -> 'n001234')
                    class_id = os.path.splitext(img_name)[0]
                    current_true_ids.append(class_id.strip())

        # --------------------------------------------------------------------------
        # 2.4 Evaluate Current Dataset Baseline Accuracy
        # --------------------------------------------------------------------------
        print(f"\nEvaluating baseline accuracy on {ds_name} ({len(current_image_paths)} images)...")
        correct_preds_nn1 = 0
        correct_preds_nn2 = 0

        with tqdm(total=len(current_image_paths), desc=f"Evaluating {ds_name}", unit="img") as pbar:
            for img_path, true_id in zip(current_image_paths, current_true_ids):
                try:
                    img        = Image.open(img_path).convert('RGB')
                    img_tensor_nn1 = preprocess_nn1(img).unsqueeze(0).to(DEVICE)
                    img_tensor_nn2 = preprocess_nn2(img).unsqueeze(0).to(DEVICE)

                    with torch.no_grad():
                        # NN1 Evaluation
                        preds_1    = nn1(img_tensor_nn1)
                        pred_idx_1 = np.array(preds_1[0].detach().cpu().numpy()).argmax()
                        # NN2 Evaluation 
                        preds_2    = nn2(img_tensor_nn2)
                        pred_idx_2 = np.array(preds_2[0].detach().cpu().numpy()).argmax()

                    # Resolve True Label Mapping
                    true_name  = id_to_name.get(true_id, "Unknown")
                    true_clean = true_name.lower().replace(" ", "_").replace(",_", "_").replace(",", "_")

                    pred_clean_1 = clean_pred_label(pred_idx_1)
                    pred_clean_2 = clean_pred_label(pred_idx_2)

                    if pred_clean_1 == true_clean:
                        correct_preds_nn1 += 1
                        
                    if pred_clean_2 == true_clean:
                        correct_preds_nn2 += 1

                except Exception as e:
                    print(f"Error processing {img_path}: {e}")

                pbar.update(1)

        total_images   = len(current_image_paths)
        clean_accuracy_nn1      = (correct_preds_nn1 / total_images) * 100 if total_images > 0 else 0.0
        clean_accuracy_nn2      = (correct_preds_nn2 / total_images) * 100 if total_images > 0 else 0.0
        
        # Save results into temporary tracking dictionary
        baseline_results[ds_name] = {
            "total":       total_images,
            "nn1_acc":     clean_accuracy_nn1,
            "nn1_correct": correct_preds_nn1,
            "nn2_acc":     clean_accuracy_nn2,
            "nn2_correct": correct_preds_nn2,
            "paths":       current_image_paths,
            "ids":         current_true_ids
        }

    # ------------------------------------------------------------------------------
    # 2.6 Assign Global Variables and Print Reports
    # ------------------------------------------------------------------------------
    # Extract specific values for each global placeholder variable safely
    BASELINE_ACC_1000 = baseline_results.get("DATASET_1000", {}).get("nn1_acc", 0.0)
    BASELINE_ACC_100  = baseline_results.get("DATASET_100", {}).get("nn1_acc", 0.0)
    
    BASELINE_ACC_NN2_1000 = baseline_results.get("DATASET_1000", {}).get("nn2_acc", 0.0)
    BASELINE_ACC_NN2_100  = baseline_results.get("DATASET_100", {}).get("nn2_acc", 0.0)

    # Legacy mapping to keep code backwards compatible with subsequent stages
    BASELINE_ACC    = BASELINE_ACC_1000
    image_paths     = baseline_results.get("DATASET_1000", {}).get("paths", [])
    true_folder_ids = baseline_results.get("DATASET_1000", {}).get("ids", [])

    print("\n" + "=" * 60)
    print("NN1 & NN2 - GLOBAL BASELINE EVALUATION REPORT")
    print("=" * 60)
    
    if "DATASET_1000" in baseline_results:
        res = baseline_results['DATASET_1000']
        print(f"Dataset 1000 (Nested):")
        print(f"  -> NN1 Accuracy: {res['nn1_acc']:.2f}% ({res['nn1_correct']}/{res['total']})")
        print(f"  -> NN2 Accuracy: {res['nn2_acc']:.2f}% ({res['nn2_correct']}/{res['total']})")
        
    if "DATASET_100" in baseline_results:
        res = baseline_results['DATASET_100']
        print(f"\nDataset 100 (Flat):")
        print(f"  -> NN1 Accuracy: {res['nn1_acc']:.2f}% ({res['nn1_correct']}/{res['total']})")
        print(f"  -> NN2 Accuracy: {res['nn2_acc']:.2f}% ({res['nn2_correct']}/{res['total']})")
        
    print("=" * 60 + "\n")

else:
    BASELINE_ACC_1000 = 0.87

# 3. ATTACKS

## Error-Generic Attacks


### Fast Gradient Sign Method (FGSM) — Error-Generic Attack
This step evaluates the effect of the FGSM error-generic (untargeted) attack on NN1
across a range of epsilon values. For each epsilon, adversarial examples are generated
in batches, the model accuracy is re-evaluated, and the results are saved. A Security
Evaluation Curve is plotted at the end of the sweep, and a CSV summary of all results
is written to the results directory.



In [49]:
if RUN_FGSM_GENERIC:
    # ==============================================================================
    # Fast Gradient Sign Method (FGSM) — Error-Generic Attack
    # ==============================================================================

    print("Initializing FGSM Attack Pipeline...")

    # ------------------------------------------------------------------------------
    # 1. Configuration
    # ------------------------------------------------------------------------------
    ATTACK_NAME = "FGSM"
    EPSILONS    = [0.001, 0.003, 0.005, 0.01, 0.03, 0.05, 0.1]          

    ATTACK_RESULTS_DIR = os.path.join(RESULTS_DIR, ATTACK_NAME)
    os.makedirs(ATTACK_RESULTS_DIR, exist_ok=True)

    # ------------------------------------------------------------------------------
    # 2. Evaluation Loop
    # ------------------------------------------------------------------------------
    print(f"\nStarting {ATTACK_NAME} generation across {len(EPSILONS)} epsilon values...")

    results    = []
    accuracies = []

    for eps in EPSILONS:
        print(f"\n--- Evaluating Epsilon: {eps:.3f} ---")

        fgsm = FastGradientMethod(estimator=art_classifier, eps=eps)

        correct_predictions = 0
        total_processed     = 0
        start_time          = time.time()

        for i in tqdm(range(0, len(image_paths), BATCH_SIZE), desc="Processing Batches"):
            batch_paths    = image_paths[i:i + BATCH_SIZE]
            batch_true_ids = true_folder_ids[i:i + BATCH_SIZE]

            batch_tensors  = []
            batch_filenames = []
            for path in batch_paths:
                img = Image.open(path).convert('RGB')
                batch_tensors.append(preprocess_attack(img).numpy())
                batch_filenames.append(os.path.basename(path))

            x_batch = np.stack(batch_tensors)
            x_adv   = fgsm.generate(x=x_batch)

            preds        = art_classifier.predict(x_adv)
            pred_indices = np.argmax(preds, axis=1)

            for j, pred_idx in enumerate(pred_indices):
                raw_label = LABELS[pred_idx]
                if isinstance(raw_label, np.ndarray):
                    raw_label = raw_label.flatten()[0]
                if isinstance(raw_label, bytes):
                    raw_label = raw_label.decode('utf-8')

                predicted_name = str(raw_label).replace("b'", "").replace("'", "").replace("[", "").replace("]", "").strip()
                true_name      = id_to_name.get(batch_true_ids[j], "Unknown")

                if normalize_label(predicted_name) == normalize_label(true_name):
                    correct_predictions += 1

            save_adversarial_images(x_adv, batch_true_ids, batch_filenames, ATTACK_NAME, eps)
            total_processed += len(batch_paths)

        elapsed = time.time() - start_time
        acc     = (correct_predictions / total_processed) * 100
        accuracies.append(acc)

        print(f"Accuracy at eps={eps:.3f}: {acc:.2f}%")

        results.append({
            'attack_name':          ATTACK_NAME,
            'epsilon':              eps,
            'correct_predictions':  correct_predictions,
            'total_images':         total_processed,
            'accuracy':             round(acc, 2),
            'time_elapsed_s':       round(elapsed, 2)
        })

    # ------------------------------------------------------------------------------
    # 3. Save Results CSV
    # ------------------------------------------------------------------------------
    df_results  = pd.DataFrame(results)
    csv_path    = os.path.join(ATTACK_RESULTS_DIR, f'{ATTACK_NAME}_results.csv')
    df_results.to_csv(csv_path, index=False)
    print(f"\n[INFO] Results saved at: {csv_path}")

    # ------------------------------------------------------------------------------
    # 4. Security Evaluation Curve
    # ------------------------------------------------------------------------------
    plot_security_evaluation_curve(EPSILONS, accuracies, ATTACK_NAME, ATTACK_RESULTS_DIR, BASELINE_ACC_1000)

    # ------------------------------------------------------------------------------
    # 5. Attack Summary
    # ------------------------------------------------------------------------------
    print("\n" + "=" * 50)
    print(f"{ATTACK_NAME} ATTACK SUMMARY REPORT")
    print("=" * 50)
    print(f"Evaluated Epsilon Range:  {EPSILONS[0]} to {EPSILONS[-1]}")
    print(f"Baseline Accuracy:        {BASELINE_ACC:.2f}%")
    print(f"Lowest Accuracy:          {accuracies[-1]:.2f}%")
    print("-" * 50)
    print(f"Total Model Degradation:  {(BASELINE_ACC - accuracies[-1]):.2f}%")
    print("=" * 50 + "\n")

### Basic Iterative Method (BIM) — Error-Generic Attack

This step evaluates the effect of the BIM error-generic (untargeted) attack on NN1.
BIM extends FGSM by applying multiple smaller gradient steps, each of size
`eps_step = eps / max_iter`, staying within the L-infinity ball of radius epsilon.
The attack is evaluated across a range of epsilon values for each value of `max_iter`,
producing one Security Evaluation Curve per configuration, a comparative SEC overlaying
all curves, and a hyperparameter effect plot showing accuracy as a function of `max_iter`
at fixed epsilon values. A CSV summary of all results is written to the results directory.

In [50]:
if RUN_BIM_GENERIC:
    # ==============================================================================
    # Basic Iterative Method (BIM) — Error-Generic Attack
    # ==============================================================================

    print("Initializing BIM Attack Pipeline...")

    # ------------------------------------------------------------------------------
    # 1. Configuration
    # ------------------------------------------------------------------------------
    ATTACK_NAME   = "BIM"
    EPSILONS      = [0.001, 0.003, 0.005, 0.01, 0.03, 0.05, 0.1]
    MAX_ITER_LIST = [1, 5, 10, 20]

    # One distinct color per max_iter value — reused across all plots for consistency
    ITER_COLORS = {
        1:  '#1f77b4',
        5:  "#ffbf00",
        10: '#2ca02c',
        20: '#d62728'
    }

    # Fixed epsilon values used in the hyperparameter effect plot (low, medium, high budget)
    FIXED_EPSILONS_FOR_EFFECT = [0.01, 0.05, 0.1]
    EPSILON_EFFECT_COLORS     = {
        0.01: '#9467bd',
        0.05: '#8c564b',
        0.1:  '#e377c2'
    }

    ATTACK_RESULTS_DIR = os.path.join(RESULTS_DIR, ATTACK_NAME)
    os.makedirs(ATTACK_RESULTS_DIR, exist_ok=True)

    # ------------------------------------------------------------------------------
    # 2. Evaluation Loop
    # ------------------------------------------------------------------------------
    print(f"\nStarting {ATTACK_NAME} generation across " f"{len(MAX_ITER_LIST)} max_iter values x {len(EPSILONS)} epsilon values...")

    results        = []
    all_accuracies = {}  # max_iter -> list of accuracies, one per epsilon

    for max_iter in MAX_ITER_LIST:
        print(f"\n{'='*50}")
        print(f"MAX_ITER = {max_iter}")
        print(f"{'='*50}")

        accuracies = []

        for eps in EPSILONS:
            print(f"\n--- Evaluating Epsilon: {eps:.3f} | max_iter: {max_iter} ---")

            eps_step = eps / max_iter

            bim = BasicIterativeMethod(
                estimator=art_classifier,
                eps=eps,
                eps_step=eps_step,
                max_iter=max_iter
            )

            correct_predictions = 0
            total_processed     = 0
            start_time          = time.time()

            for i in tqdm(range(0, len(image_paths), BATCH_SIZE), desc="Processing Batches"):
                batch_paths     = image_paths[i:i + BATCH_SIZE]
                batch_true_ids  = true_folder_ids[i:i + BATCH_SIZE]

                batch_tensors   = []
                batch_filenames = []
                for path in batch_paths:
                    img = Image.open(path).convert('RGB')
                    batch_tensors.append(preprocess_attack(img).numpy())
                    batch_filenames.append(os.path.basename(path))

                x_batch = np.stack(batch_tensors)
                x_adv   = bim.generate(x=x_batch)

                preds        = art_classifier.predict(x_adv)
                pred_indices = np.argmax(preds, axis=1)

                for j, pred_idx in enumerate(pred_indices):
                    raw_label = LABELS[pred_idx]
                    if isinstance(raw_label, np.ndarray):
                        raw_label = raw_label.flatten()[0]
                    if isinstance(raw_label, bytes):
                        raw_label = raw_label.decode('utf-8')

                    predicted_name = (str(raw_label)
                                    .replace("b'", "").replace("'", "")
                                    .replace("[", "").replace("]", "").strip())
                    true_name = id_to_name.get(batch_true_ids[j], "Unknown")

                    if normalize_label(predicted_name) == normalize_label(true_name):
                        correct_predictions += 1

                save_adversarial_images(
                    x_adv, batch_true_ids, batch_filenames,
                    f"{ATTACK_NAME}_iter{max_iter}", eps
                )
                total_processed += len(batch_paths)

            elapsed = time.time() - start_time
            acc     = (correct_predictions / total_processed) * 100
            accuracies.append(acc)

            print(f"Accuracy at eps={eps:.3f} | max_iter={max_iter}: {acc:.2f}%")

            results.append({
                'attack_name':         ATTACK_NAME,
                'max_iter':            max_iter,
                'eps_step':            round(eps / max_iter, 6),
                'epsilon':             eps,
                'correct_predictions': correct_predictions,
                'total_images':        total_processed,
                'accuracy':            round(acc, 2),
                'time_elapsed_s':      round(elapsed, 2)
            })

        all_accuracies[max_iter] = accuracies

        # --------------------------------------------------------------------------
        # 3. Individual SEC for this max_iter value
        # --------------------------------------------------------------------------
        color = ITER_COLORS[max_iter]

        fig, ax = plt.subplots(figsize=(9, 6))

        ax.plot([0.0, EPSILONS[0]], [BASELINE_ACC, accuracies[0]],
                linestyle='-', color=color, linewidth=1.5)
        ax.plot(EPSILONS, accuracies,
                marker='o', linestyle='-', color=color,
                linewidth=1.5, markersize=6,
                label=f'NN1 Performance ({ATTACK_NAME}, max_iter={max_iter})')
        ax.plot(0.0, BASELINE_ACC,
                marker='o', markerfacecolor=color,
                markeredgecolor='#2ca02c', markeredgewidth=1,
                markersize=6, linestyle='None',
                label=f'Baseline Accuracy ({BASELINE_ACC:.2f}%)')

        ax.set_title(f'Security Evaluation Curve — {ATTACK_NAME} (max_iter={max_iter})',fontsize=16, fontweight='bold', pad=15)
        ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
        ax.set_ylabel('Model Accuracy (%)', fontsize=14)
        ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
        ax.set_xticks(np.arange(0.0, 0.11, 0.01))
        ax.set_yticks(np.arange(0, 101, 10))
        ax.tick_params(axis='both', labelsize=12)
        ax.set_ylim(-5, 105)
        ax.set_xlim(-0.005, 0.105)
        ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2,
                label=r'Max Allowable $\epsilon$ (0.1)')
        ax.legend(loc='upper right', fontsize=12)
        plt.tight_layout()

        plot_path = os.path.join(ATTACK_RESULTS_DIR, f'{ATTACK_NAME}_iter{max_iter}_SEC.png')
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\n[INFO] Individual SEC saved at: {plot_path}")

    # ------------------------------------------------------------------------------
    # 4. Comparative SEC — all max_iter curves overlaid, epsilon on x-axis
    # ------------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(9, 6))

    ax.plot(0.0, BASELINE_ACC,
            marker='o', markerfacecolor='black',
            markeredgecolor='#2ca02c', markeredgewidth=1,
            markersize=6, linestyle='None',
            label=f'Baseline Accuracy ({BASELINE_ACC:.2f}%)')

    for max_iter, accuracies in all_accuracies.items():
        color = ITER_COLORS[max_iter]
        ax.plot([0.0, EPSILONS[0]], [BASELINE_ACC, accuracies[0]],
                linestyle='-', color=color, linewidth=1.5)
        ax.plot(EPSILONS, accuracies,
                marker='o', linestyle='-', color=color,
                linewidth=1.5, markersize=6,
                label=f'{ATTACK_NAME} max_iter={max_iter}')

    ax.set_title(f'Security Evaluation Curve — {ATTACK_NAME} (Comparative)',fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
    ax.set_ylabel('Model Accuracy (%)', fontsize=14)
    ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
    ax.set_xticks(np.arange(0.0, 0.11, 0.01))
    ax.set_yticks(np.arange(0, 101, 10))
    ax.tick_params(axis='both', labelsize=12)
    ax.set_ylim(-5, 105)
    ax.set_xlim(-0.005, 0.105)
    ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2,label=r'Max Allowable $\epsilon$ (0.1)')
    ax.legend(loc='upper right', fontsize=12)
    plt.tight_layout()

    comparative_path = os.path.join(ATTACK_RESULTS_DIR, f'{ATTACK_NAME}_comparative_SEC.png')
    plt.savefig(comparative_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n[INFO] Comparative SEC saved at: {comparative_path}")

    # ------------------------------------------------------------------------------
    # 5. Hyperparameter Effect Plot — Accuracy vs max_iter at fixed epsilon values
    #    eps_step = eps / max_iter for each point, all other parameters fixed.
    # ------------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(9, 6))

    for fixed_eps in FIXED_EPSILONS_FOR_EFFECT:
        eps_idx     = EPSILONS.index(fixed_eps)
        accs_at_eps = [all_accuracies[m][eps_idx] for m in MAX_ITER_LIST]

        ax.plot(MAX_ITER_LIST, accs_at_eps,
                marker='o', linestyle='-',
                color=EPSILON_EFFECT_COLORS[fixed_eps],
                linewidth=1.5, markersize=6,
                label=rf'$\epsilon$ = {fixed_eps}')

    ax.axhline(y=BASELINE_ACC, color='#2ca02c', linestyle='--', linewidth=1.5, label=f'Baseline Accuracy ({BASELINE_ACC:.2f}%)')

    ax.set_title(f'Effect of max_iter on Model Accuracy — {ATTACK_NAME}',fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel('Number of Iterations (max_iter)', fontsize=14)
    ax.set_ylabel('Model Accuracy (%)', fontsize=14)
    ax.set_xticks(MAX_ITER_LIST)
    ax.set_yticks(np.arange(0, 101, 10))
    ax.tick_params(axis='both', labelsize=12)
    ax.set_ylim(-5, 105)
    ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
    ax.legend(loc='upper right', fontsize=12)
    plt.tight_layout()

    effect_plot_path = os.path.join(ATTACK_RESULTS_DIR, f'{ATTACK_NAME}_maxiter_effect.png')
    plt.savefig(effect_plot_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n[INFO] Hyperparameter effect plot saved at: {effect_plot_path}")

    # ------------------------------------------------------------------------------
    # 6. Save Results CSV
    # ------------------------------------------------------------------------------
    df_results = pd.DataFrame(results)
    csv_path   = os.path.join(ATTACK_RESULTS_DIR, f'{ATTACK_NAME}_results.csv')
    df_results.to_csv(csv_path, index=False)
    print(f"\n[INFO] Results CSV saved at: {csv_path}")

    # ------------------------------------------------------------------------------
    # 7. Attack Summary
    # ------------------------------------------------------------------------------
    print("\n" + "=" * 50)
    print(f"{ATTACK_NAME} ATTACK SUMMARY REPORT")
    print("=" * 50)
    print(f"Evaluated Epsilon Range:  {EPSILONS[0]} to {EPSILONS[-1]}")
    print(f"Max Iter Values:          {MAX_ITER_LIST}")
    print(f"Baseline Accuracy:        {BASELINE_ACC:.2f}%")
    for max_iter, accuracies in all_accuracies.items():
        print(f"  max_iter={max_iter:2d} — Lowest Accuracy: {accuracies[-1]:.2f}%  " f"(Degradation: {BASELINE_ACC - accuracies[-1]:.2f}%)")
    print("=" * 50 + "\n")

### Projected Gradient Descent (PGD) — Error-Generic Attack

This step evaluates the effect of the PGD error-generic (untargeted) attack on NN1.
PGD extends BIM by adding random restarts: before each optimization, the starting
point is perturbed by a random offset sampled uniformly within the L-infinity ball
of radius epsilon. The parameter `num_random_init` controls how many random restarts
are performed, with `num_random_init=0` reducing PGD to standard BIM.
The attack is evaluated across all combinations of `epsilon`, `max_iter`, and
`num_random_init`, producing individual SECs, comparative plots, and hyperparameter
effect plots. A CSV summary of all results is written to the results directory.

In [51]:
if RUN_PGD_GENERIC:
    # ==============================================================================
    # STEP 3 Projected Gradient Descent (PGD) — Error-Generic Attack
    # ==============================================================================

    print("Initializing PGD Attack Pipeline...")

    # ------------------------------------------------------------------------------
    # 1. Configuration
    # ------------------------------------------------------------------------------
    ATTACK_NAME        = "PGD"
    EPSILONS           = [0.001, 0.003, 0.005, 0.01, 0.03, 0.05, 0.1]
    MAX_ITER_LIST      = [1, 5, 10, 20, 40]
    NUM_RANDOM_INITS   = [0, 1, 3, 5]

    # One color per max_iter — reused across all SEC plots
    ITER_COLORS = {
        1:  '#1f77b4',
        5:  '#ffbf00',
        10: '#2ca02c',
        20: '#d62728'
    }

    # One color per num_random_init — reused across all random_init plots
    RAND_COLORS = {
        0: '#1f77b4',
        1: '#ff7f0e',
        3: '#2ca02c',
        5: '#d62728'
    }

    # Fixed epsilon values for hyperparameter effect plots (low, medium, high budget)
    FIXED_EPSILONS_FOR_EFFECT = [0.01, 0.05, 0.1]
    EPSILON_EFFECT_COLORS     = {
        0.01: '#9467bd',
        0.05: '#8c564b',
        0.1:  '#e377c2'
    }

    ATTACK_RESULTS_DIR = os.path.join(RESULTS_DIR, ATTACK_NAME)
    os.makedirs(ATTACK_RESULTS_DIR, exist_ok=True)

    # ------------------------------------------------------------------------------
    # 2. Evaluation Loop
    # all_accuracies[num_random_init][max_iter] -> list of accuracies, one per epsilon
    # ------------------------------------------------------------------------------
    print(f"\nStarting {ATTACK_NAME} generation across "
            f"{len(NUM_RANDOM_INITS)} num_random_init values x "
            f"{len(MAX_ITER_LIST)} max_iter values x "
            f"{len(EPSILONS)} epsilon values...")

    results        = []
    all_accuracies = {nri: {} for nri in NUM_RANDOM_INITS}

    for num_random_init in NUM_RANDOM_INITS:
        print(f"\n{'#'*50}")
        print(f"NUM_RANDOM_INIT = {num_random_init}")
        print(f"{'#'*50}")

        for max_iter in MAX_ITER_LIST:
            print(f"\n{'='*50}")
            print(f"MAX_ITER = {max_iter} | NUM_RANDOM_INIT = {num_random_init}")
            print(f"{'='*50}")

            accuracies = []

            for eps in EPSILONS:
                print(f"\n--- Evaluating Epsilon: {eps:.3f} | max_iter: {max_iter} | num_random_init: {num_random_init} ---")

                eps_step = eps / max_iter

                pgd = ProjectedGradientDescentPyTorch(
                    estimator=art_classifier,
                    norm=np.inf,
                    eps=eps,
                    eps_step=eps_step,
                    max_iter=max_iter,
                    num_random_init=num_random_init,
                    targeted=False,
                    batch_size=BATCH_SIZE,
                    verbose=False
                )

                correct_predictions = 0
                total_processed     = 0
                start_time          = time.time()

                for i in tqdm(range(0, len(image_paths), BATCH_SIZE), desc="Processing Batches"):
                    batch_paths     = image_paths[i:i + BATCH_SIZE]
                    batch_true_ids  = true_folder_ids[i:i + BATCH_SIZE]

                    batch_tensors   = []
                    batch_filenames = []
                    for path in batch_paths:
                        img = Image.open(path).convert('RGB')
                        batch_tensors.append(preprocess_attack(img).numpy())
                        batch_filenames.append(os.path.basename(path))

                    x_batch = np.stack(batch_tensors)
                    x_adv   = pgd.generate(x=x_batch)

                    preds        = art_classifier.predict(x_adv)
                    pred_indices = np.argmax(preds, axis=1)

                    for j, pred_idx in enumerate(pred_indices):
                        raw_label = LABELS[pred_idx]
                        if isinstance(raw_label, np.ndarray):
                            raw_label = raw_label.flatten()[0]
                        if isinstance(raw_label, bytes):
                            raw_label = raw_label.decode('utf-8')

                        predicted_name = (str(raw_label)
                                        .replace("b'", "").replace("'", "")
                                        .replace("[", "").replace("]", "").strip())
                        true_name = id_to_name.get(batch_true_ids[j], "Unknown")

                        if normalize_label(predicted_name) == normalize_label(true_name):
                            correct_predictions += 1

                    save_adversarial_images(
                        x_adv, batch_true_ids, batch_filenames,
                        f"{ATTACK_NAME}_nri{num_random_init}_iter{max_iter}", eps
                    )
                    total_processed += len(batch_paths)

                elapsed = time.time() - start_time
                acc     = (correct_predictions / total_processed) * 100
                accuracies.append(acc)

                print(f"Accuracy at eps={eps:.3f} | max_iter={max_iter} | num_random_init={num_random_init}: {acc:.2f}%")

                results.append({
                    'attack_name':         ATTACK_NAME,
                    'num_random_init':     num_random_init,
                    'max_iter':            max_iter,
                    'eps_step':            round(eps / max_iter, 6),
                    'epsilon':             eps,
                    'correct_predictions': correct_predictions,
                    'total_images':        total_processed,
                    'accuracy':            round(acc, 2),
                    'time_elapsed_s':      round(elapsed, 2)
                })

            all_accuracies[num_random_init][max_iter] = accuracies

            # ----------------------------------------------------------------------
            # 3. Individual SEC for this (num_random_init, max_iter) combination
            # ----------------------------------------------------------------------
            color = ITER_COLORS[max_iter]

            fig, ax = plt.subplots(figsize=(9, 6))

            ax.plot([0.0, EPSILONS[0]], [BASELINE_ACC, accuracies[0]],
                    linestyle='-', color=color, linewidth=1.5)
            ax.plot(EPSILONS, accuracies,
                    marker='o', linestyle='-', color=color,
                    linewidth=1.5, markersize=6,
                    label=f'NN1 Performance ({ATTACK_NAME}, max_iter={max_iter}, nri={num_random_init})')
            ax.plot(0.0, BASELINE_ACC,
                    marker='o', markerfacecolor=color,
                    markeredgecolor='#2ca02c', markeredgewidth=1,
                    markersize=6, linestyle='None',
                    label=f'Baseline Accuracy ({BASELINE_ACC:.2f}%)')

            ax.set_title(f'Security Evaluation Curve — {ATTACK_NAME} 'f'(max_iter={max_iter}, num_random_init={num_random_init})',fontsize=16, fontweight='bold', pad=15)
            ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
            ax.set_ylabel('Model Accuracy (%)', fontsize=14)
            ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
            ax.set_xticks(np.arange(0.0, 0.11, 0.01))
            ax.set_yticks(np.arange(0, 101, 10))
            ax.tick_params(axis='both', labelsize=12)
            ax.set_ylim(-5, 105)
            ax.set_xlim(-0.005, 0.105)
            ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2, label=r'Max Allowable $\epsilon$ (0.1)')
            ax.legend(loc='upper right', fontsize=12)
            plt.tight_layout()

            plot_path = os.path.join(
                ATTACK_RESULTS_DIR,
                f'{ATTACK_NAME}_nri{num_random_init}_iter{max_iter}_SEC.png'
            )
            plt.savefig(plot_path, dpi=300, bbox_inches='tight')
            plt.show()
            print(f"\n[INFO] Individual SEC saved at: {plot_path}")

    # ------------------------------------------------------------------------------
    # 4. Comparative SEC per num_random_init — max_iter varies, epsilon on x-axis
    #    For each fixed num_random_init, overlay all max_iter curves.
    #    Ceteris paribus: num_random_init fixed, max_iter varies.
    # ------------------------------------------------------------------------------
    for num_random_init in NUM_RANDOM_INITS:
        fig, ax = plt.subplots(figsize=(9, 6))

        ax.plot(0.0, BASELINE_ACC,
                marker='o', markerfacecolor='black',
                markeredgecolor='#2ca02c', markeredgewidth=1,
                markersize=6, linestyle='None',
                label=f'Baseline Accuracy ({BASELINE_ACC:.2f}%)')

        for max_iter in MAX_ITER_LIST:
            color      = ITER_COLORS[max_iter]
            accuracies = all_accuracies[num_random_init][max_iter]
            ax.plot([0.0, EPSILONS[0]], [BASELINE_ACC, accuracies[0]],
                    linestyle='-', color=color, linewidth=1.5)
            ax.plot(EPSILONS, accuracies,
                    marker='o', linestyle='-', color=color,
                    linewidth=1.5, markersize=6,
                    label=f'max_iter={max_iter}')

        ax.set_title(f'SEC — {ATTACK_NAME} | num_random_init={num_random_init} (max_iter varies)', fontsize=16, fontweight='bold', pad=15)
        ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
        ax.set_ylabel('Model Accuracy (%)', fontsize=14)
        ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
        ax.set_xticks(np.arange(0.0, 0.11, 0.01))
        ax.set_yticks(np.arange(0, 101, 10))
        ax.tick_params(axis='both', labelsize=12)
        ax.set_ylim(-5, 105)
        ax.set_xlim(-0.005, 0.105)
        ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2, label=r'Max Allowable $\epsilon$ (0.1)')
        ax.legend(loc='upper right', fontsize=12)
        plt.tight_layout()

        path = os.path.join(ATTACK_RESULTS_DIR,
                            f'{ATTACK_NAME}_nri{num_random_init}_comparative_maxiter_SEC.png')
        plt.savefig(path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\n[INFO] Comparative SEC (nri={num_random_init}, max_iter varies) saved at: {path}")

    # ------------------------------------------------------------------------------
    # 5. Comparative SEC per max_iter — num_random_init varies, epsilon on x-axis
    #    For each fixed max_iter, overlay all num_random_init curves.
    #    Ceteris paribus: max_iter fixed, num_random_init varies.
    # ------------------------------------------------------------------------------
    for max_iter in MAX_ITER_LIST:
        fig, ax = plt.subplots(figsize=(9, 6))

        ax.plot(0.0, BASELINE_ACC,
                marker='o', markerfacecolor='black',
                markeredgecolor='#2ca02c', markeredgewidth=1,
                markersize=6, linestyle='None',
                label=f'Baseline Accuracy ({BASELINE_ACC:.2f}%)')

        for num_random_init in NUM_RANDOM_INITS:
            color      = RAND_COLORS[num_random_init]
            accuracies = all_accuracies[num_random_init][max_iter]
            ax.plot([0.0, EPSILONS[0]], [BASELINE_ACC, accuracies[0]],
                    linestyle='-', color=color, linewidth=1.5)
            ax.plot(EPSILONS, accuracies,
                    marker='o', linestyle='-', color=color,
                    linewidth=1.5, markersize=6,
                    label=f'num_random_init={num_random_init}')

        ax.set_title(f'SEC — {ATTACK_NAME} | max_iter={max_iter} (num_random_init varies)', fontsize=16, fontweight='bold', pad=15)
        ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
        ax.set_ylabel('Model Accuracy (%)', fontsize=14)
        ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
        ax.set_xticks(np.arange(0.0, 0.11, 0.01))
        ax.set_yticks(np.arange(0, 101, 10))
        ax.tick_params(axis='both', labelsize=12)
        ax.set_ylim(-5, 105)
        ax.set_xlim(-0.005, 0.105)
        ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2, label=r'Max Allowable $\epsilon$ (0.1)')
        ax.legend(loc='upper right', fontsize=12)
        plt.tight_layout()

        path = os.path.join(ATTACK_RESULTS_DIR,
                            f'{ATTACK_NAME}_iter{max_iter}_comparative_nri_SEC.png')
        plt.savefig(path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\n[INFO] Comparative SEC (max_iter={max_iter}, nri varies) saved at: {path}")

    # ------------------------------------------------------------------------------
    # 6. Hyperparameter Effect Plot — Accuracy vs max_iter at fixed epsilon values
    #    num_random_init fixed at maximum value (5) — ceteris paribus.
    # ------------------------------------------------------------------------------
    FIXED_NRI_FOR_EFFECT = max(NUM_RANDOM_INITS)

    fig, ax = plt.subplots(figsize=(9, 6))

    for fixed_eps in FIXED_EPSILONS_FOR_EFFECT:
        eps_idx     = EPSILONS.index(fixed_eps)
        accs_at_eps = [all_accuracies[FIXED_NRI_FOR_EFFECT][m][eps_idx] for m in MAX_ITER_LIST]

        ax.plot(MAX_ITER_LIST, accs_at_eps,
                marker='o', linestyle='-',
                color=EPSILON_EFFECT_COLORS[fixed_eps],
                linewidth=1.5, markersize=6,
                label=rf'$\epsilon$ = {fixed_eps}')

    ax.axhline(y=BASELINE_ACC, color='#2ca02c', linestyle='--',linewidth=1.5, label=f'Baseline Accuracy ({BASELINE_ACC:.2f}%)')

    ax.set_title(f'Effect of max_iter on Model Accuracy — {ATTACK_NAME} 'f'(num_random_init={FIXED_NRI_FOR_EFFECT})',fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel('Number of Iterations (max_iter)', fontsize=14)
    ax.set_ylabel('Model Accuracy (%)', fontsize=14)
    ax.set_xticks(MAX_ITER_LIST)
    ax.set_yticks(np.arange(0, 101, 10))
    ax.tick_params(axis='both', labelsize=12)
    ax.set_ylim(-5, 105)
    ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
    ax.legend(loc='upper right', fontsize=12)
    plt.tight_layout()

    effect_iter_path = os.path.join(ATTACK_RESULTS_DIR,
                                    f'{ATTACK_NAME}_maxiter_effect_nri{FIXED_NRI_FOR_EFFECT}.png')
    plt.savefig(effect_iter_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n[INFO] max_iter effect plot saved at: {effect_iter_path}")

    # ------------------------------------------------------------------------------
    # 7. Hyperparameter Effect Plot — Accuracy vs num_random_init at fixed epsilon values
    #    max_iter fixed at maximum value (40) — ceteris paribus.
    # ------------------------------------------------------------------------------
    FIXED_ITER_FOR_EFFECT = max(MAX_ITER_LIST)

    fig, ax = plt.subplots(figsize=(9, 6))

    for fixed_eps in FIXED_EPSILONS_FOR_EFFECT:
        eps_idx     = EPSILONS.index(fixed_eps)
        accs_at_eps = [all_accuracies[nri][FIXED_ITER_FOR_EFFECT][eps_idx] for nri in NUM_RANDOM_INITS]

        ax.plot(NUM_RANDOM_INITS, accs_at_eps,
                marker='o', linestyle='-',
                color=EPSILON_EFFECT_COLORS[fixed_eps],
                linewidth=1.5, markersize=6,
                label=rf'$\epsilon$ = {fixed_eps}')

    ax.axhline(y=BASELINE_ACC, color='#2ca02c', linestyle='--',linewidth=1.5, label=f'Baseline Accuracy ({BASELINE_ACC:.2f}%)')

    ax.set_title(f'Effect of num_random_init on Model Accuracy — {ATTACK_NAME} 'f'(max_iter={FIXED_ITER_FOR_EFFECT})',fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel('Number of Random Restarts (num_random_init)', fontsize=14)
    ax.set_ylabel('Model Accuracy (%)', fontsize=14)
    ax.set_xticks(NUM_RANDOM_INITS)
    ax.set_yticks(np.arange(0, 101, 10))
    ax.tick_params(axis='both', labelsize=12)
    ax.set_ylim(-5, 105)
    ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
    ax.legend(loc='upper right', fontsize=12)
    plt.tight_layout()

    effect_nri_path = os.path.join(ATTACK_RESULTS_DIR,f'{ATTACK_NAME}_nri_effect_iter{FIXED_ITER_FOR_EFFECT}.png')
    plt.savefig(effect_nri_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n[INFO] num_random_init effect plot saved at: {effect_nri_path}")

    # ------------------------------------------------------------------------------
    # 8. Save Results CSV
    # ------------------------------------------------------------------------------
    df_results = pd.DataFrame(results)
    csv_path   = os.path.join(ATTACK_RESULTS_DIR, f'{ATTACK_NAME}_results.csv')
    df_results.to_csv(csv_path, index=False)
    print(f"\n[INFO] Results CSV saved at: {csv_path}")

    # ------------------------------------------------------------------------------
    # 9. Attack Summary
    # ------------------------------------------------------------------------------
    print("\n" + "=" * 50)
    print(f"{ATTACK_NAME} ATTACK SUMMARY REPORT")
    print("=" * 50)
    print(f"Evaluated Epsilon Range:   {EPSILONS[0]} to {EPSILONS[-1]}")
    print(f"Max Iter Values:           {MAX_ITER_LIST}")
    print(f"Num Random Init Values:    {NUM_RANDOM_INITS}")
    print(f"Baseline Accuracy:         {BASELINE_ACC:.2f}%")
    print("-" * 50)
    for num_random_init in NUM_RANDOM_INITS:
        for max_iter in MAX_ITER_LIST:
            acc_worst = all_accuracies[num_random_init][max_iter][-1]
            print(f"  nri={num_random_init} | max_iter={max_iter:2d} — "
                f"Lowest Accuracy: {acc_worst:.2f}%  "
                f"(Degradation: {BASELINE_ACC - acc_worst:.2f}%)")
    print("=" * 50 + "\n")


### Carlini & Wagner L-inf (CW) — Error-Generic Attack

This step evaluates the Carlini & Wagner L-infinity attack in its untargeted
(error-generic) variant on NN1. CW is an optimization-based attack that searches
for the minimum perturbation sufficient to cause misclassification, subject to an
L-infinity constraint of 0.1.

Unlike gradient-sign attacks, CW may fail to find a valid perturbation within the
budget for some images. Images for which CW exceeds the budget are counted as
attack failures and contribute to the residual accuracy — they are never excluded
from the denominator.

The attack is evaluated across a grid of hyperparameters: `max_iter`,
`learning_rate`, and `confidence`. Due to the high computational cost, the dataset
used is the 1-per-person subset (100 images). A Security Evaluation Curve is
generated plotting accuracy against mean L-inf applied, and a CSV summary of all
hyperparameter combinations is saved to the results directory.

In [ ]:
if RUN_CW_GENERIC:
    # ==========================================================================
    # STEP 3.4: Carlini & Wagner L-inf (CW) — Error-Generic Attack
    # ==========================================================================

    print("Initializing CW L-inf Error-Generic Attack Pipeline...")

    # --------------------------------------------------------------------------
    # 1. Configuration
    # --------------------------------------------------------------------------
    ATTACK_NAME   = "CW_Untargeted"
    EPSILON_LIMIT = 0.1
    NUM_CLASSES   = 8631

    MAX_ITER_LIST      = [1, 5, 10, 20]
    LEARNING_RATE_LIST = [0.01, 0.1]
    CONFIDENCE_LIST    = [0.0]

    ITER_COLORS = {
        1:  '#1f77b4',
        5:  '#ff7f0e',
        10: '#2ca02c',
        20: '#d62728'
    }

    ATTACK_RESULTS_DIR = os.path.join(RESULTS_DIR, ATTACK_NAME)
    ADV_BASE_DIR       = os.path.join(ADVERSARIAL_DATA_DIR, ATTACK_NAME)
    os.makedirs(ATTACK_RESULTS_DIR, exist_ok=True)
    os.makedirs(ADV_BASE_DIR, exist_ok=True)

    # --------------------------------------------------------------------------
    # 2. Load 100-image dataset (1 per subject)
    # --------------------------------------------------------------------------
    print("\nLoading 1-per-person dataset...")

    preprocess_cw = transforms.Compose([
        transforms.Resize((160, 160)),
        transforms.ToTensor()
    ])

    x_data      = []
    image_names = []
    true_ids    = []

    image_files = sorted([
        f for f in os.listdir(DATA_DIR_100)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])

    if len(image_files) != 100:
        print(f"[WARNING] Found {len(image_files)} images instead of 100.")

    for img_name in image_files:
        img_path = os.path.join(DATA_DIR_100, img_name)
        img      = Image.open(img_path).convert('RGB')
        x_data.append(preprocess_cw(img).numpy())
        image_names.append(img_name)
        true_ids.append(os.path.splitext(img_name)[0].strip())

    x_data = np.array(x_data)
    print(f"Loaded {len(x_data)} images. Shape: {x_data.shape}")

    # --------------------------------------------------------------------------
    # 3. Helper — label parsing
    # --------------------------------------------------------------------------
    def parse_label(raw_label):
        """Convert a raw label (bytes, ndarray, str) to a normalized string."""
        if isinstance(raw_label, np.ndarray):
            raw_label = raw_label.flatten()[0]
        if isinstance(raw_label, bytes):
            raw_label = raw_label.decode('utf-8')
        return normalize_label(
            str(raw_label)
            .replace("b'", "").replace("'", "")
            .replace("[", "").replace("]", "")
            .strip()
        )

    # --------------------------------------------------------------------------
    # 4. Checkpoint — load already-completed results from CSV (if present)
    # --------------------------------------------------------------------------
    csv_path = os.path.join(ATTACK_RESULTS_DIR, f'{ATTACK_NAME}_results.csv')

    CSV_COLUMNS = [
        'attack_name', 'max_iter', 'learning_rate', 'confidence',
        'accuracy', 'saved_images', 'failed_images',
        'total_images', 'mean_linf_applied', 'time_elapsed_s'
    ]

    if os.path.exists(csv_path):
        df_checkpoint = pd.read_csv(csv_path)
        for col in CSV_COLUMNS:
            if col not in df_checkpoint.columns:
                df_checkpoint[col] = np.nan
        results = df_checkpoint.to_dict('records')
        completed = {
            (row['max_iter'], row['learning_rate'], row['confidence'])
            for row in results
        }
        print(f"\n[RESUME] CSV found: {len(completed)} combinations already completed "
            f"out of {len(MAX_ITER_LIST) * len(LEARNING_RATE_LIST) * len(CONFIDENCE_LIST)} total.")
        print("[RESUME] Completed combinations:")
        for (mi, lr, cf) in sorted(completed):
            print(f"         max_iter={mi}, lr={lr}, confidence={cf}")
    else:
        results   = []
        completed = set()
        print("\n[INFO] No checkpoint found — starting from scratch.")

    # Rebuild all_combo_data from already-loaded results (needed for final plots)
    all_combo_data = [
        {
            'label':     f"iter={r['max_iter']}, lr={r['learning_rate']}, c={r['confidence']}",
            'max_iter':  r['max_iter'],
            'mean_linf': r['mean_linf_applied'],
            'accuracy':  r['accuracy']
        }
        for r in results
    ]

    # --------------------------------------------------------------------------
    # 5. Hyperparameter Grid Search (with per-combination checkpoint)
    # --------------------------------------------------------------------------
    print("\nStarting CW L-inf grid search...")

    total_combinations = (len(MAX_ITER_LIST)
                        * len(LEARNING_RATE_LIST)
                        * len(CONFIDENCE_LIST))
    current_combo = 1

    for max_iter in MAX_ITER_LIST:
        for lr in LEARNING_RATE_LIST:
            for conf in CONFIDENCE_LIST:

                combo_key = (max_iter, lr, conf)

                if combo_key in completed:
                    print(f"\n[SKIP {current_combo}/{total_combinations}] "
                        f"max_iter={max_iter} | lr={lr} | confidence={conf} "
                        f"— already completed, skipping.")
                    current_combo += 1
                    continue

                print(f"\n[{current_combo}/{total_combinations}] "
                    f"max_iter={max_iter} | lr={lr} | confidence={conf}")

                lr_str         = str(lr).replace('.', '_')
                conf_str       = str(conf).replace('.', '_')
                combo_dir_name = f"iter_{max_iter}_lr_{lr_str}_c_{conf_str}"
                adv_combo_dir  = os.path.join(ADV_BASE_DIR, combo_dir_name)
                os.makedirs(adv_combo_dir, exist_ok=True)

                attack_cw = CarliniLInfMethod(
                    classifier=art_classifier,
                    targeted=False,
                    max_iter=max_iter,
                    learning_rate=lr,
                    confidence=conf,
                    verbose=False
                )

                start_time = time.time()

                x_adv_list = []
                for i in range(len(x_data)):
                    single_adv = attack_cw.generate(x=x_data[i:i+1])
                    x_adv_list.append(single_adv)
                    if (i + 1) % 10 == 0:
                        print(f"  Progress: {i+1}/{len(x_data)} images generated...")

                x_adv_raw = np.concatenate(x_adv_list, axis=0)

                # --------------------------------------------------------------
                # Evaluation — robust accuracy with fixed denominator.
                #
                # Semantics follow the standard definition (Croce et al., 2020;
                # Carlini et al., 2019): robust accuracy is the fraction of ALL
                # test images correctly classified under the adversary's best
                # attempt within the perturbation budget.
                #
                # - If the attack produces a perturbation within budget
                #   (L-inf <= EPSILON_LIMIT): clip to [0,1], predict on the
                #   clipped adversarial image, and save it to disk.
                # - If the attack fails to stay within budget (L-inf >
                #   EPSILON_LIMIT): the adversary is assumed to present the
                #   original clean image instead. Predict on x_data[i] and
                #   count toward the numerator if correct. Do not save.
                # - The denominator is always len(x_data) = 100.
                # - The budget check is performed on the raw perturbation
                #   (pre-clip); statistics (l_inf_list) are recorded on the
                #   clipped perturbation for the saved images only.
                # --------------------------------------------------------------
                correct        = 0
                saved_count    = 0
                failed_count   = 0
                l_inf_list     = []

                for i in range(len(x_adv_raw)):
                    perturbation_raw = x_adv_raw[i] - x_data[i]
                    l_inf_raw        = float(np.max(np.abs(perturbation_raw)))

                    true_clean = normalize_label(
                        id_to_name.get(true_ids[i], "Unknown")
                    )

                    if l_inf_raw > EPSILON_LIMIT + 1e-5:
                        # Attack failed to stay within budget: evaluate on the
                        # original clean image (the adversary has no valid
                        # adversarial example to present).
                        failed_count += 1
                        pred_out  = art_classifier.predict(x_data[i][np.newaxis])
                        pred_idx  = np.argmax(pred_out, axis=1)[0]
                        pred_clean = parse_label(LABELS[pred_idx])
                        if pred_clean == true_clean:
                            correct += 1
                        continue

                    # Attack succeeded within budget: clip, predict, save.
                    adv_clipped   = np.clip(x_adv_raw[i], 0.0, 1.0)
                    l_inf_applied = float(np.max(np.abs(adv_clipped - x_data[i])))
                    l_inf_list.append(l_inf_applied)

                    pred_out   = art_classifier.predict(adv_clipped[np.newaxis])
                    pred_idx   = np.argmax(pred_out, axis=1)[0]
                    pred_clean = parse_label(LABELS[pred_idx])

                    if pred_clean == true_clean:
                        correct += 1

                    adv_save = np.transpose(adv_clipped, (1, 2, 0))
                    adv_save = (adv_save * 255.0).astype(np.uint8)
                    Image.fromarray(adv_save).save(
                        os.path.join(adv_combo_dir, image_names[i])
                    )
                    saved_count += 1

                elapsed  = time.time() - start_time
                accuracy = (correct / len(x_data)) * 100

                mean_linf = float(np.mean(l_inf_list)) if l_inf_list else 0.0

                print(f"  Saved (within budget):  {saved_count}/100 | "
                      f"Failed (over budget):    {failed_count}/100")
                print(f"  Robust Accuracy:        {accuracy:.2f}% | "
                      f"Mean L-inf (applied):    {mean_linf:.4f} | "
                      f"Time: {elapsed:.1f}s")

                new_row = {
                    'attack_name':       ATTACK_NAME,
                    'max_iter':          max_iter,
                    'learning_rate':     lr,
                    'confidence':        conf,
                    'accuracy':          round(accuracy, 2),
                    'saved_images':      saved_count,
                    'failed_images':     failed_count,
                    'total_images':      len(x_data),
                    'mean_linf_applied': round(mean_linf, 4),
                    'time_elapsed_s':    round(elapsed, 2)
                }

                results.append(new_row)
                completed.add(combo_key)
                pd.DataFrame(results).to_csv(csv_path, index=False)
                print(f"  [CHECKPOINT] Saved to CSV "
                    f"({len(results)}/{total_combinations} combinations done).")

                all_combo_data.append({
                    'label':     f'iter={max_iter}, lr={lr}, c={conf}',
                    'max_iter':  max_iter,
                    'mean_linf': mean_linf,
                    'accuracy':  accuracy
                })

                current_combo += 1

    # --------------------------------------------------------------------------
    # 6. Completion check
    # --------------------------------------------------------------------------
    if len(results) < total_combinations:
        print(f"\n[WARNING] Only {len(results)}/{total_combinations} combinations "
            f"in CSV — some plots may be incomplete.")
    else:
        print(f"\n[INFO] All {total_combinations} combinations completed.")

    df_results = pd.DataFrame(results)

    # --------------------------------------------------------------------------
    # 7. Security Evaluation Curve — Accuracy vs Mean L-inf Applied
    # --------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(9, 6))

    ax.axhline(y=BASELINE_ACC_100, color='#2ca02c', linestyle='--',
            linewidth=1.5,
            label=f'Baseline Accuracy ({BASELINE_ACC_100:.2f}%)')

    for max_iter in MAX_ITER_LIST:
        if max_iter not in ITER_COLORS:
            continue
        color  = ITER_COLORS[max_iter]
        combos = [c for c in all_combo_data if c['max_iter'] == max_iter]
        if not combos:
            continue
        combos.sort(key=lambda c: c['mean_linf'])
        x_vals = [c['mean_linf'] for c in combos]
        y_vals = [c['accuracy']  for c in combos]

        ax.plot(x_vals, y_vals,
                marker='o', linestyle='-', color=color,
                linewidth=1.5, markersize=6,
                label=f'max_iter={max_iter}')

    ax.axvline(x=EPSILON_LIMIT, color='black', linestyle=':', linewidth=2,
            label=r'Max Allowable $\epsilon$ (0.1)')

    ax.set_title(
        f'Security Evaluation Curve — {ATTACK_NAME}\n'
        f'(Accuracy on images within budget L-inf ≤ {EPSILON_LIMIT})',
        fontsize=16, fontweight='bold', pad=15
    )
    ax.set_xlabel(r'Mean L-inf Perturbation (applied, within budget)', fontsize=14)
    ax.set_ylabel('Model Accuracy (%)', fontsize=14)
    ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
    ax.set_xticks(np.arange(0.0, 0.11, 0.01))
    ax.set_yticks(np.arange(0, 101, 10))
    ax.tick_params(axis='both', labelsize=12)
    ax.set_ylim(-5, 105)
    ax.set_xlim(-0.005, 0.105)
    ax.legend(loc='upper right', fontsize=12)
    plt.tight_layout()

    sec_path = os.path.join(ATTACK_RESULTS_DIR, f'{ATTACK_NAME}_SEC.png')
    plt.savefig(sec_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n[INFO] SEC saved at: {sec_path}")

    # --------------------------------------------------------------------------
    # 8. Accuracy vs learning_rate — one plot per confidence value
    #
    # Iterates over every value in CONFIDENCE_LIST so that no plot is ever
    # skipped or left empty if the list is changed.
    # --------------------------------------------------------------------------
    for conf_val in CONFIDENCE_LIST:
        conf_str_plot = str(conf_val).replace('.', '_')

        fig, ax = plt.subplots(figsize=(9, 6))

        plotted_any = False
        for max_iter in MAX_ITER_LIST:
            if max_iter not in ITER_COLORS:
                continue
            color  = ITER_COLORS[max_iter]
            subset = df_results[
                (df_results['max_iter']   == max_iter) &
                (df_results['confidence'] == conf_val)
            ].sort_values('learning_rate')
            if subset.empty:
                continue

            ax.plot(subset['learning_rate'].tolist(),
                    subset['accuracy'].tolist(),
                    marker='o', linestyle='-', color=color,
                    linewidth=1.5, markersize=6,
                    label=f'max_iter={max_iter}')
            plotted_any = True

        if not plotted_any:
            plt.close()
            print(f"\n[WARNING] Step 8 — no data for confidence={conf_val}, "
                f"plot skipped.")
            continue

        ax.axhline(y=BASELINE_ACC_100, color='#2ca02c', linestyle='--',
                linewidth=1.5,
                label=f'Baseline Accuracy ({BASELINE_ACC_100:.2f}%)')

        ax.set_title(
            f'Effect of learning_rate on Accuracy — {ATTACK_NAME}\n'
            f'(confidence={conf_val}, accuracy on images within budget)',
            fontsize=16, fontweight='bold', pad=15
        )
        ax.set_xlabel('Learning Rate', fontsize=14)
        ax.set_ylabel('Model Accuracy (%)', fontsize=14)
        ax.set_xscale('log')
        ax.set_xticks(LEARNING_RATE_LIST)
        ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
        ax.set_yticks(np.arange(0, 101, 10))
        ax.tick_params(axis='both', labelsize=12)
        ax.set_ylim(-5, 105)
        ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
        ax.legend(loc='upper right', fontsize=12)
        plt.tight_layout()

        lr_effect_path = os.path.join(
            ATTACK_RESULTS_DIR,
            f'{ATTACK_NAME}_lr_effect_conf{conf_str_plot}.png'
        )
        plt.savefig(lr_effect_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\n[INFO] Accuracy vs lr plot (confidence={conf_val}) "
            f"saved at: {lr_effect_path}")

    # --------------------------------------------------------------------------
    # 9. Accuracy vs confidence — one plot per learning_rate value
    #
    # Iterates over every value in LEARNING_RATE_LIST so that no plot is ever
    # skipped or left empty if the list is changed.
    # --------------------------------------------------------------------------
    for lr_val in LEARNING_RATE_LIST:
        lr_str_plot = str(lr_val).replace('.', '_')

        fig, ax = plt.subplots(figsize=(9, 6))

        plotted_any = False
        for max_iter in MAX_ITER_LIST:
            if max_iter not in ITER_COLORS:
                continue
            color  = ITER_COLORS[max_iter]
            subset = df_results[
                (df_results['max_iter']      == max_iter) &
                (df_results['learning_rate'] == lr_val)
            ].sort_values('confidence')
            if subset.empty:
                continue

            ax.plot(subset['confidence'].tolist(),
                    subset['accuracy'].tolist(),
                    marker='o', linestyle='-', color=color,
                    linewidth=1.5, markersize=6,
                    label=f'max_iter={max_iter}')
            plotted_any = True

        if not plotted_any:
            plt.close()
            print(f"\n[WARNING] Step 9 — no data for learning_rate={lr_val}, "
                f"plot skipped.")
            continue

        ax.axhline(y=BASELINE_ACC_100, color='#2ca02c', linestyle='--',
                linewidth=1.5,
                label=f'Baseline Accuracy ({BASELINE_ACC_100:.2f}%)')

        ax.set_title(
            f'Effect of confidence on Accuracy — {ATTACK_NAME}\n'
            f'(learning_rate={lr_val}, accuracy on images within budget)',
            fontsize=16, fontweight='bold', pad=15
        )
        ax.set_xlabel('Confidence', fontsize=14)
        ax.set_ylabel('Model Accuracy (%)', fontsize=14)
        ax.set_xticks(CONFIDENCE_LIST)
        ax.set_yticks(np.arange(0, 101, 10))
        ax.tick_params(axis='both', labelsize=12)
        ax.set_ylim(-5, 105)
        ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
        ax.legend(loc='upper right', fontsize=12)
        plt.tight_layout()

        conf_effect_path = os.path.join(
            ATTACK_RESULTS_DIR,
            f'{ATTACK_NAME}_conf_effect_lr{lr_str_plot}.png'
        )
        plt.savefig(conf_effect_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\n[INFO] Accuracy vs confidence plot (lr={lr_val}) "
            f"saved at: {conf_effect_path}")

    # --------------------------------------------------------------------------
    # 10. Attack Summary
    # --------------------------------------------------------------------------
    valid_rows = df_results[df_results['valid_images'] > 0]
    best_row   = valid_rows.loc[valid_rows['accuracy'].idxmin()] \
                if not valid_rows.empty else df_results.iloc[0]

    print("\n" + "=" * 65)
    print(f"{ATTACK_NAME} ATTACK SUMMARY REPORT")
    print("=" * 65)
    print(f"Dataset:           1-per-person subset (100 images)")
    print(f"Baseline Accuracy: {BASELINE_ACC_100:.2f}%")
    print(f"Budget L-inf:      <= {EPSILON_LIMIT} (failures evaluated on clean image)")
    print(f"Grid:              {total_combinations} combinations evaluated")
    print("-" * 65)
    print(f"{'max_iter':<12} {'lr':<8} {'conf':<8} {'Accuracy':>10} "
        f"{'Mean L-inf':>12} {'Valid':>8} {'Skip':>6}")
    print("-" * 65)
    for r in results:
        print(f"  {r['max_iter']:<10} {r['learning_rate']:<8} {r['confidence']:<8} "
              f"{r['accuracy']:>10.2f}% "
              f"{r['mean_linf_applied']:>12.4f} "
              f"{r['saved_images']:>7}/100"
              f"{r['failed_images']:>6}")
    print("-" * 65)
    print(f"Best Configuration (lowest accuracy on valid images):")
    print(f"  max_iter:      {best_row['max_iter']}")
    print(f"  learning_rate: {best_row['learning_rate']}")
    print(f"  confidence:    {best_row['confidence']}")
    print(f"  Accuracy:      {best_row['accuracy']:.2f}%")
    print(f"  Mean L-inf:    {best_row['mean_linf_applied']:.4f}")
    print(f"  Valid:         {best_row['valid_images']}/100")
    print("=" * 65 + "\n")

### DeepFool — Error-Generic Attack

This step evaluates the DeepFool untargeted attack on NN1. DeepFool iteratively
computes the minimum perturbation required to cross the nearest decision boundary.
The parameter `epsilon` in DeepFool is not a strict L-infinity budget but an
overshoot threshold added to the perturbation to prevent numerical instability.

Because DeepFool does not strictly bound the perturbation during generation, a 
strict L-infinity budget of 0.1 is enforced a posteriori. Images for which the 
generated perturbation exceeds this budget are filtered out and excluded from 
both the numerator and the denominator when calculating model accuracy.

Due to the high computational cost, the 1-per-person subset (100 images) is used.
The attack is evaluated across a range of `max_iter` values with a fixed overshoot 
`epsilon`. Two visualizations are produced: a bar chart comparing model accuracy 
across different `max_iter` values, and an aggregated scatter plot showing the 
direct relationship between mean L-inf perturbation and model accuracy. A CSV 
summary of all results is saved to the results directory.

In [ ]:
if RUN_DEEPFOOL_GENERIC:
    # ========================================================================== 
    # STEP 3.6: DeepFool — Error-Generic Attack
    # ========================================================================== 

    print("Initializing DeepFool Error-Generic Attack Pipeline...")

    # --------------------------------------------------------------------------
    # 1. Configuration
    # --------------------------------------------------------------------------
    ATTACK_NAME   = "DeepFool"
    EPSILON_LIMIT = 0.1
    MAX_ITER_LIST = [1, 5, 10, 20, 50, 100, 175]
    EPSILON       = 1e-6   # single value, not a list

    ITER_COLORS = {
        1:   '#1f77b4',
        5:   '#ff7f0e',
        10:  '#2ca02c',
        20: "#f2ff00",
        50:  '#d62728',
        100: '#9467bd',
        175: "#4b8c6c"
    }

    ATTACK_RESULTS_DIR = os.path.join(RESULTS_DIR, ATTACK_NAME)
    ADV_BASE_DIR       = os.path.join(ADVERSARIAL_DATA_DIR, ATTACK_NAME)
    os.makedirs(ATTACK_RESULTS_DIR, exist_ok=True)
    os.makedirs(ADV_BASE_DIR, exist_ok=True)

    # --------------------------------------------------------------------------
    # 2. Load 100-image dataset (1 per subject)
    # --------------------------------------------------------------------------
    print("\nLoading 1-per-person dataset...")

    preprocess_df = transforms.Compose([
        transforms.Resize((160, 160)),
        transforms.ToTensor()
    ])

    x_data      = []
    image_names = []
    true_ids    = []

    image_files = sorted([
        f for f in os.listdir(DATA_DIR_100)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])

    if len(image_files) != 100:
        print(f"[WARNING] Found {len(image_files)} images instead of 100.")

    for img_name in image_files:
        img_path = os.path.join(DATA_DIR_100, img_name)
        img      = Image.open(img_path).convert('RGB')
        x_data.append(preprocess_df(img).numpy())
        image_names.append(img_name)
        true_ids.append(os.path.splitext(img_name)[0].strip())

    x_data = np.array(x_data)
    print(f"Loaded {len(x_data)} images. Shape: {x_data.shape}")

    # --------------------------------------------------------------------------
    # 3. Checkpoint — load previously completed results from CSV (if exists)
    # --------------------------------------------------------------------------
    csv_path = os.path.join(ATTACK_RESULTS_DIR, f'{ATTACK_NAME}_results.csv')

    CSV_COLUMNS = [
        'attack_name', 'max_iter', 'epsilon', 'accuracy',
        'correct_predictions', 'saved_images', 'failed_budget',
        'evaluated_images', 'total_images', 'mean_linf_applied', 'time_elapsed_s'
    ]

    if os.path.exists(csv_path):
        df_checkpoint = pd.read_csv(csv_path)
        for col in CSV_COLUMNS:
            if col not in df_checkpoint.columns:
                df_checkpoint[col] = np.nan
        results = df_checkpoint.to_dict('records')
        # The resume key is only max_iter (EPSILON is fixed and unique)
        completed = { row['max_iter'] for row in results }
        print(f"\n[RESUME] CSV found: {len(completed)} combinations already completed "
                f"out of {len(MAX_ITER_LIST)} total.")
        print(f"[RESUME] max_iter already completed: {sorted(completed)}")
    else:
        results   = []
        completed = set()
        print("\n[INFO] No checkpoint found — starting from scratch.")

    # Rebuild all_data from already loaded results
    all_data = {m: [] for m in MAX_ITER_LIST}
    for r in results:
        mi = r['max_iter']
        if mi in all_data:
            all_data[mi].append({
                'epsilon':   r['epsilon'],
                'accuracy':  r['accuracy'],
                'mean_linf': r['mean_linf_applied']
            })

    # --------------------------------------------------------------------------
    # 4. Evaluation Loop
    # --------------------------------------------------------------------------
    total_combinations = len(MAX_ITER_LIST)
    current_combo      = 1

    for max_iter in MAX_ITER_LIST:
        print(f"\n{'#'*50}")
        print(f"MAX_ITER = {max_iter}")
        print(f"{'#'*50}")

        # --- SKIP if already completed ---
        if max_iter in completed:
            print(f"[SKIP {current_combo}/{total_combinations}] "
                    f"max_iter={max_iter} — already completed, skipping.")
            current_combo += 1
            continue

        print(f"\n[{current_combo}/{total_combinations}] "
                f"max_iter={max_iter} | epsilon={EPSILON}")   # FIX #1

        adv_combo_dir = os.path.join(
            ADV_BASE_DIR,
            f"iter_{max_iter}_eps_{str(EPSILON).replace('.', '_')}"  # FIX #1
        )
        os.makedirs(adv_combo_dir, exist_ok=True)

        deepfool = DeepFool(
            classifier=art_classifier,
            max_iter=max_iter,
            epsilon=EPSILON,    
            nb_grads=10,
            batch_size=1,
            verbose=False
        )

        start_time = time.time()

        x_adv_list = []
        for i in range(len(x_data)):
            single_adv = deepfool.generate(x=x_data[i:i+1])
            x_adv_list.append(single_adv)
            if (i + 1) % 10 == 0:
                print(f"  Progress: {i+1}/{len(x_data)} images generated...")

        x_adv_raw = np.concatenate(x_adv_list, axis=0)

        perturbation_raw    = x_adv_raw - x_data
        correct_predictions = 0
        saved_count         = 0
        failed_budget       = 0
        l_inf_list          = []

        for i in range(len(x_adv_raw)):
            l_inf_img = np.max(np.abs(perturbation_raw[i]))

            if l_inf_img > EPSILON_LIMIT + 1e-5:
                failed_budget += 1
                continue

            adv_clipped = np.clip(x_adv_raw[i], 0.0, 1.0)
            l_inf_list.append(l_inf_img)

            pred_clipped     = art_classifier.predict(adv_clipped[np.newaxis])
            pred_idx_clipped = np.argmax(pred_clipped, axis=1)[0]
            raw_label        = LABELS[pred_idx_clipped]
            if isinstance(raw_label, np.ndarray):
                raw_label = raw_label.flatten()[0]
            if isinstance(raw_label, bytes):
                raw_label = raw_label.decode('utf-8')
            pred_name = str(raw_label).replace("b'", "").replace("'", "").strip()
            true_name = id_to_name.get(true_ids[i], "Unknown")

            if normalize_label(pred_name) == normalize_label(true_name):
                correct_predictions += 1

            adv_save = np.transpose(adv_clipped, (1, 2, 0))
            adv_save = (adv_save * 255.0).astype(np.uint8)
            Image.fromarray(adv_save).save(
                os.path.join(adv_combo_dir, image_names[i])
            )
            saved_count += 1

        elapsed   = time.time() - start_time
        evaluated = len(x_data) - failed_budget
        accuracy  = (correct_predictions / evaluated * 100) if evaluated > 0 else 0.0
        mean_linf = float(np.mean(l_inf_list)) if l_inf_list else 0.0

        print(f"  Evaluated: {evaluated}/100 | "
                f"Budget exceeded (ignored): {failed_budget}/100")
        print(f"  Accuracy (on evaluated): {accuracy:.2f}% | "
                f"Mean L-inf: {mean_linf:.4f} | "
                f"Time: {elapsed:.1f}s")

        new_row = {
            'attack_name':         ATTACK_NAME,
            'max_iter':            max_iter,
            'epsilon':             EPSILON,    # FIX #1
            'accuracy':            round(accuracy, 2),
            'correct_predictions': correct_predictions,
            'saved_images':        saved_count,
            'failed_budget':       failed_budget,
            'evaluated_images':    evaluated,
            'total_images':        len(x_data),
            'mean_linf_applied':   round(mean_linf, 4),
            'time_elapsed_s':      round(elapsed, 2)
        }

        # --- CHECKPOINT: write CSV immediately ---
        results.append(new_row)
        completed.add(max_iter)
        pd.DataFrame(results).to_csv(csv_path, index=False)
        print(f"  [CHECKPOINT] Saved to CSV "
                f"({len(results)}/{total_combinations} completed).")

        all_data[max_iter].append({
            'epsilon':   EPSILON,
            'accuracy':  accuracy,
            'mean_linf': mean_linf
        })

        current_combo += 1

    # --------------------------------------------------------------------------
    # 5. Completion check
    # --------------------------------------------------------------------------
    if len(results) < total_combinations:
        print(f"\n[WARNING] Only {len(results)}/{total_combinations} combinations "
                f"in the CSV — some plots may be incomplete.")
    else:
        print(f"\n[INFO] All {total_combinations} combinations completed.")

    df_results = pd.DataFrame(results)

    # --------------------------------------------------------------------------
    # 6. Individual plot per max_iter — bar chart accuracy ONLY
    # --------------------------------------------------------------------------
    fig, ax1 = plt.subplots(figsize=(10, 6))

    iter_labels  = [str(m) for m in MAX_ITER_LIST]
    acc_vals     = [all_data[m][0]['accuracy'] if all_data[m] else 0 for m in MAX_ITER_LIST]
    bar_colors   = [ITER_COLORS[m] for m in MAX_ITER_LIST]

    x_pos  = np.arange(len(MAX_ITER_LIST))
    width  = 0.6  # Wider bars since they are no longer paired

    # Plot only the accuracy bars
    bars1 = ax1.bar(x_pos, acc_vals, width=width,
                    color=bar_colors, alpha=0.85, label='Model Accuracy (%)')

    # Add baseline accuracy line
    ax1.axhline(y=BASELINE_ACC_100, color='#2ca02c', linestyle='--',
                linewidth=1.5, label=f'Baseline Accuracy ({BASELINE_ACC_100:.2f}%)')

    ax1.set_title(
        f'DeepFool — Accuracy per max_iter\n'
        f'(epsilon={EPSILON}, accuracy on images within budget ≤ {EPSILON_LIMIT})',
        fontsize=16, fontweight='bold', pad=15
    )
    ax1.set_xlabel('max_iter', fontsize=14)
    ax1.set_ylabel('Model Accuracy (%)', fontsize=14)
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(iter_labels, fontsize=12)
    # Find the maximum accuracy among the bars to dynamically scale the y-axis
    max_acc = max(acc_vals) if acc_vals else 100
    # Add a 10% padding above the highest bar for better visual spacing
    y_upper_limit = max_acc + 10
    # Increase granularity by placing a tick every 5% instead of 10%
    tick_step = 5
    ax1.set_yticks(np.arange(0, y_upper_limit + tick_step, tick_step))
    ax1.set_ylim(-2, y_upper_limit)
    ax1.tick_params(axis='both', labelsize=12)
    ax1.grid(True, axis='y', linestyle='-', linewidth=0.5, alpha=0.7)

    # Simplified legend
    handles = (
        [plt.Line2D([0], [0], color='#2ca02c', linestyle='--', linewidth=1.5)] +
        [plt.Rectangle((0,0),1,1, color=ITER_COLORS[m], alpha=0.85) for m in MAX_ITER_LIST]
    )
    labels_ = (
        [f'Baseline ({BASELINE_ACC_100:.2f}%)'] +
        [f'max_iter={m}' for m in MAX_ITER_LIST]
    )
    ax1.legend(handles, labels_, loc='upper right', fontsize=10)

    plt.tight_layout()
    bar_path = os.path.join(
        ATTACK_RESULTS_DIR,
        f'{ATTACK_NAME}_acc_per_iter_eps{str(EPSILON).replace(".", "_")}.png'
    )
    plt.savefig(bar_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n[INFO] Bar chart saved at: {bar_path}")

    # --------------------------------------------------------------------------
    # 7. Aggregated plot — Mean L-inf vs Accuracy (scatter, one point per iter)
    # --------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(9, 6))

    ax.axhline(y=BASELINE_ACC_100, color='#2ca02c', linestyle='--',
                linewidth=1.5,
                label=f'Baseline Accuracy ({BASELINE_ACC_100:.2f}%)')
    ax.axvline(x=EPSILON_LIMIT, color='black', linestyle=':', linewidth=2,
                label=r'Max Allowable L-inf (0.1)')

    for max_iter in MAX_ITER_LIST:
        if not all_data[max_iter]:
            continue
        d     = all_data[max_iter][0]
        color = ITER_COLORS[max_iter]
        ax.scatter(d['mean_linf'], d['accuracy'],
                    color=color, s=80, zorder=5,
                    label=f'max_iter={max_iter}')
        # Label next to the point
        ax.annotate(f"iter={max_iter}",
                    xy=(d['mean_linf'], d['accuracy']),
                    xytext=(6, 4), textcoords='offset points',
                    fontsize=10, color=color)

    ax.set_title(
        f'DeepFool — Mean L-inf vs Model Accuracy\n'
        f'(epsilon={EPSILON}, one point per max_iter, '
        f'accuracy on images within budget)',
        fontsize=16, fontweight='bold', pad=15
    )
    ax.set_xlabel(r'Mean L-inf Perturbation (raw, within budget)', fontsize=14)
    ax.set_ylabel('Model Accuracy (%) [on evaluated images]', fontsize=14)
    # Calculate dynamic x-axis limits based on the actual mean_linf data points
    all_linf_vals = [all_data[m][0]['mean_linf'] for m in MAX_ITER_LIST if all_data[m]]
    if all_linf_vals:
        min_linf = min(all_linf_vals)
        max_linf = max(all_linf_vals)
        # Add dynamic padding (e.g., 20% of the range) to avoid plotting points on the edges
        padding = (max_linf - min_linf) * 0.20
        # Fallback padding if all points collapse to the exact same x coordinate
        if padding == 0:
            padding = 0.005
        ax.set_xlim(min_linf - padding, max_linf + padding)

    # Let matplotlib handle the x-ticks automatically based on the new zoom level
    ax.set_yticks(np.arange(0, 101, 10))
    ax.tick_params(axis='both', labelsize=12)
    ax.set_ylim(-5, 105)
    ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
    ax.legend(loc='upper right', fontsize=12)
    plt.tight_layout()

    agg_path = os.path.join(ATTACK_RESULTS_DIR,
                            f'{ATTACK_NAME}_aggregated_linf_vs_acc.png')
    plt.savefig(agg_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n[INFO] Aggregated plot saved at: {agg_path}")

    # --------------------------------------------------------------------------
    # 8. Attack Summary
    # --------------------------------------------------------------------------
    valid_rows = df_results[df_results['evaluated_images'] > 0]
    best_row   = valid_rows.loc[valid_rows['accuracy'].idxmin()] \
                    if not valid_rows.empty else df_results.iloc[0]

    print("\n" + "=" * 65)
    print(f"{ATTACK_NAME} ATTACK SUMMARY REPORT")
    print("=" * 65)
    print(f"Dataset:           1-per-person subset (100 images)")
    print(f"Baseline Accuracy: {BASELINE_ACC_100:.2f}%")
    print(f"Budget L-inf:      <= {EPSILON_LIMIT} (images beyond budget ignored)")
    print(f"Epsilon (overshoot): {EPSILON}")
    print(f"Grid:              {len(MAX_ITER_LIST)} max_iter")   # FIX #3/#5
    print("-" * 65)
    print(f"{'max_iter':<12} {'Accuracy':>10} "
            f"{'Mean L-inf':>12} {'Evaluated':>10} {'Ignored':>8}")
    print("-" * 65)
    for r in results:
        print(f"  {r['max_iter']:<10} "
                f"{r['accuracy']:>10.2f}% "
                f"{r['mean_linf_applied']:>12.4f} "
                f"{r['evaluated_images']:>9}/100"
                f"{r['failed_budget']:>8}/100")
    print("-" * 65)
    print(f"Best Configuration (lowest accuracy):")
    print(f"  max_iter:   {best_row['max_iter']}")
    print(f"  Accuracy:   {best_row['accuracy']:.2f}%")
    print(f"  Mean L-inf: {best_row['mean_linf_applied']:.4f}")
    print(f"  Evaluated:  {best_row['evaluated_images']}/100")
    print("=" * 65 + "\n")

## Error Specific Attacks 

### Fast Gradient Sign Method (FGSM) — Error-Specific (Targeted) Attack

This step evaluates the FGSM attack in its targeted (error-specific) variant on NN1.
Two targeting strategies are evaluated:
- **Fixed**: every image is pushed toward a single fixed target class (impersonation attack).
- **LeastLikely**: each image is pushed toward the class assigned the lowest probability
  by the clean model, maximizing the semantic distance of the misclassification.

For targeted attacks the metric reported is the Attack Success Rate (ASR), defined as
the fraction of images for which the model predicts exactly the intended target class.
A CSV summary and a Targeted Security Evaluation Curve are saved for each mode.


In [54]:
if RUN_FGSM_SPECIFIC:
    # ==============================================================================
    # STEP Fast Gradient Sign Method (FGSM) — Error-Specific (Targeted) Attack
    # ==============================================================================

    print("Initializing FGSM Targeted Attack Pipeline...")

    # ------------------------------------------------------------------------------
    # 1. Configuration
    # ------------------------------------------------------------------------------
    ATTACK_NAME_BASE   = "FGSM_Targeted"
    EPSILONS           = [0.001, 0.003, 0.005, 0.01, 0.03, 0.05, 0.1]
    raw = LABELS[2026]
    if isinstance(raw, np.ndarray): raw = raw.flatten()[0]
    if isinstance(raw, bytes):      raw = raw.decode('utf-8')
    print(f"Class at index 2026: {raw}")
    TARGET_MODES       = ["LeastLikely", "Fixed"]

    # ------------------------------------------------------------------------------
    # 2. Evaluation Loop
    # ------------------------------------------------------------------------------
    for mode in TARGET_MODES:
        ATTACK_NAME        = f"{ATTACK_NAME_BASE}_{mode}"
        ATTACK_RESULTS_DIR = os.path.join(RESULTS_DIR, ATTACK_NAME)
        os.makedirs(ATTACK_RESULTS_DIR, exist_ok=True)

        # Pre-compute least likely targets ONCE before the epsilon loop
        if mode == "LeastLikely":
            print("\nPre-computing least likely targets for all images...")
            ll_targets_all = []
            for i in range(0, len(image_paths), BATCH_SIZE):
                batch_paths_pre   = image_paths[i:i + BATCH_SIZE]
                batch_tensors_pre = []
                for path in batch_paths_pre:
                    img = Image.open(path).convert('RGB')
                    batch_tensors_pre.append(preprocess_attack(img).numpy())
                x_pre = np.stack(batch_tensors_pre)
                ll_targets_all.extend(get_least_likely_indices(x_pre))
            save_least_likely_targets(ll_targets_all, ATTACK_NAME, filenames=image_paths)
            print(f"Pre-computed and saved {len(ll_targets_all)} targets.")
        else:
            ll_targets_all = None

        print(f"\n{'='*60}")
        print(f"Starting {ATTACK_NAME} across {len(EPSILONS)} epsilon values")
        print(f"{'='*60}")

        results              = []
        attack_success_rates = []

        for eps in EPSILONS:
            print(f"\n--- Evaluating Epsilon: {eps:.3f} | Mode: {mode} ---")

            fgsm = FastGradientMethod(estimator=art_classifier, eps=eps, targeted=True)

            successful_attacks = 0
            total_processed    = 0
            start_time         = time.time()

            for i in tqdm(range(0, len(image_paths), BATCH_SIZE), desc="Processing Batches"):
                batch_paths    = image_paths[i:i + BATCH_SIZE]
                batch_true_ids = true_folder_ids[i:i + BATCH_SIZE]

                batch_tensors   = []
                batch_filenames = []
                for path in batch_paths:
                    img = Image.open(path).convert('RGB')
                    batch_tensors.append(preprocess_attack(img).numpy())
                    batch_filenames.append(os.path.basename(path))

                x_batch = np.stack(batch_tensors)

                # Use pre-computed targets — slice the correct portion for this batch
                if mode == "LeastLikely":
                    batch_target_indices = ll_targets_all[i:i + BATCH_SIZE]
                elif mode == "Fixed":
                    batch_target_indices = [FIXED_TARGET_CLASS] * len(batch_paths)

                y_target_one_hot = np.zeros((len(batch_paths), NUM_CLASSES))
                for k, t_idx in enumerate(batch_target_indices):
                    y_target_one_hot[k, int(t_idx)] = 1.0

                x_adv        = fgsm.generate(x=x_batch, y=y_target_one_hot)
                preds        = art_classifier.predict(x_adv)
                pred_indices = np.argmax(preds, axis=1)

                for j, pred_idx in enumerate(pred_indices):
                    if pred_idx == batch_target_indices[j]:
                        successful_attacks += 1

                save_adversarial_images(x_adv, batch_true_ids, batch_filenames, ATTACK_NAME, eps)
                total_processed += len(batch_paths)

            elapsed        = time.time() - start_time
            asr_percentage = (successful_attacks / total_processed) * 100
            attack_success_rates.append(asr_percentage)

            print(f"ASR at eps={eps:.3f}: {asr_percentage:.2f}%")

            results.append({
                'attack_name':         ATTACK_NAME,
                'target_mode':         mode,
                'epsilon':             eps,
                'successful_attacks':  successful_attacks,
                'total_images':        total_processed,
                'attack_success_rate': round(asr_percentage, 2),
                'time_elapsed_s':      round(elapsed, 2)
            })
            
        # --------------------------------------------------------------------------
        # 3. Save Results CSV
        # --------------------------------------------------------------------------
        df_results = pd.DataFrame(results)
        csv_path   = os.path.join(ATTACK_RESULTS_DIR, f'{ATTACK_NAME}_results.csv')
        df_results.to_csv(csv_path, index=False)
        print(f"\n[INFO] Results CSV saved at: {csv_path}")

        # --------------------------------------------------------------------------
        # 4. Targeted Security Evaluation Curve
        # --------------------------------------------------------------------------
        plot_targeted_sec(EPSILONS, attack_success_rates, ATTACK_NAME, ATTACK_RESULTS_DIR)

        # --------------------------------------------------------------------------
        # 5. Attack Summary
        # --------------------------------------------------------------------------
        print("\n" + "=" * 50)
        print(f"{ATTACK_NAME} ATTACK SUMMARY REPORT")
        print("=" * 50)
        print(f"Target Mode:             {mode}")
        print(f"Evaluated Epsilon Range: {EPSILONS[0]} to {EPSILONS[-1]}")
        print(f"Highest ASR:             {attack_success_rates[-1]:.2f}% at eps={EPSILONS[-1]}")
        print("=" * 50 + "\n")

### Basic Iterative Method (BIM) — Error-Specific (Targeted) Attack

This step evaluates BIM in its targeted (error-specific) variant on NN1 across two
targeting strategies (Fixed and LeastLikely) and a range of `max_iter` values.
The step size is set to `eps / max_iter` for each configuration, keeping all other
parameters fixed (ceteris paribus). The Attack Success Rate (ASR) is reported for
each combination. Individual and comparative Targeted SECs and a hyperparameter
effect plot are saved alongside a CSV summary.

In [55]:
if RUN_BIM_SPECIFIC: 
    # ==============================================================================
    # Basic Iterative Method (BIM) — Error-Specific (Targeted) Attack
    # ==============================================================================

    print("Initializing BIM Targeted Attack Pipeline...")

    # ------------------------------------------------------------------------------
    # 1. Configuration
    # ------------------------------------------------------------------------------
    ATTACK_NAME_BASE   = "BIM_Targeted"
    EPSILONS           = [0.001, 0.003, 0.005, 0.01, 0.03, 0.05, 0.1]
    MAX_ITER_LIST      = [1, 5, 10, 20]
    TARGET_MODES       = ["LeastLikely", "Fixed"]

    ITER_COLORS = {
        1:  '#1f77b4',
        5:  '#ff7f0e',
        10: '#2ca02c',
        20: '#d62728'
    }

    FIXED_EPSILONS_FOR_EFFECT = [0.01, 0.05, 0.1]
    EPSILON_EFFECT_COLORS     = {
        0.01: '#9467bd',
        0.05: '#8c564b',
        0.1:  '#e377c2'
    }

    # ------------------------------------------------------------------------------
    # 2. Evaluation Loop
    # ------------------------------------------------------------------------------
    for mode in TARGET_MODES:
        ATTACK_NAME        = f"{ATTACK_NAME_BASE}_{mode}"
        ATTACK_RESULTS_DIR = os.path.join(RESULTS_DIR, ATTACK_NAME)
        os.makedirs(ATTACK_RESULTS_DIR, exist_ok=True)
        
        # Pre-compute least likely targets ONCE before the epsilon loop
        if mode == "LeastLikely":
            print("\nPre-computing least likely targets for all images...")
            ll_targets_all = []
            for i in range(0, len(image_paths), BATCH_SIZE):
                batch_paths_pre   = image_paths[i:i + BATCH_SIZE]
                batch_tensors_pre = []
                for path in batch_paths_pre:
                    img = Image.open(path).convert('RGB')
                    batch_tensors_pre.append(preprocess_attack(img).numpy())
                x_pre = np.stack(batch_tensors_pre)
                ll_targets_all.extend(get_least_likely_indices(x_pre))
            save_least_likely_targets(ll_targets_all, ATTACK_NAME, filenames=image_paths)
            print(f"Pre-computed and saved {len(ll_targets_all)} targets.")
        else:
            ll_targets_all = None

        print(f"\n{'*'*70}")
        print(f"Starting {ATTACK_NAME} across "
            f"{len(MAX_ITER_LIST)} max_iter values x {len(EPSILONS)} epsilon values")
        print(f"{'*'*70}")

        results  = []
        all_asrs = {}  # max_iter -> list of ASRs, one per epsilon

        for max_iter in MAX_ITER_LIST:
            print(f"\n{'='*50}")
            print(f"MAX_ITER = {max_iter} | MODE = {mode}")
            print(f"{'='*50}")

            attack_success_rates = []

            for eps in EPSILONS:
                print(f"\n--- Evaluating Epsilon: {eps:.3f} | max_iter: {max_iter} ---")

                eps_step = eps / max_iter

                bim = BasicIterativeMethod(
                    estimator=art_classifier,
                    eps=eps,
                    eps_step=eps_step,
                    max_iter=max_iter,
                    targeted=True,
                    batch_size=BATCH_SIZE
                )

                successful_attacks = 0
                total_processed    = 0
                start_time         = time.time()

                for i in tqdm(range(0, len(image_paths), BATCH_SIZE), desc="Processing Batches"):
                    batch_paths    = image_paths[i:i + BATCH_SIZE]
                    batch_true_ids = true_folder_ids[i:i + BATCH_SIZE]

                    batch_tensors   = []
                    batch_filenames = []
                    for path in batch_paths:
                        img = Image.open(path).convert('RGB')
                        batch_tensors.append(preprocess_attack(img).numpy())
                        batch_filenames.append(os.path.basename(path))

                    x_batch = np.stack(batch_tensors)

                    if mode == "LeastLikely":
                        batch_target_indices = ll_targets_all[i:i + BATCH_SIZE]
                    elif mode == "Fixed":
                        batch_target_indices = [FIXED_TARGET_CLASS] * len(batch_paths)

                    y_target_one_hot = np.zeros((len(batch_paths), NUM_CLASSES))
                    for k, t_idx in enumerate(batch_target_indices):
                        y_target_one_hot[k, int(t_idx)] = 1.0

                    x_adv        = bim.generate(x=x_batch, y=y_target_one_hot)
                    preds        = art_classifier.predict(x_adv)
                    pred_indices = np.argmax(preds, axis=1)

                    for j, pred_idx in enumerate(pred_indices):
                        if pred_idx == batch_target_indices[j]:
                            successful_attacks += 1

                    save_adversarial_images(x_adv, batch_true_ids, batch_filenames,f"{ATTACK_NAME}_iter{max_iter}", eps)
                    total_processed += len(batch_paths)

                elapsed = time.time() - start_time
                asr     = (successful_attacks / total_processed) * 100
                attack_success_rates.append(asr)

                print(f"ASR at eps={eps:.3f} | max_iter={max_iter}: {asr:.2f}%")

                results.append({
                    'attack_name':         ATTACK_NAME,
                    'target_mode':         mode,
                    'max_iter':            max_iter,
                    'eps_step':            round(eps_step, 6),
                    'epsilon':             eps,
                    'successful_attacks':  successful_attacks,
                    'total_images':        total_processed,
                    'attack_success_rate': round(asr, 2),
                    'time_elapsed_s':      round(elapsed, 2)
                })

            all_asrs[max_iter] = attack_success_rates

            # ----------------------------------------------------------------------
            # 3. Individual Targeted SEC for this max_iter value
            # ----------------------------------------------------------------------
            color = ITER_COLORS[max_iter]

            fig, ax = plt.subplots(figsize=(9, 6))
            ax.plot([0.0, EPSILONS[0]], [0.0, attack_success_rates[0]],
                    linestyle='-', color=color, linewidth=1.5)
            ax.plot(EPSILONS, attack_success_rates,
                    marker='o', linestyle='-', color=color,
                    linewidth=1.5, markersize=6,
                    label=f'ASR ({ATTACK_NAME}, max_iter={max_iter})')
            ax.plot(0.0, 0.0,
                    marker='o', markerfacecolor=color,
                    markeredgecolor='#2ca02c', markeredgewidth=1,
                    markersize=6, linestyle='None', label='Baseline ASR (~0%)')

            ax.set_title(f'Targeted SEC — {ATTACK_NAME} (max_iter={max_iter})',
                        fontsize=16, fontweight='bold', pad=15)
            ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
            ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
            ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
            ax.set_xticks(np.arange(0.0, 0.11, 0.01))
            ax.set_yticks(np.arange(0, 101, 10))
            ax.tick_params(axis='both', labelsize=12)
            ax.set_ylim(-5, 105)
            ax.set_xlim(-0.005, 0.105)
            ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2,
                    label=r'Max Allowable $\epsilon$ (0.1)')
            ax.legend(loc='upper left', fontsize=12)
            plt.tight_layout()

            plot_path = os.path.join(ATTACK_RESULTS_DIR,
                                    f'{ATTACK_NAME}_iter{max_iter}_SEC.png')
            plt.savefig(plot_path, dpi=300, bbox_inches='tight')
            plt.show()
            print(f"\n[INFO] Individual SEC saved at: {plot_path}")

        # --------------------------------------------------------------------------
        # 4. Comparative Targeted SEC — all max_iter curves overlaid
        # --------------------------------------------------------------------------
        fig, ax = plt.subplots(figsize=(9, 6))
        ax.plot(0.0, 0.0, marker='o', markerfacecolor='black',
                markeredgecolor='#2ca02c', markeredgewidth=1,
                markersize=6, linestyle='None', label='Baseline ASR (~0%)')

        for max_iter, asrs in all_asrs.items():
            color = ITER_COLORS[max_iter]
            ax.plot([0.0, EPSILONS[0]], [0.0, asrs[0]],
                    linestyle='-', color=color, linewidth=1.5)
            ax.plot(EPSILONS, asrs,
                    marker='o', linestyle='-', color=color,
                    linewidth=1.5, markersize=6, label=f'max_iter={max_iter}')

        ax.set_title(f'Targeted SEC — {ATTACK_NAME} (Comparative)',
                    fontsize=16, fontweight='bold', pad=15)
        ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
        ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
        ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
        ax.set_xticks(np.arange(0.0, 0.11, 0.01))
        ax.set_yticks(np.arange(0, 101, 10))
        ax.tick_params(axis='both', labelsize=12)
        ax.set_ylim(-5, 105)
        ax.set_xlim(-0.005, 0.105)
        ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2,
                label=r'Max Allowable $\epsilon$ (0.1)')
        ax.legend(loc='upper left', fontsize=12)
        plt.tight_layout()

        comparative_path = os.path.join(ATTACK_RESULTS_DIR,
                                        f'{ATTACK_NAME}_comparative_SEC.png')
        plt.savefig(comparative_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\n[INFO] Comparative SEC saved at: {comparative_path}")

        # --------------------------------------------------------------------------
        # 5. Hyperparameter Effect Plot — ASR vs max_iter at fixed epsilon values
        #    All other parameters fixed — ceteris paribus.
        # --------------------------------------------------------------------------
        fig, ax = plt.subplots(figsize=(9, 6))

        for fixed_eps in FIXED_EPSILONS_FOR_EFFECT:
            eps_idx     = EPSILONS.index(fixed_eps)
            asrs_at_eps = [all_asrs[m][eps_idx] for m in MAX_ITER_LIST]
            ax.plot(MAX_ITER_LIST, asrs_at_eps,
                    marker='o', linestyle='-',
                    color=EPSILON_EFFECT_COLORS[fixed_eps],
                    linewidth=1.5, markersize=6,
                    label=rf'$\epsilon$ = {fixed_eps}')

        ax.axhline(y=0.0, color='#2ca02c', linestyle='--',
                linewidth=1.5, label='Baseline ASR (~0%)')
        ax.set_title(f'Effect of max_iter on ASR — {ATTACK_NAME}',
                    fontsize=16, fontweight='bold', pad=15)
        ax.set_xlabel('Number of Iterations (max_iter)', fontsize=14)
        ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
        ax.set_xticks(MAX_ITER_LIST)
        ax.set_yticks(np.arange(0, 101, 10))
        ax.tick_params(axis='both', labelsize=12)
        ax.set_ylim(-5, 105)
        ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
        ax.legend(loc='upper left', fontsize=12)
        plt.tight_layout()

        effect_path = os.path.join(ATTACK_RESULTS_DIR,
                                f'{ATTACK_NAME}_maxiter_effect.png')
        plt.savefig(effect_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\n[INFO] Hyperparameter effect plot saved at: {effect_path}")

        # --------------------------------------------------------------------------
        # 6. Save Results CSV
        # --------------------------------------------------------------------------
        df_results = pd.DataFrame(results)
        csv_path   = os.path.join(ATTACK_RESULTS_DIR, f'{ATTACK_NAME}_results.csv')
        df_results.to_csv(csv_path, index=False)
        print(f"\n[INFO] Results CSV saved at: {csv_path}")

        # --------------------------------------------------------------------------
        # 7. Attack Summary
        # --------------------------------------------------------------------------
        print("\n" + "=" * 50)
        print(f"{ATTACK_NAME} ATTACK SUMMARY REPORT")
        print("=" * 50)
        print(f"Target Mode:               {mode}")
        print(f"Evaluated Epsilon Range:   {EPSILONS[0]} to {EPSILONS[-1]}")
        print(f"Max Iter Values:           {MAX_ITER_LIST}")
        print("-" * 50)
        for max_iter, asrs in all_asrs.items():
            print(f"  max_iter={max_iter:2d} — Highest ASR: {asrs[-1]:.2f}% (at max eps)")
        print("=" * 50 + "\n")

### Projected Gradient Descent (PGD) — Error-Specific (Targeted) Attack

This step evaluates PGD in its targeted (error-specific) variant on NN1 across two
targeting strategies (Fixed and LeastLikely), a range of `max_iter` values, and a
range of `num_random_init` values. The step size is set to `eps / max_iter` for each
configuration (ceteris paribus). Individual and comparative Targeted SECs and
hyperparameter effect plots for both `max_iter` and `num_random_init` are saved
alongside a full CSV summary of all results.




In [56]:
if RUN_PGD_SPECIFIC:
    
    # ==============================================================================
    #  Projected Gradient Descent (PGD) — Error-Specific (Targeted) Attack
    # ==============================================================================

    print("Initializing PGD Targeted Attack Pipeline...")

    # ------------------------------------------------------------------------------
    # 1. Configuration
    # ------------------------------------------------------------------------------
    ATTACK_NAME_BASE   = "PGD_Targeted"
    EPSILONS           = [0.001, 0.003, 0.005, 0.01, 0.03, 0.05, 0.1]
    MAX_ITER_LIST      = [1, 5, 10, 20, 40]
    NUM_RANDOM_INITS   = [0, 1, 3, 5]
    TARGET_MODES       = ["LeastLikely", "Fixed"]

    ITER_COLORS = {
        1:  '#1f77b4',
        5:  '#ffbf00',
        10: '#2ca02c',
        20: '#d62728',
        40: '#9467bd'
    }

    RAND_COLORS = {
        0: '#1f77b4',
        1: '#ff7f0e',
        3: '#2ca02c',
        5: '#d62728'
    }

    FIXED_EPSILONS_FOR_EFFECT = [0.01, 0.05, 0.1]
    EPSILON_EFFECT_COLORS     = {
        0.01: '#9467bd',
        0.05: '#8c564b',
        0.1:  '#e377c2'
    }

    # ------------------------------------------------------------------------------
    # 2. Evaluation Loop
    # ------------------------------------------------------------------------------
    for mode in TARGET_MODES:
        ATTACK_NAME        = f"{ATTACK_NAME_BASE}_{mode}"
        ATTACK_RESULTS_DIR = os.path.join(RESULTS_DIR, ATTACK_NAME)
        os.makedirs(ATTACK_RESULTS_DIR, exist_ok=True)
        
        # Pre-compute least likely targets ONCE before the epsilon loop
        if mode == "LeastLikely":
            print("\nPre-computing least likely targets for all images...")
            ll_targets_all = []
            for i in range(0, len(image_paths), BATCH_SIZE):
                batch_paths_pre   = image_paths[i:i + BATCH_SIZE]
                batch_tensors_pre = []
                for path in batch_paths_pre:
                    img = Image.open(path).convert('RGB')
                    batch_tensors_pre.append(preprocess_attack(img).numpy())
                x_pre = np.stack(batch_tensors_pre)
                ll_targets_all.extend(get_least_likely_indices(x_pre))
            save_least_likely_targets(ll_targets_all, ATTACK_NAME, filenames=image_paths)
            print(f"Pre-computed and saved {len(ll_targets_all)} targets.")
        else:
            ll_targets_all = None
            
        print(f"\n{'*'*70}")
        print(f"Starting {ATTACK_NAME} across "
            f"{len(NUM_RANDOM_INITS)} nri x "
            f"{len(MAX_ITER_LIST)} iters x "
            f"{len(EPSILONS)} epsilons")
        print(f"{'*'*70}")

        results        = []
        all_asrs       = {nri: {} for nri in NUM_RANDOM_INITS}

        for num_random_init in NUM_RANDOM_INITS:
            print(f"\n{'#'*50}")
            print(f"NUM_RANDOM_INIT = {num_random_init} | MODE = {mode}")
            print(f"{'#'*50}")

            for max_iter in MAX_ITER_LIST:
                print(f"\n{'='*50}")
                print(f"MAX_ITER = {max_iter} | NUM_RANDOM_INIT = {num_random_init}")
                print(f"{'='*50}")

                attack_success_rates = []

                for eps in EPSILONS:
                    print(f"\n--- eps={eps:.3f} | max_iter={max_iter} | nri={num_random_init} ---")

                    eps_step = eps / max_iter

                    pgd = ProjectedGradientDescentPyTorch(
                        estimator=art_classifier,
                        norm=np.inf,
                        eps=eps,
                        eps_step=eps_step,
                        max_iter=max_iter,
                        num_random_init=num_random_init,
                        targeted=True,
                        batch_size=BATCH_SIZE,
                        verbose=False
                    )

                    successful_attacks = 0
                    total_processed    = 0
                    start_time         = time.time()

                    for i in tqdm(range(0, len(image_paths), BATCH_SIZE), desc="Processing Batches"):
                        batch_paths    = image_paths[i:i + BATCH_SIZE]
                        batch_true_ids = true_folder_ids[i:i + BATCH_SIZE]

                        batch_tensors   = []
                        batch_filenames = []
                        for path in batch_paths:
                            img = Image.open(path).convert('RGB')
                            batch_tensors.append(preprocess_attack(img).numpy())
                            batch_filenames.append(os.path.basename(path))

                        x_batch = np.stack(batch_tensors)

                        if mode == "LeastLikely":
                            batch_target_indices = ll_targets_all[i:i + BATCH_SIZE]
                        elif mode == "Fixed":
                            batch_target_indices = [FIXED_TARGET_CLASS] * len(batch_paths)

                        y_target_one_hot = np.zeros((len(batch_paths), NUM_CLASSES))
                        for k, t_idx in enumerate(batch_target_indices):
                            y_target_one_hot[k, int(t_idx)] = 1.0

                        x_adv        = pgd.generate(x=x_batch, y=y_target_one_hot)
                        preds        = art_classifier.predict(x_adv)
                        pred_indices = np.argmax(preds, axis=1)

                        for j, pred_idx in enumerate(pred_indices):
                            if pred_idx == batch_target_indices[j]:
                                successful_attacks += 1

                        save_adversarial_images(
                            x_adv, batch_true_ids, batch_filenames,
                            f"{ATTACK_NAME}_nri{num_random_init}_iter{max_iter}", eps
                        )
                        total_processed += len(batch_paths)

                    elapsed = time.time() - start_time
                    asr     = (successful_attacks / total_processed) * 100
                    attack_success_rates.append(asr)

                    print(f"ASR at eps={eps:.3f} | max_iter={max_iter} | nri={num_random_init}: {asr:.2f}%")

                    results.append({
                        'attack_name':         ATTACK_NAME,
                        'target_mode':         mode,
                        'num_random_init':     num_random_init,
                        'max_iter':            max_iter,
                        'eps_step':            round(eps / max_iter, 6),
                        'epsilon':             eps,
                        'successful_attacks':  successful_attacks,
                        'total_images':        total_processed,
                        'attack_success_rate': round(asr, 2),
                        'time_elapsed_s':      round(elapsed, 2)
                    })

                all_asrs[num_random_init][max_iter] = attack_success_rates

                # ------------------------------------------------------------------
                # 3. Individual Targeted SEC
                # ------------------------------------------------------------------
                color = ITER_COLORS[max_iter]

                fig, ax = plt.subplots(figsize=(9, 6))
                ax.plot([0.0, EPSILONS[0]], [0.0, attack_success_rates[0]],
                        linestyle='-', color=color, linewidth=1.5)
                ax.plot(EPSILONS, attack_success_rates,
                        marker='o', linestyle='-', color=color,
                        linewidth=1.5, markersize=6,
                        label=f'ASR ({ATTACK_NAME}, max_iter={max_iter}, nri={num_random_init})')
                ax.plot(0.0, 0.0,
                        marker='o', markerfacecolor=color,
                        markeredgecolor='#2ca02c', markeredgewidth=1,
                        markersize=6, linestyle='None', label='Baseline ASR (~0%)')

                ax.set_title(f'Targeted SEC — {ATTACK_NAME}\n'
                            f'(max_iter={max_iter}, num_random_init={num_random_init})',
                            fontsize=16, fontweight='bold', pad=15)
                ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
                ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
                ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
                ax.set_xticks(np.arange(0.0, 0.11, 0.01))
                ax.set_yticks(np.arange(0, 101, 10))
                ax.tick_params(axis='both', labelsize=12)
                ax.set_ylim(-5, 105)
                ax.set_xlim(-0.005, 0.105)
                ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2,
                        label=r'Max Allowable $\epsilon$ (0.1)')
                ax.legend(loc='upper left', fontsize=12)
                plt.tight_layout()

                plot_path = os.path.join(
                    ATTACK_RESULTS_DIR,
                    f'{ATTACK_NAME}_nri{num_random_init}_iter{max_iter}_SEC.png'
                )
                plt.savefig(plot_path, dpi=300, bbox_inches='tight')
                plt.show()
                print(f"\n[INFO] Individual SEC saved at: {plot_path}")

        # --------------------------------------------------------------------------
        # 4. Comparative SEC per num_random_init — max_iter varies
        # --------------------------------------------------------------------------
        for num_random_init in NUM_RANDOM_INITS:
            fig, ax = plt.subplots(figsize=(9, 6))
            ax.plot(0.0, 0.0, marker='o', markerfacecolor='black',
                    markeredgecolor='#2ca02c', markeredgewidth=1,
                    markersize=6, linestyle='None', label='Baseline ASR (~0%)')

            for max_iter in MAX_ITER_LIST:
                color = ITER_COLORS[max_iter]
                asrs  = all_asrs[num_random_init][max_iter]
                ax.plot([0.0, EPSILONS[0]], [0.0, asrs[0]],
                        linestyle='-', color=color, linewidth=1.5)
                ax.plot(EPSILONS, asrs,
                        marker='o', linestyle='-', color=color,
                        linewidth=1.5, markersize=6, label=f'max_iter={max_iter}')

            ax.set_title(f'Targeted SEC — {ATTACK_NAME} | nri={num_random_init} (max_iter varies)',
                        fontsize=16, fontweight='bold', pad=15)
            ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
            ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
            ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
            ax.set_xticks(np.arange(0.0, 0.11, 0.01))
            ax.set_yticks(np.arange(0, 101, 10))
            ax.tick_params(axis='both', labelsize=12)
            ax.set_ylim(-5, 105)
            ax.set_xlim(-0.005, 0.105)
            ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2,
                    label=r'Max Allowable $\epsilon$ (0.1)')
            ax.legend(loc='upper left', fontsize=12)
            plt.tight_layout()

            path = os.path.join(
                ATTACK_RESULTS_DIR,
                f'{ATTACK_NAME}_nri{num_random_init}_comparative_maxiter_SEC.png'
            )
            plt.savefig(path, dpi=300, bbox_inches='tight')
            plt.show()
            print(f"\n[INFO] Comparative SEC (nri={num_random_init}) saved at: {path}")

        # --------------------------------------------------------------------------
        # 5. Comparative SEC per max_iter — num_random_init varies
        # --------------------------------------------------------------------------
        for max_iter in MAX_ITER_LIST:
            fig, ax = plt.subplots(figsize=(9, 6))
            ax.plot(0.0, 0.0, marker='o', markerfacecolor='black',
                    markeredgecolor='#2ca02c', markeredgewidth=1,
                    markersize=6, linestyle='None', label='Baseline ASR (~0%)')

            for num_random_init in NUM_RANDOM_INITS:
                color = RAND_COLORS[num_random_init]
                asrs  = all_asrs[num_random_init][max_iter]
                ax.plot([0.0, EPSILONS[0]], [0.0, asrs[0]],
                        linestyle='-', color=color, linewidth=1.5)
                ax.plot(EPSILONS, asrs,
                        marker='o', linestyle='-', color=color,
                        linewidth=1.5, markersize=6, label=f'nri={num_random_init}')

            ax.set_title(f'Targeted SEC — {ATTACK_NAME} | max_iter={max_iter} (nri varies)',
                        fontsize=16, fontweight='bold', pad=15)
            ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
            ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
            ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
            ax.set_xticks(np.arange(0.0, 0.11, 0.01))
            ax.set_yticks(np.arange(0, 101, 10))
            ax.tick_params(axis='both', labelsize=12)
            ax.set_ylim(-5, 105)
            ax.set_xlim(-0.005, 0.105)
            ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2,
                    label=r'Max Allowable $\epsilon$ (0.1)')
            ax.legend(loc='upper left', fontsize=12)
            plt.tight_layout()

            path = os.path.join(
                ATTACK_RESULTS_DIR,
                f'{ATTACK_NAME}_iter{max_iter}_comparative_nri_SEC.png'
            )
            plt.savefig(path, dpi=300, bbox_inches='tight')
            plt.show()
            print(f"\n[INFO] Comparative SEC (max_iter={max_iter}) saved at: {path}")

        # --------------------------------------------------------------------------
        # 6. Hyperparameter Effect Plot — ASR vs max_iter at fixed epsilon
        #    num_random_init fixed at maximum value — ceteris paribus.
        # --------------------------------------------------------------------------
        FIXED_NRI_FOR_EFFECT  = max(NUM_RANDOM_INITS)

        fig, ax = plt.subplots(figsize=(9, 6))
        for fixed_eps in FIXED_EPSILONS_FOR_EFFECT:
            eps_idx     = EPSILONS.index(fixed_eps)
            asrs_at_eps = [all_asrs[FIXED_NRI_FOR_EFFECT][m][eps_idx] for m in MAX_ITER_LIST]
            ax.plot(MAX_ITER_LIST, asrs_at_eps,
                    marker='o', linestyle='-',
                    color=EPSILON_EFFECT_COLORS[fixed_eps],
                    linewidth=1.5, markersize=6,
                    label=rf'$\epsilon$ = {fixed_eps}')

        ax.axhline(y=0.0, color='#2ca02c', linestyle='--',
                linewidth=1.5, label='Baseline ASR (~0%)')
        ax.set_title(f'Effect of max_iter on ASR — {ATTACK_NAME}\n'
                    f'(num_random_init={FIXED_NRI_FOR_EFFECT})',
                    fontsize=16, fontweight='bold', pad=15)
        ax.set_xlabel('Number of Iterations (max_iter)', fontsize=14)
        ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
        ax.set_xticks(MAX_ITER_LIST)
        ax.set_yticks(np.arange(0, 101, 10))
        ax.tick_params(axis='both', labelsize=12)
        ax.set_ylim(-5, 105)
        ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
        ax.legend(loc='upper left', fontsize=12)
        plt.tight_layout()

        effect_iter_path = os.path.join(
            ATTACK_RESULTS_DIR,
            f'{ATTACK_NAME}_maxiter_effect_nri{FIXED_NRI_FOR_EFFECT}.png'
        )
        plt.savefig(effect_iter_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\n[INFO] max_iter effect plot saved at: {effect_iter_path}")

        # --------------------------------------------------------------------------
        # 7. Hyperparameter Effect Plot — ASR vs num_random_init at fixed epsilon
        #    max_iter fixed at maximum value — ceteris paribus.
        # --------------------------------------------------------------------------
        FIXED_ITER_FOR_EFFECT = max(MAX_ITER_LIST)

        fig, ax = plt.subplots(figsize=(9, 6))
        for fixed_eps in FIXED_EPSILONS_FOR_EFFECT:
            eps_idx     = EPSILONS.index(fixed_eps)
            asrs_at_eps = [all_asrs[nri][FIXED_ITER_FOR_EFFECT][eps_idx] for nri in NUM_RANDOM_INITS]
            ax.plot(NUM_RANDOM_INITS, asrs_at_eps,
                    marker='o', linestyle='-',
                    color=EPSILON_EFFECT_COLORS[fixed_eps],
                    linewidth=1.5, markersize=6,
                    label=rf'$\epsilon$ = {fixed_eps}')

        ax.axhline(y=0.0, color='#2ca02c', linestyle='--',
                linewidth=1.5, label='Baseline ASR (~0%)')
        ax.set_title(f'Effect of num_random_init on ASR — {ATTACK_NAME}\n'
                    f'(max_iter={FIXED_ITER_FOR_EFFECT})',
                    fontsize=16, fontweight='bold', pad=15)
        ax.set_xlabel('Number of Random Restarts (num_random_init)', fontsize=14)
        ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
        ax.set_xticks(NUM_RANDOM_INITS)
        ax.set_yticks(np.arange(0, 101, 10))
        ax.tick_params(axis='both', labelsize=12)
        ax.set_ylim(-5, 105)
        ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
        ax.legend(loc='upper left', fontsize=12)
        plt.tight_layout()

        effect_nri_path = os.path.join(
            ATTACK_RESULTS_DIR,
            f'{ATTACK_NAME}_nri_effect_iter{FIXED_ITER_FOR_EFFECT}.png'
        )
        plt.savefig(effect_nri_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\n[INFO] num_random_init effect plot saved at: {effect_nri_path}")

        # --------------------------------------------------------------------------
        # 8. Save Results CSV
        # --------------------------------------------------------------------------
        df_results = pd.DataFrame(results)
        csv_path   = os.path.join(ATTACK_RESULTS_DIR, f'{ATTACK_NAME}_results.csv')
        df_results.to_csv(csv_path, index=False)
        print(f"\n[INFO] Results CSV saved at: {csv_path}")

        # --------------------------------------------------------------------------
        # 9. Attack Summary
        # --------------------------------------------------------------------------
        print("\n" + "=" * 50)
        print(f"{ATTACK_NAME} ATTACK SUMMARY REPORT")
        print("=" * 50)
        print(f"Target Mode:               {mode}")
        print(f"Evaluated Epsilon Range:   {EPSILONS[0]} to {EPSILONS[-1]}")
        print(f"Max Iter Values:           {MAX_ITER_LIST}")
        print(f"Num Random Init Values:    {NUM_RANDOM_INITS}")
        print("-" * 50)
        for num_random_init in NUM_RANDOM_INITS:
            for max_iter in MAX_ITER_LIST:
                asr_best = all_asrs[num_random_init][max_iter][-1]
                print(f"  nri={num_random_init} | max_iter={max_iter:2d} — "
                    f"Highest ASR: {asr_best:.2f}% (at max eps)")
        print("=" * 50 + "\n")

### Carlini & Wagner L-inf (CW) — Error-Specific (Targeted) Attack

This step evaluates the Carlini & Wagner L-infinity attack in its targeted
(error-specific) variant on NN1. Two targeting strategies are evaluated (subject to computational constraints):
- **Fixed**: all images are pushed toward a single predetermined target class.
- **LeastLikely**: each image is pushed toward the class assigned the lowest
  probability by the clean model, maximizing the semantic distance of the
  misclassification

The metric reported is the Attack Success Rate (ASR), defined as the fraction
of images for which the model predicts exactly the intended target class.
Images for which CW exceeds the L-infinity budget are counted as attack failures
and are never excluded from the ASR denominator.

Due to the high computational cost of CW, the 1-per-person subset (100 images)
is used. The attack is evaluated across a grid of `max_iter`, `learning_rate`,
and `confidence`. Targeted SECs, hyperparameter effect plots, and a CSV summary
are saved for each targeting mode.

In [ ]:
if RUN_CW_SPECIFIC:

    # ==========================================================================
    # STEP 3.5: Carlini & Wagner L-inf (CW) — Error-Specific (Targeted) Attack
    # ==========================================================================

    print("Initializing CW L-inf Targeted Attack Pipeline...")

    # --------------------------------------------------------------------------
    # 1. Configuration
    # --------------------------------------------------------------------------
    ATTACK_NAME_BASE  = "CW_Targeted"
    EPSILON_LIMIT     = 0.1
    NUM_CLASSES       = 8631   # FIX #8 — era mancante

    TARGET_MODE_SELECTION = "Fixed"

    if TARGET_MODE_SELECTION == "Both":
        TARGET_MODES = ["LeastLikely", "Fixed"]
    elif TARGET_MODE_SELECTION in ("Fixed", "LeastLikely"):
        TARGET_MODES = [TARGET_MODE_SELECTION]
    else:
        raise ValueError(
            f"Invalid TARGET_MODE_SELECTION: '{TARGET_MODE_SELECTION}'. "
            "Choose 'Fixed', 'LeastLikely', or 'Both'."
        )

    MAX_ITER_LIST      = [1, 20, 50]
    LEARNING_RATE_LIST = [0.01, 0.1]
    CONFIDENCE_LIST    = [0.0]

    # FIX #5 — colori allineati ai valori effettivi di MAX_ITER_LIST
    ITER_COLORS = {
        1: '#1f77b4',
        20: '#ff7f0e',
        50: '#2ca02c'
    }

    # --------------------------------------------------------------------------
    # 2. Load 100-image dataset (1 per subject)
    # --------------------------------------------------------------------------
    print("\nLoading 1-per-person dataset...")

    preprocess_cw = transforms.Compose([
        transforms.Resize((160, 160)),
        transforms.ToTensor()
    ])

    x_data             = []
    image_names        = []
    image_paths_for_ll = []
    true_ids           = []

    image_files = sorted([
        f for f in os.listdir(DATA_DIR_100)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])

    if len(image_files) != 100:
        print(f"[WARNING] Found {len(image_files)} images instead of 100.")

    for img_name in image_files:
        img_path = os.path.join(DATA_DIR_100, img_name)
        img      = Image.open(img_path).convert('RGB')
        x_data.append(preprocess_cw(img).numpy())
        image_names.append(img_name)
        stem = os.path.splitext(img_name)[0]
        image_paths_for_ll.append(os.path.join(stem, img_name))
        true_ids.append(stem)

    x_data = np.array(x_data)
    print(f"Loaded {len(x_data)} images. Shape: {x_data.shape}")

    # --------------------------------------------------------------------------
    # 3. Helper — label parsing centralizzato
    # --------------------------------------------------------------------------
    def parse_label(raw_label):
        if isinstance(raw_label, np.ndarray):
            raw_label = raw_label.flatten()[0]
        if isinstance(raw_label, bytes):
            raw_label = raw_label.decode('utf-8')
        return normalize_label(
            str(raw_label)
            .replace("b'", "").replace("'", "")
            .replace("[", "").replace("]", "")
            .strip()
        )

    # --------------------------------------------------------------------------
    # 4. Evaluation Loop (over target modes)
    # --------------------------------------------------------------------------
    for mode in TARGET_MODES:
        ATTACK_NAME        = f"{ATTACK_NAME_BASE}_{mode}"
        ATTACK_RESULTS_DIR = os.path.join(RESULTS_DIR, ATTACK_NAME)
        ADV_BASE_DIR       = os.path.join(ADVERSARIAL_DATA_DIR, ATTACK_NAME)
        os.makedirs(ATTACK_RESULTS_DIR, exist_ok=True)
        os.makedirs(ADV_BASE_DIR, exist_ok=True)

        # ----------------------------------------------------------------------
        # Checkpoint — carica risultati già completati dal CSV (se esiste)
        # FIX #6 — sostituisce la logica "cancella e ricomincia" con resume
        # ----------------------------------------------------------------------
        csv_path = os.path.join(
            ATTACK_RESULTS_DIR, f'{ATTACK_NAME}_results.csv'
        )

        CSV_COLUMNS = [
            'attack_name', 'target_mode', 'max_iter', 'learning_rate',
            'confidence', 'attack_success_rate', 'successful_attacks',
            'valid_images', 'saved_images', 'failed_budget', 'total_images',
            'mean_linf_raw', 'time_elapsed_s'
        ]

        if os.path.exists(csv_path):
            df_checkpoint = pd.read_csv(csv_path)
            for col in CSV_COLUMNS:
                if col not in df_checkpoint.columns:
                    df_checkpoint[col] = np.nan
            mode_results = df_checkpoint.to_dict('records')
            completed = {
                (row['max_iter'], row['learning_rate'], row['confidence'])
                for row in mode_results
            }
            print(f"\n[RESUME] CSV trovato per mode={mode}: "
                f"{len(completed)} combinazioni già completate su "
                f"{len(MAX_ITER_LIST)*len(LEARNING_RATE_LIST)*len(CONFIDENCE_LIST)} totali.")
            print(f"[RESUME] Completate: {sorted(completed)}")
        else:
            mode_results = []
            completed    = set()
            print(f"\n[INFO] Nessun checkpoint per mode={mode} — esecuzione da zero.")

        # Ricostruisce all_combo_data dai risultati già caricati
        all_combo_data = [
            {
                'label':         f"iter={r['max_iter']}, lr={r['learning_rate']}, "
                                f"c={r['confidence']}",
                'max_iter':      r['max_iter'],
                'learning_rate': r['learning_rate'],
                'confidence':    r['confidence'],
                'mean_linf':     r['mean_linf_raw'],
                'asr':           r['attack_success_rate']
            }
            for r in mode_results
        ]

        print(f"\n{'*'*70}")
        print(f"Starting {ATTACK_NAME} — "
            f"{len(MAX_ITER_LIST)} max_iter x "
            f"{len(LEARNING_RATE_LIST)} lr x "
            f"{len(CONFIDENCE_LIST)} confidence combinations")
        print(f"{'*'*70}")

        # Pre-computa target indices ONCE per mode
        if mode == "LeastLikely":
            print("\nPre-computing least likely target indices...")
            least_likely_targets = get_least_likely_indices(x_data)
            save_least_likely_targets(
                least_likely_targets,
                ATTACK_NAME,
                filenames=image_paths_for_ll
            )
            print(f"Least likely targets computed and saved for "
                f"{len(least_likely_targets)} images.")
        else:
            least_likely_targets = None

        total_combinations = (
            len(MAX_ITER_LIST)
            * len(LEARNING_RATE_LIST)
            * len(CONFIDENCE_LIST)
        )
        current_combo = 1

        for max_iter in MAX_ITER_LIST:
            for lr in LEARNING_RATE_LIST:
                for conf in CONFIDENCE_LIST:

                    combo_key = (max_iter, lr, conf)

                    # --- SKIP se già completata ---
                    if combo_key in completed:
                        print(f"\n[SKIP {current_combo}/{total_combinations}] "
                            f"max_iter={max_iter} | lr={lr} | confidence={conf} "
                            f"| mode={mode} — già completata, salto.")
                        current_combo += 1
                        continue

                    print(f"\n[{current_combo}/{total_combinations}] "
                        f"max_iter={max_iter} | lr={lr} | "
                        f"confidence={conf} | mode={mode}")

                    lr_str         = str(lr).replace('.', '_')
                    conf_str       = str(conf).replace('.', '_')
                    combo_dir_name = f"iter_{max_iter}_lr_{lr_str}_c_{conf_str}"
                    adv_combo_dir  = os.path.join(ADV_BASE_DIR, combo_dir_name)
                    os.makedirs(adv_combo_dir, exist_ok=True)

                    # Costruisce target one-hot e lista indici
                    y_target_one_hot     = np.zeros((len(x_data), NUM_CLASSES))
                    batch_target_indices = []

                    for k in range(len(x_data)):
                        if mode == "Fixed":
                            target_idx = FIXED_TARGET_CLASS
                        else:
                            target_idx = least_likely_targets[k]  # FIX #7 — safe: None solo se Fixed
                        batch_target_indices.append(target_idx)
                        y_target_one_hot[k, int(target_idx)] = 1.0

                    attack_cw = CarliniLInfMethod(
                        classifier=art_classifier,
                        targeted=True,
                        max_iter=max_iter,
                        learning_rate=lr,
                        confidence=conf,
                        verbose=False
                    )

                    start_time = time.time()

                    x_adv_list = []
                    for i in range(len(x_data)):
                        single_adv = attack_cw.generate(
                            x=x_data[i:i+1],
                            y=y_target_one_hot[i:i+1]
                        )
                        x_adv_list.append(single_adv)
                        if (i + 1) % 10 == 0:
                            print(f"  Progress: {i+1}/{len(x_data)} "
                                f"images generated...")

                    x_adv_raw = np.concatenate(x_adv_list, axis=0)

                    # ----------------------------------------------------------
                    # Valutazione con budget filter RIGOROSO
                    #
                    # Semantica:
                    #   - L-inf calcolata sulla perturbazione RAW (pre-clip)
                    #     sia per il filtro che per la statistica → coerenza
                    #   - Se L-inf > EPSILON_LIMIT: immagine IGNORATA
                    #     (non entra né in numeratore né in denominatore ASR)
                    #   - Se L-inf <= EPSILON_LIMIT: clip a [0,1], predici,
                    #     salva, conta per ASR
                    #   - Predizione SEMPRE su immagine clippata
                    # ----------------------------------------------------------
                    successful_attacks = 0
                    valid_count        = 0   # denominatore reale dell'ASR
                    saved_count        = 0
                    skipped_count      = 0
                    l_inf_list         = []  # perturbazioni raw delle img valide

                    for i in range(len(x_adv_raw)):
                        # FIX #2 — perturbazione raw usata sia per filtro
                        # che per la statistica (coerente)
                        perturbation_raw = x_adv_raw[i] - x_data[i]
                        l_inf_raw        = float(np.max(np.abs(perturbation_raw)))

                        # Oltre budget: completamente ignorata
                        if l_inf_raw > EPSILON_LIMIT + 1e-5:
                            skipped_count += 1
                            continue

                        # Immagine valida: clip, predici su clippata, salva
                        valid_count += 1
                        l_inf_list.append(l_inf_raw)   # FIX #2 — raw, coerente

                        adv_clipped      = np.clip(x_adv_raw[i], 0.0, 1.0)
                        pred_out         = art_classifier.predict(
                            adv_clipped[np.newaxis]
                        )
                        pred_idx_clipped = np.argmax(pred_out, axis=1)[0]

                        # Salva immagine avversariale valida
                        adv_save = np.transpose(adv_clipped, (1, 2, 0))
                        adv_save = (adv_save * 255.0).astype(np.uint8)
                        Image.fromarray(adv_save).save(
                            os.path.join(adv_combo_dir, image_names[i])
                        )
                        saved_count += 1

                        # ASR: successo se predetto = target
                        if pred_idx_clipped == int(batch_target_indices[i]):
                            successful_attacks += 1

                    elapsed   = time.time() - start_time
                    mean_linf = float(np.mean(l_inf_list)) if l_inf_list else 0.0

                    # FIX #1 — denominatore = valid_count, non len(x_data)
                    if valid_count > 0:
                        asr = (successful_attacks / valid_count) * 100
                    else:
                        asr = 0.0
                        print("  [WARNING] Nessuna immagine dentro budget.")

                    print(f"  Valide (dentro budget): {valid_count}/100 | "
                        f"Ignorate (oltre budget): {skipped_count}/100")
                    print(f"  ASR (su valide):        {asr:.2f}% | "
                        f"Mean L-inf (raw):        {mean_linf:.4f} | "
                        f"Time: {elapsed:.1f}s")

                    new_row = {
                        'attack_name':         ATTACK_NAME,
                        'target_mode':         mode,
                        'max_iter':            max_iter,
                        'learning_rate':       lr,
                        'confidence':          conf,
                        'attack_success_rate': round(asr, 2),
                        'successful_attacks':  successful_attacks,
                        'valid_images':        valid_count,   # FIX #4
                        'saved_images':        saved_count,
                        'failed_budget':       skipped_count,
                        'total_images':        len(x_data),
                        'mean_linf_raw':       round(mean_linf, 4),  # FIX #2
                        'time_elapsed_s':      round(elapsed, 2)
                    }

                    # --- CHECKPOINT: scrivi CSV subito dopo ogni combinazione ---
                    mode_results.append(new_row)
                    completed.add(combo_key)
                    pd.DataFrame(mode_results).to_csv(csv_path, index=False)
                    print(f"  [CHECKPOINT] Salvato nel CSV "
                        f"({len(mode_results)}/{total_combinations} completate).")

                    all_combo_data.append({
                        'label':         f'iter={max_iter}, lr={lr}, c={conf}',
                        'max_iter':      max_iter,
                        'learning_rate': lr,
                        'confidence':    conf,
                        'mean_linf':     mean_linf,
                        'asr':           asr
                    })

                    current_combo += 1

        # ----------------------------------------------------------------------
        # Verifica completamento
        # ----------------------------------------------------------------------
        if len(mode_results) < total_combinations:
            print(f"\n[WARNING] Solo {len(mode_results)}/{total_combinations} "
                f"combinazioni nel CSV — plot potrebbero essere incompleti.")
        else:
            print(f"\n[INFO] Tutte le {total_combinations} combinazioni completate.")

        df_results = pd.DataFrame(mode_results)

        # ----------------------------------------------------------------------
        # 5. Security Evaluation Curve (SEC) — ASR vs mean L-inf
        # ----------------------------------------------------------------------
        fig, ax = plt.subplots(figsize=(9, 6))

        ax.axhline(y=0.0, color='#2ca02c', linestyle='--',
                linewidth=1.5, label='Baseline ASR (~0%)')

        for max_iter in MAX_ITER_LIST:
            color  = ITER_COLORS[max_iter]   # FIX #5 — chiavi esistono ora
            combos = [c for c in all_combo_data if c['max_iter'] == max_iter]
            if not combos:
                continue
            combos.sort(key=lambda c: c['mean_linf'])
            x_vals = [c['mean_linf'] for c in combos]
            y_vals = [c['asr']       for c in combos]

            ax.plot(x_vals, y_vals,
                    marker='o', linestyle='-', color=color,
                    linewidth=1.5, markersize=6,
                    label=f'max_iter={max_iter}')

        ax.axvline(x=EPSILON_LIMIT, color='black', linestyle=':', linewidth=2,
                label=r'Max Allowable $\epsilon$ (0.1)')

        ax.set_title(
            f'Security Evaluation Curve — {ATTACK_NAME}\n'
            f'(ASR su immagini dentro budget L-inf ≤ {EPSILON_LIMIT})',
            fontsize=16, fontweight='bold', pad=15
        )
        ax.set_xlabel(
            r'Mean L-inf Perturbation (raw, dentro budget)',
            fontsize=14
        )
        ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
        ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
        ax.set_xticks(np.arange(0.0, 0.11, 0.01))
        ax.set_yticks(np.arange(0, 101, 10))
        ax.tick_params(axis='both', labelsize=12)
        ax.set_ylim(-5, 105)
        ax.set_xlim(-0.005, 0.105)
        ax.legend(loc='upper left', fontsize=12)
        plt.tight_layout()

        sec_path = os.path.join(
            ATTACK_RESULTS_DIR, f'{ATTACK_NAME}_SEC.png'
        )
        plt.savefig(sec_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"[INFO] SEC saved at: {sec_path}")

        # ----------------------------------------------------------------------
        # 6. ASR vs learning_rate — una curva per max_iter, conf fissa
        # ----------------------------------------------------------------------
        for conf_val in CONFIDENCE_LIST:
            conf_str_plot = str(conf_val).replace('.', '_')
            fig, ax = plt.subplots(figsize=(9, 6))

            for max_iter in MAX_ITER_LIST:
                color  = ITER_COLORS[max_iter]   # FIX #5
                subset = df_results[
                    (df_results['max_iter']   == max_iter) &
                    (df_results['confidence'] == conf_val)
                ].sort_values('learning_rate')
                if subset.empty:
                    continue

                ax.plot(subset['learning_rate'].tolist(),
                        subset['attack_success_rate'].tolist(),
                        marker='o', linestyle='-', color=color,
                        linewidth=1.5, markersize=6,
                        label=f'max_iter={max_iter}')

            ax.axhline(y=0.0, color='#2ca02c', linestyle='--',
                    linewidth=1.5, label='Baseline ASR (~0%)')
            ax.set_title(
                f'ASR vs Learning Rate — {ATTACK_NAME}\n'
                f'(confidence={conf_val}, ASR su immagini dentro budget)',
                fontsize=16, fontweight='bold', pad=15
            )
            ax.set_xlabel('Learning Rate', fontsize=14)
            ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
            ax.set_xscale('log')
            ax.set_xticks(LEARNING_RATE_LIST)
            ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
            ax.set_yticks(np.arange(0, 101, 10))
            ax.tick_params(axis='both', labelsize=12)
            ax.set_ylim(-5, 105)
            ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
            ax.legend(loc='upper left', fontsize=12)
            plt.tight_layout()

            plot_path = os.path.join(
                ATTACK_RESULTS_DIR,
                f'{ATTACK_NAME}_asr_vs_lr_conf{conf_str_plot}.png'
            )
            plt.savefig(plot_path, dpi=300, bbox_inches='tight')
            plt.show()
            print(f"[INFO] ASR vs lr plot (conf={conf_val}) saved at: {plot_path}")

        # ----------------------------------------------------------------------
        # 7. ASR vs confidence — una curva per max_iter, lr fisso
        # ----------------------------------------------------------------------
        for lr_val in LEARNING_RATE_LIST:
            lr_str_plot = str(lr_val).replace('.', '_')
            fig, ax = plt.subplots(figsize=(9, 6))

            for max_iter in MAX_ITER_LIST:
                color  = ITER_COLORS[max_iter]   # FIX #5
                subset = df_results[
                    (df_results['max_iter']      == max_iter) &
                    (df_results['learning_rate'] == lr_val)
                ].sort_values('confidence')
                if subset.empty:
                    continue

                ax.plot(subset['confidence'].tolist(),
                        subset['attack_success_rate'].tolist(),
                        marker='o', linestyle='-', color=color,
                        linewidth=1.5, markersize=6,
                        label=f'max_iter={max_iter}')

            ax.axhline(y=0.0, color='#2ca02c', linestyle='--',
                    linewidth=1.5, label='Baseline ASR (~0%)')
            ax.set_title(
                f'ASR vs Confidence — {ATTACK_NAME}\n'
                f'(learning_rate={lr_val}, ASR su immagini dentro budget)',
                fontsize=16, fontweight='bold', pad=15
            )
            ax.set_xlabel('Confidence', fontsize=14)
            ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
            ax.set_xticks(CONFIDENCE_LIST)
            ax.set_yticks(np.arange(0, 101, 10))
            ax.tick_params(axis='both', labelsize=12)
            ax.set_ylim(-5, 105)
            ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
            ax.legend(loc='upper left', fontsize=12)
            plt.tight_layout()

            plot_path = os.path.join(
                ATTACK_RESULTS_DIR,
                f'{ATTACK_NAME}_asr_vs_conf_lr{lr_str_plot}.png'
            )
            plt.savefig(plot_path, dpi=300, bbox_inches='tight')
            plt.show()
            print(f"[INFO] ASR vs conf plot (lr={lr_val}) saved at: {plot_path}")

        # ----------------------------------------------------------------------
        # 8. Attack Summary
        # ----------------------------------------------------------------------
        valid_rows = df_results[df_results['valid_images'] > 0]
        best_row   = valid_rows.loc[valid_rows['attack_success_rate'].idxmax()] \
                    if not valid_rows.empty else df_results.iloc[0]

        print("\n" + "=" * 65)
        print(f"{ATTACK_NAME} ATTACK SUMMARY REPORT")
        print("=" * 65)
        print(f"Target Mode:       {mode}")
        print(f"Dataset:           1-per-person subset (100 images)")
        print(f"Budget L-inf:      <= {EPSILON_LIMIT} (immagini oltre budget ignorate)")
        print(f"Grid:              {total_combinations} combinations evaluated")
        print("-" * 65)
        print(f"{'max_iter':<12} {'lr':<8} {'conf':<8} {'ASR':>8} "
            f"{'Mean L-inf':>12} {'Valide':>8} {'Skip':>6}")
        print("-" * 65)
        for r in mode_results:
            print(f"  {r['max_iter']:<10} {r['learning_rate']:<8} "
                f"{r['confidence']:<8} "
                f"{r['attack_success_rate']:>8.2f}% "
                f"{r['mean_linf_raw']:>12.4f} "
                f"{r['valid_images']:>7}/100"
                f"{r['failed_budget']:>6}")
        print("-" * 65)
        print(f"Best Configuration (highest ASR su immagini valide):")
        print(f"  max_iter:      {best_row['max_iter']}")
        print(f"  learning_rate: {best_row['learning_rate']}")
        print(f"  confidence:    {best_row['confidence']}")
        print(f"  ASR:           {best_row['attack_success_rate']:.2f}%")
        print(f"  Mean L-inf:    {best_row['mean_linf_raw']:.4f}")
        print(f"  Valide:        {best_row['valid_images']}/100")
        print("=" * 65 + "\n")

# 4 & 5: TRANSFERABILITY EVALUATION 

This section evaluates whether adversarial examples generated against NN1
(InceptionResnetV1) transfer to NN2 (ResNet-50), i.e., whether they cause
misclassification on a model they were not optimized for.

For error-generic attacks, transferability is measured as the drop in NN2
accuracy relative to its clean baseline. For error-specific attacks,
transferability is measured as the Attack Success Rate on NN2 — the fraction
of images for which NN2 predicts exactly the target class that was imposed
during generation against NN1.

Adversarial images are preprocessed for NN2 using nearest-neighbor interpolation
to preserve the L-infinity perturbation as faithfully as possible during the
required resize from 160x160 to 224x224.

### Transferability — FGSM (Error-Generic)

Evaluates NN2 accuracy on adversarial examples generated by FGSM against NN1,
across all epsilon values. A comparative SEC overlays NN1 and NN2 accuracy curves
to visualize the transferability gap.

In [58]:
if RUN_FGSM_TRANSFERABILITY_GENERIC:
    # ==============================================================================
    # STEP 5.1: Transferability — FGSM (Error-Generic)
    # ==============================================================================

    print("Evaluating FGSM transferability on NN2...")

    ATTACK_NAME        = "FGSM"
    EPSILONS           = [0.001, 0.003, 0.005, 0.01, 0.03, 0.05, 0.1]

    TRANSFER_RESULTS_DIR = os.path.join(RESULTS_DIR, f"{ATTACK_NAME}_Transferability")
    os.makedirs(TRANSFER_RESULTS_DIR, exist_ok=True)

    ADV_ATTACK_DIR = os.path.join(ADVERSARIAL_DATA_DIR, ATTACK_NAME)

    # Load NN1 results CSV to retrieve NN1 accuracies for the comparative SEC
    nn1_results_path = os.path.join(RESULTS_DIR, ATTACK_NAME, f"{ATTACK_NAME}_results.csv")
    df_nn1           = pd.read_csv(nn1_results_path)

    results      = []
    nn2_accuracies = []
    nn1_accuracies = []

    for eps in EPSILONS:
        eps_folder    = f"eps_{eps:.3f}".replace('.', '_')
        adv_eps_dir   = os.path.join(ADV_ATTACK_DIR, eps_folder)

        if not os.path.exists(adv_eps_dir):
            print(f"[WARNING] Directory not found: {adv_eps_dir}. Skipping.")
            continue

        print(f"\n--- Evaluating Epsilon: {eps:.3f} ---")

        correct_nn2    = 0
        total_processed = 0
        start_time      = time.time()

        for class_id in os.listdir(adv_eps_dir):
            class_folder = os.path.join(adv_eps_dir, class_id)
            if not os.path.isdir(class_folder):
                continue

            true_name  = id_to_name.get(class_id.strip(), "Unknown")
            true_clean = normalize_label(true_name)

            for img_name in os.listdir(class_folder):
                if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                    continue

                img_path = os.path.join(class_folder, img_name)
                try:
                    img        = Image.open(img_path).convert('RGB')
                    tensor_nn2 = preprocess_nn2(img).unsqueeze(0).to(DEVICE)

                    with torch.no_grad():
                        pred_idx_2 = nn2(tensor_nn2)[0].detach().cpu().numpy().argmax()

                    pred_clean_2 = clean_pred_label(pred_idx_2)

                    if pred_clean_2 == true_clean:
                        correct_nn2 += 1

                    total_processed += 1

                except Exception as e:
                    print(f"Error processing {img_path}: {e}")

        elapsed    = time.time() - start_time
        acc_nn2    = (correct_nn2 / total_processed) * 100 if total_processed > 0 else 0.0
        nn2_accuracies.append(acc_nn2)

        # Retrieve corresponding NN1 accuracy from saved CSV
        nn1_row = df_nn1[df_nn1['epsilon'] == eps]
        acc_nn1 = nn1_row['accuracy'].values[0] if len(nn1_row) > 0 else 0.0
        nn1_accuracies.append(acc_nn1)

        print(f"NN2 Accuracy at eps={eps:.3f}: {acc_nn2:.2f}%  |  NN1: {acc_nn1:.2f}%")

        results.append({
            'attack_name':   ATTACK_NAME,
            'epsilon':       eps,
            'nn1_accuracy':  round(acc_nn1, 2),
            'nn2_accuracy':  round(acc_nn2, 2),
            'nn2_correct':   correct_nn2,
            'total_images':  total_processed,
            'time_elapsed_s': round(elapsed, 2)
        })

    # ------------------------------------------------------------------------------
    # Comparative SEC — NN1 vs NN2
    # ------------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(9, 6))

    ax.plot(0.0, BASELINE_ACC_1000,
            marker='o', markerfacecolor='#1f77b4',
            markeredgecolor='#2ca02c', markeredgewidth=1,
            markersize=6, linestyle='None',
            label=f'NN1 Baseline ({BASELINE_ACC_1000:.2f}%)')
    ax.plot(0.0, BASELINE_ACC_NN2_1000,
            marker='o', markerfacecolor='#ff7f0e',
            markeredgecolor='#2ca02c', markeredgewidth=1,
            markersize=6, linestyle='None',
            label=f'NN2 Baseline ({BASELINE_ACC_NN2_1000:.2f}%)')

    ax.plot([0.0, EPSILONS[0]], [BASELINE_ACC_1000, nn1_accuracies[0]],
            linestyle='-', color='#1f77b4', linewidth=1.5)
    ax.plot(EPSILONS, nn1_accuracies,
            marker='o', linestyle='-', color='#1f77b4',
            linewidth=1.5, markersize=6, label=f'NN1 Under Attack')

    ax.plot([0.0, EPSILONS[0]], [BASELINE_ACC_NN2_1000, nn2_accuracies[0]],
            linestyle='-', color='#ff7f0e', linewidth=1.5)
    ax.plot(EPSILONS, nn2_accuracies,
            marker='o', linestyle='-', color='#ff7f0e',
            linewidth=1.5, markersize=6, label=f'NN2 Transferability')

    ax.set_title(f'Transferability SEC — {ATTACK_NAME} (NN1 vs NN2)', fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
    ax.set_ylabel('Model Accuracy (%)', fontsize=14)
    ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
    ax.set_xticks(np.arange(0.0, 0.11, 0.01))
    ax.set_yticks(np.arange(0, 101, 10))
    ax.tick_params(axis='both', labelsize=12)
    ax.set_ylim(-5, 105)
    ax.set_xlim(-0.005, 0.105)
    ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2, label=r'Max Allowable $\epsilon$ (0.1)')
    ax.legend(loc='upper right', fontsize=12)
    plt.tight_layout()

    plot_path = os.path.join(TRANSFER_RESULTS_DIR, f'{ATTACK_NAME}_transferability_SEC.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n[INFO] Transferability SEC saved at: {plot_path}")

    # ------------------------------------------------------------------------------
    # Save Results CSV
    # ------------------------------------------------------------------------------
    df_results = pd.DataFrame(results)
    csv_path   = os.path.join(TRANSFER_RESULTS_DIR, f'{ATTACK_NAME}_transferability_results.csv')
    df_results.to_csv(csv_path, index=False)
    print(f"\n[INFO] Results CSV saved at: {csv_path}")

    # ------------------------------------------------------------------------------
    # Summary
    # ------------------------------------------------------------------------------
    print("\n" + "=" * 55)
    print(f"FGSM TRANSFERABILITY SUMMARY")
    print("=" * 55)
    print(f"NN1 Baseline: {BASELINE_ACC_1000:.2f}%  |  NN2 Baseline: {BASELINE_ACC_NN2_1000:.2f}%")
    print(f"{'Epsilon':<10} {'NN1 Acc':>10} {'NN2 Acc':>10} {'Transfer Gap':>14}")
    print("-" * 55)
    for eps, a1, a2 in zip(EPSILONS, nn1_accuracies, nn2_accuracies):
        print(f"{eps:<10.3f} {a1:>10.2f}% {a2:>10.2f}% {(a1 - a2):>+13.2f}%")
    print("=" * 55 + "\n")

### Transferability — BIM (Error-Generic)

Evaluates NN2 accuracy on adversarial examples generated by BIM against NN1,
across all combinations of epsilon and max_iter. A comparative SEC is generated
for each max_iter value, overlaying NN1 and NN2 accuracy curves.
A summary table reports the transferability gap for each configuration.

In [59]:
if RUN_BIM_TRANSFERABILITY_GENERIC:
    # ==============================================================================
    # STEP 5.2: Transferability — BIM (Error-Generic)
    # ==============================================================================

    print("Evaluating BIM transferability on NN2...")

    ATTACK_NAME    = "BIM"
    EPSILONS       = [0.001, 0.003, 0.005, 0.01, 0.03, 0.05, 0.1]
    MAX_ITER_LIST  = [1, 5, 10, 20]

    ITER_COLORS_NN1 = {1: '#1f77b4', 5: '#ff7f0e', 10: '#2ca02c', 20: '#d62728'}
    ITER_COLORS_NN2 = {1: '#aec7e8', 5: '#ffbb78', 10: '#98df8a', 20: '#ff9896'}

    TRANSFER_RESULTS_DIR = os.path.join(RESULTS_DIR, f"{ATTACK_NAME}_Transferability")
    os.makedirs(TRANSFER_RESULTS_DIR, exist_ok=True)

    nn1_results_path = os.path.join(RESULTS_DIR, ATTACK_NAME, f"{ATTACK_NAME}_results.csv")
    df_nn1           = pd.read_csv(nn1_results_path)

    results            = []
    all_nn1_accuracies = {}
    all_nn2_accuracies = {}

    for max_iter in MAX_ITER_LIST:
        folder_name  = f"{ATTACK_NAME}_iter{max_iter}"
        adv_iter_dir = os.path.join(ADVERSARIAL_DATA_DIR, folder_name)

        if not os.path.exists(adv_iter_dir):
            print(f"[WARNING] Directory not found: {adv_iter_dir}. Skipping.")
            continue

        print(f"\n{'='*50}")
        print(f"MAX_ITER = {max_iter}")
        print(f"{'='*50}")

        nn1_accs = []
        nn2_accs = []

        for eps in EPSILONS:
            eps_folder  = f"eps_{eps:.3f}".replace('.', '_')
            adv_eps_dir = os.path.join(adv_iter_dir, eps_folder)

            if not os.path.exists(adv_eps_dir):
                print(f"[WARNING] Directory not found: {adv_eps_dir}. Skipping.")
                continue

            print(f"\n--- Epsilon: {eps:.3f} | max_iter: {max_iter} ---")

            correct_nn2     = 0
            total_processed = 0
            start_time      = time.time()

            for class_id in os.listdir(adv_eps_dir):
                class_folder = os.path.join(adv_eps_dir, class_id)
                if not os.path.isdir(class_folder):
                    continue

                true_name  = id_to_name.get(class_id.strip(), "Unknown")
                true_clean = normalize_label(true_name)

                for img_name in os.listdir(class_folder):
                    if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                        continue

                    img_path = os.path.join(class_folder, img_name)
                    try:
                        img        = Image.open(img_path).convert('RGB')
                        tensor_nn2 = preprocess_nn2(img).unsqueeze(0).to(DEVICE)

                        with torch.no_grad():
                            pred_idx_2 = nn2(tensor_nn2)[0].detach().cpu().numpy().argmax()

                        if clean_pred_label(pred_idx_2) == true_clean:
                            correct_nn2 += 1

                        total_processed += 1

                    except Exception as e:
                        print(f"Error processing {img_path}: {e}")

            elapsed = time.time() - start_time
            acc_nn2 = (correct_nn2 / total_processed) * 100 if total_processed > 0 else 0.0

            nn1_row = df_nn1[(df_nn1['epsilon'] == eps) & (df_nn1['max_iter'] == max_iter)]
            acc_nn1 = nn1_row['accuracy'].values[0] if len(nn1_row) > 0 else 0.0

            nn1_accs.append(acc_nn1)
            nn2_accs.append(acc_nn2)

            print(f"NN2 Accuracy: {acc_nn2:.2f}%  |  NN1: {acc_nn1:.2f}%")

            results.append({
                'attack_name':    ATTACK_NAME,
                'max_iter':       max_iter,
                'epsilon':        eps,
                'nn1_accuracy':   round(acc_nn1, 2),
                'nn2_accuracy':   round(acc_nn2, 2),
                'nn2_correct':    correct_nn2,
                'total_images':   total_processed,
                'time_elapsed_s': round(elapsed, 2)
            })

        all_nn1_accuracies[max_iter] = nn1_accs
        all_nn2_accuracies[max_iter] = nn2_accs

        # --------------------------------------------------------------------------
        # Individual Comparative SEC for this max_iter
        # --------------------------------------------------------------------------
        c_nn1 = ITER_COLORS_NN1[max_iter]
        c_nn2 = ITER_COLORS_NN2[max_iter]

        fig, ax = plt.subplots(figsize=(9, 6))

        ax.plot(0.0, BASELINE_ACC_1000,
                marker='o', markerfacecolor=c_nn1,
                markeredgecolor='#2ca02c', markeredgewidth=1,
                markersize=6, linestyle='None',
                label=f'NN1 Baseline ({BASELINE_ACC_1000:.2f}%)')
        ax.plot(0.0, BASELINE_ACC_NN2_1000,
                marker='o', markerfacecolor=c_nn2,
                markeredgecolor='#2ca02c', markeredgewidth=1,
                markersize=6, linestyle='None',
                label=f'NN2 Baseline ({BASELINE_ACC_NN2_1000:.2f}%)')

        ax.plot([0.0, EPSILONS[0]], [BASELINE_ACC_1000, nn1_accs[0]],
                linestyle='-', color=c_nn1, linewidth=1.5)
        ax.plot(EPSILONS, nn1_accs,
                marker='o', linestyle='-', color=c_nn1,
                linewidth=1.5, markersize=6,
                label=f'NN1 Under Attack (max_iter={max_iter})')

        ax.plot([0.0, EPSILONS[0]], [BASELINE_ACC_NN2_1000, nn2_accs[0]],
                linestyle='--', color=c_nn2, linewidth=1.5)
        ax.plot(EPSILONS, nn2_accs,
                marker='s', linestyle='--', color=c_nn2,
                linewidth=1.5, markersize=6,
                label=f'NN2 Transferability (max_iter={max_iter})')

        ax.set_title(f'Transferability SEC — {ATTACK_NAME} (max_iter={max_iter})', fontsize=16, fontweight='bold', pad=15)
        ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
        ax.set_ylabel('Model Accuracy (%)', fontsize=14)
        ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
        ax.set_xticks(np.arange(0.0, 0.11, 0.01))
        ax.set_yticks(np.arange(0, 101, 10))
        ax.tick_params(axis='both', labelsize=12)
        ax.set_ylim(-5, 105)
        ax.set_xlim(-0.005, 0.105)
        ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2, label=r'Max Allowable $\epsilon$ (0.1)')
        ax.legend(loc='upper right', fontsize=12)
        plt.tight_layout()

        plot_path = os.path.join(TRANSFER_RESULTS_DIR, f'{ATTACK_NAME}_iter{max_iter}_transferability_SEC.png')
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\n[INFO] Transferability SEC saved at: {plot_path}")

    # ------------------------------------------------------------------------------
    # Comparative overview — all max_iter, NN1 vs NN2
    # ------------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(9, 6))

    ax.plot(0.0, BASELINE_ACC_1000,
            marker='o', color='black', markersize=6,
            linestyle='None', label=f'NN1 Baseline ({BASELINE_ACC_1000:.2f}%)')
    ax.plot(0.0, BASELINE_ACC_NN2_1000,
            marker='s', color='gray', markersize=6,
            linestyle='None', label=f'NN2 Baseline ({BASELINE_ACC_NN2_1000:.2f}%)')

    for max_iter in MAX_ITER_LIST:
        if max_iter not in all_nn1_accuracies:
            continue
        c_nn1 = ITER_COLORS_NN1[max_iter]
        c_nn2 = ITER_COLORS_NN2[max_iter]

        ax.plot([0.0, EPSILONS[0]],
                [BASELINE_ACC_1000, all_nn1_accuracies[max_iter][0]],
                linestyle='-', color=c_nn1, linewidth=1.5)
        ax.plot(EPSILONS, all_nn1_accuracies[max_iter],
                marker='o', linestyle='-', color=c_nn1,
                linewidth=1.5, markersize=5,
                label=f'NN1 iter={max_iter}')

        ax.plot([0.0, EPSILONS[0]],
                [BASELINE_ACC_NN2_1000, all_nn2_accuracies[max_iter][0]],
                linestyle='--', color=c_nn2, linewidth=1.5)
        ax.plot(EPSILONS, all_nn2_accuracies[max_iter],
                marker='s', linestyle='--', color=c_nn2,
                linewidth=1.5, markersize=5,
                label=f'NN2 iter={max_iter}')

    ax.set_title(f'Transferability SEC — {ATTACK_NAME} (All max_iter, Comparative)', fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
    ax.set_ylabel('Model Accuracy (%)', fontsize=14)
    ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
    ax.set_xticks(np.arange(0.0, 0.11, 0.01))
    ax.set_yticks(np.arange(0, 101, 10))
    ax.tick_params(axis='both', labelsize=12)
    ax.set_ylim(-5, 105)
    ax.set_xlim(-0.005, 0.105)
    ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2, label=r'Max Allowable $\epsilon$ (0.1)')
    ax.legend(loc='upper right', fontsize=9)
    plt.tight_layout()

    comparative_path = os.path.join(TRANSFER_RESULTS_DIR, f'{ATTACK_NAME}_transferability_comparative_SEC.png')
    plt.savefig(comparative_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n[INFO] Comparative transferability SEC saved at: {comparative_path}")

    # ------------------------------------------------------------------------------
    # Save Results CSV
    # ------------------------------------------------------------------------------
    df_results = pd.DataFrame(results)
    csv_path   = os.path.join(TRANSFER_RESULTS_DIR, f'{ATTACK_NAME}_transferability_results.csv')
    df_results.to_csv(csv_path, index=False)
    print(f"\n[INFO] Results CSV saved at: {csv_path}")

    # ------------------------------------------------------------------------------
    # Summary
    # ------------------------------------------------------------------------------
    print("\n" + "=" * 60)
    print(f"BIM TRANSFERABILITY SUMMARY")
    print("=" * 60)
    print(f"NN1 Baseline: {BASELINE_ACC_1000:.2f}%  |  NN2 Baseline: {BASELINE_ACC_NN2_1000:.2f}%")
    for max_iter in MAX_ITER_LIST:
        if max_iter not in all_nn1_accuracies:
            continue
        a1 = all_nn1_accuracies[max_iter][-1]
        a2 = all_nn2_accuracies[max_iter][-1]
        print(f"  max_iter={max_iter:2d} @ eps=0.1 — NN1: {a1:.2f}%  NN2: {a2:.2f}%  " f"Gap: {(a1-a2):+.2f}%")
    print("=" * 60 + "\n")

### Transferability — PGD (Error-Generic)

Evaluates NN2 accuracy on adversarial examples generated by PGD against NN1,
across all combinations of epsilon, max_iter, and num_random_init.
Individual comparative SECs are generated for each (max_iter, num_random_init)
combination. A summary table reports the transferability gap for the strongest
configuration (max max_iter, max num_random_init).

In [60]:
if RUN_PGD_TRANSFERABILITY_GENERIC:
    # ==============================================================================
    # STEP 5.3: Transferability — PGD (Error-Generic)
    # ==============================================================================

    print("Evaluating PGD transferability on NN2...")

    ATTACK_NAME      = "PGD"
    EPSILONS         = [0.001, 0.003, 0.005, 0.01, 0.03, 0.05, 0.1]
    MAX_ITER_LIST    = [1, 5, 10, 20]
    NUM_RANDOM_INITS = [0, 1, 3, 5]

    ITER_COLORS_NN1 = {
        1: '#1f77b4', 5: '#ffbf00', 10: '#2ca02c', 20: '#d62728', 40: '#9467bd'
    }
    ITER_COLORS_NN2 = {
        1: '#aec7e8', 5: '#ffe599', 10: '#98df8a', 20: '#ff9896', 40: '#c5b0d5'
    }

    TRANSFER_RESULTS_DIR = os.path.join(RESULTS_DIR, f"{ATTACK_NAME}_Transferability")
    os.makedirs(TRANSFER_RESULTS_DIR, exist_ok=True)

    nn1_results_path = os.path.join(RESULTS_DIR, ATTACK_NAME, f"{ATTACK_NAME}_results.csv")
    df_nn1           = pd.read_csv(nn1_results_path)

    results            = []
    all_nn1_accuracies = {nri: {} for nri in NUM_RANDOM_INITS}
    all_nn2_accuracies = {nri: {} for nri in NUM_RANDOM_INITS}

    for num_random_init in NUM_RANDOM_INITS:
        for max_iter in MAX_ITER_LIST:
            folder_name  = f"{ATTACK_NAME}_nri{num_random_init}_iter{max_iter}"
            adv_iter_dir = os.path.join(ADVERSARIAL_DATA_DIR, folder_name)

            if not os.path.exists(adv_iter_dir):
                print(f"[WARNING] Directory not found: {adv_iter_dir}. Skipping.")
                continue

            print(f"\n{'='*50}")
            print(f"NUM_RANDOM_INIT = {num_random_init} | MAX_ITER = {max_iter}")
            print(f"{'='*50}")

            nn1_accs = []
            nn2_accs = []

            for eps in EPSILONS:
                eps_folder  = f"eps_{eps:.3f}".replace('.', '_')
                adv_eps_dir = os.path.join(adv_iter_dir, eps_folder)

                if not os.path.exists(adv_eps_dir):
                    print(f"[WARNING] Directory not found: {adv_eps_dir}. Skipping.")
                    continue

                correct_nn2     = 0
                total_processed = 0
                start_time      = time.time()

                for class_id in os.listdir(adv_eps_dir):
                    class_folder = os.path.join(adv_eps_dir, class_id)
                    if not os.path.isdir(class_folder):
                        continue

                    true_name  = id_to_name.get(class_id.strip(), "Unknown")
                    true_clean = normalize_label(true_name)

                    for img_name in os.listdir(class_folder):
                        if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                            continue

                        img_path = os.path.join(class_folder, img_name)
                        try:
                            img        = Image.open(img_path).convert('RGB')
                            tensor_nn2 = preprocess_nn2(img).unsqueeze(0).to(DEVICE)

                            with torch.no_grad():
                                pred_idx_2 = nn2(tensor_nn2)[0].detach().cpu().numpy().argmax()

                            if clean_pred_label(pred_idx_2) == true_clean:
                                correct_nn2 += 1

                            total_processed += 1

                        except Exception as e:
                            print(f"Error processing {img_path}: {e}")

                elapsed = time.time() - start_time
                acc_nn2 = (correct_nn2 / total_processed) * 100 if total_processed > 0 else 0.0

                nn1_row = df_nn1[
                    (df_nn1['epsilon'] == eps) &
                    (df_nn1['max_iter'] == max_iter) &
                    (df_nn1['num_random_init'] == num_random_init)
                ]
                acc_nn1 = nn1_row['accuracy'].values[0] if len(nn1_row) > 0 else 0.0

                nn1_accs.append(acc_nn1)
                nn2_accs.append(acc_nn2)

                print(f"eps={eps:.3f} | NN1: {acc_nn1:.2f}%  NN2: {acc_nn2:.2f}%")

                results.append({
                    'attack_name':      ATTACK_NAME,
                    'num_random_init':  num_random_init,
                    'max_iter':         max_iter,
                    'epsilon':          eps,
                    'nn1_accuracy':     round(acc_nn1, 2),
                    'nn2_accuracy':     round(acc_nn2, 2),
                    'nn2_correct':      correct_nn2,
                    'total_images':     total_processed,
                    'time_elapsed_s':   round(elapsed, 2)
                })

            all_nn1_accuracies[num_random_init][max_iter] = nn1_accs
            all_nn2_accuracies[num_random_init][max_iter] = nn2_accs

            # ----------------------------------------------------------------------
            # Individual Comparative SEC for this (nri, max_iter) combination
            # ----------------------------------------------------------------------
            c_nn1 = ITER_COLORS_NN1[max_iter]
            c_nn2 = ITER_COLORS_NN2[max_iter]

            fig, ax = plt.subplots(figsize=(9, 6))

            ax.plot(0.0, BASELINE_ACC_1000,
                    marker='o', markerfacecolor=c_nn1,
                    markeredgecolor='#2ca02c', markeredgewidth=1,
                    markersize=6, linestyle='None',
                    label=f'NN1 Baseline ({BASELINE_ACC_1000:.2f}%)')
            ax.plot(0.0, BASELINE_ACC_NN2_1000,
                    marker='o', markerfacecolor=c_nn2,
                    markeredgecolor='#2ca02c', markeredgewidth=1,
                    markersize=6, linestyle='None',
                    label=f'NN2 Baseline ({BASELINE_ACC_NN2_1000:.2f}%)')

            ax.plot([0.0, EPSILONS[0]], [BASELINE_ACC_1000, nn1_accs[0]],
                    linestyle='-', color=c_nn1, linewidth=1.5)
            ax.plot(EPSILONS, nn1_accs,
                    marker='o', linestyle='-', color=c_nn1,
                    linewidth=1.5, markersize=6,
                    label=f'NN1 (nri={num_random_init}, iter={max_iter})')

            ax.plot([0.0, EPSILONS[0]], [BASELINE_ACC_NN2_1000, nn2_accs[0]],
                    linestyle='--', color=c_nn2, linewidth=1.5)
            ax.plot(EPSILONS, nn2_accs,
                    marker='s', linestyle='--', color=c_nn2,
                    linewidth=1.5, markersize=6,
                    label=f'NN2 (nri={num_random_init}, iter={max_iter})')

            ax.set_title(f'Transferability SEC — {ATTACK_NAME}\n'
                        f'(nri={num_random_init}, max_iter={max_iter})',
                        fontsize=16, fontweight='bold', pad=15)
            ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
            ax.set_ylabel('Model Accuracy (%)', fontsize=14)
            ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
            ax.set_xticks(np.arange(0.0, 0.11, 0.01))
            ax.set_yticks(np.arange(0, 101, 10))
            ax.tick_params(axis='both', labelsize=12)
            ax.set_ylim(-5, 105)
            ax.set_xlim(-0.005, 0.105)
            ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2, label=r'Max Allowable $\epsilon$ (0.1)')
            ax.legend(loc='upper right', fontsize=11)
            plt.tight_layout()

            plot_path = os.path.join(
                TRANSFER_RESULTS_DIR,
                f'{ATTACK_NAME}_nri{num_random_init}_iter{max_iter}_transferability_SEC.png'
            )
            plt.savefig(plot_path, dpi=300, bbox_inches='tight')
            plt.show()
            print(f"\n[INFO] SEC saved at: {plot_path}")

    # ------------------------------------------------------------------------------
    # Save Results CSV
    # ------------------------------------------------------------------------------
    df_results = pd.DataFrame(results)
    csv_path   = os.path.join(TRANSFER_RESULTS_DIR, f'{ATTACK_NAME}_transferability_results.csv')
    df_results.to_csv(csv_path, index=False)
    print(f"\n[INFO] Results CSV saved at: {csv_path}")

    # ------------------------------------------------------------------------------
    # Summary — strongest configuration only
    # ------------------------------------------------------------------------------
    best_nri  = max(NUM_RANDOM_INITS)
    best_iter = max(MAX_ITER_LIST)

    print("\n" + "=" * 60)
    print(f"PGD TRANSFERABILITY SUMMARY (Strongest Config: nri={best_nri}, iter={best_iter})")
    print("=" * 60)
    print(f"NN1 Baseline: {BASELINE_ACC_1000:.2f}%  |  NN2 Baseline: {BASELINE_ACC_NN2_1000:.2f}%")
    if best_nri in all_nn1_accuracies and best_iter in all_nn1_accuracies[best_nri]:
        a1 = all_nn1_accuracies[best_nri][best_iter][-1]
        a2 = all_nn2_accuracies[best_nri][best_iter][-1]
        print(f"  @ eps=0.1 — NN1: {a1:.2f}%  NN2: {a2:.2f}%  Gap: {(a1-a2):+.2f}%")
    print("=" * 60 + "\n")

### Transferability — Carlini & Wagner L-inf (Error-Generic)

Evaluates NN2 accuracy on adversarial examples generated by CW Untargeted against NN1,
across all combinations of max_iter, learning_rate, and confidence. Since CW does not
enforce a strict L-infinity budget but minimizes perturbation internally, the comparative
plot uses mean L-inf actually applied on the x-axis, consistent with the CW and DeepFool
evaluation approach. A hyperparameter effect plot for learning_rate and confidence is also
generated. Only images saved within the L-inf <= 0.1 constraint are evaluated.

In [61]:
if RUN_CW_TRANSFERABILITY_GENERIC:
    # ==============================================================================
    # Transferability — Carlini & Wagner L-inf (Error-Generic)
    # ==============================================================================

    print("Evaluating CW Untargeted transferability on NN2...")

    ATTACK_NAME        = "CW_Untargeted"
    MAX_ITER_LIST      = [1, 5, 10, 20]
    LEARNING_RATE_LIST = [0.01, 0.1]
    CONFIDENCE_LIST    = [0.0]
    EVAL_BATCH_SIZE    = 64

    ITER_COLORS = {
        1:  '#1f77b4',
        5:  '#ff7f0e',
        10: '#2ca02c',
        20: '#d62728'
    }
    TRANSFER_RESULTS_DIR = os.path.join(RESULTS_DIR, f"{ATTACK_NAME}_Transferability")
    os.makedirs(TRANSFER_RESULTS_DIR, exist_ok=True)

    nn1_results_path = os.path.join(RESULTS_DIR, ATTACK_NAME,
                                    f"{ATTACK_NAME}_results.csv")
    if not os.path.exists(nn1_results_path):
        raise FileNotFoundError(
            f"NN1 results CSV not found at {nn1_results_path}. "
            "Run Step 3.4 first."
        )
    df_nn1 = pd.read_csv(nn1_results_path)

    results        = []
    all_combo_data = []  # {max_iter, lr, conf, mean_linf, nn1_acc, nn2_acc}

    total_combinations = len(MAX_ITER_LIST) * len(LEARNING_RATE_LIST) * len(CONFIDENCE_LIST)
    current_combo      = 1

    for max_iter in MAX_ITER_LIST:
        for lr in LEARNING_RATE_LIST:
            for conf in CONFIDENCE_LIST:
                print(f"\n[{current_combo}/{total_combinations}] "
                    f"max_iter={max_iter} | lr={lr} | confidence={conf}")

                lr_str         = str(lr).replace('.', '_')
                conf_str       = str(conf).replace('.', '_')
                combo_dir_name = f"iter_{max_iter}_lr_{lr_str}_c_{conf_str}"
                adv_combo_dir  = os.path.join(
                    ADVERSARIAL_DATA_DIR, ATTACK_NAME, combo_dir_name
                )

                if not os.path.exists(adv_combo_dir):
                    print(f"  [WARNING] Directory not found: {adv_combo_dir}. Skipping.")
                    current_combo += 1
                    continue

                # ------------------------------------------------------------------
                # Load the reference image list from DATA_DIR_100 (same order
                # used during generation) to guarantee a fixed denominator of 100.
                # For each image:
                #   - If an adversarial file exists in adv_combo_dir: evaluate NN2
                #     on the adversarial image.
                #   - If the file is missing (attack exceeded budget during
                #     generation): evaluate NN2 on the original clean image.
                #     The attack failed, so the adversary presents the clean input.
                # ------------------------------------------------------------------
                reference_files = sorted([
                    f for f in os.listdir(DATA_DIR_100)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
                ])

                if not reference_files:
                    print(f"  [WARNING] No reference images found in DATA_DIR_100. Skipping.")
                    current_combo += 1
                    continue

                eval_imgs  = []
                eval_tids  = []

                for img_name in reference_files:
                    true_id  = os.path.splitext(img_name)[0].strip()
                    adv_path = os.path.join(adv_combo_dir, img_name)

                    if os.path.exists(adv_path):
                        # Adversarial image exists — attack stayed within budget.
                        src_path = adv_path
                    else:
                        # Adversarial image missing — attack exceeded budget.
                        # Evaluate on the original clean image instead.
                        src_path = os.path.join(DATA_DIR_100, img_name)

                    try:
                        img = Image.open(src_path).convert('RGB')
                        eval_imgs.append(preprocess_nn2(img))
                        eval_tids.append(true_id)
                    except Exception as e:
                        print(f"  Error loading {img_name}: {e}")

                # ------------------------------------------------------------------
                # Run inference in batches on NN2 — denominator is always 100.
                # ------------------------------------------------------------------
                correct_nn2 = 0
                start_time  = time.time()

                for b in range(0, len(eval_imgs), EVAL_BATCH_SIZE):
                    batch_tensor = torch.stack(
                        eval_imgs[b:b + EVAL_BATCH_SIZE]
                    ).to(DEVICE)
                    batch_tids = eval_tids[b:b + EVAL_BATCH_SIZE]

                    with torch.no_grad():
                        pred_indices = nn2(batch_tensor).detach().cpu().numpy().argmax(axis=1)

                    for pred_idx_2, true_id in zip(pred_indices, batch_tids):
                        true_name  = id_to_name.get(true_id, "Unknown")
                        true_clean = normalize_label(true_name)
                        pred_clean = clean_pred_label(pred_idx_2)

                        if pred_clean == true_clean:
                            correct_nn2 += 1

                elapsed = time.time() - start_time
                acc_nn2 = (correct_nn2 / len(reference_files)) * 100

                # Retrieve NN1 accuracy and mean_linf from generation CSV
                nn1_row   = df_nn1[
                    (df_nn1['max_iter']       == max_iter) &
                    (df_nn1['learning_rate']  == lr)       &
                    (df_nn1['confidence']     == conf)
                ]
                acc_nn1   = nn1_row['accuracy'].values[0]          if len(nn1_row) > 0 else 0.0
                mean_linf = nn1_row['mean_linf_applied'].values[0] if len(nn1_row) > 0 else 0.0

                adv_found = sum(
                    1 for f in reference_files
                    if os.path.exists(os.path.join(adv_combo_dir, f))
                )
                print(f"  Adversarial found: {adv_found}/100 | " f"Clean fallback: {len(reference_files) - adv_found}/100 | "
                    f"NN1: {acc_nn1:.2f}% | NN2: {acc_nn2:.2f}% | " f"Mean L-inf: {mean_linf:.4f} | Time: {elapsed:.1f}s")

                results.append({
                    'attack_name':    ATTACK_NAME,
                    'max_iter':       max_iter,
                    'learning_rate':  lr,
                    'confidence':     conf,
                    'nn1_accuracy':   round(acc_nn1, 2),
                    'nn2_accuracy':   round(acc_nn2, 2),
                    'nn2_correct':    correct_nn2,
                    'total_images':   len(reference_files),
                    'mean_linf':      round(mean_linf, 4),
                    'time_elapsed_s': round(elapsed, 2)
                })

                all_combo_data.append({
                    'max_iter':  max_iter,
                    'lr':        lr,
                    'conf':      conf,
                    'mean_linf': mean_linf,
                    'nn1_acc':   acc_nn1,
                    'nn2_acc':   acc_nn2
                })

                current_combo += 1

    # ------------------------------------------------------------------------------
    # Save Results CSV
    # ------------------------------------------------------------------------------
    df_results = pd.DataFrame(results)
    csv_path   = os.path.join(TRANSFER_RESULTS_DIR,
                            f'{ATTACK_NAME}_transferability_results.csv')
    df_results.to_csv(csv_path, index=False)
    print(f"\n[INFO] Results CSV saved at: {csv_path}")

    # ------------------------------------------------------------------------------
    # Comparative SEC — Mean L-inf vs Accuracy, one curve per max_iter
    # Each point is one (lr, confidence) combination.
    # Ceteris paribus: for each curve max_iter is fixed, lr and confidence vary.
    # NN1 and NN2 curves shown together to visualize transferability gap.
    # ------------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(9, 6))

    ax.axhline(y=BASELINE_ACC_100, color='#2ca02c', linestyle='--',
            linewidth=1.5, label=f'NN1 Baseline ({BASELINE_ACC_100:.2f}%)')
    ax.axhline(y=BASELINE_ACC_NN2_100, color='#8c8c8c', linestyle='--',
            linewidth=1.5, label=f'NN2 Baseline ({BASELINE_ACC_NN2_100:.2f}%)')
    ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2,
            label=r'Max Allowable L-inf (0.1)')

    for max_iter in MAX_ITER_LIST:
        combos = [c for c in all_combo_data if c['max_iter'] == max_iter]
        if not combos:
            continue

        combos_sorted = sorted(combos, key=lambda c: c['mean_linf'])
        lif_vals      = [c['mean_linf'] for c in combos_sorted]
        nn1_vals      = [c['nn1_acc']   for c in combos_sorted]
        nn2_vals      = [c['nn2_acc']   for c in combos_sorted]

        c_nn1 = ITER_COLORS[max_iter]
        # Lighter shade for NN2
        c_nn2 = {
            1:  '#aec7e8',
            5:  '#ffbb78',
            10: '#98df8a',
            20: "#d35a5a"
        }[max_iter]

        ax.plot(lif_vals, nn1_vals,
                marker='o', linestyle='-', color=c_nn1,
                linewidth=1.5, markersize=5,
                label=f'NN1 max_iter={max_iter}')
        ax.plot(lif_vals, nn2_vals,
                marker='s', linestyle='--', color=c_nn2,
                linewidth=1.5, markersize=5,
                label=f'NN2 max_iter={max_iter}')

    ax.set_title(f'Transferability SEC — {ATTACK_NAME}\n'
                f'(X-axis: mean perturbation applied, not fixed budget)',
                fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel(r'Mean L-inf Perturbation Actually Applied (not fixed $\epsilon$)',
                fontsize=14)
    ax.set_ylabel('Model Accuracy (%)', fontsize=14)
    ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
    ax.set_xticks(np.arange(0.0, 0.11, 0.01))
    ax.set_yticks(np.arange(0, 101, 10))
    ax.tick_params(axis='both', labelsize=12)
    ax.set_ylim(-5, 105)
    ax.set_xlim(-0.005, 0.105)
    ax.legend(loc='upper right', fontsize=9)
    plt.tight_layout()

    sec_path = os.path.join(TRANSFER_RESULTS_DIR,
                            f'{ATTACK_NAME}_transferability_SEC.png')
    plt.savefig(sec_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n[INFO] Transferability SEC saved at: {sec_path}")

    # ------------------------------------------------------------------------------
    # Hyperparameter Effect — NN1 vs NN2 accuracy vs learning_rate
    # confidence fixed at 0.0 — ceteris paribus.
    # ------------------------------------------------------------------------------
    FIXED_CONF_FOR_EFFECT = 0.0

    fig, ax = plt.subplots(figsize=(9, 6))

    ax.axhline(y=BASELINE_ACC_100, color='#2ca02c', linestyle='--',
            linewidth=1.5, label=f'NN1 Baseline ({BASELINE_ACC_100:.2f}%)')
    ax.axhline(y=BASELINE_ACC_NN2_100, color='#8c8c8c', linestyle='--',
            linewidth=1.5, label=f'NN2 Baseline ({BASELINE_ACC_NN2_100:.2f}%)')

    for max_iter in MAX_ITER_LIST:
        c_nn1 = ITER_COLORS[max_iter]
        c_nn2 = {1: '#aec7e8', 5: '#ffbb78', 10: '#98df8a', 20: '#d35a5a'}[max_iter]

        subset = df_results[
            (df_results['max_iter']      == max_iter) &
            (df_results['confidence']    == FIXED_CONF_FOR_EFFECT)
        ].sort_values('learning_rate')

        ax.plot(subset['learning_rate'].tolist(),
                subset['nn1_accuracy'].tolist(),
                marker='o', linestyle='-', color=c_nn1,
                linewidth=1.5, markersize=6,
                label=f'NN1 max_iter={max_iter}')
        ax.plot(subset['learning_rate'].tolist(),
                subset['nn2_accuracy'].tolist(),
                marker='s', linestyle='--', color=c_nn2,
                linewidth=1.5, markersize=6,
                label=f'NN2 max_iter={max_iter}')

    ax.set_title(f'Effect of learning_rate on Accuracy — {ATTACK_NAME} Transferability\n'
                f'(confidence={FIXED_CONF_FOR_EFFECT})',
                fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel('Learning Rate', fontsize=14)
    ax.set_ylabel('Model Accuracy (%)', fontsize=14)
    ax.set_xscale('log')
    ax.set_xticks(LEARNING_RATE_LIST)
    ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
    ax.set_yticks(np.arange(0, 101, 10))
    ax.tick_params(axis='both', labelsize=12)
    ax.set_ylim(-5, 105)
    ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
    ax.legend(loc='upper right', fontsize=9)
    plt.tight_layout()

    lr_path = os.path.join(TRANSFER_RESULTS_DIR,
                        f'{ATTACK_NAME}_transferability_lr_effect.png')
    plt.savefig(lr_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n[INFO] Learning rate effect plot saved at: {lr_path}")

    # ------------------------------------------------------------------------------
    # Hyperparameter Effect — NN1 vs NN2 accuracy vs confidence
    # learning_rate fixed at 0.01 — ceteris paribus.
    # ------------------------------------------------------------------------------
    FIXED_LR_FOR_EFFECT = 0.01

    fig, ax = plt.subplots(figsize=(9, 6))

    ax.axhline(y=BASELINE_ACC_100, color='#2ca02c', linestyle='--',
            linewidth=1.5, label=f'NN1 Baseline ({BASELINE_ACC_100:.2f}%)')
    ax.axhline(y=BASELINE_ACC_NN2_100, color='#8c8c8c', linestyle='--',
            linewidth=1.5, label=f'NN2 Baseline ({BASELINE_ACC_NN2_100:.2f}%)')

    for max_iter in MAX_ITER_LIST:
        c_nn1 = ITER_COLORS[max_iter]
        c_nn2 = {1: '#aec7e8', 5: '#ffbb78', 10: '#98df8a', 20: '#d35a5a'}[max_iter]

        subset = df_results[
            (df_results['max_iter']      == max_iter) &
            (df_results['learning_rate'] == FIXED_LR_FOR_EFFECT)
        ].sort_values('confidence')

        ax.plot(subset['confidence'].tolist(),
                subset['nn1_accuracy'].tolist(),
                marker='o', linestyle='-', color=c_nn1,
                linewidth=1.5, markersize=6,
                label=f'NN1 max_iter={max_iter}')
        ax.plot(subset['confidence'].tolist(),
                subset['nn2_accuracy'].tolist(),
                marker='s', linestyle='--', color=c_nn2,
                linewidth=1.5, markersize=6,
                label=f'NN2 max_iter={max_iter}')

    ax.set_title(f'Effect of confidence on Accuracy — {ATTACK_NAME} Transferability\n'
                f'(learning_rate={FIXED_LR_FOR_EFFECT})',
                fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel('Confidence', fontsize=14)
    ax.set_ylabel('Model Accuracy (%)', fontsize=14)
    ax.set_xticks(CONFIDENCE_LIST)
    ax.set_yticks(np.arange(0, 101, 10))
    ax.tick_params(axis='both', labelsize=12)
    ax.set_ylim(-5, 105)
    ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
    ax.legend(loc='upper right', fontsize=9)
    plt.tight_layout()

    conf_path = os.path.join(TRANSFER_RESULTS_DIR,
                            f'{ATTACK_NAME}_transferability_conf_effect.png')
    plt.savefig(conf_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n[INFO] Confidence effect plot saved at: {conf_path}")

    # ------------------------------------------------------------------------------
    # Summary
    # ------------------------------------------------------------------------------
    print("\n" + "=" * 65)
    print(f"CW UNTARGETED TRANSFERABILITY SUMMARY")
    print("=" * 65)
    print(f"NN1 Baseline: {BASELINE_ACC_100:.2f}%  |  "
        f"NN2 Baseline: {BASELINE_ACC_NN2_100:.2f}%")
    print(f"{'max_iter':<10} {'lr':<8} {'conf':<8} "
        f"{'NN1 Acc':>10} {'NN2 Acc':>10} {'Gap':>10} {'L-inf':>10}")
    print("-" * 65)
    for r in results:
        print(f"  {r['max_iter']:<8} {r['learning_rate']:<8} "
            f"{r['confidence']:<8} "
            f"{r['nn1_accuracy']:>10.2f}% "
            f"{r['nn2_accuracy']:>10.2f}% "
            f"{(r['nn1_accuracy']-r['nn2_accuracy']):>+9.2f}% "
            f"{r['mean_linf']:>10.4f}")
    print("=" * 65 + "\n")

### Transferability — DeepFool (Error-Generic)

Evaluates NN2 accuracy on adversarial examples generated by DeepFool against NN1,
across all max_iter values. Since DeepFool does not have a fixed epsilon budget,
the comparative plot uses max_iter on the x-axis rather than epsilon.
A summary table reports the transferability gap for each configuration.

In [62]:
if RUN_DEEPFOOL_TRANSFERABILITY_GENERIC:
    # ==============================================================================
    # STEP Transferability — DeepFool (Error-Generic)
    # ==============================================================================

    print("Evaluating DeepFool transferability on NN2...")

    ATTACK_NAME     = "DeepFool"
    EPSILON         = 1e-6   # Must match the value used during generation.
    MAX_ITER_LIST   = [1, 5, 10, 20, 50, 100, 175]
    EVAL_BATCH_SIZE = 64

    ITER_COLORS_NN1 = {
        1:   '#1f77b4',
        5:   '#ff7f0e',
        10:  '#2ca02c',
        20:  '#f2ff00',
        50:  '#d62728',
        100: '#9467bd',
        175: '#4b8c6c'
    }
    ITER_COLORS_NN2 = {
        1:   '#aec7e8',
        5:   '#ffbb78',
        10:  '#98df8a',
        20:  '#ffffa0',
        50:  '#ff9896',
        100: '#c5b0d5',
        175: '#a8d4bf'
    }

    TRANSFER_RESULTS_DIR = os.path.join(RESULTS_DIR, f"{ATTACK_NAME}_Transferability")
    os.makedirs(TRANSFER_RESULTS_DIR, exist_ok=True)

    nn1_results_path = os.path.join(RESULTS_DIR, ATTACK_NAME, f"{ATTACK_NAME}_results.csv")
    df_nn1           = pd.read_csv(nn1_results_path)

    results  = []
    all_data = {m: [] for m in MAX_ITER_LIST}

    # Load reference file list once — same order used during generation.
    reference_files = sorted([
        f for f in os.listdir(DATA_DIR_100)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])
    if not reference_files:
        raise FileNotFoundError(
            f"No reference images found in DATA_DIR_100: {DATA_DIR_100}"
        )

    for max_iter in MAX_ITER_LIST:
        print(f"\n{'#'*50}")
        print(f"MAX_ITER = {max_iter}")
        print(f"{'#'*50}")

        adv_combo_dir = os.path.join(
            ADVERSARIAL_DATA_DIR, ATTACK_NAME,
            f"iter_{max_iter}_eps_{str(EPSILON).replace('.', '_')}"
        )

        if not os.path.exists(adv_combo_dir):
            print(f"  [WARNING] Directory not found: {adv_combo_dir}. Skipping.")
            continue

        print(f"  max_iter={max_iter} | epsilon={EPSILON}")

        # ------------------------------------------------------------------
        # Build eval list from reference_files — fixed denominator of 100.
        #
        # For each reference image:
        #   - If the adversarial file exists in adv_combo_dir: evaluate NN2
        #     on the adversarial image.
        #   - If the file is missing (attack exceeded budget during
        #     generation): evaluate NN2 on the original clean image instead.
        #     The attack failed; the model sees the unperturbed input.
        # ------------------------------------------------------------------
        eval_imgs  = []
        eval_tids  = []

        for img_name in reference_files:
            true_id  = os.path.splitext(img_name)[0].strip()
            adv_path = os.path.join(adv_combo_dir, img_name)

            if os.path.exists(adv_path):
                src_path = adv_path
            else:
                src_path = os.path.join(DATA_DIR_100, img_name)

            try:
                img = Image.open(src_path).convert('RGB')
                eval_imgs.append(preprocess_nn2(img))
                eval_tids.append(true_id)
            except Exception as e:
                print(f"  Error loading {img_name}: {e}")

        if not eval_imgs:
            print(f"  [WARNING] No images to evaluate. Skipping.")
            continue

        # Run inference in batches on NN2 — denominator is always 100.
        correct_nn2 = 0
        start_time  = time.time()

        for b in range(0, len(eval_imgs), EVAL_BATCH_SIZE):
            batch_tensor = torch.stack(
                eval_imgs[b:b + EVAL_BATCH_SIZE]
            ).to(DEVICE)
            batch_tids_b = eval_tids[b:b + EVAL_BATCH_SIZE]

            with torch.no_grad():
                pred_indices = nn2(batch_tensor).detach().cpu().numpy().argmax(axis=1)

            for pred_idx_2, true_id in zip(pred_indices, batch_tids_b):
                true_name  = id_to_name.get(true_id, "Unknown")
                true_clean = normalize_label(true_name)
                pred_clean = clean_pred_label(pred_idx_2)

                if pred_clean == true_clean:
                    correct_nn2 += 1

        elapsed = time.time() - start_time
        acc_nn2 = (correct_nn2 / len(reference_files)) * 100

        # Retrieve NN1 accuracy and mean_linf from generation CSV.
        nn1_row   = df_nn1[df_nn1['max_iter'] == max_iter]
        acc_nn1   = nn1_row['accuracy'].values[0]          if len(nn1_row) > 0 else 0.0
        mean_linf = nn1_row['mean_linf_applied'].values[0] if len(nn1_row) > 0 else 0.0

        adv_found = sum(
            1 for f in reference_files
            if os.path.exists(os.path.join(adv_combo_dir, f))
        )
        print(f"  Adversarial found: {adv_found}/100 | " f"Clean fallback: {len(reference_files) - adv_found}/100 | "
            f"NN1: {acc_nn1:.2f}% | NN2: {acc_nn2:.2f}% | " f"Mean L-inf: {mean_linf:.4f} | Time: {elapsed:.1f}s")

        results.append({
            'attack_name':    ATTACK_NAME,
            'max_iter':       max_iter,
            'epsilon':        EPSILON,
            'nn1_accuracy':   round(acc_nn1, 2),
            'nn2_accuracy':   round(acc_nn2, 2),
            'nn2_correct':    correct_nn2,
            'total_images':   len(reference_files),
            'mean_linf':      round(mean_linf, 4),
            'time_elapsed_s': round(elapsed, 2)
        })

        all_data[max_iter].append({
            'nn1_acc':   acc_nn1,
            'nn2_acc':   acc_nn2,
            'mean_linf': mean_linf
        })
    # ------------------------------------------------------------------------------
    # Bar chart — NN1 vs NN2 accuracy per max_iter
    # Each pair of bars shows the transferability gap for one max_iter value.
    # ------------------------------------------------------------------------------
    iter_labels = [str(m) for m in MAX_ITER_LIST]
    nn1_vals    = [all_data[m][0]['nn1_acc'] if all_data[m] else 0.0 for m in MAX_ITER_LIST]
    nn2_vals    = [all_data[m][0]['nn2_acc'] if all_data[m] else 0.0 for m in MAX_ITER_LIST]

    x_pos = np.arange(len(MAX_ITER_LIST))
    width = 0.35

    fig, ax = plt.subplots(figsize=(11, 6))

    bars_nn1 = ax.bar(x_pos - width / 2, nn1_vals, width, color=[ITER_COLORS_NN1[m] for m in MAX_ITER_LIST], alpha=0.85, label='NN1 Accuracy')
    bars_nn2 = ax.bar(x_pos + width / 2, nn2_vals, width, color=[ITER_COLORS_NN2[m] for m in MAX_ITER_LIST], alpha=0.85, label='NN2 Transferability')

    ax.axhline(y=BASELINE_ACC_100, color='#2ca02c', linestyle='--', linewidth=1.5, label=f'NN1 Baseline ({BASELINE_ACC_100:.2f}%)')
    ax.axhline(y=BASELINE_ACC_NN2_100, color='#8c8c8c', linestyle='--',linewidth=1.5, label=f'NN2 Baseline ({BASELINE_ACC_NN2_100:.2f}%)')

    ax.set_title(
        f'DeepFool Transferability — NN1 vs NN2 Accuracy per max_iter\n'
        f'(epsilon={EPSILON}, robust accuracy with fixed denominator)',
        fontsize=16, fontweight='bold', pad=15
    )
    ax.set_xlabel('max_iter', fontsize=14)
    ax.set_ylabel('Model Accuracy (%)', fontsize=14)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(iter_labels, fontsize=12)
    ax.set_yticks(np.arange(0, 101, 10))
    ax.set_ylim(-2, 105)
    ax.tick_params(axis='both', labelsize=12)
    ax.grid(True, axis='y', linestyle='-', linewidth=0.5, alpha=0.7)
    ax.legend(loc='upper right', fontsize=11)
    plt.tight_layout()

    bar_path = os.path.join(
        TRANSFER_RESULTS_DIR,
        f'{ATTACK_NAME}_transferability_bar_nn1_vs_nn2.png'
    )
    plt.savefig(bar_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n[INFO] Bar chart saved at: {bar_path}")
    
    # ------------------------------------------------------------------------------
    # Aggregated plot — Mean L-inf vs Accuracy for NN1 and NN2, all max_iter
    # Shows the perturbation-accuracy tradeoff and transferability gap together
    # ------------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(9, 6))

    ax.axhline(y=BASELINE_ACC_100, color='#2ca02c', linestyle='--',
            linewidth=1.5, label=f'NN1 Baseline ({BASELINE_ACC_100:.2f}%)')
    ax.axhline(y=BASELINE_ACC_NN2_100, color='#8c8c8c', linestyle='--',
            linewidth=1.5, label=f'NN2 Baseline ({BASELINE_ACC_NN2_100:.2f}%)')
    ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2,
            label=r'Max Allowable L-inf (0.1)')

    for max_iter in MAX_ITER_LIST:
        combos = all_data[max_iter]
        if not combos:
            continue

        d     = all_data[max_iter][0]
        c_nn1 = ITER_COLORS_NN1[max_iter]
        c_nn2 = ITER_COLORS_NN2[max_iter]
        ax.scatter(d['mean_linf'], d['nn1_acc'], color=c_nn1, s=80, marker='o', zorder=5, label=f'NN1 iter={max_iter}')
        ax.scatter(d['mean_linf'], d['nn2_acc'], color=c_nn2, s=80, marker='s', zorder=5, label=f'NN2 iter={max_iter}')
        ax.annotate(f"iter={max_iter}", xy=(d['mean_linf'], d['nn1_acc']), xytext=(6, 4), textcoords='offset points',fontsize=9, color=c_nn1)

    ax.set_title(f'DeepFool Transferability — Mean L-inf vs Accuracy\n'f'(Aggregated, all configurations)',fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel(r'Mean L-inf Perturbation Actually Applied', fontsize=14)
    ax.set_ylabel('Model Accuracy (%)', fontsize=14)
    ax.set_xticks(np.arange(0.0, 0.11, 0.01))
    ax.set_yticks(np.arange(0, 101, 10))
    ax.tick_params(axis='both', labelsize=12)
    ax.set_ylim(-5, 105)
    ax.set_xlim(-0.005, 0.105)
    ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
    ax.legend(loc='upper right', fontsize=9)
    plt.tight_layout()

    agg_path = os.path.join(
        TRANSFER_RESULTS_DIR,
        f'{ATTACK_NAME}_transferability_aggregated.png'
    )
    plt.savefig(agg_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n[INFO] Aggregated transferability plot saved at: {agg_path}")

    # ------------------------------------------------------------------------------
    # Save Results CSV
    # ------------------------------------------------------------------------------
    df_results = pd.DataFrame(results)
    csv_path   = os.path.join(TRANSFER_RESULTS_DIR,
                            f'{ATTACK_NAME}_transferability_results.csv')
    df_results.to_csv(csv_path, index=False)
    print(f"\n[INFO] Results CSV saved at: {csv_path}")

    # ------------------------------------------------------------------------------
    # Summary
    # ------------------------------------------------------------------------------
    print("\n" + "=" * 65)
    print(f"DEEPFOOL TRANSFERABILITY SUMMARY")
    print("=" * 65)
    print(f"NN1 Baseline: {BASELINE_ACC_100:.2f}%  |  "
        f"NN2 Baseline: {BASELINE_ACC_NN2_100:.2f}%")
    print(f"{'max_iter':<10} {'epsilon':<10} {'NN1 Acc':>10} "
        f"{'NN2 Acc':>10} {'Gap':>10} {'Mean L-inf':>12}")
    print("-" * 65)
    for r in results:
        print(f"  {r['max_iter']:<8} {r['epsilon']:<10} "
            f"{r['nn1_accuracy']:>10.2f}% "
            f"{r['nn2_accuracy']:>10.2f}% "
            f"{(r['nn1_accuracy']-r['nn2_accuracy']):>+9.2f}% "
            f"{r['mean_linf']:>12.4f}")
    print("=" * 65 + "\n")

### Error Generic Transferability Diagnostic

In [ ]:
# ==============================================================================
# Diagnostic: Transferability Error-Generic verification
# ==============================================================================

ATTACK_NAME_GENERIC = "BIM"   # Change to "BIM" for BIM generic
MAX_ITER_CHECK      = 20       # Only used for BIM — ignored for FGSM
eps                 = 0.1
eps_folder          = f"eps_{eps:.3f}".replace('.', '_')

if ATTACK_NAME_GENERIC == "FGSM":
    adv_eps_dir = os.path.join(ADVERSARIAL_DATA_DIR, ATTACK_NAME_GENERIC, eps_folder)
else:
    adv_eps_dir = os.path.join(ADVERSARIAL_DATA_DIR,
                            f"{ATTACK_NAME_GENERIC}_iter{MAX_ITER_CHECK}",
                            eps_folder)

print("=" * 60)
print(f"SECTION 1 — Directory and file freshness")
print("=" * 60)
print(f"ADVERSARIAL_DATA_DIR: {ADVERSARIAL_DATA_DIR}")
print(f"Checking path:        {adv_eps_dir}")
print(f"Directory exists:     {os.path.exists(adv_eps_dir)}")

if os.path.exists(adv_eps_dir):
    import datetime
    all_files = []
    for class_id in os.listdir(adv_eps_dir):
        class_folder = os.path.join(adv_eps_dir, class_id)
        if not os.path.isdir(class_folder):
            continue
        for f in os.listdir(class_folder):
            all_files.append(os.path.join(class_folder, f))

    print(f"Total adversarial files found: {len(all_files)}")
    if all_files:
        mtimes   = [os.path.getmtime(f) for f in all_files]
        newest   = max(mtimes)
        oldest   = min(mtimes)
        print(f"Newest file modified: "
            f"{datetime.datetime.fromtimestamp(newest).strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"Oldest file modified: "
            f"{datetime.datetime.fromtimestamp(oldest).strftime('%Y-%m-%d %H:%M:%S')}")
else:
    print("[ERROR] Directory does not exist. Cannot proceed.")
    raise SystemExit

print("\n" + "=" * 60)
print("SECTION 2 — NN1 accuracy on adversarial images (sample)")
print("=" * 60)
print("Verifying that NN1 is actually fooled by the adversarial images...")

nn1_correct = 0
nn1_total   = 0
sample_log  = []

for class_id in sorted(os.listdir(adv_eps_dir))[:10]:
    class_folder = os.path.join(adv_eps_dir, class_id)
    if not os.path.isdir(class_folder):
        continue

    true_name  = id_to_name.get(class_id.strip(), "Unknown")
    true_clean = normalize_label(true_name)

    for img_name in sorted(os.listdir(class_folder))[:3]:
        if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue

        img_path = os.path.join(class_folder, img_name)
        img      = Image.open(img_path).convert('RGB')
        t1       = preprocess_attack(img).numpy()[np.newaxis]

        pred_nn1     = np.argmax(art_classifier.predict(t1), axis=1)[0]
        raw_label    = LABELS[pred_nn1]
        if isinstance(raw_label, np.ndarray): raw_label = raw_label.flatten()[0]
        if isinstance(raw_label, bytes):      raw_label = raw_label.decode('utf-8')
        pred_name    = str(raw_label).replace("b'","").replace("'","").strip()
        pred_clean   = normalize_label(pred_name)

        is_correct = (pred_clean == true_clean)
        nn1_correct += int(is_correct)
        nn1_total   += 1
        sample_log.append((img_name, class_id, true_clean, pred_clean, is_correct))

for img_name, class_id, true_c, pred_c, correct in sample_log:
    status = "OK (not fooled)" if correct else "FOOLED"
    print(f"  {img_name} | {class_id} | True: {true_c:<30} "
          f"Pred: {pred_c:<30} [{status}]")

print(f"\nNN1 still correct on adversarial images: "
    f"{nn1_correct}/{nn1_total} ({100*nn1_correct/nn1_total:.1f}%)")
print(f"NN1 fooled: "
    f"{nn1_total - nn1_correct}/{nn1_total} "
    f"({100*(nn1_total - nn1_correct)/nn1_total:.1f}%)")

print("\n" + "=" * 60)
print("SECTION 3 — NN2 accuracy on adversarial images (sample)")
print("=" * 60)
print("Verifying NN2 predictions on the same adversarial images...")

nn2_correct = 0
nn2_total   = 0

for class_id in sorted(os.listdir(adv_eps_dir))[:10]:
    class_folder = os.path.join(adv_eps_dir, class_id)
    if not os.path.isdir(class_folder):
        continue

    true_name  = id_to_name.get(class_id.strip(), "Unknown")
    true_clean = normalize_label(true_name)

    for img_name in sorted(os.listdir(class_folder))[:3]:
        if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue

        img_path = os.path.join(class_folder, img_name)
        img      = Image.open(img_path).convert('RGB')
        t2       = preprocess_nn2(img).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            pred_nn2  = nn2(t2)[0].detach().cpu().numpy().argmax()

        pred_clean_2 = clean_pred_label(pred_nn2)
        is_correct   = (pred_clean_2 == true_clean)
        nn2_correct += int(is_correct)
        nn2_total   += 1

        print(f"  {img_name} | {class_id} | True: {true_clean:<30} "
            f"Pred: {pred_clean_2:<30} "
            f"[{'OK' if is_correct else 'FOOLED'}]")

print(f"\nNN2 still correct on adversarial images: "
    f"{nn2_correct}/{nn2_total} ({100*nn2_correct/nn2_total:.1f}%)")
print(f"NN2 fooled: "
    f"{nn2_total - nn2_correct}/{nn2_total} "
    f"({100*(nn2_total - nn2_correct)/nn2_total:.1f}%)")

print("\n" + "=" * 60)
print("SECTION 4 — Cross-check NN1 vs NN2 on same adversarial images")
print("=" * 60)
print("Comparing NN1 and NN2 predictions side by side...")

both_fooled  = 0
only_nn1     = 0
only_nn2     = 0
neither      = 0
total_cross  = 0

for class_id in sorted(os.listdir(adv_eps_dir))[:10]:
    class_folder = os.path.join(adv_eps_dir, class_id)
    if not os.path.isdir(class_folder):
        continue

    true_name  = id_to_name.get(class_id.strip(), "Unknown")
    true_clean = normalize_label(true_name)

    for img_name in sorted(os.listdir(class_folder))[:3]:
        if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue

        img_path = os.path.join(class_folder, img_name)
        img      = Image.open(img_path).convert('RGB')

        # NN1
        t1       = preprocess_attack(img).numpy()[np.newaxis]
        pred_nn1 = np.argmax(art_classifier.predict(t1), axis=1)[0]
        raw      = LABELS[pred_nn1]
        if isinstance(raw, np.ndarray): raw = raw.flatten()[0]
        if isinstance(raw, bytes):      raw = raw.decode('utf-8')
        pred_c1  = normalize_label(str(raw).replace("b'","").replace("'","").strip())

        # NN2
        t2       = preprocess_nn2(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            pred_nn2 = nn2(t2)[0].detach().cpu().numpy().argmax()
        pred_c2  = clean_pred_label(pred_nn2)

        nn1_fooled = (pred_c1 != true_clean)
        nn2_fooled = (pred_c2 != true_clean)

        if nn1_fooled and nn2_fooled:
            both_fooled += 1
        elif nn1_fooled and not nn2_fooled:
            only_nn1 += 1
        elif not nn1_fooled and nn2_fooled:
            only_nn2 += 1
        else:
            neither += 1
        total_cross += 1

print(f"Total images checked: {total_cross}")
print(f"  Both fooled:    {both_fooled:3d} ({100*both_fooled/total_cross:.1f}%)  "
    f"<-- transferability")
print(f"  Only NN1 fooled:{only_nn1:3d} ({100*only_nn1/total_cross:.1f}%)  "
    f"<-- no transfer")
print(f"  Only NN2 fooled:{only_nn2:3d} ({100*only_nn2/total_cross:.1f}%)  "
    f"<-- anomaly")
print(f"  Neither fooled: {neither:3d} ({100*neither/total_cross:.1f}%)  "
    f"<-- attack failed on NN1")

print("\n" + "=" * 60)
print("SECTION 5 — Verify clean_pred_label function for NN2")
print("=" * 60)
print("Checking that clean_pred_label correctly normalizes NN2 predictions...")

for img_path in image_paths[:5]:
    class_id   = os.path.basename(os.path.dirname(img_path))
    true_name  = id_to_name.get(class_id, "Unknown")
    true_clean = normalize_label(true_name)

    img = Image.open(img_path).convert('RGB')
    t2  = preprocess_nn2(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred_idx = nn2(t2)[0].detach().cpu().numpy().argmax()

    pred_normalized = clean_pred_label(pred_idx)

    # Also show raw label for debugging
    raw = LABELS[pred_idx]
    if isinstance(raw, np.ndarray): raw = raw.flatten()[0]
    if isinstance(raw, bytes):      raw = raw.decode('utf-8')
    raw_str = str(raw).replace("b'","").replace("'","").strip()

    match = (pred_normalized == true_clean)
    print(f"  {os.path.basename(img_path)} | True: {true_clean:<30}")
    print(f"    Raw LABELS[{pred_idx}]: '{raw_str}'")
    print(f"    clean_pred_label output: '{pred_normalized}'")
    print(f"    Match with true: {match}")
    

# Quick check — count files actually read by the transferability code
ATTACK_NAME  = "FGSM"
eps          = 0.1
eps_folder   = f"eps_{eps:.3f}".replace('.', '_')
adv_eps_dir  = os.path.join(ADVERSARIAL_DATA_DIR, ATTACK_NAME, eps_folder)

total = 0
for class_id in os.listdir(adv_eps_dir):
    class_folder = os.path.join(adv_eps_dir, class_id)
    if not os.path.isdir(class_folder):
        continue
    for img_name in os.listdir(class_folder):
        if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
            total += 1

print(f"Files found by transferability loop: {total}")
print(f"Expected: 1000")

### Transferability — FGSM Targeted (Error-Specific)

Evaluates whether adversarial examples generated by FGSM Targeted against NN1
cause NN2 to predict the same intended target class. The Attack Success Rate (ASR)
on NN2 is compared against the ASR on NN1 via a comparative targeted SEC.
Both targeting modes (Fixed and LeastLikely) are evaluated independently.

In [64]:
if RUN_FGSM_TRANSFERABILITY_SPECIFIC:
    # ==============================================================================
    # STEP 5.4: Transferability — FGSM Targeted (Error-Specific)
    # ==============================================================================

    print("Evaluating FGSM Targeted transferability on NN2...")

    ATTACK_NAME_BASE   = "FGSM_Targeted"
    EPSILONS           = [0.001, 0.003, 0.005, 0.01, 0.03, 0.05, 0.1]
    TARGET_MODES       = ["LeastLikely", "Fixed"]
    skipped_images_least_likely = 0
    skipped_images_fixed       = 0

    for mode in TARGET_MODES:
        ATTACK_NAME          = f"{ATTACK_NAME_BASE}_{mode}"
        TRANSFER_RESULTS_DIR = os.path.join(RESULTS_DIR, f"{ATTACK_NAME}_Transferability")
        os.makedirs(TRANSFER_RESULTS_DIR, exist_ok=True)
        
        if mode == "LeastLikely":
            loaded = load_least_likely_targets(ATTACK_NAME)
            # If loaded is already a dict, use directly; if list, build map from current order
            if isinstance(loaded, dict):
                target_map = loaded
            else:
                target_map = {
                    os.path.splitext(os.path.basename(image_paths[k]))[0]: loaded[k]
                    for k in range(len(loaded))
                }
        else:
            target_map = {}

        ADV_ATTACK_DIR   = os.path.join(ADVERSARIAL_DATA_DIR, ATTACK_NAME)
        nn1_results_path = os.path.join(RESULTS_DIR, ATTACK_NAME, f"{ATTACK_NAME}_results.csv")

        if not os.path.exists(nn1_results_path):
            print(f"[WARNING] NN1 results CSV not found for {ATTACK_NAME}. Skipping.")
            continue

        df_nn1 = pd.read_csv(nn1_results_path)

        results       = []
        nn1_asrs      = []
        nn2_asrs      = []

        for eps in EPSILONS:
            eps_folder  = f"eps_{eps:.3f}".replace('.', '_')
            adv_eps_dir = os.path.join(ADV_ATTACK_DIR, eps_folder)

            if not os.path.exists(adv_eps_dir):
                print(f"[WARNING] Directory not found: {adv_eps_dir}. Skipping.")
                continue

            print(f"\n--- Epsilon: {eps:.3f} | Mode: {mode} ---")

            successful_nn2  = 0
            total_processed = 0
            start_time      = time.time()

            EVAL_BATCH_SIZE = 64

            batch_imgs    = []
            batch_names   = []
            batch_targets = []

            for class_id in os.listdir(adv_eps_dir):
                class_folder = os.path.join(adv_eps_dir, class_id)
                if not os.path.isdir(class_folder):
                    continue

                for img_name in os.listdir(class_folder):
                    if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                        continue

                    if mode == "Fixed":
                        target = FIXED_TARGET_CLASS
                    else:
                        img_stem = os.path.splitext(img_name)[0]
                        key      = f"{class_id}/{img_stem}"
                        target   = target_map.get(key, None)
                        if target is None:
                            skipped_images_least_likely += 1
                            print(f"[WARNING] No target found for {img_name}, skipping.")
                            continue

                    try:
                        img = Image.open(
                            os.path.join(class_folder, img_name)
                        ).convert('RGB')
                        batch_imgs.append(preprocess_nn2(img))
                        batch_names.append(img_name)
                        batch_targets.append(target)
                    except Exception as e:
                        print(f"Error loading {img_name}: {e}")

            # Run inference in batches
            for b in range(0, len(batch_imgs), EVAL_BATCH_SIZE):
                batch_tensor = torch.stack(
                    batch_imgs[b:b + EVAL_BATCH_SIZE]
                ).to(DEVICE)
                batch_tgts   = batch_targets[b:b + EVAL_BATCH_SIZE]

                with torch.no_grad():
                    pred_indices = nn2(batch_tensor).detach().cpu().numpy().argmax(axis=1)

                for pred_idx_2, tgt in zip(pred_indices, batch_tgts):
                    if pred_idx_2 == tgt:
                        successful_nn2 += 1
                    total_processed += 1
                    
            elapsed    = time.time() - start_time
            asr_nn2    = (successful_nn2 / total_processed) * 100 if total_processed > 0 else 0.0
            nn2_asrs.append(asr_nn2)

            nn1_row = df_nn1[df_nn1['epsilon'] == eps]
            asr_nn1 = nn1_row['attack_success_rate'].values[0] if len(nn1_row) > 0 else 0.0
            nn1_asrs.append(asr_nn1)

            print(f"NN2 ASR: {asr_nn2:.2f}%  |  NN1 ASR: {asr_nn1:.2f}%")

            results.append({
                'attack_name':   ATTACK_NAME,
                'target_mode':   mode,
                'epsilon':       eps,
                'nn1_asr':       round(asr_nn1, 2),
                'nn2_asr':       round(asr_nn2, 2),
                'nn2_successful': successful_nn2,
                'total_images':  total_processed,
                'time_elapsed_s': round(elapsed, 2)
            })

        # --------------------------------------------------------------------------
        # Comparative Targeted SEC — NN1 ASR vs NN2 ASR
        # --------------------------------------------------------------------------
        fig, ax = plt.subplots(figsize=(9, 6))

        ax.plot([0.0, EPSILONS[0]], [0.0, nn1_asrs[0]],
                linestyle='-', color='#d62728', linewidth=1.5)
        ax.plot(EPSILONS, nn1_asrs,
                marker='o', linestyle='-', color='#d62728',
                linewidth=1.5, markersize=6, label='NN1 ASR (Attack Source)')

        ax.plot([0.0, EPSILONS[0]], [0.0, nn2_asrs[0]],
                linestyle='--', color='#ff9896', linewidth=1.5)
        ax.plot(EPSILONS, nn2_asrs,
                marker='s', linestyle='--', color='#ff9896',
                linewidth=1.5, markersize=6, label='NN2 ASR (Transferability)')

        ax.plot(0.0, 0.0, marker='o', markerfacecolor='gray',
                markeredgecolor='#2ca02c', markeredgewidth=1,
                markersize=6, linestyle='None', label='Baseline ASR (~0%)')

        ax.set_title(f'Transferability Targeted SEC — {ATTACK_NAME}', fontsize=16, fontweight='bold', pad=15)
        ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
        ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
        ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
        ax.set_xticks(np.arange(0.0, 0.11, 0.01))
        ax.set_yticks(np.arange(0, 101, 10))
        ax.tick_params(axis='both', labelsize=12)
        ax.set_ylim(-5, 105)
        ax.set_xlim(-0.005, 0.105)
        ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2, label=r'Max Allowable $\epsilon$ (0.1)')
        ax.legend(loc='upper left', fontsize=12)
        plt.tight_layout()

        plot_path = os.path.join(TRANSFER_RESULTS_DIR, f'{ATTACK_NAME}_transferability_SEC.png')
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\n[INFO] Transferability SEC saved at: {plot_path}")

        # --------------------------------------------------------------------------
        # Save Results CSV
        # --------------------------------------------------------------------------
        df_results = pd.DataFrame(results)
        csv_path   = os.path.join(TRANSFER_RESULTS_DIR, f'{ATTACK_NAME}_transferability_results.csv')
        df_results.to_csv(csv_path, index=False)
        print(f"\n[INFO] Results CSV saved at: {csv_path}")

        # --------------------------------------------------------------------------
        # Summary
        # --------------------------------------------------------------------------
        print("\n" + "=" * 55)
        print(f"{ATTACK_NAME} TRANSFERABILITY SUMMARY")
        print("=" * 55)
        print(f"{'Epsilon':<10} {'NN1 ASR':>10} {'NN2 ASR':>10} {'Gap':>10}")
        print("-" * 55)
        for eps, a1, a2 in zip(EPSILONS, nn1_asrs, nn2_asrs):
            print(f"{eps:<10.3f} {a1:>10.2f}% {a2:>10.2f}% {(a2-a1):>+9.2f}%")
        print(f"Skipped Images LeastLikely: {skipped_images_least_likely}")
        print(f"Skipped Images Fixed: {skipped_images_fixed}")
        print("=" * 55 + "\n")

### Transferability — BIM Targeted (Error-Specific)

Evaluates whether adversarial examples generated by BIM Targeted against NN1
cause NN2 to predict the same intended target class. The Attack Success Rate (ASR)
on NN2 is compared against the ASR on NN1 via comparative targeted SECs, one per
max_iter value. Both targeting modes (Fixed and LeastLikely) are evaluated
independently. A CSV summary of all results is saved for each mode.

In [65]:
if RUN_BIM_TRANSFERABILITY_SPECIFIC:  
    
    # ==============================================================================
    # STEP 5.5: Transferability — BIM Targeted (Error-Specific)
    # ==============================================================================
    print("Evaluating BIM Targeted transferability on NN2...")

    ATTACK_NAME_BASE   = "BIM_Targeted"
    EPSILONS           = [0.001, 0.003, 0.005, 0.01, 0.03, 0.05, 0.1]
    MAX_ITER_LIST      = [1, 5, 10, 20]
    TARGET_MODES       = ["LeastLikely", "Fixed"]
    skipped_images_least_likely = 0
    skipped_images_fixed       = 0

    ITER_COLORS_NN1 = {1: '#d62728', 5: '#8c1a1a', 10: '#e07070', 20: '#a83232'}
    ITER_COLORS_NN2 = {1: '#ff9896', 5: '#ffbcba', 10: '#ffd7d6', 20: '#ffe8e8'}

    for mode in TARGET_MODES:
        ATTACK_NAME          = f"{ATTACK_NAME_BASE}_{mode}"
        TRANSFER_RESULTS_DIR = os.path.join(RESULTS_DIR, f"{ATTACK_NAME}_Transferability")
        os.makedirs(TRANSFER_RESULTS_DIR, exist_ok=True)
        
        if mode == "LeastLikely":
            loaded = load_least_likely_targets(ATTACK_NAME)
            # If loaded is already a dict, use directly; if list, build map from current order
            if isinstance(loaded, dict):
                target_map = loaded
            else:
                target_map = {
                    os.path.splitext(os.path.basename(image_paths[k]))[0]: loaded[k]
                    for k in range(len(loaded))
                }
        else:
            target_map = {}

        nn1_results_path = os.path.join(RESULTS_DIR, ATTACK_NAME,
                                        f"{ATTACK_NAME}_results.csv")
        if not os.path.exists(nn1_results_path):
            print(f"[WARNING] NN1 results CSV not found for {ATTACK_NAME}. Skipping.")
            continue

        df_nn1 = pd.read_csv(nn1_results_path)

        print(f"\n{'*'*70}")
        print(f"Starting {ATTACK_NAME} transferability across " f"{len(MAX_ITER_LIST)} max_iter values x {len(EPSILONS)} epsilon values")
        print(f"{'*'*70}")

        results            = []
        all_nn1_asrs       = {}
        all_nn2_asrs       = {}

        for max_iter in MAX_ITER_LIST:
            folder_name  = f"{ATTACK_NAME}_iter{max_iter}"
            adv_iter_dir = os.path.join(ADVERSARIAL_DATA_DIR, folder_name)

            if not os.path.exists(adv_iter_dir):
                print(f"[WARNING] Directory not found: {adv_iter_dir}. Skipping.")
                continue

            print(f"\n{'='*50}")
            print(f"MAX_ITER = {max_iter} | MODE = {mode}")
            print(f"{'='*50}")

            nn1_asrs = []
            nn2_asrs = []

            for eps in EPSILONS:
                eps_folder  = f"eps_{eps:.3f}".replace('.', '_')
                adv_eps_dir = os.path.join(adv_iter_dir, eps_folder)

                if not os.path.exists(adv_eps_dir):
                    print(f"[WARNING] Directory not found: {adv_eps_dir}. Skipping.")
                    continue

                print(f"\n--- Epsilon: {eps:.3f} | max_iter: {max_iter} ---")

                successful_nn2  = 0
                total_processed = 0
                start_time      = time.time()

                EVAL_BATCH_SIZE = 64
                batch_imgs    = []
                batch_names   = []
                batch_targets = []

                for class_id in os.listdir(adv_eps_dir):
                    class_folder = os.path.join(adv_eps_dir, class_id)
                    if not os.path.isdir(class_folder):
                        continue

                    for img_name in os.listdir(class_folder):
                        if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                            continue

                        if mode == "Fixed":
                            target = FIXED_TARGET_CLASS
                        else:
                            img_stem = os.path.splitext(img_name)[0]
                            key      = f"{class_id}/{img_stem}"
                            target   = target_map.get(key, None)
                            if target is None:
                                skipped_images_least_likely += 1
                                print(f"[WARNING] No target found for {img_name}, skipping.")
                                continue

                        try:
                            img = Image.open(
                                os.path.join(class_folder, img_name)
                            ).convert('RGB')
                            batch_imgs.append(preprocess_nn2(img))
                            batch_names.append(img_name)
                            batch_targets.append(target)
                        except Exception as e:
                            print(f"Error loading {img_name}: {e}")

                # Run inference in batches
                for b in range(0, len(batch_imgs), EVAL_BATCH_SIZE):
                    batch_tensor = torch.stack(
                        batch_imgs[b:b + EVAL_BATCH_SIZE]
                    ).to(DEVICE)
                    batch_tgts   = batch_targets[b:b + EVAL_BATCH_SIZE]

                    with torch.no_grad():
                        pred_indices = nn2(batch_tensor).detach().cpu().numpy().argmax(axis=1)

                    for pred_idx_2, tgt in zip(pred_indices, batch_tgts):
                        if pred_idx_2 == tgt:
                            successful_nn2 += 1
                        total_processed += 1

                elapsed = time.time() - start_time
                asr_nn2 = (successful_nn2 / total_processed) * 100 if total_processed > 0 else 0.0

                nn1_row = df_nn1[
                    (df_nn1['epsilon'] == eps) &
                    (df_nn1['max_iter'] == max_iter)
                ]
                asr_nn1 = nn1_row['attack_success_rate'].values[0] if len(nn1_row) > 0 else 0.0

                nn1_asrs.append(asr_nn1)
                nn2_asrs.append(asr_nn2)

                print(f"NN2 ASR: {asr_nn2:.2f}%  |  NN1 ASR: {asr_nn1:.2f}%")

                results.append({
                    'attack_name':    ATTACK_NAME,
                    'target_mode':    mode,
                    'max_iter':       max_iter,
                    'epsilon':        eps,
                    'nn1_asr':        round(asr_nn1, 2),
                    'nn2_asr':        round(asr_nn2, 2),
                    'nn2_successful': successful_nn2,
                    'total_images':   total_processed,
                    'time_elapsed_s': round(elapsed, 2)
                })

            all_nn1_asrs[max_iter] = nn1_asrs
            all_nn2_asrs[max_iter] = nn2_asrs

            # ----------------------------------------------------------------------
            # Individual Comparative Targeted SEC for this max_iter
            # ----------------------------------------------------------------------
            c_nn1 = ITER_COLORS_NN1[max_iter]
            c_nn2 = ITER_COLORS_NN2[max_iter]

            fig, ax = plt.subplots(figsize=(9, 6))

            ax.plot(0.0, 0.0, marker='o', markerfacecolor='gray',
                    markeredgecolor='#2ca02c', markeredgewidth=1,
                    markersize=6, linestyle='None', label='Baseline ASR (~0%)')

            ax.plot([0.0, EPSILONS[0]], [0.0, nn1_asrs[0]],
                    linestyle='-', color=c_nn1, linewidth=1.5)
            ax.plot(EPSILONS, nn1_asrs,
                    marker='o', linestyle='-', color=c_nn1,
                    linewidth=1.5, markersize=6,
                    label=f'NN1 ASR (max_iter={max_iter})')

            ax.plot([0.0, EPSILONS[0]], [0.0, nn2_asrs[0]],
                    linestyle='--', color=c_nn2, linewidth=1.5)
            ax.plot(EPSILONS, nn2_asrs,
                    marker='s', linestyle='--', color=c_nn2,
                    linewidth=1.5, markersize=6,
                    label=f'NN2 ASR (max_iter={max_iter})')

            ax.set_title(f'Transferability Targeted SEC — {ATTACK_NAME}\n'
                        f'(max_iter={max_iter})',
                        fontsize=16, fontweight='bold', pad=15)
            ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
            ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
            ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
            ax.set_xticks(np.arange(0.0, 0.11, 0.01))
            ax.set_yticks(np.arange(0, 101, 10))
            ax.tick_params(axis='both', labelsize=12)
            ax.set_ylim(-5, 105)
            ax.set_xlim(-0.005, 0.105)
            ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2, label=r'Max Allowable $\epsilon$ (0.1)')
            ax.legend(loc='upper left', fontsize=12)
            plt.tight_layout()

            plot_path = os.path.join(
                TRANSFER_RESULTS_DIR,
                f'{ATTACK_NAME}_iter{max_iter}_transferability_SEC.png'
            )
            plt.savefig(plot_path, dpi=300, bbox_inches='tight')
            plt.show()
            print(f"\n[INFO] SEC saved at: {plot_path}")

        # --------------------------------------------------------------------------
        # Comparative overview — all max_iter, NN1 ASR vs NN2 ASR
        # --------------------------------------------------------------------------
        fig, ax = plt.subplots(figsize=(9, 6))

        ax.plot(0.0, 0.0, marker='o', markerfacecolor='gray',
                markeredgecolor='#2ca02c', markeredgewidth=1,
                markersize=6, linestyle='None', label='Baseline ASR (~0%)')

        for max_iter in MAX_ITER_LIST:
            if max_iter not in all_nn1_asrs:
                continue
            c_nn1 = ITER_COLORS_NN1[max_iter]
            c_nn2 = ITER_COLORS_NN2[max_iter]

            ax.plot([0.0, EPSILONS[0]], [0.0, all_nn1_asrs[max_iter][0]],
                    linestyle='-', color=c_nn1, linewidth=1.5)
            ax.plot(EPSILONS, all_nn1_asrs[max_iter],
                    marker='o', linestyle='-', color=c_nn1,
                    linewidth=1.5, markersize=5,
                    label=f'NN1 iter={max_iter}')

            ax.plot([0.0, EPSILONS[0]], [0.0, all_nn2_asrs[max_iter][0]],
                    linestyle='--', color=c_nn2, linewidth=1.5)
            ax.plot(EPSILONS, all_nn2_asrs[max_iter],
                    marker='s', linestyle='--', color=c_nn2,
                    linewidth=1.5, markersize=5,
                    label=f'NN2 iter={max_iter}')

        ax.set_title(f'Transferability Targeted SEC — {ATTACK_NAME} (Comparative)', fontsize=16, fontweight='bold', pad=15)
        ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
        ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
        ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
        ax.set_xticks(np.arange(0.0, 0.11, 0.01))
        ax.set_yticks(np.arange(0, 101, 10))
        ax.tick_params(axis='both', labelsize=12)
        ax.set_ylim(-5, 105)
        ax.set_xlim(-0.005, 0.105)
        ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2, label=r'Max Allowable $\epsilon$ (0.1)')
        ax.legend(loc='upper left', fontsize=9)
        plt.tight_layout()

        comparative_path = os.path.join(
            TRANSFER_RESULTS_DIR,
            f'{ATTACK_NAME}_transferability_comparative_SEC.png'
        )
        plt.savefig(comparative_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\n[INFO] Comparative SEC saved at: {comparative_path}")

        # --------------------------------------------------------------------------
        # Save Results CSV
        # --------------------------------------------------------------------------
        df_results = pd.DataFrame(results)
        csv_path   = os.path.join(TRANSFER_RESULTS_DIR, f'{ATTACK_NAME}_transferability_results.csv')
        df_results.to_csv(csv_path, index=False)
        print(f"\n[INFO] Results CSV saved at: {csv_path}")

        # --------------------------------------------------------------------------
        # Summary
        # --------------------------------------------------------------------------
        print("\n" + "=" * 60)
        print(f"{ATTACK_NAME} TRANSFERABILITY SUMMARY")
        print("=" * 60)
        for max_iter in MAX_ITER_LIST:
            if max_iter not in all_nn1_asrs:
                continue
            a1 = all_nn1_asrs[max_iter][-1]
            a2 = all_nn2_asrs[max_iter][-1]
            print(f"  max_iter={max_iter:2d} @ eps=0.1 — "
                f"NN1 ASR: {a1:.2f}%  NN2 ASR: {a2:.2f}%  "
                f"Gap: {(a2-a1):+.2f}%")
        print(f"Skipped Images LeastLikely: {skipped_images_least_likely}")
        print(f"Skipped Images Fixed: {skipped_images_fixed}")  
        print("=" * 60 + "\n")

### Transferability — PGD Targeted (Error-Specific)

Evaluates whether adversarial examples generated by PGD Targeted against NN1
cause NN2 to predict the same intended target class, across all combinations of
epsilon, max_iter, and num_random_init. Individual comparative targeted SECs are
generated for each (max_iter, num_random_init) combination. A summary table
reports the transferability gap for the strongest configuration.
Both targeting modes (Fixed and LeastLikely) are evaluated independently.

In [66]:
if RUN_PGD_TRANSFERABILITY_SPECIFIC:

    # ==============================================================================
    # STEP 5.6: Transferability — PGD Targeted (Error-Specific)
    # ==============================================================================

    print("Evaluating PGD Targeted transferability on NN2...")

    ATTACK_NAME_BASE   = "PGD_Targeted"
    EPSILONS           = [0.001, 0.003, 0.005, 0.01, 0.03, 0.05, 0.1]
    MAX_ITER_LIST      = [1, 5, 10, 20, 40]
    NUM_RANDOM_INITS   = [0, 1, 3, 5]
    BATCH_SIZE         = 32
    TARGET_MODES       = ["LeastLikely", "Fixed"]
    skipped_images_least_likely = 0
    skipped_images_fixed = 0

    ITER_COLORS_NN1 = {
        1: '#d62728', 5: '#8c1a1a', 10: '#e07070', 20: '#a83232', 40: '#6b0000'
    }
    ITER_COLORS_NN2 = {
        1: '#ff9896', 5: '#ffbcba', 10: '#ffd7d6', 20: '#ffe8e8', 40: '#fff0f0'
    }

    for mode in TARGET_MODES:
        ATTACK_NAME          = f"{ATTACK_NAME_BASE}_{mode}"
        TRANSFER_RESULTS_DIR = os.path.join(RESULTS_DIR, f"{ATTACK_NAME}_Transferability")
        os.makedirs(TRANSFER_RESULTS_DIR, exist_ok=True)

        if mode == "LeastLikely":
            loaded = load_least_likely_targets(ATTACK_NAME)
            # If loaded is already a dict, use directly; if list, build map from current order
            if isinstance(loaded, dict):
                target_map = loaded
            else:
                target_map = {
                    os.path.splitext(os.path.basename(image_paths[k]))[0]: loaded[k]
                    for k in range(len(loaded))
                }
        else:
            target_map = {}
            
        nn1_results_path = os.path.join(RESULTS_DIR, ATTACK_NAME,
                                        f"{ATTACK_NAME}_results.csv")
        if not os.path.exists(nn1_results_path):
            print(f"[WARNING] NN1 results CSV not found for {ATTACK_NAME}. Skipping.")
            continue

        df_nn1 = pd.read_csv(nn1_results_path)

        print(f"\n{'*'*70}")
        print(f"Starting {ATTACK_NAME} transferability across "
                f"{len(NUM_RANDOM_INITS)} nri x "
                f"{len(MAX_ITER_LIST)} iters x "
                f"{len(EPSILONS)} epsilons")
        print(f"{'*'*70}")

        results            = []
        all_nn1_asrs       = {nri: {} for nri in NUM_RANDOM_INITS}
        all_nn2_asrs       = {nri: {} for nri in NUM_RANDOM_INITS}

        for num_random_init in NUM_RANDOM_INITS:
            for max_iter in MAX_ITER_LIST:
                folder_name  = f"{ATTACK_NAME}_nri{num_random_init}_iter{max_iter}"
                adv_iter_dir = os.path.join(ADVERSARIAL_DATA_DIR, folder_name)

                if not os.path.exists(adv_iter_dir):
                    print(f"[WARNING] Directory not found: {adv_iter_dir}. Skipping.")
                    continue

                print(f"\n{'='*50}")
                print(f"NUM_RANDOM_INIT = {num_random_init} | MAX_ITER = {max_iter}")
                print(f"{'='*50}")

                nn1_asrs = []
                nn2_asrs = []

                for eps in EPSILONS:
                    eps_folder  = f"eps_{eps:.3f}".replace('.', '_')
                    adv_eps_dir = os.path.join(adv_iter_dir, eps_folder)

                    if not os.path.exists(adv_eps_dir):
                        print(f"[WARNING] Directory not found: {adv_eps_dir}. Skipping.")
                        continue

                    print(f"\n--- eps={eps:.3f} | max_iter={max_iter} | nri={num_random_init} ---")

                    successful_nn2  = 0
                    total_processed = 0
                    start_time      = time.time()

                    EVAL_BATCH_SIZE = 64
                    batch_imgs    = []
                    batch_names   = []
                    batch_targets = []

                    for class_id in os.listdir(adv_eps_dir):
                        class_folder = os.path.join(adv_eps_dir, class_id)
                        if not os.path.isdir(class_folder):
                            continue

                        for img_name in os.listdir(class_folder):
                            if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                                continue

                            if mode == "Fixed":
                                target = FIXED_TARGET_CLASS
                            else:
                                img_stem = os.path.splitext(img_name)[0]
                                key      = f"{class_id}/{img_stem}"   # <- chiave univoca
                                target   = target_map.get(key, None)
                                if target is None:
                                    skipped_images_least_likely += 1
                                    print(f"[WARNING] No target found for {img_name}, skipping.")
                                    continue

                            try:
                                img = Image.open(
                                    os.path.join(class_folder, img_name)
                                ).convert('RGB')
                                batch_imgs.append(preprocess_nn2(img))
                                batch_names.append(img_name)
                                batch_targets.append(target)
                            except Exception as e:
                                print(f"Error loading {img_name}: {e}")

                    # Run inference in batches
                    for b in range(0, len(batch_imgs), EVAL_BATCH_SIZE):
                        batch_tensor = torch.stack(
                            batch_imgs[b:b + EVAL_BATCH_SIZE]
                        ).to(DEVICE)
                        batch_tgts   = batch_targets[b:b + EVAL_BATCH_SIZE]

                        with torch.no_grad():
                            pred_indices = nn2(batch_tensor).detach().cpu().numpy().argmax(axis=1)

                        for pred_idx_2, tgt in zip(pred_indices, batch_tgts):
                            if pred_idx_2 == tgt:
                                successful_nn2 += 1
                            total_processed += 1

                    elapsed = time.time() - start_time
                    asr_nn2 = (successful_nn2 / total_processed) * 100 if total_processed > 0 else 0.0

                    nn1_row = df_nn1[
                        (df_nn1['epsilon'] == eps) &
                        (df_nn1['max_iter'] == max_iter) &
                        (df_nn1['num_random_init'] == num_random_init)
                    ]
                    asr_nn1 = nn1_row['attack_success_rate'].values[0] if len(nn1_row) > 0 else 0.0

                    nn1_asrs.append(asr_nn1)
                    nn2_asrs.append(asr_nn2)

                    print(f"NN2 ASR: {asr_nn2:.2f}%  |  NN1 ASR: {asr_nn1:.2f}%")

                    results.append({
                        'attack_name':    ATTACK_NAME,
                        'target_mode':    mode,
                        'num_random_init': num_random_init,
                        'max_iter':       max_iter,
                        'epsilon':        eps,
                        'nn1_asr':        round(asr_nn1, 2),
                        'nn2_asr':        round(asr_nn2, 2),
                        'nn2_successful': successful_nn2,
                        'total_images':   total_processed,
                        'time_elapsed_s': round(elapsed, 2)
                    })

                all_nn1_asrs[num_random_init][max_iter] = nn1_asrs
                all_nn2_asrs[num_random_init][max_iter] = nn2_asrs

                # ------------------------------------------------------------------
                # Individual Comparative Targeted SEC for this (nri, max_iter)
                # ------------------------------------------------------------------
                c_nn1 = ITER_COLORS_NN1[max_iter]
                c_nn2 = ITER_COLORS_NN2[max_iter]

                fig, ax = plt.subplots(figsize=(9, 6))

                ax.plot(0.0, 0.0, marker='o', markerfacecolor='gray',
                        markeredgecolor='#2ca02c', markeredgewidth=1,
                        markersize=6, linestyle='None', label='Baseline ASR (~0%)')

                ax.plot([0.0, EPSILONS[0]], [0.0, nn1_asrs[0]],
                        linestyle='-', color=c_nn1, linewidth=1.5)
                ax.plot(EPSILONS, nn1_asrs,
                        marker='o', linestyle='-', color=c_nn1,
                        linewidth=1.5, markersize=6,
                        label=f'NN1 ASR (nri={num_random_init}, iter={max_iter})')

                ax.plot([0.0, EPSILONS[0]], [0.0, nn2_asrs[0]],
                        linestyle='--', color=c_nn2, linewidth=1.5)
                ax.plot(EPSILONS, nn2_asrs,
                        marker='s', linestyle='--', color=c_nn2,
                        linewidth=1.5, markersize=6,
                        label=f'NN2 ASR (nri={num_random_init}, iter={max_iter})')

                ax.set_title(f'Transferability Targeted SEC — {ATTACK_NAME}\n'
                            f'(nri={num_random_init}, max_iter={max_iter})',
                            fontsize=16, fontweight='bold', pad=15)
                ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
                ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
                ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
                ax.set_xticks(np.arange(0.0, 0.11, 0.01))
                ax.set_yticks(np.arange(0, 101, 10))
                ax.tick_params(axis='both', labelsize=12)
                ax.set_ylim(-5, 105)
                ax.set_xlim(-0.005, 0.105)
                ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2, label=r'Max Allowable $\epsilon$ (0.1)')
                ax.legend(loc='upper left', fontsize=11)
                plt.tight_layout()

                plot_path = os.path.join(
                    TRANSFER_RESULTS_DIR,
                    f'{ATTACK_NAME}_nri{num_random_init}_iter{max_iter}_transferability_SEC.png'
                )
                plt.savefig(plot_path, dpi=300, bbox_inches='tight')
                plt.show()
                print(f"\n[INFO] SEC saved at: {plot_path}")

        # --------------------------------------------------------------------------
        # Save Results CSV
        # --------------------------------------------------------------------------
        df_results = pd.DataFrame(results)
        csv_path   = os.path.join(TRANSFER_RESULTS_DIR, f'{ATTACK_NAME}_transferability_results.csv')
        df_results.to_csv(csv_path, index=False)
        print(f"\n[INFO] Results CSV saved at: {csv_path}")

        # --------------------------------------------------------------------------
        # Summary — strongest configuration only
        # --------------------------------------------------------------------------
        best_nri  = max(NUM_RANDOM_INITS)
        best_iter = max(MAX_ITER_LIST)

        print("\n" + "=" * 60)
        print(f"{ATTACK_NAME} TRANSFERABILITY SUMMARY " f"(Strongest Config: nri={best_nri}, iter={best_iter})")
        print("=" * 60)
        if (best_nri in all_nn1_asrs and
                best_iter in all_nn1_asrs[best_nri]):
            a1 = all_nn1_asrs[best_nri][best_iter][-1]
            a2 = all_nn2_asrs[best_nri][best_iter][-1]
            print(f"  @ eps=0.1 — NN1 ASR: {a1:.2f}%  " f"NN2 ASR: {a2:.2f}%  Gap: {(a2-a1):+.2f}%")
        print(f"Skipped Images Least Likely: {skipped_images_least_likely}")
        print(f"Skipped Images Fixed: {skipped_images_fixed}")
        print("=" * 60 + "\n")

### Transferability — Carlini & Wagner L-inf Targeted (Error-Specific)

Evaluates whether adversarial examples generated by CW Targeted LeastLikely against NN1
cause NN2 to predict the same intended target class. The Attack Success Rate (ASR) on NN2
is compared against the ASR on NN1 via a comparative SEC plot. Since CW does not enforce
a strict L-infinity budget, the x-axis uses the mean L-inf perturbation actually applied,
consistent with the CW Untargeted transferability evaluation. Only images saved within
the L-inf <= 0.1 constraint are evaluated.

In [67]:
if RUN_CW_TRANSFERABILITY_SPECIFIC:
    # ==============================================================================
    # STEP 5.7: Transferability — Carlini & Wagner L-inf Targeted (Error-Specific)
    # ==============================================================================

    print("Evaluating CW Targeted transferability on NN2...")

    ATTACK_NAME_BASE  = "CW_Targeted"
    MAX_ITER_LIST      = [1, 20, 50]
    LEARNING_RATE_LIST = [0.01, 0.1]
    CONFIDENCE_LIST    = [0.0]

    TARGET_MODE_SELECTION = "Fixed"   # "Fixed", "LeastLikely", or "Both"

    if TARGET_MODE_SELECTION == "Both":
        TARGET_MODES = ["LeastLikely", "Fixed"]
    elif TARGET_MODE_SELECTION in ("Fixed", "LeastLikely"):
        TARGET_MODES = [TARGET_MODE_SELECTION]
    else:
        raise ValueError(
            f"Invalid TARGET_MODE_SELECTION: '{TARGET_MODE_SELECTION}'. "
            "Choose 'Fixed', 'LeastLikely', or 'Both'."
        )

    ITER_COLORS_NN1 = {1: '#d62728', 20: '#8c1a1a', 50: '#e07070'}
    ITER_COLORS_NN2 = {1: '#ff9896', 20: '#ffbcba', 50: '#ffd7d6'}

    # Load reference file list once — same order used during generation.
    reference_files = sorted([
        f for f in os.listdir(DATA_DIR_100)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])
    if not reference_files:
        raise FileNotFoundError(
            f"No reference images found in DATA_DIR_100: {DATA_DIR_100}"
        )
    for mode in TARGET_MODES:
        ATTACK_NAME          = f"{ATTACK_NAME_BASE}_{mode}"
        TRANSFER_RESULTS_DIR = os.path.join(
            RESULTS_DIR, f"{ATTACK_NAME}_Transferability"
        )
        ADV_BASE_DIR         = os.path.join(ADVERSARIAL_DATA_DIR, ATTACK_NAME)
        os.makedirs(TRANSFER_RESULTS_DIR, exist_ok=True)

        print(f"\n{'*'*70}")
        print(f"Transferability evaluation — {ATTACK_NAME}")
        print(f"{'*'*70}")

        nn1_results_path = os.path.join(
            RESULTS_DIR, ATTACK_NAME, f"{ATTACK_NAME}_results.csv"
        )
        if not os.path.exists(nn1_results_path):
            raise FileNotFoundError(
                f"NN1 results CSV not found at {nn1_results_path}. "
                "Run Step 3.5 first."
            )
        df_nn1 = pd.read_csv(nn1_results_path)

        # Load target map — only needed for LeastLikely mode.
        if mode == "LeastLikely":
            loaded     = load_least_likely_targets(ATTACK_NAME)
            target_map = loaded if isinstance(loaded, dict) else None
            if target_map is None:
                raise ValueError(
                    "Loaded targets are not in dict format. "
                    "Run the remapping utility script first."
                )
            print(f"Loaded target map: {len(target_map)} entries.")
            sample_key = list(target_map.keys())[0]
            print(f"Key format check — first key: '{sample_key}'")
            if '/' not in sample_key:
                raise ValueError(
                    "Target map uses old format (no class_id prefix). "
                    "Run the remapping utility script first."
                )
            print("Key format: class_id/filename_stem — CORRECT")
        else:
            target_map = None

        results        = []
        all_combo_data = []

        total_combinations = (
            len(MAX_ITER_LIST) * len(LEARNING_RATE_LIST) * len(CONFIDENCE_LIST)
        )
        current_combo = 1

        for max_iter in MAX_ITER_LIST:
            for lr in LEARNING_RATE_LIST:
                for conf in CONFIDENCE_LIST:
                    print(f"\n[{current_combo}/{total_combinations}] "
                        f"max_iter={max_iter} | lr={lr} | confidence={conf}")

                    lr_str         = str(lr).replace('.', '_')
                    conf_str       = str(conf).replace('.', '_')
                    combo_dir_name = f"iter_{max_iter}_lr_{lr_str}_c_{conf_str}"
                    adv_combo_dir  = os.path.join(ADV_BASE_DIR, combo_dir_name)

                    if not os.path.exists(adv_combo_dir):
                        print(f"  [WARNING] Directory not found: {adv_combo_dir}. Skipping.")
                        current_combo += 1
                        continue

                    # ------------------------------------------------------------------
                    # Build eval list from reference_files — fixed denominator of 100.
                    #
                    # For each reference image:
                    #   - If the adversarial file exists in adv_combo_dir: evaluate
                    #     NN2 on it and check against the target class.
                    #   - If the file is missing (attack exceeded budget during
                    #     generation): the attack failed — count as non-successful.
                    #     The image is not loaded at all; ASR is not incremented.
                    #
                    # For Fixed mode: target is FIXED_TARGET_CLASS for all images.
                    # For LeastLikely mode: target is read from target_map using
                    #   key format "class_id/class_id.jpg" (with extension),
                    #   matching the format used by save_least_likely_targets.
                    # ------------------------------------------------------------------
                    eval_imgs    = []
                    eval_targets = []
                    eval_is_adv  = []
                    skipped      = 0

                    for img_name in reference_files:
                        true_id  = os.path.splitext(img_name)[0].strip()
                        adv_path = os.path.join(adv_combo_dir, img_name)

                        if mode == "Fixed":
                            target = FIXED_TARGET_CLASS
                        else:
                            key    = f"{true_id}/{img_name}"
                            target = target_map.get(key, None)
                            if target is None:
                                skipped += 1
                                continue

                        if os.path.exists(adv_path):
                            src_path = adv_path
                            is_adv   = True
                        else:
                            src_path = os.path.join(DATA_DIR_100, img_name)
                            is_adv   = False

                        try:
                            img = Image.open(src_path).convert('RGB')
                            eval_imgs.append(preprocess_nn2(img))
                            eval_targets.append(target)
                            eval_is_adv.append(is_adv)
                        except Exception as e:
                            print(f"  Error loading {img_name}: {e}")

                    if skipped > 0:
                        print(f"  [WARNING] Skipped {skipped} images — "
                            f"target not found in target_map.")

                    if not eval_imgs:
                        print(f"  [WARNING] No images to evaluate. Skipping.")
                        current_combo += 1
                        continue

                    # ------------------------------------------------------------------
                    # Run inference in batches on NN2.
                    # Only adversarial images can contribute to successful_nn2.
                    # Clean fallback images (attack failed) never count as successes.
                    # ------------------------------------------------------------------
                    successful_nn2 = 0
                    start_time     = time.time()

                    for b in range(0, len(eval_imgs), BATCH_SIZE):
                        batch_tensor = torch.stack(
                            eval_imgs[b:b + BATCH_SIZE]
                        ).to(DEVICE)
                        batch_tgts   = eval_targets[b:b + BATCH_SIZE]
                        batch_is_adv = eval_is_adv[b:b + BATCH_SIZE]

                        with torch.no_grad():
                            pred_indices = (
                                nn2(batch_tensor).detach().cpu().numpy().argmax(axis=1)
                            )

                        for pred_idx_2, tgt, is_adv in zip(
                            pred_indices, batch_tgts, batch_is_adv
                        ):
                            if is_adv and pred_idx_2 == tgt:
                                successful_nn2 += 1

                    elapsed = time.time() - start_time
                    asr_nn2 = (successful_nn2 / len(reference_files)) * 100

                    nn1_row   = df_nn1[
                        (df_nn1['max_iter']      == max_iter) &
                        (df_nn1['learning_rate'] == lr)       &
                        (df_nn1['confidence']    == conf)
                    ]
                    asr_nn1   = nn1_row['attack_success_rate'].values[0] if len(nn1_row) > 0 else 0.0
                    mean_linf = nn1_row['mean_linf_raw'].values[0]       if len(nn1_row) > 0 else 0.0

                    adv_found = sum(1 for v in eval_is_adv if v)
                    print(f"  Adversarial found: {adv_found}/{len(reference_files)} | "
                        f"Clean fallback: {len(reference_files) - adv_found - skipped}"
                        f"/{len(reference_files)} | "
                        f"NN1 ASR: {asr_nn1:.2f}% | NN2 ASR: {asr_nn2:.2f}% | "
                        f"Mean L-inf: {mean_linf:.4f} | Time: {elapsed:.1f}s")

                    results.append({
                        'attack_name':    ATTACK_NAME,
                        'target_mode':    mode,
                        'max_iter':       max_iter,
                        'learning_rate':  lr,
                        'confidence':     conf,
                        'nn1_asr':        round(asr_nn1, 2),
                        'nn2_asr':        round(asr_nn2, 2),
                        'nn2_successful': successful_nn2,
                        'adv_found':      adv_found,
                        'total_images':   len(reference_files),
                        'mean_linf':      round(mean_linf, 4),
                        'time_elapsed_s': round(elapsed, 2)
                    })
                    all_combo_data.append({
                        'max_iter':  max_iter,
                        'lr':        lr,
                        'conf':      conf,
                        'mean_linf': mean_linf,
                        'nn1_asr':   asr_nn1,
                        'nn2_asr':   asr_nn2
                    })

                    current_combo += 1

        # ----------------------------------------------------------------------
        # Save Results CSV — inside for mode
        # ----------------------------------------------------------------------
        df_results = pd.DataFrame(results)
        csv_path   = os.path.join(TRANSFER_RESULTS_DIR,
                                f'{ATTACK_NAME}_transferability_results.csv')
        df_results.to_csv(csv_path, index=False)
        print(f"\n[INFO] Results CSV saved at: {csv_path}")

        # ----------------------------------------------------------------------
        # Comparative SEC — inside for mode
        # ----------------------------------------------------------------------
        fig, ax = plt.subplots(figsize=(9, 6))

        ax.axhline(y=0.0, color='#2ca02c', linestyle='--',
                linewidth=1.5, label='Baseline ASR (~0%)')
        ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2,
                label=r'Max Allowable L-inf (0.1)')

        for max_iter in MAX_ITER_LIST:
            combos = [c for c in all_combo_data if c['max_iter'] == max_iter]
            if not combos:
                continue
            combos_sorted = sorted(combos, key=lambda c: c['mean_linf'])
            lif_vals      = [c['mean_linf'] for c in combos_sorted]
            nn1_vals      = [c['nn1_asr']   for c in combos_sorted]
            nn2_vals      = [c['nn2_asr']   for c in combos_sorted]
            c_nn1 = ITER_COLORS_NN1[max_iter]
            c_nn2 = ITER_COLORS_NN2[max_iter]
            ax.plot(lif_vals, nn1_vals,
                    marker='o', linestyle='-', color=c_nn1,
                    linewidth=1.5, markersize=5,
                    label=f'NN1 ASR max_iter={max_iter}')
            ax.plot(lif_vals, nn2_vals,
                    marker='s', linestyle='--', color=c_nn2,
                    linewidth=1.5, markersize=5,
                    label=f'NN2 ASR max_iter={max_iter}')

        ax.set_title(f'Transferability SEC — {ATTACK_NAME}\n'
                    f'(X-axis: mean perturbation applied, not fixed budget)',
                    fontsize=16, fontweight='bold', pad=15)
        ax.set_xlabel(r'Mean L-inf Perturbation Actually Applied (not fixed $\epsilon$)',
                    fontsize=14)
        ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
        ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
        ax.set_xticks(np.arange(0.0, 0.11, 0.01))
        ax.set_yticks(np.arange(0, 101, 10))
        ax.tick_params(axis='both', labelsize=12)
        ax.set_ylim(-5, 105)
        ax.set_xlim(-0.005, 0.105)
        ax.legend(loc='upper left', fontsize=9)
        plt.tight_layout()

        sec_path = os.path.join(TRANSFER_RESULTS_DIR,
                                f'{ATTACK_NAME}_transferability_SEC.png')
        plt.savefig(sec_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\n[INFO] Transferability SEC saved at: {sec_path}")

        # ----------------------------------------------------------------------
        # Hyperparameter Effect — ASR vs learning_rate — inside for mode
        # ----------------------------------------------------------------------
        FIXED_CONF_FOR_EFFECT = 0.0

        fig, ax = plt.subplots(figsize=(9, 6))
        ax.axhline(y=0.0, color='#2ca02c', linestyle='--',
                linewidth=1.5, label='Baseline ASR (~0%)')

        for max_iter in MAX_ITER_LIST:
            c_nn1  = ITER_COLORS_NN1[max_iter]
            c_nn2  = ITER_COLORS_NN2[max_iter]
            subset = df_results[
                (df_results['max_iter']   == max_iter) &
                (df_results['confidence'] == FIXED_CONF_FOR_EFFECT)
            ].sort_values('learning_rate')
            ax.plot(subset['learning_rate'].tolist(), subset['nn1_asr'].tolist(),
                    marker='o', linestyle='-', color=c_nn1,
                    linewidth=1.5, markersize=6, label=f'NN1 max_iter={max_iter}')
            ax.plot(subset['learning_rate'].tolist(), subset['nn2_asr'].tolist(),
                    marker='s', linestyle='--', color=c_nn2,
                    linewidth=1.5, markersize=6, label=f'NN2 max_iter={max_iter}')

        ax.set_title(f'Effect of learning_rate on ASR — {ATTACK_NAME} Transferability\n'
                    f'(confidence={FIXED_CONF_FOR_EFFECT})',
                    fontsize=16, fontweight='bold', pad=15)
        ax.set_xlabel('Learning Rate', fontsize=14)
        ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
        ax.set_xscale('log')
        ax.set_xticks(LEARNING_RATE_LIST)
        ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
        ax.set_yticks(np.arange(0, 101, 10))
        ax.tick_params(axis='both', labelsize=12)
        ax.set_ylim(-5, 105)
        ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
        ax.legend(loc='upper left', fontsize=9)
        plt.tight_layout()

        lr_path = os.path.join(TRANSFER_RESULTS_DIR,
                            f'{ATTACK_NAME}_transferability_lr_effect.png')
        plt.savefig(lr_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\n[INFO] Learning rate effect plot saved at: {lr_path}")

        # ----------------------------------------------------------------------
        # Hyperparameter Effect — ASR vs confidence — inside for mode
        # ----------------------------------------------------------------------
        FIXED_LR_FOR_EFFECT = 0.01

        fig, ax = plt.subplots(figsize=(9, 6))
        ax.axhline(y=0.0, color='#2ca02c', linestyle='--',
                linewidth=1.5, label='Baseline ASR (~0%)')

        for max_iter in MAX_ITER_LIST:
            c_nn1  = ITER_COLORS_NN1[max_iter]
            c_nn2  = ITER_COLORS_NN2[max_iter]
            subset = df_results[
                (df_results['max_iter']      == max_iter) &
                (df_results['learning_rate'] == FIXED_LR_FOR_EFFECT)
            ].sort_values('confidence')
            ax.plot(subset['confidence'].tolist(), subset['nn1_asr'].tolist(),
                    marker='o', linestyle='-', color=c_nn1,
                    linewidth=1.5, markersize=6, label=f'NN1 max_iter={max_iter}')
            ax.plot(subset['confidence'].tolist(), subset['nn2_asr'].tolist(),
                    marker='s', linestyle='--', color=c_nn2,
                    linewidth=1.5, markersize=6, label=f'NN2 max_iter={max_iter}')

        ax.set_title(f'Effect of confidence on ASR — {ATTACK_NAME} Transferability\n'
                    f'(learning_rate={FIXED_LR_FOR_EFFECT})',
                    fontsize=16, fontweight='bold', pad=15)
        ax.set_xlabel('Confidence', fontsize=14)
        ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
        ax.set_xticks(CONFIDENCE_LIST)
        ax.set_yticks(np.arange(0, 101, 10))
        ax.tick_params(axis='both', labelsize=12)
        ax.set_ylim(-5, 105)
        ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
        ax.legend(loc='upper left', fontsize=9)
        plt.tight_layout()

        conf_path = os.path.join(TRANSFER_RESULTS_DIR,
                                f'{ATTACK_NAME}_transferability_conf_effect.png')
        plt.savefig(conf_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\n[INFO] Confidence effect plot saved at: {conf_path}")

        # ----------------------------------------------------------------------
        # Summary — inside for mode
        # ----------------------------------------------------------------------
        print("\n" + "=" * 65)
        print(f"CW TARGETED {mode.upper()} TRANSFERABILITY SUMMARY")
        print("=" * 65)
        print(f"{'max_iter':<10} {'lr':<8} {'conf':<8} "
            f"{'NN1 ASR':>10} {'NN2 ASR':>10} {'Gap':>10} {'L-inf':>10}")
        print("-" * 65)
        for r in results:
            print(f"  {r['max_iter']:<8} {r['learning_rate']:<8} "
                f"{r['confidence']:<8} "
                f"{r['nn1_asr']:>10.2f}% "
                f"{r['nn2_asr']:>10.2f}% "
                f"{(r['nn1_asr'] - r['nn2_asr']):>+9.2f}% "
                f"{r['mean_linf']:>10.4f}")
        print("=" * 65 + "\n")

### Error Specific Transferability Diagnostic

In [ ]:
# Quick check — BIM Targeted LeastLikely iter20, eps=0.1
import datetime

csv_path = os.path.join(RESULTS_DIR, "BIM_Targeted_LeastLikely",
                        "BIM_Targeted_LeastLikely_results.csv")
df = pd.read_csv(csv_path)
row = df[(df['max_iter'] == 20) & (df['epsilon'] == 0.1)]
print(f"BIM LeastLikely | max_iter=20 | eps=0.1")
print(f"ASR: {row['attack_success_rate'].values[0]:.2f}%")
print(f"Successful attacks: {row['successful_attacks'].values[0]}/{row['total_images'].values[0]}")

# Check file freshness for BIM iter20
adv_dir = os.path.join(ADVERSARIAL_DATA_DIR,
                       "BIM_Targeted_LeastLikely_iter20",
                       "eps_0_100")
all_files = []
for class_id in os.listdir(adv_dir):
    cf = os.path.join(adv_dir, class_id)
    if os.path.isdir(cf):
        for f in os.listdir(cf):
            all_files.append(os.path.join(cf, f))

mtimes = [os.path.getmtime(f) for f in all_files]
print(f"\nFiles in BIM_iter20 eps=0.1: {len(all_files)}")
print(f"Newest: {datetime.datetime.fromtimestamp(max(mtimes)).strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Oldest: {datetime.datetime.fromtimestamp(min(mtimes)).strftime('%Y-%m-%d %H:%M:%S')}")

# Also check first few images directly
loaded     = load_least_likely_targets("BIM_Targeted_LeastLikely")
target_map = loaded if isinstance(loaded, dict) else {}

nn1_hits = 0
total    = 0
for class_id in sorted(os.listdir(adv_dir))[:3]:
    cf = os.path.join(adv_dir, class_id)
    if not os.path.isdir(cf):
        continue
    for img_name in sorted(os.listdir(cf))[:3]:
        if not img_name.lower().endswith('.png'):
            continue
        img_stem = os.path.splitext(img_name)[0]
        key      = f"{class_id}/{img_stem}"
        target   = target_map.get(key, -1)
        if target == -1:
            print(f"  [MISSING] {key}")
            continue
        img  = Image.open(os.path.join(cf, img_name)).convert('RGB')
        t1   = preprocess_attack(img).numpy()[np.newaxis]
        pred = int(np.argmax(art_classifier.predict(t1), axis=1)[0])
        hit  = (pred == target)
        nn1_hits += int(hit)
        total    += 1
        raw = LABELS[target]
        if isinstance(raw, np.ndarray): raw = raw.flatten()[0]
        if isinstance(raw, bytes):      raw = raw.decode('utf-8')
        tgt_name = str(raw).replace("b'","").replace("'","").strip()
        print(f"  {img_name} | key={key} | target={tgt_name} | "
              f"pred={LABELS[pred]} | hit={hit}")

print(f"\nNN1 hit rate on BIM iter20: {nn1_hits}/{total}")

# 6 DEFENSES

In the defense phase two defense mechanism will be implemented and evaluated for NN1:
- Detector
- Jpeg compression

## 6.0 DATASET PREPARATION

The defense phase of the model requires a dataset that combines both genuine and adversarial images. This section prepares that dataset, ensuring it is balanced and properly split into training, validation, and test sets.


### 6.0.1 Defense Dataset Preparation

This cell builds the dataset used in the defense phase of the model by combining adversarial and genuine images.

In particular:
- it copies a subset of adversarial images into the output folder;
- it selects genuine images from the original dataset only for the class IDs listed in `selected_subjects`;
- it balances the dataset with additional genuine images taken from the external pool;
- it saves everything under `detector_dataset/` and prints a final summary of the copied samples.


In [ ]:
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(DEFENSE_SPLITS_DIR, exist_ok=True)
print(f"[INFO] Attack configurations defined: {len(ATTACK_CONFIGS)}")
print(f"       Max adversarial images: {sum(c['n_samples'] for c in ATTACK_CONFIGS)}")

genuine_out_dir     = os.path.join(DEFENSE_DATASET_DIR, 'genuine')
adversarial_out_dir = os.path.join(DEFENSE_DATASET_DIR, 'adversarial')

if os.path.exists(DEFENSE_DATASET_DIR) and os.path.exists(os.path.join(DEFENSE_DATASET_DIR, 'genuine')) and os.path.exists(os.path.join(DEFENSE_DATASET_DIR, 'adversarial')):
    print("[SKIP] detector_dataset already exists locally. Delete the folder to rebuild.")
else:
    genuine_out_dir     = os.path.join(DEFENSE_DATASET_DIR, 'genuine')
    adversarial_out_dir = os.path.join(DEFENSE_DATASET_DIR, 'adversarial')
    os.makedirs(genuine_out_dir,     exist_ok=True)
    os.makedirs(adversarial_out_dir, exist_ok=True)

    # --------------------------------------------------------------------------
    # Helper: collect image paths from a source directory
    # --------------------------------------------------------------------------
    def collect_images(directory: str, structure: str) -> list:
        """
        Returns a sorted list of absolute paths to all images found in the
        given directory.
        For 'class_subfolders' structure, descends one level into class_id
        subdirectories (FGSM/BIM/PGD layout). For 'flat' structure, lists
        only the top-level files (CW/DeepFool layout).
        """
        paths = []
        if structure == 'class_subfolders':
            for class_dir in sorted(os.listdir(directory)):
                class_path = os.path.join(directory, class_dir)
                if os.path.isdir(class_path):
                    for fname in sorted(os.listdir(class_path)):
                        if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                            paths.append(os.path.join(class_path, fname))
        elif structure == 'flat':
            for fname in sorted(os.listdir(directory)):
                if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                    paths.append(os.path.join(directory, fname))
        return paths

    random.seed(SEED)

    # ==========================================================================
    # PART A — Adversarial images
    # ==========================================================================
    print("=" * 65)
    print(" PART A — ADVERSARIAL IMAGES")
    print("=" * 65)

    total_adversarial = 0

    for cfg in ATTACK_CONFIGS:
        folder    = cfg['folder']
        subfolder = cfg['subfolder']
        structure = cfg['structure']
        n_samples = cfg['n_samples']
        src_path = (
            os.path.join(ADVERSARIAL_DATA_DIR, folder, subfolder)
            if subfolder is not None
            else os.path.join(ADVERSARIAL_DATA_DIR, folder)
        )
        if not os.path.exists(src_path):
            raise FileNotFoundError(
                f"[ERROR] Configuration not found on disk: {folder}/{subfolder}\n"
                f"        Expected path: {src_path}"
            )
        all_imgs = collect_images(src_path, structure)
        sampled  = random.sample(all_imgs, min(n_samples, len(all_imgs)))
        safe_folder    = folder.replace('/', '_')
        safe_subfolder = subfolder.replace('/', '_') if subfolder else 'root'
        prefix         = f"{safe_folder}__{safe_subfolder}__"
        copied = 0
        for src_img in sampled:
            fname = os.path.basename(src_img)
            if structure == 'class_subfolders':
                parent_dir = os.path.basename(os.path.dirname(src_img))
                dst_name   = f"{prefix}{parent_dir}__{fname}"
            else:
                dst_name = f"{prefix}{fname}"
            shutil.copy2(src_img, os.path.join(adversarial_out_dir, dst_name))
            copied += 1
            
        total_adversarial += copied  
        print(f"   [OK] {(folder + '/' + str(subfolder)):<55} "
            f"available: {len(all_imgs):>4} | sampled: {copied:>4}")

    print(f"\n   Total adversarial : {total_adversarial}")

    # ==========================================================================
    # PART B — Genuine images from dataset_100_subjects (original 1000)
    # ==========================================================================
    print("\n" + "=" * 65)
    print(" PART B — GENUINE IMAGES (dataset_100_subjects)")
    print("=" * 65)
    valid_class_ids = set(selected_subjects['Class_ID'].astype(str).str.strip())
    print(f"   Valid class IDs from selected_subjects.csv: {len(valid_class_ids)}")
    print(valid_class_ids)
    genuine_pool_orig = []
    for class_dir in sorted(os.listdir(DATA_DIR_1000)):
        if class_dir.strip() not in valid_class_ids:
            continue
        class_path = os.path.join(DATA_DIR_1000, class_dir)
        if os.path.isdir(class_path):
            for fname in sorted(os.listdir(class_path)):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    genuine_pool_orig.append(
                        (class_dir.strip(), os.path.join(class_path, fname))
                    )

    sampled_orig = random.sample(genuine_pool_orig, min(1000, len(genuine_pool_orig)))

    copied_orig = 0
    for class_id, src_img in sampled_orig:
        fname    = os.path.basename(src_img)
        dst_name = f"orig__{class_id}__{fname}"
        shutil.copy2(src_img, os.path.join(genuine_out_dir, dst_name))
        copied_orig += 1

    print(f"   Pool size : {len(genuine_pool_orig)}")
    print(f"   Copied    : {copied_orig}")


    # ==========================================================================
    # PART C — Genuine images from extra pool
    # ==========================================================================
    print("\n" + "=" * 65)
    print(" PART C — GENUINE IMAGES (extra from VGGFace2)")
    print("=" * 65)
    copied_extra = 0
    if not os.path.exists(EXTRA_GENUINE_DIR):
        raise FileNotFoundError(
            f"[ERROR] EXTRA_GENUINE_DIR not found: {EXTRA_GENUINE_DIR}\n"
            f"        Run cell 4 to extract extra genuine images from VGGFace2."
        )
    genuine_pool_extra = []
    for class_dir in sorted(os.listdir(EXTRA_GENUINE_DIR)):
        if class_dir.strip() not in valid_class_ids:
            continue
        class_path = os.path.join(EXTRA_GENUINE_DIR, class_dir)
        if os.path.isdir(class_path):
            for fname in sorted(os.listdir(class_path)):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    genuine_pool_extra.append(
                        (class_dir.strip(), os.path.join(class_path, fname))
                    )
    extra_budget  = max(0, total_adversarial - copied_orig)
    sampled_extra = random.sample(
        genuine_pool_extra,
        min(extra_budget, len(genuine_pool_extra))
    )
    for class_id, src_img in sampled_extra:
        fname    = os.path.basename(src_img)
        dst_name = f"extra__{class_id}__{fname}"
        shutil.copy2(src_img, os.path.join(genuine_out_dir, dst_name))
        copied_extra += 1

    print(f"   Extra pool size    : {len(genuine_pool_extra)}")
    print(f"   Budget (to balance): {extra_budget}")
    print(f"   Copied             : {copied_extra}")


    # ==========================================================================
    # SUMMARY — counted directly from output directories
    # ==========================================================================
    genuine_on_disk     = len([f for f in os.listdir(genuine_out_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
    adversarial_on_disk = len([f for f in os.listdir(adversarial_out_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
    ratio = genuine_on_disk / adversarial_on_disk if adversarial_on_disk > 0 else float('nan')

    print("\n" + "=" * 65)
    print(" DATASET CONSTRUCTION SUMMARY")
    print("=" * 65)
    print(f"   Genuine total         : {genuine_on_disk}")
    print(f"     orig__ prefix       : {len([f for f in os.listdir(genuine_out_dir) if f.startswith('orig__')])}")
    print(f"     extra__ prefix      : {len([f for f in os.listdir(genuine_out_dir) if f.startswith('extra__')])}")
    print(f"   Adversarial total     : {adversarial_on_disk}")
    print(f"   Genuine/Adversarial   : {ratio:.3f}")
    print(f"   Output directory      : {DEFENSE_DATASET_DIR}")
    print("=" * 65)




### 6.0.2 Correctness checks

**Pre-flight diagnostic**: verifies that all required sources are present
and correctly structured before the construction cell is run. The following
checks are performed:

1. **Adversarial configurations**: every entry in `ATTACK_CONFIGS` is
   resolved to a path on disk. A missing path is reported as `FAIL`.
2. **Subject cross-check**: for `class_subfolders` configurations, every
   `class_id` subfolder found on disk is verified against the CSV.
3. **Extra genuine pool**: the extra genuine directory is present and
   contains images from the correct 100 subjects.
4. **Balance estimate**: the expected genuine-to-adversarial ratio is
   computed and reported.

A final `PASS / WARN / FAIL` verdict is printed for each check. The
construction cell must not be run if any `FAIL` is present.

In [ ]:
PASS  = "[PASS]"
WARN  = "[WARN]"
FAIL  = "[FAIL]"

all_verdicts = []

def verdict(tag, message):
    all_verdicts.append(tag)
    print(f"  {tag} {message}")

df_csv = pd.read_csv(CSV_PATH)
df_csv['Class_ID'] = df_csv['Class_ID'].astype(str).str.strip()
csv_ids = set(df_csv['Class_ID'].tolist())


# ==========================================================================
# CHECK 0 — dataset_100_subjects structure
# ==========================================================================
print("\n" + "=" * 65)
print(" CHECK 0 — dataset_100_subjects structure")
print("=" * 65)

if not os.path.exists(DATA_DIR_1000):
    verdict(FAIL, f"DATA_DIR_1000 not found: {DATA_DIR_1000}")
else:
    found_dirs = {
        d.strip()
        for d in os.listdir(DATA_DIR_1000)
        if os.path.isdir(os.path.join(DATA_DIR_1000, d))
    }

    missing_from_disk  = csv_ids - found_dirs
    unexpected_on_disk = found_dirs - csv_ids

    if not missing_from_disk:
        verdict(PASS, "All 100 expected subject folders found on disk.")
    else:
        verdict(FAIL, f"{len(missing_from_disk)} subject(s) in CSV but missing " f"on disk: {sorted(missing_from_disk)[:5]}")

    if not unexpected_on_disk:
        verdict(PASS, "No unexpected subject folders on disk.")
    else:
        verdict(WARN, f"{len(unexpected_on_disk)} folder(s) on disk not in CSV "f"(will be ignored): {sorted(unexpected_on_disk)[:5]}")

    empty_subjects = []
    img_counts     = {}
    for class_id in csv_ids:
        sp = os.path.join(DATA_DIR_1000, class_id)
        if os.path.isdir(sp):
            imgs = [f for f in os.listdir(sp)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            img_counts[class_id] = len(imgs)
            if len(imgs) == 0:
                empty_subjects.append(class_id)

    total_orig = sum(img_counts.values())
    if empty_subjects:
        verdict(WARN, f"{len(empty_subjects)} subject folder(s) are empty: " f"{empty_subjects[:5]}")
    else:
        verdict(PASS, f"No empty subject folders. Total images: {total_orig}")
        
        
# ==========================================================================
# CHECK 1 — ATTACK_CONFIGS check
# ==========================================================================
print("\n" + "=" * 65)
print(" CHECK 1 — Adversarial configurations")
print("=" * 65)

print(f"  {'CONFIG':<60} {'STATUS':<10} {'IMAGES':>8}")
print("  " + "-" * 80)

configs_found       = 0
configs_missing     = 0
total_adv_available = 0
total_adv_budget    = 0

for cfg in ATTACK_CONFIGS:
    folder    = cfg['folder']
    subfolder = cfg['subfolder']
    structure = cfg['structure']
    n_samples = cfg['n_samples']

    src_path = (
        os.path.join(ADVERSARIAL_DATA_DIR, folder, subfolder)
        if subfolder is not None
        else os.path.join(ADVERSARIAL_DATA_DIR, folder)
    )

    config_label = f"{folder}/{subfolder}"

    if not os.path.exists(src_path):
        print(f"  {config_label:<60} {FAIL:<10} {'PATH NOT FOUND':>8}")
        verdict(FAIL, f"Configuration not found on disk: {config_label}")
        configs_missing  += 1
        total_adv_budget += n_samples
        continue

    if structure == 'class_subfolders':
        n_imgs = sum(
            len([f for f in os.listdir(os.path.join(src_path, d))if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
            for d in os.listdir(src_path)
            if os.path.isdir(os.path.join(src_path, d))
        )
    else:
        n_imgs = len([f for f in os.listdir(src_path)if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

    sampled = min(n_samples, n_imgs)
    total_adv_available += sampled
    total_adv_budget    += n_samples
    configs_found       += 1

    tag  = PASS if n_imgs >= n_samples else WARN
    note = f"{sampled}/{n_samples}" if n_imgs < n_samples else f"{sampled}"
    print(f"  {config_label:<60} {tag:<10} {note:>8}")

print(f"\n  Configurations found              : {configs_found}/{len(ATTACK_CONFIGS)}")
print(f"  Adversarial images available      : {total_adv_available}")
print(f"  Adversarial images budget         : {total_adv_budget}")


# ==========================================================================
# CHECK 2 — Subject cross-check for adversarial class_subfolders configs
# ==========================================================================
print("\n" + "=" * 65)
print(" CHECK 2 — Subject cross-check (adversarial vs CSV)")
print("=" * 65)

cross_check_issues = 0
for cfg in ATTACK_CONFIGS:
    if cfg['structure'] != 'class_subfolders':
        continue
    folder    = cfg['folder']
    subfolder = cfg['subfolder']
    src_path  = os.path.join(ADVERSARIAL_DATA_DIR, folder, subfolder)
    if not os.path.exists(src_path):
        continue

    adv_ids    = {
        d.strip()
        for d in os.listdir(src_path)
        if os.path.isdir(os.path.join(src_path, d))
    }
    unexpected = adv_ids - csv_ids
    if unexpected:
        verdict(WARN, f"{folder}/{subfolder}: {len(unexpected)} class_id(s) not " f"in CSV (will be ignored): {sorted(unexpected)[:3]}")
        cross_check_issues += 1

if cross_check_issues == 0:
    verdict(PASS, "All adversarial class_id folders match the CSV.")


# ==========================================================================
# CHECK 3 — Extra genuine pool
# ==========================================================================
print("\n" + "=" * 65)
print(" CHECK 3 — Extra genuine pool")
print("=" * 65)

if not os.path.exists(EXTRA_GENUINE_DIR):
    verdict(FAIL, f"EXTRA_GENUINE_DIR not found: {EXTRA_GENUINE_DIR}. "f"Run cell 4 to extract extra genuine images from VGGFace2.")
    total_extra_available = 0
else:
    extra_ids = {
        d.strip()
        for d in os.listdir(EXTRA_GENUINE_DIR)
        if os.path.isdir(os.path.join(EXTRA_GENUINE_DIR, d))
    }
    unexpected_extra      = extra_ids - csv_ids
    total_extra_available = sum(
        len([f for f in os.listdir(os.path.join(EXTRA_GENUINE_DIR, d)) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        for d in extra_ids & csv_ids
        if os.path.isdir(os.path.join(EXTRA_GENUINE_DIR, d))
    )

    verdict(
        PASS if not unexpected_extra else WARN,
        f"Extra pool: {len(extra_ids & csv_ids)} valid subjects, "
        f"{total_extra_available} images."
        + (f" {len(unexpected_extra)} unexpected subject(s) will be ignored."
            if unexpected_extra else "")
    )


# ==========================================================================
# CHECK 4 — Balance estimate
# ==========================================================================
print("\n" + "=" * 65)
print(" CHECK 4 — Balance estimate")
print("=" * 65)

est_adversarial   = total_adv_available
est_genuine_orig  = min(1000, total_orig) if 'total_orig' in dir() else 0
est_genuine_extra = min(
    max(0, est_adversarial - est_genuine_orig),
    total_extra_available
)
est_genuine_total = est_genuine_orig + est_genuine_extra
est_ratio         = est_genuine_total / est_adversarial if est_adversarial > 0 else float('nan')

print(f"  Estimated adversarial      : {est_adversarial}")
print(f"  Estimated genuine (orig)   : {est_genuine_orig}")
print(f"  Estimated genuine (extra)  : {est_genuine_extra}")
print(f"  Estimated genuine total    : {est_genuine_total}")
print(f"  Estimated ratio G/A        : {est_ratio:.3f}")

if 0.9 <= est_ratio <= 1.1:
    verdict(PASS, f"Dataset will be well balanced (ratio={est_ratio:.3f}).")
elif 0.7 <= est_ratio <= 1.3:
    verdict(WARN, f"Dataset slightly unbalanced (ratio={est_ratio:.3f}). "f"Consider adjusting n_samples or N_EXTRA_PER_SUBJECT.")
else:
    verdict(WARN, f"Dataset significantly unbalanced (ratio={est_ratio:.3f}). "f"Adjust configuration before proceeding.")


# ==========================================================================
# FINAL VERDICT
# ==========================================================================
print("\n" + "=" * 65)
print(" FINAL VERDICT")
print("=" * 65)

n_fail = all_verdicts.count(FAIL)
n_warn = all_verdicts.count(WARN)
n_pass = all_verdicts.count(PASS)

print(f"  PASS : {n_pass}")
print(f"  WARN : {n_warn}")
print(f"  FAIL : {n_fail}")

if n_fail > 0:
    print("\n  >> STOP: fix FAIL issues before running the construction cell.")
elif n_warn > 0:
    print("\n  >> CAUTION: review WARN issues, then proceed if acceptable.")
else:
    print("\n  >> All checks passed. Safe to proceed with dataset construction.")


### 6.0.3 — Dataset Split and DataLoader Construction

This cell loads the detector dataset from disk, performs a subject-based
(identity-disjoint) split into train, validation, and test sets, and
constructs the corresponding DataLoaders.

The split is performed on the 100 subject identities rather than on
individual images, ensuring that no subject present in the training set
appears in the validation or test sets. This prevents identity leakage
and guarantees that the detector is evaluated on genuinely unseen subjects.

The dataset folder structure is:

detector_dataset/

├── genuine/


│   └── {prefix}{class_id}{filename}.ext


└── adversarial/


└── {attack}{subfolder}{class_id}__{filename}.ext

Class labels: genuine = 0, adversarial = 1.
Split: 70 subjects train / 20 subjects val / 10 subjects test.


In [ ]:
# ==============================================================================
# STEP 1 — Load and validate subject list from CSV
# ==============================================================================
df_csv = pd.read_csv(CSV_PATH)
df_csv['Class_ID'] = df_csv['Class_ID'].astype(str).str.strip()
valid_class_ids = sorted(df_csv['Class_ID'].tolist())

if len(valid_class_ids) != 100:
    raise ValueError(f"[ERROR] Expected 100 subjects in CSV, found {len(valid_class_ids)}.")

print(f"[INFO] Loaded {len(valid_class_ids)} subject IDs from CSV.")



# ==============================================================================
# STEP 2 — Scan dataset directory and build per-subject file index
#
# Filename conventions:
#   genuine    : orig__{class_id}__{filename}.ext
#                extra__{class_id}__{filename}.ext
#   adversarial: {attack}__{subfolder}__{class_id}__{filename}.ext
#
# In both cases class_id is always the third token when splitting by '__',
# except for genuine files where it is the second token. We extract it
# by checking the prefix.
# ==============================================================================

genuine_dir     = os.path.join(DEFENSE_DATASET_DIR, 'genuine')
adversarial_dir = os.path.join(DEFENSE_DATASET_DIR, 'adversarial')

for d in [genuine_dir, adversarial_dir]:
    if not os.path.exists(d):
        raise FileNotFoundError(f"[ERROR] Directory not found: {d}")

def extract_class_id_genuine(filename: str) -> str:
    """
    Extracts the class_id from a genuine image filename.
    Expected formats:
        orig__{class_id}__{filename}.ext
        extra__{class_id}__{filename}.ext
    """
    parts = filename.split('__')
    if len(parts) < 3:
        raise ValueError(f"Unexpected genuine filename format: {filename}")
    return parts[1]


def extract_class_id_adversarial(filename: str) -> str:
    """
    Extracts the class_id from an adversarial image filename.

    Two formats are supported:

    class_subfolders layout (FGSM, BIM, PGD):
        {attack}__{subfolder}__{class_id}__{filename}.ext
        → class_id is parts[2], filename is parts[3]

    flat layout (CW, DeepFool):
        {attack}__{subfolder}__{class_id}.ext
        → class_id is the stem of parts[2] (no parts[3] exists)
    """
    parts = filename.split('__')
    if len(parts) < 3:
        raise ValueError(f"Unexpected adversarial filename format: {filename}")
    if len(parts) >= 4:
        # class_subfolders layout: class_id is a standalone token
        return parts[2]
    else:
        # flat layout: parts[2] is '{class_id}.ext', strip the extension
        return os.path.splitext(parts[2])[0]

# subject_files[class_id] = list of (abs_path, label) tuples
subject_files = {cid: [] for cid in valid_class_ids}

skipped_genuine     = 0
skipped_adversarial = 0
skipped_genuine_list     = []
skipped_adversarial_list = []

# Process genuine images
for fname in os.listdir(genuine_dir):
    if not fname.lower().endswith(('.png', '.jpg', '.jpeg')):
        continue
    try:
        cid = extract_class_id_genuine(fname)
    except ValueError as e:
        skipped_genuine += 1
        skipped_genuine_list.append((fname, f"Format error: {e}"))
        continue
        
    if cid not in subject_files:
        skipped_genuine += 1
        skipped_genuine_list.append((fname, f"Class ID '{cid}' not in valid_class_ids"))
        continue
        
    subject_files[cid].append((os.path.join(genuine_dir, fname), 0))

# Process adversarial images
for fname in os.listdir(adversarial_dir):
    if not fname.lower().endswith(('.png', '.jpg', '.jpeg')):
        continue
    try:
        cid = extract_class_id_adversarial(fname)
    except ValueError as e:
        skipped_adversarial += 1
        skipped_adversarial_list.append((fname, f"Format error: {e}"))
        continue
        
    if cid not in subject_files:
        skipped_adversarial += 1
        skipped_adversarial_list.append((fname, f"Class ID '{cid}' not in valid_class_ids"))
        continue
        
    subject_files[cid].append((os.path.join(adversarial_dir, fname), 1))

# ==============================================================================
# Print summary and skipped file details
# ==============================================================================
total_images = sum(len(v) for v in subject_files.values())
empty_subjects = [cid for cid, files in subject_files.items() if len(files) == 0]

print(f"[INFO] Total images indexed : {total_images}")
print(f"[INFO] Skipped genuine      : {skipped_genuine}")
print(f"[INFO] Skipped adversarial  : {skipped_adversarial}")

# Print the specific adversarial images that were skipped
if skipped_adversarial_list:
    print("\n[WARN] Detailed list of skipped ADVERSARIAL images:")
    for fname, reason in skipped_adversarial_list:
        print(f"       -> {fname}")
        print(f"          Reason: {reason}")

# Optional: Print the specific genuine images that were skipped
if skipped_genuine_list:
    print("\n[WARN] Detailed list of skipped GENUINE images:")
    for fname, reason in skipped_genuine_list:
        print(f"       -> {fname}")
        print(f"          Reason: {reason}")

if empty_subjects:
    print(f"\n[WARN] {len(empty_subjects)} subject(s) have no images: {empty_subjects[:5]}")
else:
    print(f"\n[INFO] All 100 subjects have at least one image.")

# ==============================================================================
# STEP 3 — Subject-based split (identity-disjoint)
# ==============================================================================
rng = random.Random(SEED)
shuffled_ids = valid_class_ids.copy()
rng.shuffle(shuffled_ids)

n_train = int(len(shuffled_ids) * TRAIN_RATIO)        # 70
n_val   = int(len(shuffled_ids) * VAL_RATIO)          # 20
# remaining subjects go to test                       # 10

train_ids = set(shuffled_ids[:n_train])
val_ids   = set(shuffled_ids[n_train:n_train + n_val])
test_ids  = set(shuffled_ids[n_train + n_val:])

print(f"\n[INFO] Subject split:")
print(f"       Train : {len(train_ids)} subjects")
print(f"       Val   : {len(val_ids)} subjects")
print(f"       Test  : {len(test_ids)} subjects")

# Flatten into (path, label) lists per split
def collect_split(id_set):
    files = []
    for cid in id_set:
        files.extend(subject_files[cid])
    return files

train_files = collect_split(train_ids)
val_files   = collect_split(val_ids)
test_files  = collect_split(test_ids)

# Shuffle within each split for good measure
rng.shuffle(train_files)
rng.shuffle(val_files)
rng.shuffle(test_files)

for split_name, split_files in [('Train', train_files), ('Val',   val_files), ('Test',  test_files)]:
    n_genuine     = sum(1 for _, lbl in split_files if lbl == 0)
    n_adversarial = sum(1 for _, lbl in split_files if lbl == 1)
    print(f"       {split_name:<6}: {len(split_files):>5} images " f"(genuine: {n_genuine}, adversarial: {n_adversarial})")

# ==============================================================================
# STEP 4 — Save split index to CSV for reproducibility
# ==============================================================================
for split_name, split_files in [('train', train_files), ('val',   val_files), ('test',  test_files)]:
    df_split = pd.DataFrame(split_files, columns=['path', 'label'])
    df_split['class_id'] = df_split['path'].apply(
        lambda p: extract_class_id_genuine(os.path.basename(p))
        if os.path.basename(p).startswith(('orig__', 'extra__'))
        else extract_class_id_adversarial(os.path.basename(p))
    )
    csv_out = os.path.join(DEFENSE_SPLITS_DIR, f'detector_{split_name}_split.csv')
    df_split.to_csv(csv_out, index=False)
    print(f"[INFO] Split index saved: {csv_out}")

# ==============================================================================
# STEP 5 — Dataset class
# ==============================================================================
class DetectorDataset(Dataset):
    """
    PyTorch Dataset for the adversarial detector.

    Loads images from disk as float tensors in [0, 1] at 160x160
    resolution. Images are resized to 160x160 if not already at that
    size. Corrupted or unreadable images are replaced with a zero
    tensor and logged to stderr to avoid silent failures.

    Args:
        file_list (list): List of (abs_path, label) tuples.
        augment (bool): If True, applies random horizontal flip.
                        Should be False for all splits per convention.
    """
    def __init__(self, file_list: list, augment: bool = False):
        self.file_list = file_list
        self.augment   = augment

        base_transforms = [
            transforms.Resize((160, 160)),
            transforms.ToTensor()
        ]
        if augment:
            base_transforms = [
                transforms.RandomHorizontalFlip(p=0.5)
            ] + base_transforms

        self.transform = transforms.Compose(base_transforms)

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        path, label = self.file_list[idx]
        try:
            img = Image.open(path).convert('RGB')
            tensor = self.transform(img)
        except Exception as e:
            print(f"\n[ERROR] Failed to load image: {path} — {e}")
            tensor = torch.zeros(3, 160, 160)
        return tensor, torch.tensor(label, dtype=torch.long)


from torch.utils.data import WeightedRandomSampler

def build_weighted_sampler(file_list: list, boost_families: list, boost_factor: float = 3.0) -> WeightedRandomSampler:
    """
    Builds a WeightedRandomSampler that oversamples adversarial images
    belonging to the specified attack families by boost_factor.

    Args:
        file_list    : List of (path, label) tuples.
        boost_families: Attack family names to oversample (e.g. ['CW', 'DeepFool']).
        boost_factor : Sampling weight multiplier for boosted families.

    Returns:
        WeightedRandomSampler instance.
    """
    weights = []
    for path, label in file_list:
        fname  = os.path.basename(path)
        weight = 1.0

        if label == 1:  # adversarial only
            raw = fname.split('__')[0]
            for suffix in ['_Targeted_Fixed', '_Targeted_LeastLikely', '_Untargeted', '_Targeted']:
                raw = raw.replace(suffix, '')
            raw    = re.sub(r'_nri\d+',  '', raw)
            raw    = re.sub(r'_iter\d+', '', raw)
            raw    = re.sub(r'_\d+$',    '', raw)
            family = raw

            if family in boost_families:
                weight = boost_factor

        weights.append(weight)

    weights = torch.DoubleTensor(weights)
    return WeightedRandomSampler( weights, num_samples=len(weights), replacement=True )
    
    
# ==============================================================================
# STEP 6 — DataLoaders
# ==============================================================================
sampler = build_weighted_sampler(train_files, boost_families=['CW', 'DeepFool'], boost_factor=3.0)
train_dataset = DetectorDataset(train_files, augment=False)
val_dataset   = DetectorDataset(val_files,   augment=False)
test_dataset  = DetectorDataset(test_files,  augment=False)

simple_train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE_ADV_DETECTOR, shuffle=False,  num_workers=0, pin_memory=True)
weighted_train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE_ADV_DETECTOR, sampler=sampler, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE_ADV_DETECTOR, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE_ADV_DETECTOR, shuffle=False, num_workers=0, pin_memory=True)

if CLASS_WEIGHTED_LOSS:
    train_loader = weighted_train_loader
else:
    train_loader = simple_train_loader
    
print(f"\n[INFO] DataLoaders ready.")
print(f"       Train batches : {len(train_loader)}")
print(f"       Val batches   : {len(val_loader)}")
print(f"       Test batches  : {len(test_loader)}")


## 6.1 DETECTOR

### Step 6.1.1 — Training Loop

This cell implements the two-phase fine-tuning pipeline for the
AdversarialDetector. Two training experiments are executed sequentially
and their results are compared:

- **Experiment A — Full fine-tuning** (`freeze_mode='none'`): all backbone
  parameters are trainable from the start. A single learning rate is used
  throughout.

- **Experiment B — Two-phase fine-tuning** (`freeze_mode='head_only'` →
  `freeze_mode='partial'`): Phase 1 trains only the classification head
  for a few epochs to warm it up before touching the backbone. Phase 2
  unfreezes the last `N_UNFREEZE_BLOCKS` InvertedResidual blocks and
  continues training with a reduced learning rate.

Both experiments use the same DataLoaders, loss function, and random seed.
Early stopping patience is tuned independently per phase.

### Metrics tracked per epoch
- Training loss
- Validation loss
- Validation accuracy
- Validation precision, recall, F1
- Validation AUROC


#### Helpers

In [72]:
# ==============================================================================
# HELPERS
# ==============================================================================

def evaluate(model, loader, criterion, device):
    """
    Evaluates the model on a DataLoader.

    Returns a dictionary with loss, accuracy, precision, recall, F1
    and AUROC computed over the full loader.
    """
    model.eval()
    total_loss = 0.0
    all_labels = []
    all_preds  = []
    all_scores = []

    with torch.no_grad():
        for x, y in loader:
            x, y    = x.to(device), y.to(device)
            logits   = model(x)
            loss     = criterion(logits, y)
            total_loss += loss.item()

            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            preds = np.argmax(
                torch.softmax(logits, dim=1).cpu().numpy(), axis=1
            )
            all_labels.extend(y.cpu().numpy())
            all_preds.extend(preds)
            all_scores.extend(probs)

    all_labels = np.array(all_labels)
    all_preds  = np.array(all_preds)
    all_scores = np.array(all_scores)

    avg_loss  = total_loss / len(loader)
    accuracy  = (all_preds == all_labels).mean() * 100
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall    = recall_score(all_labels, all_preds, zero_division=0)
    f1        = f1_score(all_labels, all_preds, zero_division=0)
    try:
        auroc = roc_auc_score(all_labels, all_scores)
    except ValueError:
        auroc = float('nan')

    return {
        'loss':      avg_loss,
        'accuracy':  accuracy,
        'precision': precision,
        'recall':    recall,
        'f1':        f1,
        'auroc':     auroc
    }


def evaluate_per_attack(model, loader, device, threshold=0.5):
    """
    Evaluates detection performance broken down by attack family and variant.

    The dictionary key is the full config prefix (e.g. FGSM_Targeted_Fixed,
    BIM_iter5) before iter/nri stripping, so that _aggregate_per_variant
    can correctly identify the variant from it.
    """
    model.eval()

    config_labels = {}
    config_preds  = {}

    with torch.no_grad():
        for x, y, paths in loader:
            x      = x.to(device)
            logits = model(x)
            probs  = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            preds  = (probs >= threshold).astype(int)

            for i, path in enumerate(paths):
                fname  = os.path.basename(path)
                label  = int(y[i].item())
                pred   = int(preds[i])

                if fname.startswith('orig__') or fname.startswith('extra__'):
                    config_key = 'genuine'
                else:
                    # Preserve variant info in the key — strip only
                    # iter/nri numeric suffixes, NOT variant suffixes
                    raw = fname.split('__')[0]
                    raw = re.sub(r'_nri\d+',  '', raw)
                    raw = re.sub(r'_iter\d+', '', raw)
                    raw = re.sub(r'_\d+$',    '', raw)
                    config_key = raw  # e.g. FGSM, FGSM_Targeted_Fixed, BIM, PGD_Targeted_Fixed

                if config_key not in config_labels:
                    config_labels[config_key] = []
                    config_preds[config_key]  = []
                config_labels[config_key].append(label)
                config_preds[config_key].append(pred)

    results = {}
    for config_key in config_labels:
        lbl = np.array(config_labels[config_key])
        prd = np.array(config_preds[config_key])

        if config_key == 'genuine':
            fp_rate = (prd == 1).sum() / len(prd) * 100
            results[config_key] = {
                'total':          len(prd),
                'detection_rate': 100 - fp_rate,
                'precision':      float('nan'),
                'recall':         float('nan'),
                'f1':             float('nan'),
            }
        else:
            adv_mask       = (lbl == 1)
            adv_lbl        = lbl[adv_mask]
            adv_prd        = prd[adv_mask]
            detection_rate = adv_prd.mean() * 100 if len(adv_prd) > 0 else 0.0
            results[config_key] = {
                'total':          int(adv_mask.sum()),
                'detection_rate': detection_rate,
                'precision':      precision_score(adv_lbl, adv_prd, zero_division=0),
                'recall':         recall_score(adv_lbl, adv_prd, zero_division=0),
                'f1':             f1_score(adv_lbl, adv_prd, zero_division=0),
            }
    return results


def run_training(model, optimizer, criterion, train_loader, val_loader,
                 device, n_epochs, patience, experiment_name,
                 checkpoint_path=None):
    """
    Executes a full training run with early stopping.

    Returns best model weights (by minimum validation loss) and the
    history dictionary containing per-epoch train and validation metrics.
    If checkpoint_path is provided, best weights are written to disk
    immediately on every improvement.
    """
    best_weights  = copy.deepcopy(model.state_dict())
    best_val_loss = float('inf')
    best_val_acc  = 0.0
    patience_ctr  = 0

    history = {
        'train_loss':    [], 'train_accuracy':  [],
        'val_loss':      [], 'val_accuracy':    [],
        'val_precision': [], 'val_recall':      [],
        'val_f1':        [], 'val_auroc':        []
    }

    n_train_batches = len(train_loader)
    n_train         = len(train_loader.dataset)
    n_val           = len(val_loader.dataset)

    print(f"\n{'=' * 65}")
    print(f"  {experiment_name}")
    print(f"{'=' * 65}")
    print(f"  Architecture         : {model.__class__.__name__}")
    print(f"  Trainable parameters : {model.count_trainable_params():,}")
    print(f"  Batch size           : {train_loader.batch_size}")
    print(f"  Train samples        : {n_train}")
    print(f"  Val samples          : {n_val}")
    print(f"  Train batches/epoch  : {n_train_batches}")
    print(f"  Max epochs           : {n_epochs}")
    print(f"  Early stop patience  : {patience}")
    print(f"  Learning rate        : {optimizer.param_groups[0]['lr']}")
    print(f"  Weight decay         : {optimizer.param_groups[0]['weight_decay']}")
    if checkpoint_path:
        print(f"  Checkpoint path      : {checkpoint_path}")
    print(f"{'=' * 65}\n")

    for epoch in range(1, n_epochs + 1):
        model.train()
        epoch_loss   = 0.0
        epoch_correct = 0
        epoch_total   = 0
        start         = time.time()
        batches_done  = 0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss   = criterion(logits, y)
            loss.backward()
            optimizer.step()

            epoch_loss    += loss.item()
            preds          = torch.argmax(logits, dim=1)
            epoch_correct += (preds == y).sum().item()
            epoch_total   += y.size(0)
            batches_done  += 1

            if batches_done % max(1, n_train_batches // 5) == 0 \
                    or batches_done == n_train_batches:
                avg_so_far = epoch_loss / batches_done
                print(f"  [{experiment_name}] Epoch {epoch:>3}/{n_epochs} | "
                      f"Batch {batches_done:>3}/{n_train_batches} | "
                      f"Train Loss: {avg_so_far:.4f}",
                      end='\r')

        print(' ' * 100, end='\r')

        train_loss = epoch_loss   / n_train_batches
        train_acc  = epoch_correct / epoch_total * 100
        val_m      = evaluate(model, val_loader, criterion, device)
        elapsed    = time.time() - start

        history['train_loss'].append(train_loss)
        history['train_accuracy'].append(train_acc)
        history['val_loss'].append(val_m['loss'])
        history['val_accuracy'].append(val_m['accuracy'])
        history['val_precision'].append(val_m['precision'])
        history['val_recall'].append(val_m['recall'])
        history['val_f1'].append(val_m['f1'])
        history['val_auroc'].append(val_m['auroc'])

        is_best     = val_m['loss'] < best_val_loss
        best_marker = ' ★ NEW BEST' if is_best else ''

        print(f"  [{experiment_name}] Epoch {epoch:>3}/{n_epochs} | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_m['loss']:.4f} | Val Acc: {val_m['accuracy']:.2f}% | "
              f"F1: {val_m['f1']:.4f} | AUROC: {val_m['auroc']:.4f} | "
              f"Time: {elapsed:.1f}s{best_marker}")

        if is_best:
            best_val_loss = val_m['loss']
            best_val_acc  = val_m['accuracy']
            best_weights  = copy.deepcopy(model.state_dict())
            patience_ctr  = 0
            if checkpoint_path is not None:
                torch.save(best_weights, checkpoint_path)
                print(f"  [{experiment_name}] Checkpoint saved "
                      f"(val_loss={best_val_loss:.4f})")
        else:
            patience_ctr += 1
            print(f"  [{experiment_name}] No improvement "
                  f"{patience_ctr}/{patience} | "
                  f"Best — Val Loss: {best_val_loss:.4f} | "
                  f"Val Acc: {best_val_acc:.2f}%")
            if patience_ctr >= patience:
                print(f"\n  [{experiment_name}] Early stopping at epoch {epoch}.")
                break

    print(f"\n  [{experiment_name}] Training complete.")
    print(f"  [{experiment_name}] Best — Val Loss: {best_val_loss:.4f} | "
          f"Val Acc: {best_val_acc:.2f}%")
    return best_weights, history


def plot_experiment(history, title, save_path,
                    n_train, n_val, batch_size, lr):
    """
    Produces a 2x3 grid of plots for a single experiment.
    Each subplot shows both train and validation curves where applicable.
    Dataset and training details are embedded in the figure title.
    """
    epochs = range(1, len(history['train_loss']) + 1)
    info   = (f"Train: {n_train} | Val: {n_val} | "
              f"Batch: {batch_size} | LR: {lr} | "
              f"Epochs run: {len(epochs)}")

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle(f"{title}\n{info}", fontsize=13, fontweight='bold')

    # Loss
    axes[0, 0].plot(epochs, history['train_loss'], label='Train',
                    color='#1f77b4', linewidth=1.5)
    axes[0, 0].plot(epochs, history['val_loss'],   label='Val',
                    color='#ff7f0e', linewidth=1.5)
    axes[0, 0].set_title('Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Accuracy
    axes[0, 1].plot(epochs, history['train_accuracy'], label='Train',
                    color='#1f77b4', linewidth=1.5)
    axes[0, 1].plot(epochs, history['val_accuracy'],   label='Val',
                    color='#2ca02c', linewidth=1.5)
    axes[0, 1].set_title('Accuracy (%)')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylim(0, 105)
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # AUROC
    axes[0, 2].plot(epochs, history['val_auroc'], label='Val',
                    color='#9467bd', linewidth=1.5)
    axes[0, 2].set_title('AUROC (Validation)')
    axes[0, 2].set_xlabel('Epoch')
    axes[0, 2].set_ylim(0, 1.05)
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)

    # Precision
    axes[1, 0].plot(epochs, history['val_precision'], label='Val',
                    color='#d62728', linewidth=1.5)
    axes[1, 0].set_title('Precision (Validation)')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylim(0, 1.05)
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # Recall
    axes[1, 1].plot(epochs, history['val_recall'], label='Val',
                    color='#8c564b', linewidth=1.5)
    axes[1, 1].set_title('Recall (Validation)')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylim(0, 1.05)
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    # F1
    axes[1, 2].plot(epochs, history['val_f1'], label='Val',
                    color='#e377c2', linewidth=1.5)
    axes[1, 2].set_title('F1 Score (Validation)')
    axes[1, 2].set_xlabel('Epoch')
    axes[1, 2].set_ylim(0, 1.05)
    axes[1, 2].legend()
    axes[1, 2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"[INFO] Plot saved: {save_path}")


def _aggregate_per_variant(per_attack_results: dict) -> dict:
    """
    Aggregates per-configuration detection rates into per-variant
    (family + mode) mean and std.

    Variant classification rule:
        - If the config key contains 'Untargeted' or 'untargeted',
          it is classified as untargeted.
        - If the config key contains 'Targeted' or 'targeted' (but NOT
          'Untargeted'), it is classified as targeted.
        - Otherwise it is classified as untargeted.
    """
    import re
    variant_data = {}

    for config_key, metrics in per_attack_results.items():
        if config_key == 'genuine':
            continue

        # Determine variant — check Untargeted BEFORE Targeted to avoid
        # 'Targeted' matching as substring of 'Untargeted'
        if 'Untargeted' in config_key or 'untargeted' in config_key:
            variant = 'untargeted'
        elif 'Targeted' in config_key or 'targeted' in config_key:
            variant = 'targeted'
        else:
            variant = 'untargeted'

        # Extract family
        family = config_key
        for suffix in [
            '_Targeted_Fixed', '_Targeted_LeastLikely',
            '_Targeted', '_Untargeted'
        ]:
            family = family.replace(suffix, '')

        family = re.sub(r'_nri\d+',  '', family)
        family = re.sub(r'_iter\d+', '', family)
        family = re.sub(r'_\d+$',    '', family)

        if family not in variant_data:
            variant_data[family] = {}
        if variant not in variant_data[family]:
            variant_data[family][variant] = []

        variant_data[family][variant].append(metrics['detection_rate'])

    aggregated = {}
    for family, variants in variant_data.items():
        aggregated[family] = {}
        for variant, rates in variants.items():
            aggregated[family][variant] = {
                'mean': float(np.mean(rates)),
                'std':  float(np.std(rates)),
                'n':    len(rates)
            }

    return aggregated



def plot_per_attack_single(per_attack_results: dict,
                           experiment_name: str,
                           save_path: str,
                           csv_path: str) -> None:
    """
    Produces a grouped bar chart of per-attack detection rate for a
    single experiment. Configurations are aggregated by family and
    variant (untargeted / targeted) showing mean detection rate with
    std error bars. Genuine FPR is not shown in the plot — it is
    reported in the training summary only.
    """
    aggregated = _aggregate_per_variant(per_attack_results)

    FAMILY_ORDER   = ['FGSM', 'BIM', 'PGD', 'CW', 'DeepFool']
    VARIANT_COLORS = {
        'untargeted': '#1f77b4',
        'targeted':   '#ff7f0e',
    }
    VARIANT_LABELS = {
        'untargeted': 'Untargeted',
        'targeted':   'Targeted',
    }

    families = [f for f in FAMILY_ORDER if f in aggregated]
    if not families:
        print(f"[WARN] No adversarial families found for {experiment_name}.")
        return

    all_variants = []
    for f in families:
        for v in aggregated[f]:
            if v not in all_variants:
                all_variants.append(v)

    n_variants = len(all_variants)
    width      = 0.8 / n_variants
    offsets    = np.linspace(
        -(n_variants - 1) / 2 * width,
         (n_variants - 1) / 2 * width,
        n_variants
    )

    fig, ax = plt.subplots(figsize=(max(10, len(families) * 2.5), 6))
    fig.suptitle(
        f'Per-Attack Detection Rate — {experiment_name}\n'
        f'(Validation Set)',
        fontsize=12, fontweight='bold'
    )

    x_pos = np.arange(len(families))

    for v_idx, variant in enumerate(all_variants):
        means = []
        stds  = []
        for family in families:
            if variant in aggregated[family]:
                means.append(aggregated[family][variant]['mean'])
                stds.append(aggregated[family][variant]['std'])
            else:
                means.append(0.0)
                stds.append(0.0)

        color = VARIANT_COLORS.get(variant, '#9467bd')
        label = VARIANT_LABELS.get(variant, variant)
        bars  = ax.bar(
            x_pos + offsets[v_idx], means, width,
            yerr=stds, capsize=3,
            label=label, color=color,
            alpha=0.85, edgecolor='white', linewidth=0.5,
            error_kw={'elinewidth': 1.2, 'ecolor': 'black', 'alpha': 0.6}
        )
        for bar, mean_val, std_val in zip(bars, means, stds):
            if mean_val > 0:
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + std_val * 0.3 + 1.0,
                    f'{mean_val:.1f}',
                    ha='center', va='bottom', fontsize=7.5
                )

    ax.set_xticks(x_pos)
    ax.set_xticklabels(families, fontsize=11)
    ax.set_xlabel('Attack Family', fontsize=12)
    ax.set_ylabel('Detection Rate (%)', fontsize=12)
    ax.set_ylim(0, 120)
    ax.legend(fontsize=9, loc='lower right')
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"[INFO] Per-attack plot saved: {save_path}")

    # Save per-attack CSV (aggregated detection rates)
    rows = []
    for family in families:
        for variant, stats in aggregated[family].items():
            rows.append({
                'family':               family,
                'variant':              variant,
                'n_configs':            stats['n'],
                'detection_rate_mean':  round(stats['mean'], 2),
                'detection_rate_std':   round(stats['std'],  2),
            })
    pd.DataFrame(rows).to_csv(csv_path, index=False)
    print(f"[INFO] Per-attack CSV saved: {csv_path}")
    
    


def plot_comparison(hist_a, hist_b, save_path):
    """
    Produces a 2x2 overlay plot comparing Experiment A (MobileNetV3)
    and Experiment B (EfficientNet-B0) on validation loss, accuracy,
    F1 and AUROC.
    """
    epochs_a = range(1, len(hist_a['val_loss']) + 1)
    epochs_b = range(1, len(hist_b['val_loss']) + 1)

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    fig.suptitle(
        'Experiment Comparison: MobileNetV3-Large vs EfficientNet-B0',
        fontsize=14, fontweight='bold'
    )

    for ax, key, title, ylim in [
        (axes[0, 0], 'val_loss',     'Validation Loss',         None),
        (axes[0, 1], 'val_accuracy', 'Validation Accuracy (%)', (0, 105)),
        (axes[1, 0], 'val_f1',       'Validation F1',           (0, 1.05)),
        (axes[1, 1], 'val_auroc',    'Validation AUROC',        (0, 1.05)),
    ]:
        ax.plot(epochs_a, hist_a[key],
                label='A — MobileNetV3-Large', color='#1f77b4', linewidth=1.5)
        ax.plot(epochs_b, hist_b[key],
                label='B — EfficientNet-B0',   color='#d62728', linewidth=1.5)
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        if ylim:
            ax.set_ylim(ylim)
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"[INFO] Comparison plot saved: {save_path}")




def save_training_history_csv(history: dict, experiment_name: str, save_path: str) -> None:
    """
    Saves the full epoch-level training history to CSV.

    The CSV contains all metrics needed to replicate the training plots
    without re-running training:
        epoch, train_loss, train_accuracy, val_loss, val_accuracy,
        val_precision, val_recall, val_f1, val_auroc, best_found_so_far.

    The best_found_so_far column is True for the epoch at which the
    minimum validation loss was achieved (and all subsequent epochs
    where val_loss equalled that minimum).
    """
    n_epochs   = len(history['train_loss'])
    min_val_loss = min(history['val_loss'])

    rows = []
    for i in range(n_epochs):
        rows.append({
            'experiment':        experiment_name,
            'epoch':             i + 1,
            'train_loss':        round(history['train_loss'][i],    4),
            'train_accuracy':    round(history['train_accuracy'][i], 4),
            'val_loss':          round(history['val_loss'][i],       4),
            'val_accuracy':      round(history['val_accuracy'][i],   4),
            'val_precision':     round(history['val_precision'][i],  4),
            'val_recall':        round(history['val_recall'][i],     4),
            'val_f1':            round(history['val_f1'][i],         4),
            'val_auroc':         round(history['val_auroc'][i],      4),
            'best_found_so_far': history['val_loss'][i] == min_val_loss,
        })

    pd.DataFrame(rows).to_csv(save_path, index=False)
    print(f"[INFO] Training history CSV saved: {save_path}")

# ==============================================================================
# VAL DATALOADER WITH PATHS (needed for per-attack breakdown)
# ==============================================================================

class DetectorDatasetWithPaths(Dataset):
    """
    Variant of DetectorDataset returning (tensor, label, path).
    Used for per-attack validation breakdown.
    """
    def __init__(self, file_list: list):
        self.file_list = file_list
        self.transform = transforms.Compose([
            transforms.Resize((160, 160)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        path, label = self.file_list[idx]
        try:
            img    = Image.open(path).convert('RGB')
            tensor = self.transform(img)
        except Exception as e:
            print(f"\n[ERROR] Failed to load: {path} — {e}")
            tensor = torch.zeros(3, 160, 160)
        return tensor, torch.tensor(label, dtype=torch.long), path


#### Training Loop

In [73]:
path_a       = os.path.join(MODELS_DIR, 'detector_A_mobilenet.pth')
path_b       = os.path.join(MODELS_DIR, 'detector_B_efficientnet.pth')

if RUN_DETECTOR_TRAINING:
    
    if RUN_BOTH_TRAININGS:
        RUN_TRAINING_A = True
        RUN_TRAINING_B = True

    if not RUN_TRAINING_A and not RUN_TRAINING_B:
        raise ValueError("[ERROR] At least one training flag must be True.")

    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    os.makedirs(os.path.join(RESULTS_DIR, 'training_results'), exist_ok=True)
    RESULTS_PATH = os.path.join(RESULTS_DIR, 'training_results')


    val_dataset_with_paths = DetectorDatasetWithPaths(val_files)
    val_loader_with_paths  = torch.utils.data.DataLoader( val_dataset_with_paths, batch_size=BATCH_SIZE_ADV_DETECTOR, shuffle=False, num_workers=0, pin_memory=True)

    # ==============================================================================
    # EXPERIMENT A — MobileNetV3-Large
    # ==============================================================================
    history_a    = None
    per_attack_a = None

    if RUN_TRAINING_A:
        print("=" * 65)
        print(" EXPERIMENT A — MobileNetV3-Large")
        print("=" * 65)

        model_a     = AdversarialDetector(pretrained=True, device=DEVICE)
        criterion_a = nn.CrossEntropyLoss()
        optimizer_a = optim.Adam(
            model_a.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
        )

        best_weights_a, history_a = run_training(
            model=model_a,
            optimizer=optimizer_a,
            criterion=criterion_a,
            train_loader=train_loader,
            val_loader=val_loader,
            device=DEVICE,
            n_epochs=EPOCHS,
            patience=PATIENCE_A,
            experiment_name='Experiment A — MobileNetV3',
            checkpoint_path=path_a
        )

        torch.save(best_weights_a, path_a)
        print(f"[INFO] Experiment A weights saved: {path_a}")

        plot_experiment(
            history_a,
            title='Experiment A — MobileNetV3-Large',
            save_path=os.path.join(RESULTS_PATH, 'detector_A_history.png'),
            n_train=len(train_loader.dataset),
            n_val=len(val_loader.dataset),
            batch_size=train_loader.batch_size,
            lr=LR
        )

        # Per-attack breakdown on validation set
        # Experiment A — dopo plot_experiment
        save_training_history_csv(
            history_a,
            experiment_name='Experiment A — MobileNetV3-Large',
            save_path=os.path.join(RESULTS_PATH, 'detector_A_history.csv')
        )
        model_a.load_state_dict(best_weights_a)
        per_attack_a = evaluate_per_attack(model_a, val_loader_with_paths, DEVICE)
        plot_per_attack_single(
            per_attack_a,
            experiment_name='Experiment A — MobileNetV3-Large',
            save_path=os.path.join(RESULTS_PATH, 'detector_A_per_attack.png'),
            csv_path=os.path.join(RESULTS_PATH,  'detector_A_per_attack.csv')
        )
    else:
        print("[SKIP] Experiment A — RUN_TRAINING_A is False.")


    # ==============================================================================
    # EXPERIMENT B — EfficientNet-B0
    # ==============================================================================
    history_b    = None
    per_attack_b = None

    if RUN_TRAINING_B:
        print("\n" + "=" * 65)
        print(" EXPERIMENT B — EfficientNet-B0")
        print("=" * 65)

        model_b     = EfficientNetDetector(pretrained=True, device=DEVICE)
        criterion_b = nn.CrossEntropyLoss()
        optimizer_b = optim.Adam(
            model_b.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
        )

        best_weights_b, history_b = run_training(
            model=model_b,
            optimizer=optimizer_b,
            criterion=criterion_b,
            train_loader=train_loader,
            val_loader=val_loader,
            device=DEVICE,
            n_epochs=EPOCHS,
            patience=PATIENCE_B,
            experiment_name='Experiment B — EfficientNet',
            checkpoint_path=path_b
        )

        torch.save(best_weights_b, path_b)
        print(f"[INFO] Experiment B weights saved: {path_b}")

        plot_experiment(
            history_b,
            title='Experiment B — EfficientNet-B0',
            save_path=os.path.join(RESULTS_PATH, 'detector_B_history.png'),
            n_train=len(train_loader.dataset),
            n_val=len(val_loader.dataset),
            batch_size=train_loader.batch_size,
            lr=LR
        )

        # Experiment B — dopo plot_experiment
        save_training_history_csv(
            history_b,
            experiment_name='Experiment B — EfficientNet-B0',
            save_path=os.path.join(RESULTS_PATH, 'detector_B_history.csv')
        )

        model_b.load_state_dict(best_weights_b)
        per_attack_b = evaluate_per_attack(model_b, val_loader_with_paths, DEVICE)
        plot_per_attack_single(
            per_attack_b,
            experiment_name='Experiment B — EfficientNet-B0',
            save_path=os.path.join(RESULTS_PATH, 'detector_B_per_attack.png'),
            csv_path=os.path.join(RESULTS_PATH,  'detector_B_per_attack.csv')
        )
    else:
        print("[SKIP] Experiment B — RUN_TRAINING_B is False.")


    # ==============================================================================
    # COMPARATIVE PLOTS
    # ==============================================================================
    if history_a is not None and history_b is not None:
        plot_comparison(
            history_a, history_b,
            save_path=os.path.join(RESULTS_PATH, 'detector_AB_comparison.png')
        )
    else:
        print("[SKIP] Comparison plot requires both experiments.")

    # ==============================================================================
    # SUMMARY
    # ==============================================================================
    print("\n" + "=" * 65)
    print(" TRAINING SUMMARY")
    print("=" * 65)

    for name, history, path, per_attack in [
        ('A — MobileNetV3-Large', history_a, path_a, per_attack_a),
        ('B — EfficientNet-B0',   history_b, path_b, per_attack_b),
    ]:
        if history is None:
            continue
        print(f"\n  Experiment {name}")
        print(f"    Best Val Accuracy : {max(history['val_accuracy']):.2f}%")
        print(f"    Best Val F1       : {max(history['val_f1']):.4f}")
        print(f"    Best Val AUROC    : {max(history['val_auroc']):.4f}")
        print(f"    Epochs trained    : {len(history['train_loss'])}")
        print(f"    Weights           : {path}")
        if per_attack is not None and 'genuine' in per_attack:
            fpr = 100 - per_attack['genuine']['detection_rate']
            print(f"    Genuine FPR       : {fpr:.2f}%")
            
    print("=" * 65)

#### Validation Set Evaluation 
This cell loads the saved training history CSVs for both experiments
and reproduces all training plots without requiring a full re-run.

It also loads the per-attack validation breakdown CSVs and produces
a detection rate bar chart for each experiment, without std error bars.

In [ ]:
# ==============================================================================
# CONFIGURATION — paths to saved CSVs
# ==============================================================================
RESULTS_PATH = os.path.join(RESULTS_DIR, 'training_results')

HISTORY_CSV_A    = os.path.join(RESULTS_PATH, 'detector_A_history.csv')
HISTORY_CSV_B    = os.path.join(RESULTS_PATH, 'detector_B_history.csv')
PER_ATTACK_CSV_A = os.path.join(RESULTS_PATH, 'detector_A_per_attack.csv')
PER_ATTACK_CSV_B = os.path.join(RESULTS_PATH, 'detector_B_per_attack.csv')

print(f"Using thresold for validation = {DETECTOR_THRESHOLD}")
# ==============================================================================
# HELPER — reproduce training plots from history CSV
# ==============================================================================

def plot_experiment_from_csv(df: pd.DataFrame, title: str,
                              save_path: str) -> dict:
    """
    Reproduces the 2x3 training history plot from a loaded history CSV.
    Also computes and returns a summary dict with best-epoch metrics.

    Args:
        df        : DataFrame loaded from detector_X_history.csv.
        title     : Plot title string.
        save_path : Destination path for the saved PNG.

    Returns:
        dict with best val accuracy, F1, AUROC, and total epochs.
    """
    epochs = df['epoch'].tolist()

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    best_epoch = df.loc[df['best_found_so_far'] == True, 'epoch']
    best_ep    = int(best_epoch.iloc[-1]) if len(best_epoch) > 0 else None

    exp_name   = df['experiment'].iloc[0]
    n_epochs   = len(df)
    info       = (f"Experiment: {exp_name} | "
                  f"Epochs run: {n_epochs} | "
                  f"Best epoch: {best_ep}")
    fig.suptitle(f"{title}\n{info}", fontsize=12, fontweight='bold')

    def _add_best_line(ax, best_ep):
        if best_ep is not None:
            ax.axvline(x=best_ep, color='gray', linestyle=':',
                       linewidth=1.0, alpha=0.7, label=f'Best (ep {best_ep})')

    # Loss
    axes[0, 0].plot(epochs, df['train_loss'], label='Train',
                    color='#1f77b4', linewidth=1.5)
    axes[0, 0].plot(epochs, df['val_loss'],   label='Val',
                    color='#ff7f0e', linewidth=1.5)
    _add_best_line(axes[0, 0], best_ep)
    axes[0, 0].set_title('Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].legend(fontsize=8)
    axes[0, 0].grid(True, alpha=0.3)

    # Accuracy
    axes[0, 1].plot(epochs, df['train_accuracy'], label='Train',
                    color='#1f77b4', linewidth=1.5)
    axes[0, 1].plot(epochs, df['val_accuracy'],   label='Val',
                    color='#2ca02c', linewidth=1.5)
    _add_best_line(axes[0, 1], best_ep)
    axes[0, 1].set_title('Accuracy (%)')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylim(0, 105)
    axes[0, 1].legend(fontsize=8)
    axes[0, 1].grid(True, alpha=0.3)

    # AUROC
    axes[0, 2].plot(epochs, df['val_auroc'], label='Val',
                    color='#9467bd', linewidth=1.5)
    _add_best_line(axes[0, 2], best_ep)
    axes[0, 2].set_title('AUROC (Validation)')
    axes[0, 2].set_xlabel('Epoch')
    axes[0, 2].set_ylim(0, 1.05)
    axes[0, 2].legend(fontsize=8)
    axes[0, 2].grid(True, alpha=0.3)

    # Precision
    axes[1, 0].plot(epochs, df['val_precision'], label='Val',
                    color='#d62728', linewidth=1.5)
    _add_best_line(axes[1, 0], best_ep)
    axes[1, 0].set_title('Precision (Validation)')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylim(0, 1.05)
    axes[1, 0].legend(fontsize=8)
    axes[1, 0].grid(True, alpha=0.3)

    # Recall
    axes[1, 1].plot(epochs, df['val_recall'], label='Val',
                    color='#8c564b', linewidth=1.5)
    _add_best_line(axes[1, 1], best_ep)
    axes[1, 1].set_title('Recall (Validation)')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylim(0, 1.05)
    axes[1, 1].legend(fontsize=8)
    axes[1, 1].grid(True, alpha=0.3)

    # F1
    axes[1, 2].plot(epochs, df['val_f1'], label='Val',
                    color='#e377c2', linewidth=1.5)
    _add_best_line(axes[1, 2], best_ep)
    axes[1, 2].set_title('F1 Score (Validation)')
    axes[1, 2].set_xlabel('Epoch')
    axes[1, 2].set_ylim(0, 1.05)
    axes[1, 2].legend(fontsize=8)
    axes[1, 2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"[INFO] Plot saved: {save_path}")

    # Extract best-epoch metrics
    best_row = df[df['best_found_so_far'] == True].iloc[-1]
    return {
        'best_val_accuracy': best_row['val_accuracy'],
        'best_val_f1':       best_row['val_f1'],
        'best_val_auroc':    best_row['val_auroc'],
        'best_epoch':        int(best_row['epoch']),
        'total_epochs':      n_epochs,
        'experiment':        exp_name,
    }


def plot_per_attack_from_csv(df: pd.DataFrame, experiment_name: str,
                              save_path: str) -> None:
    """
    Produces a grouped bar chart of per-attack detection rate from the
    per-attack CSV. One bar per (family, variant) pair. No std error bars.

    Args:
        df              : DataFrame loaded from detector_X_per_attack.csv.
        experiment_name : Label for the plot title.
        save_path       : Destination path for the saved PNG.
    """
    FAMILY_ORDER   = ['FGSM', 'BIM', 'PGD', 'CW', 'DeepFool']
    VARIANT_COLORS = {'untargeted': '#1f77b4', 'targeted': '#ff7f0e'}
    VARIANT_LABELS = {'untargeted': 'Untargeted', 'targeted': 'Targeted'}

    families = [f for f in FAMILY_ORDER if f in df['family'].values]
    if not families:
        print(f"[WARN] No families found in per-attack CSV for {experiment_name}.")
        return

    all_variants = df['variant'].unique().tolist()
    n_variants   = len(all_variants)
    width        = 0.8 / n_variants
    offsets      = np.linspace(
        -(n_variants - 1) / 2 * width,
         (n_variants - 1) / 2 * width,
        n_variants
    )

    fig, ax = plt.subplots(figsize=(max(10, len(families) * 2.5), 6))
    fig.suptitle(
        f'Per-Attack Detection Rate — {experiment_name}\n'
        f'(Validation Set)',
        fontsize=12, fontweight='bold'
    )

    x_pos = np.arange(len(families))

    for v_idx, variant in enumerate(all_variants):
        means = []
        for family in families:
            row = df[(df['family'] == family) & (df['variant'] == variant)]
            means.append(
                float(row['detection_rate_mean'].iloc[0])
                if len(row) > 0 else 0.0
            )

        color = VARIANT_COLORS.get(variant, '#9467bd')
        label = VARIANT_LABELS.get(variant, variant)
        bars  = ax.bar(
            x_pos + offsets[v_idx], means, width,
            label=label, color=color,
            alpha=0.85, edgecolor='white', linewidth=0.5
        )
        for bar, mean_val in zip(bars, means):
            if mean_val > 0:
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 1.5,
                    f'{mean_val:.1f}',
                    ha='center', va='bottom', fontsize=8
                )

    ax.set_xticks(x_pos)
    ax.set_xticklabels(families, fontsize=11)
    ax.set_xlabel('Attack Family', fontsize=12)
    ax.set_ylabel('Detection Rate (%)', fontsize=12)
    ax.set_ylim(0, 120)
    ax.legend(fontsize=9, loc='lower right')
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"[INFO] Per-attack plot saved: {save_path}")


# ==============================================================================
# LOAD CSVs AND PRODUCE PLOTS
# ==============================================================================
summaries = {}

for label, hist_csv, atk_csv, hist_title in [
    ('A', HISTORY_CSV_A, PER_ATTACK_CSV_A, 'Experiment A — MobileNetV3-Large'),
    ('B', HISTORY_CSV_B, PER_ATTACK_CSV_B, 'Experiment B — EfficientNet-B0'),
]:
    print(f"\n{'=' * 65}")
    print(f" Experiment {label}")
    print(f"{'=' * 65}")

    if not os.path.exists(hist_csv):
        print(f"[SKIP] History CSV not found: {hist_csv}")
        continue

    df_hist = pd.read_csv(hist_csv)
    summary = plot_experiment_from_csv(
        df_hist,
        title=hist_title,
        save_path=os.path.join(
            RESULTS_PATH, f'detector_{label}_history_from_csv.png'
        )
    )
    summaries[label] = summary

    if not os.path.exists(atk_csv):
        print(f"[SKIP] Per-attack CSV not found: {atk_csv}")
        continue

    df_atk = pd.read_csv(atk_csv)
    plot_per_attack_from_csv(
        df_atk,
        experiment_name=hist_title,
        save_path=os.path.join(
            RESULTS_PATH, f'detector_{label}_per_attack_from_csv.png'
        )
    )

# ==============================================================================
# EER ANALYSIS — FAR vs FRR across thresholds (Validation Set)
# Requires loading the trained models for inference on val set.
# ==============================================================================

print("\n[INFO] Computing EER analysis on validation set...")

val_dataset_eer = DetectorDatasetWithPaths(val_files)
val_loader_eer  = torch.utils.data.DataLoader(
    val_dataset_eer,
    batch_size=BATCH_SIZE_ADV_DETECTOR,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

def compute_eer(model, loader, device, experiment_name, save_path):
    """
    Computes FAR, FRR across 100 thresholds and plots the trade-off
    curve with the Equal Error Rate (EER) point highlighted.

    Args:
        model           : Trained detector model.
        loader          : DataLoader returning (tensor, label, path).
        device          : Computation device.
        experiment_name : Label for the plot title.
        save_path       : Destination path for the saved PNG.

    Returns:
        dict with eer_value, eer_threshold, far_list, frr_list, thresholds.
    """
    model.eval()
    all_scores = []
    all_labels = []

    with torch.no_grad():
        for x, y, _ in loader:
            scores = model.predict_score(x.to(device)).cpu().numpy()
            all_scores.extend(scores)
            all_labels.extend(y.numpy())

    all_scores = np.array(all_scores)
    all_labels = np.array(all_labels)

    thresholds = np.linspace(0.01, 0.99, 200)
    adv_mask   = (all_labels == 1)
    gen_mask   = (all_labels == 0)
    total_adv  = np.sum(adv_mask)
    total_gen  = np.sum(gen_mask)

    far_list = []
    frr_list = []

    for t in thresholds:
        preds = (all_scores >= t).astype(int)
        far   = np.sum(preds[adv_mask] == 0) / total_adv * 100 if total_adv > 0 else 0.0
        frr   = np.sum(preds[gen_mask] == 1) / total_gen * 100 if total_gen > 0 else 0.0
        far_list.append(far)
        frr_list.append(frr)

    # EER: point where FAR and FRR are closest
    diffs         = np.abs(np.array(far_list) - np.array(frr_list))
    eer_idx       = np.argmin(diffs)
    eer_threshold = thresholds[eer_idx]
    eer_value     = (far_list[eer_idx] + frr_list[eer_idx]) / 2

    fig, ax = plt.subplots(figsize=(8, 6))
    fig.suptitle(
        f'FAR vs FRR Trade-off — {experiment_name}\n(Validation Set)',
        fontsize=13, fontweight='bold'
    )

    ax.plot(thresholds, far_list,
            label='FAR (adversarial passed as genuine)',
            color='#d62728', linewidth=2)
    ax.plot(thresholds, frr_list,
            label='FRR (genuine blocked as adversarial)',
            color='#1f77b4', linewidth=2)

    ax.axvline(x=eer_threshold, color='black', linestyle='--', alpha=0.6,
               label=f'EER = {eer_value:.2f}% at threshold {eer_threshold:.2f}')

    ax.axvline(x=DETECTOR_THRESHOLD, color='green', linestyle=':', alpha=0.8,
               label=f'Current threshold = {DETECTOR_THRESHOLD}')

    ax.scatter([eer_threshold], [eer_value], color='black',
               s=80, zorder=5, marker='D')

    ax.set_xlabel('Decision Threshold', fontsize=12)
    ax.set_ylabel('Error Rate (%)', fontsize=12)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, max(max(far_list), max(frr_list)) + 5)
    ax.legend(fontsize=9)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"[INFO] FAR vs FRR plot saved: {save_path}")

    return {
        'eer_value':     eer_value,
        'eer_threshold': eer_threshold,
    }


eer_results = {}

for exp_label, model_path, model_class, exp_name in [
    ('A', path_a, AdversarialDetector,  'Experiment A — MobileNetV3-Large'),
    ('B', path_b, EfficientNetDetector, 'Experiment B — EfficientNet-B0'),
]:
    if not os.path.exists(model_path):
        print(f"[SKIP] Experiment {exp_label}: weights not found at {model_path}")
        continue

    print(f"\n[INFO] Computing EER for Experiment {exp_label}...")
    model_eer = model_class(pretrained=False, device=DEVICE)
    model_eer.load_weights(model_path)

    eer = compute_eer(
        model_eer,
        val_loader_eer,
        DEVICE,
        experiment_name=exp_name,
        save_path=os.path.join(
            RESULTS_PATH, f'detector_{exp_label}_eer.png'
        )
    )
    eer_results[exp_label] = eer
    print(f"  EER: {eer['eer_value']:.2f}% at threshold {eer['eer_threshold']:.2f}")
    
    
# ==============================================================================
# SUMMARY TABLE
# ==============================================================================
print("\n" + "=" * 65)
print(" TRAINING SUMMARY (from CSV)")
print("=" * 65)

for label, s in summaries.items():
    print(f"\n  Experiment {label} — {s['experiment']}")
    print(f"    Best Val Accuracy : {s['best_val_accuracy']:.2f}%")
    print(f"    Best Val F1       : {s['best_val_f1']:.4f}")
    print(f"    Best Val AUROC    : {s['best_val_auroc']:.4f}")
    print(f"    Best epoch        : {s['best_epoch']}")
    print(f"    Total epochs      : {s['total_epochs']}")
    if label in eer_results:
        print(f"    EER               : {eer_results[label]['eer_value']:.2f}%")
        print(f"    EER Threshold     : {eer_results[label]['eer_threshold']:.2f}")

if len(summaries) == 2:
    better = max(summaries, key=lambda k: summaries[k]['best_val_auroc'])
    print(f"\n  Best model by AUROC: Experiment {better} — "
          f"{summaries[better]['experiment']}")

print("=" * 65)

### 6.1.2  — Test Set Evaluation

This cell loads the best weights of both experiments, evaluates them on
the held-out test set, and produces a comprehensive set of diagnostic
plots for comparison.

The test set contains subjects never seen during training or validation
(identity-disjoint split), providing an unbiased estimate of the
detector's generalisation capability.

### Outputs
- Per-model metrics: accuracy, precision, recall, F1, AUROC
- Confusion matrix for each model
- ROC curve overlay (Experiment A vs Experiment B)
- Per-attack detection rate: for each attack family present in the test
  set, the fraction of adversarial samples correctly detected

#### Helpers

In [75]:

# ==============================================================================
# HELPERS
# ==============================================================================

def evaluate_test(model, loader, device, threshold=0.5):
    """
    Runs inference on the full test DataLoader and returns per-sample
    predictions, ground-truth labels, adversarial scores, and file paths.

    Args:
        model     : trained AdversarialDetector instance.
        loader    : DataLoader returning (image_tensor, label, path) tuples.
        device    : torch.device.
        threshold : Decision threshold applied to the adversarial score.

    Returns:
        dict with keys: labels, preds, scores, paths.
    """
    model.eval()
    all_labels = []
    all_preds  = []
    all_scores = []
    all_paths  = []

    with torch.no_grad():
        for x, y, paths in loader:
            x      = x.to(device)
            logits = model(x)
            probs  = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            preds  = (probs >= threshold).astype(int)

            all_labels.extend(y.numpy())
            all_preds.extend(preds)
            all_scores.extend(probs)
            all_paths.extend(paths)

    return {
        'labels': np.array(all_labels),
        'preds':  np.array(all_preds),
        'scores': np.array(all_scores),
        'paths':  np.array(all_paths)
    }


def compute_metrics(results):
    """
    Computes classification metrics from evaluate_test output.

    Returns a dict with accuracy, precision, recall, F1, AUROC.
    """
    labels = results['labels']
    preds  = results['preds']
    scores = results['scores']

    return {
        'accuracy':  (preds == labels).mean() * 100,
        'precision': precision_score(labels, preds, zero_division=0),
        'recall':    recall_score(labels, preds, zero_division=0),
        'f1':        f1_score(labels, preds, zero_division=0),
        'auroc':     roc_auc_score(labels, scores)
    }


def get_attack_family(path: str):
    """
    Extracts a human-readable attack family label from an adversarial
    image file path. Returns None for genuine images.
    """
    fname = os.path.basename(path)
    if fname.startswith('orig__') or fname.startswith('extra__'):
        return None
    parts = fname.split('__')
    if len(parts) < 2:
        return None
    raw = parts[0]
    for suffix in ['_Targeted_Fixed', '_Targeted_LeastLikely', '_Untargeted', '_Targeted']:
        raw = raw.replace(suffix, '')
    return raw


class DetectorDatasetWithPaths(Dataset):
    """
    Variant of DetectorDataset that also returns the file path at each
    index. Used exclusively for test set evaluation to enable per-attack
    breakdown.

    Corrupted or unreadable images are replaced with a zero tensor and
    logged to stderr to avoid silent failures during evaluation.
    """
    def __init__(self, file_list: list):
        self.file_list = file_list
        self.transform = transforms.Compose([
            transforms.Resize((160, 160)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        path, label = self.file_list[idx]
        try:
            img    = Image.open(path).convert('RGB')
            tensor = self.transform(img)
        except Exception as e:
            print(f"\n[ERROR] Failed to load image: {path} — {e}")
            tensor = torch.zeros(3, 160, 160)
        return tensor, torch.tensor(label, dtype=torch.long), path


#### Test Set Evaluation

In [ ]:
if RUN_DEFENSE_DETECTOR_TESTING:

    # ==============================================================================
    # TEST DATALOADER
    # ==============================================================================
    test_dataset_with_paths = DetectorDatasetWithPaths(test_files)
    test_loader_with_paths  = DataLoader(
        test_dataset_with_paths,
        batch_size=BATCH_SIZE_ADV_DETECTOR,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )

    # ==============================================================================
    # LOAD MODELS AND RUN INFERENCE
    # ==============================================================================
    results_a = None
    results_b = None
    metrics_a = None
    metrics_b = None
    if not os.path.exists(DETECTOR_TEST_DIR):
        os.makedirs(DETECTOR_TEST_DIR, exist_ok=True)
        
    if os.path.exists(path_a):
        print("[INFO] Loading Experiment A weights (MobileNetV3-Large)...")
        model_a_eval = AdversarialDetector(pretrained=False, device=DEVICE)
        model_a_eval.load_weights(path_a)
        print("[INFO] Running inference — Experiment A...")
        results_a = evaluate_test(model_a_eval, test_loader_with_paths, DEVICE, FINETUNED_DETECTOR_THRESHOLD)
        metrics_a = compute_metrics(results_a)
    else:
        print(f"[SKIP] Experiment A weights not found at {path_a}.")

    if os.path.exists(path_b):
        print("[INFO] Loading Experiment B weights (EfficientNet-B0)...")
        model_b_eval = EfficientNetDetector(pretrained=False, device=DEVICE)
        model_b_eval.load_weights(path_b)
        print("[INFO] Running inference — Experiment B...")
        results_b = evaluate_test(model_b_eval, test_loader_with_paths, DEVICE, FINETUNED_DETECTOR_THRESHOLD)
        metrics_b = compute_metrics(results_b)
    else:
        print(f"[SKIP] Experiment B weights not found at {path_b}.")

    if results_a is None and results_b is None:
        raise RuntimeError(
            "[ERROR] No model weights found. "
            "Run at least one training experiment first."
        )

    # ==============================================================================
    # PLOT 1 — Confusion matrices
    # ==============================================================================
    available = [(r, m, t) for r, m, t in [
        (results_a, metrics_a, 'Experiment A — MobileNetV3-Large'),
        (results_b, metrics_b, 'Experiment B — EfficientNet-B0'),
    ] if r is not None]

    fig, axes = plt.subplots(1, len(available), figsize=(6 * len(available), 5))
    if len(available) == 1:
        axes = [axes]
    fig.suptitle('Confusion Matrices — Test Set', fontsize=14, fontweight='bold')

    for ax, (results, _, title) in zip(axes, available):
        cm   = confusion_matrix(results['labels'], results['preds'])
        disp = ConfusionMatrixDisplay(
            confusion_matrix=cm,
            display_labels=['Genuine', 'Adversarial']
        )
        disp.plot(ax=ax, colorbar=False, cmap='Blues')
        ax.set_title(title, fontsize=11)

    plt.tight_layout()
    cm_path = os.path.join(DETECTOR_TEST_DIR, 'detector_test_confusion_matrices.png')
    plt.savefig(cm_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"[INFO] Confusion matrices saved: {cm_path}")

    # ==============================================================================
    # PLOT 2 — ROC curves
    # ==============================================================================
    fig, ax = plt.subplots(figsize=(8, 6))

    if results_a is not None:
        fpr_a, tpr_a, _ = roc_curve(results_a['labels'], results_a['scores'])
        ax.plot(fpr_a, tpr_a, color='#1f77b4', linewidth=2,
                label=f"A — MobileNetV3-Large (AUROC={metrics_a['auroc']:.4f})")

    if results_b is not None:
        fpr_b, tpr_b, _ = roc_curve(results_b['labels'], results_b['scores'])
        ax.plot(fpr_b, tpr_b, color='#d62728', linewidth=2,
                label=f"B — EfficientNet-B0 (AUROC={metrics_b['auroc']:.4f})")

    ax.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1,
            label='Random classifier')
    ax.set_title('ROC Curve — Test Set', fontsize=14, fontweight='bold')
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    roc_path = os.path.join(DETECTOR_TEST_DIR, 'detector_test_roc.png')
    plt.savefig(roc_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"[INFO] ROC curve saved: {roc_path}")

    # ==============================================================================
    # PLOT 3 — Per-attack detection rate (Test Set)
    # Uses same aggregation logic as validation breakdown.
    # ==============================================================================

    def collect_per_attack_test(results):
        """
        Collects per-attack TP counts and totals from evaluate_test output.
        Uses the same family extraction logic as evaluate_per_attack.
        Returns two dicts: tp[family] and total[family].
        """
        import re
        tp    = {}
        total = {}
        for path, label, pred in zip(
                results['paths'], results['labels'], results['preds']):
            if label == 0:
                continue
            fname = os.path.basename(path)
            if fname.startswith('orig__') or fname.startswith('extra__'):
                continue
            raw    = fname.split('__')[0]
            family = raw
            for suffix in [
                '_Targeted_Fixed', '_Targeted_LeastLikely',
                '_Targeted', '_Untargeted'
            ]:
                family = family.replace(suffix, '')
            family = re.sub(r'_nri\d+',  '', family)
            family = re.sub(r'_iter\d+', '', family)
            family = re.sub(r'_\d+$',    '', family)

            total[family] = total.get(family, 0) + 1
            if pred == 1:
                tp[family] = tp.get(family, 0) + 1
        return tp, total

    FAMILY_ORDER = ['FGSM', 'BIM', 'PGD', 'CW', 'DeepFool']

    ref_results  = results_a if results_a is not None else results_b
    _, ref_total = collect_per_attack_test(ref_results)
    families     = [f for f in FAMILY_ORDER if f in ref_total] + \
                [f for f in sorted(ref_total) if f not in FAMILY_ORDER]

    x     = np.arange(len(families))
    width = 0.35 if (results_a is not None and results_b is not None) else 0.5

    fig, ax = plt.subplots(figsize=(max(11, len(families) * 2.2), 6))
    fig.suptitle(
        'Per-Attack Detection Rate — Test Set',
        fontsize=14, fontweight='bold'
    )

    if results_a is not None:
        tp_a, total_a = collect_per_attack_test(results_a)
        rates_a       = [tp_a.get(f, 0) / max(total_a.get(f, 1), 1) * 100
                        for f in families]
        offset        = -width / 2 if results_b is not None else 0
        bars_a        = ax.bar(x + offset, rates_a, width,
                            label='A — MobileNetV3-Large',
                            color='#1f77b4', alpha=0.85,
                            edgecolor='white', linewidth=0.5)
        for bar in bars_a:
            h = bar.get_height()
            ax.annotate(f'{h:.1f}',
                        xy=(bar.get_x() + bar.get_width() / 2, h),
                        xytext=(0, 4), textcoords='offset points',
                        ha='center', fontsize=8)

    if results_b is not None:
        tp_b, total_b = collect_per_attack_test(results_b)
        rates_b       = [tp_b.get(f, 0) / max(total_b.get(f, 1), 1) * 100
                        for f in families]
        offset        = width / 2 if results_a is not None else 0
        bars_b        = ax.bar(x + offset, rates_b, width,
                            label='B — EfficientNet-B0',
                            color='#d62728', alpha=0.85,
                            edgecolor='white', linewidth=0.5)
        for bar in bars_b:
            h = bar.get_height()
            ax.annotate(f'{h:.1f}',
                        xy=(bar.get_x() + bar.get_width() / 2, h),
                        xytext=(0, 4), textcoords='offset points',
                        ha='center', fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels(families, fontsize=11)
    ax.set_xlabel('Attack Family', fontsize=12)
    ax.set_ylabel('Detection Rate (%)', fontsize=12)
    ax.set_ylim(0, 115)
    ax.legend(fontsize=11)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    per_attack_path = os.path.join(RESULTS_PATH, 'detector_test_per_attack.png')
    plt.savefig(per_attack_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"[INFO] Per-attack plot saved: {per_attack_path}")

    # ==============================================================================
    # FINAL SUMMARY TABLE
    # ==============================================================================
    print("\n" + "=" * 65)
    print(" TEST SET EVALUATION SUMMARY")
    print("=" * 65)
    print(f"  Test subjects : {len(test_ids)}")
    print(f"  Test images   : {len(test_files)}")
    print(f"  Threshold     : {FINETUNED_DETECTOR_THRESHOLD}")
    print()

    for results, metrics, exp_label in [
        (results_a, metrics_a, 'Experiment A — MobileNetV3-Large'),
        (results_b, metrics_b, 'Experiment B — EfficientNet-B0'),
    ]:
        if results is None:
            continue
        print(f"  {exp_label}")
        print("  " + "-" * 45)
        for key, label in [
            ('accuracy',  'Accuracy (%)'),
            ('precision', 'Precision'),
            ('recall',    'Recall'),
            ('f1',        'F1 Score'),
            ('auroc',     'AUROC'),
        ]:
            fmt = (f"{metrics[key]:.2f}%"
                if key == 'accuracy' else f"{metrics[key]:.4f}")
            print(f"    {label:<22} : {fmt}")

        tp_exp, total_exp = collect_per_attack_test(results)
        # Genuine FPR
        genuine_total = sum(1 for l in results['labels'] if l == 0)
        genuine_fp    = sum(
            1 for l, p in zip(results['labels'], results['preds'])
            if l == 0 and p == 1
        )
        fpr = genuine_fp / max(genuine_total, 1) * 100
        print(f"    {'Genuine FPR':<22} : {fpr:.2f}%")
        print(f"\n    Per-attack detection rate:")
        for f in families:
            rate = tp_exp.get(f, 0) / max(total_exp.get(f, 1), 1) * 100
            print(f"      {f:<20} : {rate:.1f}%")
        print()

    if metrics_a is not None and metrics_b is not None:
        winner = ('A (MobileNetV3-Large)'
                if metrics_a['auroc'] >= metrics_b['auroc']
                else 'B (EfficientNet-B0)')
        print(f"[INFO] Best model by AUROC: Experiment {winner}")
    elif metrics_a is not None:
        print("[INFO] Only Experiment A was evaluated.")
    else:
        print("[INFO] Only Experiment B was evaluated.")

    print("=" * 65)

### 6.1.3 — System Evaluation: NN1 vs Detector + NN1

This cell evaluates the impact of the adversarial detector on the
overall face recognition system. Two runs are performed on the same
test split used for detector evaluation:

- **Run 1 — Baseline**: images are passed directly to NN1 without any
  gate. Measures NN1 raw performance on both genuine and adversarial
  samples from the test split.

- **Run 2 — Defended system**: images are first passed through the
  detector. Samples classified as adversarial are rejected and never
  reach NN1. Only samples classified as genuine are forwarded to NN1.

### Metrics reported
- Accuracy on genuine images (with and without detector)
- Block rate on adversarial images (detector gate)
- Accuracy on adversarial images that pass the gate
- Conditional accuracy: accuracy of NN1 on images that pass the gate
- System accuracy: accuracy over all images (blocked = wrong)
- Per-attack breakdown: block rate and pass-through misclassification
  for each attack family present in the test set

### Notes
- NN1 preprocessing: Resize(160) + ToTensor(), normalization to [-1,1]
  is handled internally by the model.
- The best model by AUROC from Step 6.3 is used as the detector.
- Attack families not yet in the dataset (CW, DeepFool) are reported
  as PENDING and skipped automatically.


#### Helper

In [77]:
def get_attack_family(path: str):
    """
    Extracts the canonical attack family label from an adversarial image
    path. Returns None for genuine images.

    Strips variant suffixes (Targeted_Fixed, LeastLikely, Targeted,
    Untargeted) and numeric iter/nri suffixes to return a clean family
    name (e.g. FGSM, BIM, PGD, CW, DeepFool).
    """
    fname = os.path.basename(path)
    if fname.startswith('orig__') or fname.startswith('extra__'):
        return None
    raw = fname.split('__')[0]
    for suffix in ['_Targeted_Fixed', '_Targeted_LeastLikely', '_Untargeted', '_Targeted']:
        raw = raw.replace(suffix, '')
    raw = re.sub(r'_nri\d+',  '', raw)
    raw = re.sub(r'_iter\d+', '', raw)
    raw = re.sub(r'_\d+$',    '', raw)
    return raw


def extract_class_id_from_path(fname: str) -> str:
    """
    Extracts the VGGFace2 class_id from a detector dataset filename.

    Genuine format  : orig__{class_id}__{filename}.ext
                        extra__{class_id}__{filename}.ext
    Adversarial (class_subfolders):
                        {attack}__{subfolder}__{class_id}__{filename}.ext
    Adversarial (flat, CW/DeepFool):
                        {attack}__{subfolder}__{class_id}.ext
                        (class_id is the stem of parts[2])
    """
    parts = fname.split('__')
    if fname.startswith('orig__') or fname.startswith('extra__'):
        return parts[1]
    if len(parts) >= 4:
        return parts[2]
    # flat layout: parts[2] is '{class_id}.ext'
    return os.path.splitext(parts[2])[0]


class SystemEvalDataset(Dataset):
    """
    Dataset for system-level evaluation.
    Restituisce 3 tensori: uno formattato per il detector [0, 1],
    uno formattato per la NN1 (VGGFace2) e uno formattato per la NN2 (ResNet-50).
    """
    def __init__(self, file_list: list):
        self.file_list = file_list
        self.detector_transform = transforms.Compose([
            transforms.Resize((160, 160)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        path, label = self.file_list[idx]
        try:
            img = Image.open(path).convert('RGB')
            # 1. Creiamo il tensore per il detector
            tensor_detector = self.detector_transform(img)
            # 2. Creiamo il tensore per la NN1
            tensor_nn1 = preprocess_nn1(img)
            # 2. Creiamo il tensore per la NN1
            tensor_nn2 = preprocess_nn2(img)
        except Exception as e:
            print(f"\n[ERROR] Failed to load: {path} — {e}")
            tensor_detector = torch.zeros(3, 160, 160)
            tensor_nn1 = torch.zeros(3, 160, 160)
            tensor_nn2 = torch.zeros(3, 224, 224)
            
        return tensor_detector, tensor_nn1, tensor_nn2, torch.tensor(label, dtype=torch.long), path


def nn1_predict_batch(x_batch: torch.Tensor) -> np.ndarray:
    """
    Runs NN1 on a batch of tensors in [0, 1] at 160x160.
    Returns predicted class indices as a numpy array.
    """
    with torch.no_grad():
        logits = nn1(x_batch.to(DEVICE))
        return torch.argmax(logits, dim=1).cpu().numpy()

def nn2_predict_batch(x_batch: torch.Tensor) -> np.ndarray:
    """
    Runs NN2 (ResNet-50) on a batch of tensors at 224x224.
    Returns predicted class indices as a numpy array.
    """
    with torch.no_grad():
        logits = nn2(x_batch.to(DEVICE))
        return torch.argmax(logits, dim=1).cpu().numpy()

def nn1_label_correct(pred_indices: np.ndarray, paths: list) -> np.ndarray:
    """
    Checks whether each NN1 prediction matches the true VGGFace2 identity.
    Extracts class_id from the filename, resolves the true name from the
    CSV, and compares against the predicted LABELS entry.

    Returns a boolean array of shape (N,).
    """
    correct = []
    for pred_idx, path in zip(pred_indices, paths):
        fname    = os.path.basename(path)
        class_id = extract_class_id_from_path(fname)

        true_name_arr = df_csv[df_csv['Class_ID'] == class_id]['Name'].values
        if len(true_name_arr) == 0:
            correct.append(False)
            continue
        true_name = true_name_arr[0]

        raw_label = LABELS[pred_idx]
        if isinstance(raw_label, np.ndarray):
            raw_label = raw_label.flatten()[0]
        if isinstance(raw_label, bytes):
            raw_label = raw_label.decode('utf-8')
        pred_name = (str(raw_label).replace("b'", "").replace("'", "").strip())

        correct.append(
            normalize_label(pred_name) == normalize_label(true_name)
        )
    return np.array(correct)


# ==============================================================================
# DATASET AND DATALOADER
# ==============================================================================
system_dataset = SystemEvalDataset(test_files)
system_loader  = DataLoader(system_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

#### Evaluation

In [ ]:
# ==============================================================================
# DATASET AND DATALOADER
# ==============================================================================
system_dataset = SystemEvalDataset(test_files)
system_loader  = DataLoader(system_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

if RUN_DEFENSE_DETECTOR_TESTING_WITH_NN1:
    print(f"[INFO] Detector       : {DETECTOR_TO_USE}")
    print(f"[INFO] Detector path  : {DETECTOR_PATH}")
    print(f"[INFO] Threshold      : {FINETUNED_DETECTOR_THRESHOLD}")    

    # ==============================================================================
    # LOAD DETECTOR
    # ==============================================================================
    if DETECTOR_TO_USE == 'MobileNetV3':
        detector = AdversarialDetector(pretrained=False, device=DEVICE)
    elif DETECTOR_TO_USE == 'EfficientNet':
        detector = EfficientNetDetector(pretrained=False, device=DEVICE)
    else:
        raise ValueError(f"Unknown DETECTOR_TO_USE: {DETECTOR_TO_USE}")

    detector.load_weights(DETECTOR_PATH)
    detector.eval()

    # ==============================================================================
    # RUN 1 — Baseline: NN1 and NN2 without detector
    # ==============================================================================
    print("\n" + "=" * 65)
    print(" RUN 1 — Baseline: System without detector")
    print("=" * 65)

    baseline = {
        'genuine_total': 0,
        'adversarial_total': 0,
        'nn1': {'gen_corr': 0, 'adv_corr': 0},
        'nn2': {'gen_corr': 0, 'adv_corr': 0},
        'per_attack': {}
    }

    for x_det, x_nn1, x_nn2, labels, paths in system_loader:
        # Predizioni parallele per entrambe le reti
        pred_nn1 = nn1_predict_batch(x_nn1)
        pred_nn2 = nn2_predict_batch(x_nn2)
        
        corr_mask_nn1 = nn1_label_correct(pred_nn1, list(paths))
        corr_mask_nn2 = nn1_label_correct(pred_nn2, list(paths))

        for i, (label, path) in enumerate(zip(labels.numpy(), paths)):
            family = get_attack_family(path)
            c_nn1 = int(corr_mask_nn1[i])
            c_nn2 = int(corr_mask_nn2[i])

            if label == 0:
                baseline['genuine_total'] += 1
                baseline['nn1']['gen_corr'] += c_nn1
                baseline['nn2']['gen_corr'] += c_nn2
            else:
                baseline['adversarial_total'] += 1
                baseline['nn1']['adv_corr'] += c_nn1
                baseline['nn2']['adv_corr'] += c_nn2
                
                if family is not None:
                    if family not in baseline['per_attack']:
                        baseline['per_attack'][family] = {
                            'total': 0, 'nn1_corr': 0, 'nn2_corr': 0
                        }
                    baseline['per_attack'][family]['total'] += 1
                    baseline['per_attack'][family]['nn1_corr'] += c_nn1
                    baseline['per_attack'][family]['nn2_corr'] += c_nn2

    # --- Metriche Baseline Derivate ---
    b_gen_tot = baseline['genuine_total']
    b_adv_tot = baseline['adversarial_total']

    # Baseline NN1
    base_gen_acc_nn1 = (baseline['nn1']['gen_corr'] / b_gen_tot * 100 if b_gen_tot > 0 else float('nan'))
    base_adv_acc_nn1 = (baseline['nn1']['adv_corr'] / b_adv_tot * 100 if b_adv_tot > 0 else float('nan'))
    base_sys_acc_nn1 = ((baseline['nn1']['gen_corr'] + baseline['nn1']['adv_corr']) / (b_gen_tot + b_adv_tot) * 100)

    # Baseline NN2
    base_gen_acc_nn2 = (baseline['nn2']['gen_corr'] / b_gen_tot * 100 if b_gen_tot > 0 else float('nan'))
    base_adv_acc_nn2 = (baseline['nn2']['adv_corr'] / b_adv_tot * 100 if b_adv_tot > 0 else float('nan'))
    base_sys_acc_nn2 = ((baseline['nn2']['gen_corr'] + baseline['nn2']['adv_corr']) / (b_gen_tot + b_adv_tot) * 100)

    print(f"  [NN1] Genuine acc: {base_gen_acc_nn1:.2f}% | Adv acc: {base_adv_acc_nn1:.2f}% | Overall: {base_sys_acc_nn1:.2f}%")
    print(f"  [NN2] Genuine acc: {base_gen_acc_nn2:.2f}% | Adv acc: {base_adv_acc_nn2:.2f}% | Overall: {base_sys_acc_nn2:.2f}%")

    # ==============================================================================
    # RUN 2 — Defended system: Detector + NN1 and Detector + NN2
    # ==============================================================================
    print("\n" + "=" * 65)
    print(" RUN 2 — Defended system: Detector + NN1")
    print("=" * 65)

    defended = {
        'genuine_total': 0,
        'genuine_blocked': 0,
        'adversarial_total': 0,
        'adversarial_blocked': 0,
        'nn1': {'gen_pass_corr': 0, 'gen_pass_wr': 0, 'adv_pass_corr': 0, 'adv_pass_wr': 0},
        'nn2': {'gen_pass_corr': 0, 'gen_pass_wr': 0, 'adv_pass_corr': 0, 'adv_pass_wr': 0},
        'per_attack': {}
    }

    for x_det, x_nn1, x_nn2, labels, paths in system_loader:
        with torch.no_grad():
            # 1. Il detector valuta l'immagine usando il suo tensore [0, 1]
            det_scores = detector.predict_score(x_det.to(DEVICE)).cpu().numpy()
        det_preds    = (det_scores >= FINETUNED_DETECTOR_THRESHOLD).astype(int)
        pass_indices = np.where(det_preds == 0)[0]
        
        nn1_correct_passed = {}
        nn2_correct_passed = {}

        if pass_indices.size > 0:
            paths_passed = [paths[i] for i in pass_indices]
            
            # --- VALUTAZIONE NN1 ---
            x_passed_nn1 = x_nn1[pass_indices]
            pred_nn1 = nn1_predict_batch(x_passed_nn1)
            correct_mask_nn1 = nn1_label_correct(pred_nn1, paths_passed)
            
            # --- VALUTAZIONE NN2 ---
            x_passed_nn2 = x_nn2[pass_indices]
            pred_nn2 = nn2_predict_batch(x_passed_nn2)
            correct_mask_nn2 = nn1_label_correct(pred_nn2, paths_passed)
            
            for local_i, global_i in enumerate(pass_indices):
                nn1_correct_passed[int(global_i)] = bool(correct_mask_nn1[local_i])
                nn2_correct_passed[int(global_i)] = bool(correct_mask_nn2[local_i])

        for i, (label, path) in enumerate(zip(labels.numpy(), paths)):
            family  = get_attack_family(path)
            blocked = bool(det_preds[i] == 1)

            if label == 0:
                defended['genuine_total'] += 1
                if blocked:
                    defended['genuine_blocked'] += 1
                else:
                    # Aggiorniamo NN1
                    if nn1_correct_passed.get(i, False): defended['nn1']['gen_pass_corr'] += 1
                    else: defended['nn1']['gen_pass_wr'] += 1
                    # Aggiorniamo NN2
                    if nn2_correct_passed.get(i, False): defended['nn2']['gen_pass_corr'] += 1
                    else: defended['nn2']['gen_pass_wr'] += 1
            else:
                defended['adversarial_total'] += 1
                if blocked:
                    defended['adversarial_blocked'] += 1
                else:
                    # Aggiorniamo NN1
                    if nn1_correct_passed.get(i, False): defended['nn1']['adv_pass_corr'] += 1
                    else: defended['nn1']['adv_pass_wr'] += 1
                    # Aggiorniamo NN2
                    if nn2_correct_passed.get(i, False): defended['nn2']['adv_pass_corr'] += 1
                    else: defended['nn2']['adv_pass_wr'] += 1

                if family is not None:
                    if family not in defended['per_attack']:
                        defended['per_attack'][family] = {
                            'total': 0, 'blocked': 0,
                            'nn1': {'pass_corr': 0, 'pass_wr': 0},
                            'nn2': {'pass_corr': 0, 'pass_wr': 0}
                        }
                    d = defended['per_attack'][family]
                    d['total'] += 1
                    if blocked:
                        d['blocked'] += 1
                    else:
                        # Aggiorniamo per_attack NN1
                        if nn1_correct_passed.get(i, False): d['nn1']['pass_corr'] += 1
                        else: d['nn1']['pass_wr'] += 1
                        # Aggiorniamo per_attack NN2
                        if nn2_correct_passed.get(i, False): d['nn2']['pass_corr'] += 1
                        else: d['nn2']['pass_wr'] += 1

    # --- Derived metrics ---
    g_total        = defended['genuine_total']
    a_total        = defended['adversarial_total']
    g_passed       = g_total - defended['genuine_blocked']
    a_passed       = a_total - defended['adversarial_blocked']
    # Detector metrics (Condivise)
    adv_block_rate = (defended['adversarial_blocked'] / a_total * 100 if a_total > 0 else float('nan'))
    genuine_fpr    = (defended['genuine_blocked'] / g_total * 100 if g_total > 0 else float('nan'))
    # --- Metriche NN1 ---
    genuine_nn1_acc = (defended['nn1']['gen_pass_corr'] / g_passed * 100 if g_passed > 0 else float('nan'))
    cond_correct_nn1 = defended['nn1']['gen_pass_corr'] + defended['nn1']['adv_pass_corr']
    cond_total_nn1   = g_passed + a_passed
    cond_acc_nn1     = (cond_correct_nn1 / cond_total_nn1 * 100 if cond_total_nn1 > 0 else float('nan'))
    sys_correct_nn1  = defended['nn1']['gen_pass_corr'] + defended['adversarial_blocked']
    sys_acc_nn1      = sys_correct_nn1 / (g_total + a_total) * 100
    # --- Metriche NN2 ---
    genuine_nn2_acc = (defended['nn2']['gen_pass_corr'] / g_passed * 100 if g_passed > 0 else float('nan'))
    cond_correct_nn2 = defended['nn2']['gen_pass_corr'] + defended['nn2']['adv_pass_corr']
    cond_total_nn2   = g_passed + a_passed
    cond_acc_nn2     = (cond_correct_nn2 / cond_total_nn2 * 100 if cond_total_nn2 > 0 else float('nan'))
    sys_correct_nn2  = defended['nn2']['gen_pass_corr'] + defended['adversarial_blocked']
    sys_acc_nn2      = sys_correct_nn2 / (g_total + a_total) * 100

    # ==============================================================================
    # PLOTS
    # ==============================================================================
    all_present = sorted(
        set(baseline['per_attack'].keys()) |
        set(defended['per_attack'].keys())
    )
    families_ordered = (
        [f for f in ALL_ATTACK_FAMILIES if f in all_present] +
        [f for f in all_present if f not in ALL_ATTACK_FAMILIES]
    )

    # --- Plot 1: Per-Attack Detection Rate (Detector Only) ---
    fig, ax = plt.subplots(figsize=(max(10, len(families_ordered) * 2), 6))
    fig.suptitle(f'Detector Block Rate per Attack Family ({DETECTOR_TO_USE})', fontsize=13, fontweight='bold')

    x = np.arange(len(families_ordered))
    width = 0.5
    det_rates = []
    
    for f in families_ordered:
        d = defended['per_attack'].get(f, {'blocked': 0, 'total': 0})
        total = d['total'] if d['total'] > 0 else 1
        det_rates.append(d['blocked'] / total * 100)

    bars_d = ax.bar(x, det_rates, width, color="#ff8d03", alpha=0.85, edgecolor='white', linewidth=0.5)

    for bar, rate in zip(bars_d, det_rates):
        ax.annotate(f'{rate:.1f}%', xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()), xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(families_ordered, fontsize=11)
    ax.set_xlabel('Attack Family', fontsize=12)
    ax.set_ylabel('Detection Rate (%)', fontsize=12)
    ax.set_ylim(0, 115)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    det_rate_path = os.path.join(DETECTOR_TEST_DIR, 'system_eval_attack_detection_rates.png')
    plt.savefig(det_rate_path, dpi=150, bbox_inches='tight')
    plt.show()


    # ==============================================================================
    # GRAFICI SPECIFICI PER RETE (Ciclo For su NN1 e NN2)
    # ==============================================================================
    
    networks_to_plot = [
        {
            'key': 'nn1', 'name': 'NN1 (VGGFace2)', 
            'sys_acc': sys_acc_nn1, 'gen_acc': genuine_nn1_acc,
            'base_sys_acc': base_sys_acc_nn1, 'base_gen_acc': base_gen_acc_nn1
        },
        {
            'key': 'nn2', 'name': 'NN2 (ResNet-50)', 
            'sys_acc': sys_acc_nn2, 'gen_acc': genuine_nn2_acc,
            'base_sys_acc': base_sys_acc_nn2, 'base_gen_acc': base_gen_acc_nn2
        }
    ]

    for net in networks_to_plot:
        net_key = net['key']
        net_name = net['name']
        print(f"\n[INFO] Generazione grafici per {net_name}...")

        # --- Plot 2: Global metrics comparison ---
        fig, axes = plt.subplots(1, 3, figsize=(14, 6))
        fig.suptitle(f'System Evaluation: Baseline vs Defended ({DETECTOR_TO_USE} + {net_name})', fontsize=14, fontweight='bold')

        categories  = [f'Genuine Accuracy\n({net_name} on genuine)', 'Adversarial Block Rate', 'System Accuracy']
        vals_base   = [net['base_gen_acc'], 0.0, net['base_sys_acc']]
        vals_defend = [net['gen_acc'], adv_block_rate, net['sys_acc']]
        colors_base = ['#1f77b4', '#d62728', '#2ca02c']
        colors_def  = ['#aec7e8', '#ffbb78', '#98df8a']

        for ax, cat, vb, vd, cb, cd in zip(axes, categories, vals_base, vals_defend, colors_base, colors_def):
            bars = ax.bar(['Baseline', 'Defended'], [vb, vd], color=[cb, cd], width=0.5, edgecolor='black', linewidth=0.8)
            ax.set_title(cat, fontsize=11)
            ax.set_ylim(0, 110)
            ax.set_ylabel('%')
            ax.grid(True, axis='y', alpha=0.3)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            for bar in bars:
                ax.annotate(f'{bar.get_height():.1f}%', xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()), xytext=(0, 4), textcoords='offset points', ha='center', fontsize=11)

        plt.tight_layout()
        acc_cmp_path = os.path.join(DETECTOR_TEST_DIR, f'system_eval_accuracy_comparison_{net_key}.png')
        plt.savefig(acc_cmp_path, dpi=150, bbox_inches='tight')
        plt.show()

        # --- Plot 3: Per-attack outcome breakdown (stacked bar) ---
        fig, ax = plt.subplots(figsize=(max(10, len(families_ordered) * 2), 6))
        fig.suptitle(f'Per-Attack Outcome Breakdown — Defended System ({DETECTOR_TO_USE} + {net_name})', fontsize=13, fontweight='bold')

        blocked_rates = []
        passed_ok_rates = []
        passed_fail_rates = []

        for f in families_ordered:
            d = defended['per_attack'].get(f, {'total': 0, 'blocked': 0, net_key: {'pass_corr': 0, 'pass_wr': 0}})
            total = d['total'] if d['total'] > 0 else 1

            blocked_rates.append(d['blocked'] / total * 100)
            passed_ok_rates.append(d[net_key]['pass_corr'] / total * 100)
            passed_fail_rates.append(d[net_key]['pass_wr'] / total * 100)

        x_s = np.arange(len(families_ordered))
        width = 0.5

        p1 = ax.bar(x_s, blocked_rates, width, label='Blocked by Detector', color='#2ca02c', alpha=0.9, edgecolor='white')
        p2 = ax.bar(x_s, passed_ok_rates, width, bottom=blocked_rates, label=f'Passed → {net_key.upper()} Correct', color='#1f77b4', alpha=0.9, edgecolor='white')
        p3 = ax.bar(x_s, passed_fail_rates, width, bottom=[a + b for a, b in zip(blocked_rates, passed_ok_rates)], label=f'Passed → {net_key.upper()} Wrong (ASR)', color='#d62728', alpha=0.9, edgecolor='white')

        for i in range(len(families_ordered)):
            if blocked_rates[i] > 5:
                ax.text(x_s[i], blocked_rates[i]/2, f'{blocked_rates[i]:.0f}%', ha='center', va='center', color='white', fontweight='bold', fontsize=9)
            if passed_fail_rates[i] > 5:
                y_pos = blocked_rates[i] + passed_ok_rates[i] + (passed_fail_rates[i]/2)
                ax.text(x_s[i], y_pos, f'{passed_fail_rates[i]:.0f}%', ha='center', va='center', color='white', fontweight='bold', fontsize=9)

        ax.set_xticks(x_s)
        ax.set_xticklabels(families_ordered, fontsize=11)
        ax.set_xlabel('Attack Family', fontsize=12)
        ax.set_ylabel('Percentage of Adversarial Samples (%)', fontsize=12)
        ax.set_ylim(0, 115)
        ax.legend(fontsize=10, loc='upper center', bbox_to_anchor=(0.5, 1.05), ncol=3)
        ax.grid(axis='y', linestyle='--', alpha=0.4)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.tight_layout()
        breakdown_path = os.path.join(DETECTOR_TEST_DIR, f'system_eval_breakdown_stacked_{net_key}.png')
        plt.savefig(breakdown_path, dpi=150, bbox_inches='tight')
        plt.show()

        # --- Plot 4: Attack Success Rate (ASR) ---
        fig, ax = plt.subplots(figsize=(max(10, len(families_ordered) * 2), 6))
        fig.suptitle(f'Attack Success Rate (ASR): Baseline vs Defended ({DETECTOR_TO_USE} + {net_name})\n', fontsize=13, fontweight='bold')

        x = np.arange(len(families_ordered))
        width = 0.35
        asr_baseline = []
        asr_defended = []

        for f in families_ordered:
            b_dict = baseline['per_attack'].get(f, {'total': 0, 'nn1_corr': 0, 'nn2_corr': 0})
            b_total = b_dict['total'] if b_dict['total'] > 0 else 1
            
            # Peschiamo i corretti specifici per la rete corrente (nn1_corr o nn2_corr)
            b_correct = b_dict[f"{net_key}_corr"] 
            asr_baseline.append(((b_total - b_correct) / b_total) * 100)

            d = defended['per_attack'].get(f, {'total': 0, net_key: {'pass_wr': 0}})
            d_total = d['total'] if d['total'] > 0 else 1
            asr_defended.append((d[net_key]['pass_wr'] / d_total) * 100)

        bars_base = ax.bar(x - width / 2, asr_baseline, width, label='Baseline (No Defense)', color='#d62728', alpha=0.85, edgecolor='white', linewidth=0.5)
        bars_def = ax.bar(x + width / 2, asr_defended, width, label=f'With Detector ({DETECTOR_TO_USE})', color='#1f77b4', alpha=0.85, edgecolor='white', linewidth=0.5)

        for bars in [bars_base, bars_def]:
            for bar in bars:
                ax.annotate(f'{bar.get_height():.1f}%', xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()), xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

        ax.set_xticks(x)
        ax.set_xticklabels(families_ordered, fontsize=11)
        ax.set_xlabel('Attack Family', fontsize=12)
        ax.set_ylabel('Attack Success Rate (%)', fontsize=12)
        ax.set_ylim(0, 115)
        ax.legend(fontsize=10)
        ax.grid(axis='y', linestyle='--', alpha=0.4)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.tight_layout()
        asr_path = os.path.join(DETECTOR_TEST_DIR, f'system_eval_attack_success_rate_{net_key}.png')
        plt.savefig(asr_path, dpi=150, bbox_inches='tight')
        plt.show()

    # --- Plot 5: FAR vs FRR (DET Curve equivalent) for Threshold Selection ---
    print("\n[INFO] Calculating FAR vs FRR across different thresholds...")

    # Raccogliamo tutti gli score e le vere label in un'unica passata
    all_det_scores = []
    all_true_labels = []

    with torch.no_grad():
        for x_det, x_nn1, x_nn2, labels, paths in system_loader:
            scores = detector.predict_score(x_det.to(DEVICE)).cpu().numpy()
            all_det_scores.extend(scores)
            all_true_labels.extend(labels.numpy())

    all_det_scores = np.array(all_det_scores)
    all_true_labels = np.array(all_true_labels)

    thresholds = np.linspace(0.01, 0.99, 100)
    far_list = [] # False Acceptance Rate (1 - Adversarial Block Rate)
    frr_list = [] # False Rejection Rate (Genuine FPR)

    adv_mask = (all_true_labels == 1)
    gen_mask = (all_true_labels == 0)

    total_adv = np.sum(adv_mask)
    total_gen = np.sum(gen_mask)

    for t in thresholds:
        preds = (all_det_scores >= t).astype(int)

        # FAR: Quanti attacchi sono passati (predetti 0) sul totale degli attacchi
        if total_adv > 0:
            far = np.sum(preds[adv_mask] == 0) / total_adv * 100
        else:
            far = 0.0

        # FRR: Quanti genuini sono stati bloccati (predetti 1) sul totale dei genuini
        if total_gen > 0:
            frr = np.sum(preds[gen_mask] == 1) / total_gen * 100
        else:
            frr = 0.0

        far_list.append(far)
        frr_list.append(frr)

    # Trova l'EER (Equal Error Rate) approssimato
    diffs = np.abs(np.array(far_list) - np.array(frr_list))
    eer_idx = np.argmin(diffs)
    eer_threshold = thresholds[eer_idx]
    eer_value = (far_list[eer_idx] + frr_list[eer_idx]) / 2

    fig, ax = plt.subplots(figsize=(8, 6))
    fig.suptitle(f'Trade-off: FAR vs FRR ({DETECTOR_TO_USE})', fontsize=14, fontweight='bold')

    ax.plot(thresholds, far_list, label='False Acceptance Rate (Attacks passed)', color='#d62728', linewidth=2)
    ax.plot(thresholds, frr_list, label='False Rejection Rate (Genuine blocked)', color='#1f77b4', linewidth=2)

    # Linea verticale per indicare l'EER
    ax.axvline(x=eer_threshold, color='black', linestyle='--', alpha=0.6, label=f'Equal Error Rate (EER) ~ {eer_value:.1f}%\nat Threshold = {eer_threshold:.2f}')

    # Linea verticale per la soglia attuale
    ax.axvline(x=FINETUNED_DETECTOR_THRESHOLD, color='green', linestyle=':', alpha=0.8, label=f'Current Threshold = {FINETUNED_DETECTOR_THRESHOLD}')

    ax.set_xlabel('Decision Threshold', fontsize=12)
    ax.set_ylabel('Error Rate (%)', fontsize=12)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, max(max(far_list), max(frr_list)) + 5)
    ax.legend(fontsize=10)
    ax.grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    det_curve_path = os.path.join(DETECTOR_TEST_DIR, 'system_eval_far_vs_frr.png')
    plt.savefig(det_curve_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"[INFO] FAR vs FRR trade-off plot saved: {det_curve_path}")
    print(f"[INFO] Estimated EER: {eer_value:.2f}% at threshold {eer_threshold:.2f}")


    # ==============================================================================
    # FINAL SUMMARY (Tabella unificata per NN1 e NN2)
    # ==============================================================================
    print("\n" + "=" * 75)
    print(" SYSTEM EVALUATION SUMMARY")
    print("=" * 75)
    # --- Metriche Condivise (Detector) ---
    print(f"  Detector              : {DETECTOR_TO_USE} (threshold={FINETUNED_DETECTOR_THRESHOLD})")
    print(f"  Genuine FPR (blocked) : {genuine_fpr:.2f}%")
    print(f"  Adv. Block Rate       : {adv_block_rate:.2f}%")
    print("-" * 75)
    # --- Metriche Specifiche per Rete ---
    print(f"  {'Metric (Defended System)':<35} | {'NN1 (VGG)':>15} | {'NN2 (ResNet)':>15}")
    print("-" * 75)
    print(f"  {'Genuine accuracy (passed gate)':<35} | {genuine_nn1_acc:>14.2f}% | {genuine_nn2_acc:>14.2f}%")
    print(f"  {'System accuracy (blocked=corr)':<35} | {sys_acc_nn1:>14.2f}% | {sys_acc_nn2:>14.2f}%")
    print(f"  {'Conditional accuracy (gate pass)':<35} | {cond_acc_nn1:>14.2f}% | {cond_acc_nn2:>14.2f}%")
    print()
    # --- Breakdown per Attacco ---
    print("  Per-attack breakdown (Defended System):")
    print(f"  {'Family':<10} {'Total':>6} {'Blocked':>8} {'Block%':>7} | {'NN1 PassFail':>12} | {'NN2 PassFail':>12}")
    print("-" * 75)
    for f in families_ordered:
        d = defended['per_attack'].get(
            f, {'total': 0, 'blocked': 0, 'nn1': {'pass_wr': 0}, 'nn2': {'pass_wr': 0}}
        )
        if d['total'] == 0:
            print(f"  {f:<10} {'N/A':>62}")
            continue
        block_pct = d['blocked'] / d['total'] * 100
        # Calcoliamo i fallimenti (PassFail = Passano il gate e ingannano la rete)
        nn1_fail = d['nn1']['pass_wr']
        nn1_fail_pct = nn1_fail / d['total'] * 100
        nn2_fail = d['nn2']['pass_wr']
        nn2_fail_pct = nn2_fail / d['total'] * 100
        # Formattiamo le stringhe per avere "Numero (Percentuale%)"
        nn1_str = f"{nn1_fail} ({nn1_fail_pct:.1f}%)"
        nn2_str = f"{nn2_fail} ({nn2_fail_pct:.1f}%)"
        print(f"  {f:<10} {d['total']:>6} {d['blocked']:>8} {block_pct:>6.1f}% | {nn1_str:>12} | {nn2_str:>12}")
    print("=" * 75)

## Step 6.2 — JPEG Compression Defense

### 6.2.1 A — JPEG Compression Defense: Full Dataset Evaluation with Attack Breakdown

This cell evaluates the JPEG compression defense on all images in `DEFENSE_DATASET_DIR`,
grouped by attack family. For each family and each JPEG quality level, two metrics are
reported on NN1:
- **Accuracy** on adversarial images: the fraction correctly re-classified after defense.
- **ASR** (Attack Success Rate) for Targeted Fixed attacks: the fraction of adversarial
  images still predicted as `FIXED_TARGET_CLASS` after defense is applied.

The genuine image accuracy (cost of the defense on clean inputs) is reported once,
aggregated over all genuine images regardless of origin. The undefended baseline is
always computed first for direct delta comparison.

Attack family assignment is derived from the filename prefix of each adversarial image,
using the naming convention established during dataset construction:
`{attack}__{subfolder}__{class_id}__{filename}.ext`.

#### Setup & Helpers

In [ ]:
# ------------------------------------------------------------------------------
# Classifiers with JPEG Compression Defense
# ------------------------------------------------------------------------------

criterion_defense = nn.CrossEntropyLoss()

jpeg_classifiers = {}
for q in JPEG_QUALITY_VALUES:
    jpeg_defence = JpegCompression(
        clip_values=(0.0, 1.0),
        quality=q,
        apply_fit=False,
        apply_predict=True,
        channels_first=True,
    )
    jpeg_classifiers[q] = PyTorchClassifier(
        model=nn1,
        clip_values=(0.0, 1.0),
        loss=criterion_defense,
        optimizer=None,
        input_shape=(3, 160, 160),
        nb_classes=NUM_CLASSES,
        preprocessing=(np.array([0.5, 0.5, 0.5]), np.array([0.5, 0.5, 0.5])),
        preprocessing_defences=[jpeg_defence],
        device_type=DEVICE,
    )

print("JPEG defense classifiers initialized for quality values:", JPEG_QUALITY_VALUES)



# ------------------------------------------------------------------------------
# JPEG Compression Defense: Evaluation on Detector Test Set
# ------------------------------------------------------------------------------

def clean_label(raw):
    """Apply the same label cleaning pipeline used across the project."""
    label = raw
    if isinstance(label, np.ndarray):
        label = label.flatten()[0]
    if isinstance(label, bytes):
        label = label.decode('utf-8')
    label = str(label).replace("b'", "").replace("'", "").replace("[", "").replace("]", "").strip()
    return normalize_label(label)




# ------------------------------------------------------------------------------
# Step 6.1a — JPEG Compression Defense: Full Dataset Evaluation
# ------------------------------------------------------------------------------

# ------------------------------------------------------------------------------
# Family mapping: folder prefix -> canonical family name + metric type
# Targeted Fixed families also carry ASR tracking.
# ------------------------------------------------------------------------------
FAMILY_MAP = {
    'FGSM__eps':                          ('FGSM_Untargeted',      'accuracy'),
    'FGSM_Targeted_Fixed__eps':           ('FGSM_Targeted_Fixed',  'asr'),
    'BIM_iter':                           ('BIM_Untargeted',       'accuracy'),
    'BIM_Targeted_Fixed':                 ('BIM_Targeted_Fixed',   'asr'),
    'PGD_nri':                            ('PGD_Untargeted',       'accuracy'),
    'PGD_Targeted_Fixed':                 ('PGD_Targeted_Fixed',   'asr'),
    'CW_Untargeted':                      ('CW_Untargeted',        'accuracy'),
    'DeepFool':                           ('DeepFool_Untargeted',  'accuracy'),
}

def get_family(filename: str):
    """
    Maps an adversarial image filename to its canonical attack family name
    and metric type by matching against the FAMILY_MAP prefix keys.

    Parameters
    ----------
    filename : str — basename of the adversarial image file

    Returns
    -------
    tuple (family_name: str, metric: str) or (None, None) if no match found
    """
    for prefix, (family, metric) in FAMILY_MAP.items():
        if filename.startswith(prefix):
            return family, metric
    return None, None


def evaluate_full_dataset(classifier):
    """
    Evaluate NN1 accuracy and ASR on all images in DEFENSE_DATASET_DIR
    using the provided ART classifier.

    Genuine images (label=0) contribute to a single aggregated accuracy counter.
    Adversarial images (label=1) are bucketed by attack family; each family
    tracks correct classifications and, for Targeted Fixed families, ASR counts.

    Parameters
    ----------
    classifier : PyTorchClassifier — ART classifier (defended or undefended)

    Returns
    -------
    dict with structure:
    {
        'genuine': {'correct': int, 'total': int},
        'families': {
            family_name: {
                'correct':   int,
                'asr_count': int,
                'total':     int,
                'metric':    str   # 'accuracy' or 'asr'
            }, ...
        }
    }
    """
    genuine_dir     = os.path.join(DEFENSE_DATASET_DIR, 'genuine')
    adversarial_dir = os.path.join(DEFENSE_DATASET_DIR, 'adversarial')

    results = {
        'genuine':  {'correct': 0, 'total': 0},
        'families': {},
    }

    # ------------------------------------------------------------------
    # Genuine images
    # ------------------------------------------------------------------
    for fname in sorted(os.listdir(genuine_dir)):
        if not fname.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue
        try:
            class_id   = extract_class_id_genuine(fname)
            true_clean = normalize_label(id_to_name.get(class_id.strip(), 'Unknown').strip())
            img_path   = os.path.join(genuine_dir, fname)

            img      = Image.open(img_path).convert('RGB')
            tensor   = preprocess_attack(img).unsqueeze(0).numpy()
            pred_idx = int(np.argmax(classifier.predict(tensor), axis=1)[0])
            pred_lbl = clean_label(LABELS[pred_idx]) if pred_idx < len(LABELS) else 'unknown'

            if pred_lbl == true_clean:
                results['genuine']['correct'] += 1
            results['genuine']['total'] += 1
        except Exception as e:
            print(f"Error processing genuine {fname}: {e}")

    # ------------------------------------------------------------------
    # Adversarial images — bucketed by family
    # ------------------------------------------------------------------
    for fname in sorted(os.listdir(adversarial_dir)):
        if not fname.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue

        family, metric = get_family(fname)
        if family is None:
            continue

        if family not in results['families']:
            results['families'][family] = {
                'correct':   0,
                'asr_count': 0,
                'total':     0,
                'metric':    metric,
            }

        try:
            class_id   = extract_class_id_adversarial(fname)
            true_clean = normalize_label(id_to_name.get(class_id.strip(), 'Unknown').strip())
            img_path   = os.path.join(adversarial_dir, fname)

            img      = Image.open(img_path).convert('RGB')
            tensor   = preprocess_attack(img).unsqueeze(0).numpy()
            pred_idx = int(np.argmax(classifier.predict(tensor), axis=1)[0])
            pred_lbl = clean_label(LABELS[pred_idx]) if pred_idx < len(LABELS) else 'unknown'

            if pred_lbl == true_clean:
                results['families'][family]['correct'] += 1
            if pred_idx == FIXED_TARGET_CLASS:
                results['families'][family]['asr_count'] += 1

            results['families'][family]['total'] += 1
        except Exception as e:
            print(f"Error processing adversarial {fname}: {e}")

    return results



#### Evaluation

In [ ]:
if RUN_JPEG_DEFENSE_EVAL:

    # ------------------------------------------------------------------------------
    # Run evaluation: no defense + all JPEG quality levels
    # ------------------------------------------------------------------------------
    full_eval_results = {}

    all_configs = ['no_defense'] + JPEG_QUALITY_VALUES
    for cfg in all_configs:
        label     = 'No Defense' if cfg == 'no_defense' else f'JPEG q={cfg}'
        clf       = art_classifier if cfg == 'no_defense' else jpeg_classifiers[cfg]

        print(f"\n{'=' * 60}")
        print(f"Evaluating: {label}")
        print('=' * 60)
        t0 = time.time()

        res = evaluate_full_dataset(clf)
        full_eval_results[cfg] = res

    g_acc = res['genuine']['correct'] / res['genuine']['total'] if res['genuine']['total'] > 0 else 0.0

    if cfg == 'no_defense':
        print(f"  Genuine accuracy: {g_acc * 100:.2f}%  [{res['genuine']['total']} images]")
    else:
        nd_g_acc = (full_eval_results['no_defense']['genuine']['correct'] /
                    max(full_eval_results['no_defense']['genuine']['total'], 1))
        print(f"  Genuine accuracy: {g_acc * 100:.2f}%  "
              f"(delta: {(g_acc - nd_g_acc) * 100:+.2f}%)  " f"[{res['genuine']['total']} images]")

    for family, fdata in sorted(res['families'].items()):
        metric = fdata['metric']
        total  = fdata['total']
        if total == 0:
            continue

        if metric == 'accuracy':
            score = fdata['correct'] / total
        else:
            score = fdata['asr_count'] / total

        if cfg == 'no_defense':
            delta_str = ''
        else:
            nd_fdata = full_eval_results['no_defense']['families'].get(
                family, {'correct': 0, 'asr_count': 0, 'total': 0}
            )
            nd_tot   = max(nd_fdata['total'], 1)
            nd_score = (nd_fdata['asr_count'] if metric == 'asr'
                        else nd_fdata['correct']) / nd_tot
            delta_str = f"  (delta: {(score - nd_score) * 100:+.2f}%)"

        metric_label = 'Accuracy' if metric == 'accuracy' else 'ASR     '
        print(f"  {family:<30} {metric_label} = {score * 100:.2f}%{delta_str}  [{total} imgs]")

        print(f"  Elapsed: {time.time() - t0:.1f}s")

### Step 6.2.1 B — JPEG Compression Defense: Test Set Evaluation (Common Ground)

This cell evaluates the JPEG compression defense on the same test split used to assess
the adversarial detector, enabling a direct comparison between the two defense mechanisms
on identical inputs. The test set is identity-disjoint from the training and validation
splits and contains both genuine and adversarial images drawn from all attack families.

Unlike Step 6.1a, no per-attack breakdown is produced here: the goal is a single
aggregated score on genuine and adversarial images that is directly comparable to the
detector's test-set metrics.

In [ ]:
if RUN_JPEG_DEFENSE_EVAL:

    # --------------------------------------------------------------------------
    # Step 6.2.1 B — JPEG Compression Defense: Test Set Evaluation
    # --------------------------------------------------------------------------

    def evaluate_on_file_list_with_breakdown(file_list, classifier):
        """
        Evaluates NN1 accuracy on a list of (abs_path, label) tuples using
        the provided ART classifier, with per-attack family breakdown for
        adversarial images.

        Ground-truth identity is extracted from the filename using the naming
        conventions established in the detector dataset pipeline.

        Parameters
        ----------
        file_list  : list of (abs_path: str, label: int) tuples.
        classifier : PyTorchClassifier — ART classifier (defended or undefended).

        Returns
        -------
        dict with structure:
        {
            'genuine':  {'correct': int, 'total': int},
            'families': {
                family_name: {'correct': int, 'total': int}
            }
        }
        """
        results = {
            'genuine':  {'correct': 0, 'total': 0},
            'families': {}
        }

        for img_path, label in file_list:
            fname    = os.path.basename(img_path)
            class_id = (extract_class_id_genuine(fname)
                        if fname.startswith(('orig__', 'extra__'))
                        else extract_class_id_adversarial(fname))

            true_clean = normalize_label(
                id_to_name.get(class_id.strip(), 'Unknown').strip()
            )

            try:
                img      = Image.open(img_path).convert('RGB')
                tensor   = preprocess_attack(img).unsqueeze(0).numpy()
                pred_idx = int(np.argmax(classifier.predict(tensor), axis=1)[0])
                pred_lbl = (clean_label(LABELS[pred_idx])
                            if pred_idx < len(LABELS) else 'unknown')
                is_correct = (pred_lbl == true_clean)

                if label == 0:
                    results['genuine']['correct'] += int(is_correct)
                    results['genuine']['total']   += 1
                else:
                    family = get_attack_family(img_path)
                    if family is None:
                        continue
                    if family not in results['families']:
                        results['families'][family] = {'correct': 0, 'total': 0}
                    results['families'][family]['correct'] += int(is_correct)
                    results['families'][family]['total']   += 1

            except Exception as e:
                print(f"[ERROR] {img_path}: {e}")

        return results

    # Separate genuine and adversarial entries from the test split
    test_genuine     = [(p, l) for p, l in test_files if l == 0]
    test_adversarial = [(p, l) for p, l in test_files if l == 1]

    print(f"Test set: {len(test_genuine)} genuine, {len(test_adversarial)} adversarial")
    print("=" * 60)

    testset_eval_results = {}

    for cfg in all_configs:
        label = 'No Defense' if cfg == 'no_defense' else f'JPEG q={cfg}'
        clf   = art_classifier if cfg == 'no_defense' else jpeg_classifiers[cfg]

        res = evaluate_on_file_list_with_breakdown(
            test_genuine + test_adversarial, clf
        )
        testset_eval_results[cfg] = res

        g_tot  = res['genuine']['total']
        g_acc  = res['genuine']['correct'] / g_tot if g_tot > 0 else 0.0

        adv_correct = sum(f['correct'] for f in res['families'].values())
        adv_total   = sum(f['total']   for f in res['families'].values())
        a_acc       = adv_correct / adv_total if adv_total > 0 else 0.0

        nd_g = (testset_eval_results['no_defense']['genuine']['correct'] /
                max(testset_eval_results['no_defense']['genuine']['total'], 1))
        nd_a_corr  = sum(f['correct'] for f in
                         testset_eval_results['no_defense']['families'].values())
        nd_a_total = sum(f['total']   for f in
                         testset_eval_results['no_defense']['families'].values())
        nd_a = nd_a_corr / nd_a_total if nd_a_total > 0 else 0.0

        print(f"\n  {label}:")
        if cfg == 'no_defense':
            print(f"    Genuine accuracy    : {g_acc * 100:.2f}%")
            print(f"    Adversarial accuracy: {a_acc * 100:.2f}%")
        else:
            print(f"    Genuine accuracy    : {g_acc * 100:.2f}%"
                  f"  (delta: {(g_acc - nd_g) * 100:+.2f}%)")
            print(f"    Adversarial accuracy: {a_acc * 100:.2f}%"
                  f"  (delta: {(a_acc - nd_a) * 100:+.2f}%)")

        for family in sorted(res['families']):
            fd    = res['families'][family]
            f_acc = fd['correct'] / fd['total'] if fd['total'] > 0 else 0.0
            if cfg != 'no_defense':
                nd_fd    = testset_eval_results['no_defense']['families'].get(
                    family, {'correct': 0, 'total': 0}
                )
                nd_f_acc = (nd_fd['correct'] / nd_fd['total']
                            if nd_fd['total'] > 0 else 0.0)
                delta_str = f"  (delta: {(f_acc - nd_f_acc) * 100:+.2f}%)"
            else:
                delta_str = ''
            print(f"      {family:<12}: {f_acc * 100:.2f}%{delta_str}"
                  f"  [{fd['total']} imgs]")

### Step 6.2.2 A JPEG Defense: Logging — Full Dataset Results

Results from Step 6.1a are saved to a CSV file and visualized as two grouped bar charts:
one for error-generic attack families (metric: Accuracy) and one for Targeted Fixed
families (metric: ASR). For each family, four bars are shown side by side: no defense,
JPEG q=10, q=50, q=80. A separate bar group reports genuine image accuracy.
Lower ASR bars and higher Accuracy bars both indicate effective defense.

In [ ]:
if RUN_JPEG_DEFENSE_EVAL:
    # ------------------------------------------------------------------------------
    # Step 6.2.2 B — JPEG Defense: Logging — Full Dataset Results
    # ------------------------------------------------------------------------------

    DEFENSE_RESULTS_DIR = os.path.join(RESULTS_DIR, 'JPEG_Defense')
    os.makedirs(DEFENSE_RESULTS_DIR, exist_ok=True)

    bar_labels = ['No Defense'] + [f'JPEG q={q}' for q in JPEG_QUALITY_VALUES]
    bar_colors = ['#444444', '#1f77b4', '#2ca02c', '#d62728']
    n_bars     = len(bar_labels)
    width      = 0.18
    offsets    = np.linspace(-(n_bars - 1) / 2 * width,
                            (n_bars - 1) / 2 * width, n_bars)

    # Collect all family names present across all configs
    all_families = sorted({
        fam
        for cfg in full_eval_results.values()
        for fam in cfg['families']
    })

    generic_families  = [f for f in all_families
                        if full_eval_results['no_defense']['families'].get(f, {}).get('metric') == 'accuracy']
    targeted_families = [f for f in all_families
                        if full_eval_results['no_defense']['families'].get(f, {}).get('metric') == 'asr']

    # ------------------------------------------------------------------
    # 6.3a.1  Save CSV
    # ------------------------------------------------------------------
    csv_path = os.path.join(DEFENSE_RESULTS_DIR, 'jpeg_defense_full_dataset.csv')

    with open(csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Family', 'Metric', 'No_Defense_%'] +
                        [f'JPEG_q{q}_%' for q in JPEG_QUALITY_VALUES] +
                        [f'Delta_q{q}_%' for q in JPEG_QUALITY_VALUES])

        # Genuine row
        g_nd = full_eval_results['no_defense']['genuine']['correct'] / \
            max(full_eval_results['no_defense']['genuine']['total'], 1) * 100
        g_scores = []
        for q in JPEG_QUALITY_VALUES:
            gt = full_eval_results[q]['genuine']['total']
            g_scores.append(
                full_eval_results[q]['genuine']['correct'] / max(gt, 1) * 100
            )
        writer.writerow(
            ['Genuine', 'accuracy', f'{g_nd:.2f}'] +
            [f'{s:.2f}' for s in g_scores] +
            [f'{s - g_nd:+.2f}' for s in g_scores]
        )

        # Family rows
        for family in all_families:
            metric  = full_eval_results['no_defense']['families'].get(family, {}).get('metric', 'accuracy')
            nd_data = full_eval_results['no_defense']['families'].get(family, {'correct': 0, 'asr_count': 0, 'total': 0})
            nd_tot  = max(nd_data['total'], 1)
            nd_score = (nd_data['asr_count'] if metric == 'asr' else nd_data['correct']) / nd_tot * 100

            scores = []
            for q in JPEG_QUALITY_VALUES:
                fdata = full_eval_results[q]['families'].get(family, {'correct': 0, 'asr_count': 0, 'total': 0})
                tot   = max(fdata['total'], 1)
                scores.append(
                    (fdata['asr_count'] if metric == 'asr' else fdata['correct']) / tot * 100
                )

            writer.writerow(
                [family, metric, f'{nd_score:.2f}'] +
                [f'{s:.2f}' for s in scores] +
                [f'{s - nd_score:+.2f}' for s in scores]
            )

    print(f"CSV saved to: {csv_path}")

    # ------------------------------------------------------------------
    # 6.3a.2  Plots
    # ------------------------------------------------------------------
    def _bar_group_plot(ax, families, get_score_fn, ylabel, title):
        """Render a grouped bar chart for a list of attack families."""
        group_labels = ['Genuine'] + [f.replace('_', '\n') for f in families]
        x_pos        = np.arange(len(group_labels))

        for bar_idx, (blabel, color) in enumerate(zip(bar_labels, bar_colors)):
            cfg = 'no_defense' if bar_idx == 0 else JPEG_QUALITY_VALUES[bar_idx - 1]
            vals = [
                full_eval_results[cfg]['genuine']['correct'] /
                max(full_eval_results[cfg]['genuine']['total'], 1) * 100
            ] + [get_score_fn(cfg, fam) for fam in families]

            bars = ax.bar(x_pos + offsets[bar_idx], vals, width=width,
                        color=color, label=blabel,
                        edgecolor='white', linewidth=0.5)
            for bar, val in zip(bars, vals):
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + 0.5,
                        f'{val:.0f}',
                        ha='center', va='bottom', fontsize=6.5)

        # Clean baseline on Genuine group
        ax.hlines(BASELINE_ACC_1000 * 100,
                x_pos[0] - 0.4, x_pos[0] + 0.4,
                colors='black', linestyles='dashed', linewidth=1.0)

        ax.set_xticks(x_pos)
        ax.set_xticklabels(group_labels, fontsize=8)
        ax.set_ylabel(ylabel)
        ax.set_title(title, fontweight='bold')
        ax.set_ylim(0, 115)
        ax.legend(fontsize=8, loc='upper right')
        ax.grid(axis='y', linestyle='--', alpha=0.4)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)


    def _get_acc(cfg, fam):
        fdata = full_eval_results[cfg]['families'].get(fam, {'correct': 0, 'total': 0})
        return fdata['correct'] / max(fdata['total'], 1) * 100


    def _get_asr(cfg, fam):
        fdata = full_eval_results[cfg]['families'].get(fam, {'asr_count': 0, 'total': 0})
        return fdata['asr_count'] / max(fdata['total'], 1) * 100


    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle('JPEG Compression Defense — Full Dataset Results',
                fontsize=14, fontweight='bold', y=1.01)

    _bar_group_plot(axes[0], generic_families,  _get_acc,
                    'Accuracy (%)', 'Error-Generic Attacks')
    _bar_group_plot(axes[1], targeted_families, _get_asr,
                    'ASR (%)',       'Targeted Fixed Attacks')

    plt.tight_layout()
    plot_path = os.path.join(DEFENSE_RESULTS_DIR, 'jpeg_defense_full_dataset.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Plot saved to: {plot_path}")

### Step 6.2.2 B JPEG Defense: Logging — Test Set Results (Common Ground)

Results from Step 6.1b are saved to a CSV file and visualized as a grouped bar chart.
This is the evaluation on the same test split used for the adversarial detector,
enabling direct comparison between the two defense mechanisms. Only two aggregate
metrics are reported: accuracy on genuine images and accuracy on adversarial images.

In [ ]:
if RUN_JPEG_DEFENSE_EVAL:
    # --------------------------------------------------------------------------
    # Step 6.3b — JPEG Defense: Logging — Test Set Results (Common Ground)
    # --------------------------------------------------------------------------

    FAMILY_ORDER_PLOT = ['FGSM', 'BIM', 'PGD', 'CW', 'DeepFool']

    all_families_present = sorted(
        set(f for cfg in all_configs
            for f in testset_eval_results[cfg]['families'])
    )
    families_ordered = (
        [f for f in FAMILY_ORDER_PLOT if f in all_families_present] +
        [f for f in all_families_present if f not in FAMILY_ORDER_PLOT]
    )

    def _get_accs(cfg):
        """Returns (genuine_acc_pct, adversarial_acc_pct) for a config."""
        res     = testset_eval_results[cfg]
        g_tot   = res['genuine']['total']
        g_acc   = res['genuine']['correct'] / g_tot if g_tot > 0 else 0.0
        a_corr  = sum(f['correct'] for f in res['families'].values())
        a_total = sum(f['total']   for f in res['families'].values())
        a_acc   = a_corr / a_total if a_total > 0 else 0.0
        return g_acc * 100, a_acc * 100

    genuine_vals = [_get_accs(k)[0] for k in all_configs]
    adv_vals     = [_get_accs(k)[1] for k in all_configs]
    nd_g_acc, nd_a_acc = _get_accs('no_defense')

    # ------------------------------------------------------------------
    # 6.3b.1  Save aggregate CSV
    # ------------------------------------------------------------------
    csv_path_ts = os.path.join(DEFENSE_RESULTS_DIR, 'jpeg_defense_testset.csv')
    with open(csv_path_ts, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Configuration', 'Genuine_Accuracy_%',
                         'Adversarial_Accuracy_%',
                         'Delta_Genuine_%', 'Delta_Adversarial_%'])
        for cfg in all_configs:
            lbl   = 'No Defense' if cfg == 'no_defense' else f'JPEG q={cfg}'
            g_acc, a_acc = _get_accs(cfg)
            dg = f'{g_acc - nd_g_acc:+.2f}' if cfg != 'no_defense' else '—'
            da = f'{a_acc - nd_a_acc:+.2f}' if cfg != 'no_defense' else '—'
            writer.writerow([lbl, f'{g_acc:.2f}', f'{a_acc:.2f}', dg, da])
    print(f"[INFO] Aggregate CSV saved: {csv_path_ts}")

    # ------------------------------------------------------------------
    # 6.3b.2  Save per-attack CSV
    # ------------------------------------------------------------------
    csv_path_pa = os.path.join(DEFENSE_RESULTS_DIR,
                               'jpeg_defense_testset_per_attack.csv')
    with open(csv_path_pa, 'w', newline='') as f:
        writer = csv.writer(f)
        header = (
            ['Family'] +
            [('No_Defense_%' if c == 'no_defense' else f'JPEG_q{c}_%')
             for c in all_configs] +
            [f'Delta_q{c}_%' for c in all_configs if c != 'no_defense']
        )
        writer.writerow(header)
        for family in families_ordered:
            nd_fd    = testset_eval_results['no_defense']['families'].get(
                family, {'correct': 0, 'total': 0})
            nd_f_acc = (nd_fd['correct'] / nd_fd['total'] * 100
                        if nd_fd['total'] > 0 else 0.0)
            row  = [family]
            accs = []
            for cfg in all_configs:
                fd    = testset_eval_results[cfg]['families'].get(
                    family, {'correct': 0, 'total': 0})
                f_acc = (fd['correct'] / fd['total'] * 100
                         if fd['total'] > 0 else 0.0)
                accs.append(f_acc)
                row.append(f'{f_acc:.2f}')
            for cfg, f_acc in zip(all_configs, accs):
                if cfg != 'no_defense':
                    row.append(f'{f_acc - nd_f_acc:+.2f}')
            writer.writerow(row)
    print(f"[INFO] Per-attack CSV saved: {csv_path_pa}")

    # ------------------------------------------------------------------
    # 6.3b.3  Plot 1 — Aggregate: Genuine vs Adversarial accuracy
    # ------------------------------------------------------------------
    x          = np.arange(2)
    n_bars     = len(all_configs)
    width      = 0.8 / n_bars
    offsets_ts = np.linspace(
        -(n_bars - 1) / 2 * width,
         (n_bars - 1) / 2 * width, n_bars)

    fig, ax = plt.subplots(figsize=(10, 6))
    fig.suptitle(
        'JPEG Compression Defense — Test Set Results (Common Ground)',
        fontsize=13, fontweight='bold')

    for i, (blabel, color, gv, av) in enumerate(
            zip(bar_labels, bar_colors, genuine_vals, adv_vals)):
        vals = [gv, av]
        bars = ax.bar(x + offsets_ts[i], vals, width=width,
                      color=color, label=blabel,
                      edgecolor='white', linewidth=0.5)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.5, f'{val:.1f}',
                    ha='center', va='bottom', fontsize=8)

    ax.hlines(BASELINE_ACC_1000 * 100, x[0] - 0.4, x[0] + 0.4,
              colors='black', linestyles='dashed', linewidth=1.2,
              label=f'Clean baseline ({BASELINE_ACC_1000 * 100:.1f}%)')
    ax.set_xticks(x)
    ax.set_xticklabels(['Genuine Images', 'Adversarial Images'], fontsize=12)
    ax.set_ylabel('NN1 Accuracy (%)')
    ax.set_ylim(0, 115)
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plot_path_ts = os.path.join(DEFENSE_RESULTS_DIR, 'jpeg_defense_testset.png')
    plt.savefig(plot_path_ts, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"[INFO] Aggregate plot saved: {plot_path_ts}")

    # ------------------------------------------------------------------
    # 6.3b.4  Per-quality plots — one figure per JPEG quality value
    #
    # For each quality level q, produces three plots mirroring the
    # detector evaluation:
    #   Plot A — System Evaluation: Baseline vs Defended (3 bar panels)
    #   Plot B — Attack Success Rate: Baseline vs JPEG q=X
    #   Plot C — Per-Attack Outcome Breakdown (stacked bars)
    # ------------------------------------------------------------------
    jpeg_configs = [c for c in all_configs if c != 'no_defense']

    def _family_acc(cfg, family):
        """NN1 accuracy on adversarial images of a given family, in %."""
        fd = testset_eval_results[cfg]['families'].get(
            family, {'correct': 0, 'total': 0})
        return fd['correct'] / fd['total'] * 100 if fd['total'] > 0 else 0.0

    def _family_asr(cfg, family):
        """Attack Success Rate = 100 - accuracy, in %."""
        return 100.0 - _family_acc(cfg, family)

    for q in jpeg_configs:
        q_label = f'JPEG q={q}'
        g_acc_q, a_acc_q = _get_accs(q)

        # ---- Plot A: System Evaluation — 3 panels ----
        fig, axes = plt.subplots(1, 3, figsize=(14, 6))
        fig.suptitle(
            f'System Evaluation: Baseline vs Defended ({q_label})',
            fontsize=14, fontweight='bold')

        categories  = ['Genuine Accuracy\n(NN1 on genuine)',
                        'Adversarial Block Rate\n(Accuracy Recovery)',
                        'Overall System Accuracy']
        vals_base   = [nd_g_acc,  0.0,       nd_a_acc]
        vals_defend = [g_acc_q,   a_acc_q,   (g_acc_q + a_acc_q) / 2]
        colors_base = ['#1f77b4', '#d62728', '#2ca02c']
        colors_def  = ['#aec7e8', '#ffbb78', '#98df8a']

        # Note: 'Adversarial Block Rate' for JPEG is reinterpreted as
        # adversarial accuracy recovery (NN1 acc on adv after JPEG).
        # Baseline is 0% because without defense NN1 fails on adversarial.
        # Overall system accuracy = mean of genuine and adversarial accuracy.

        for ax, cat, vb, vd, cb, cd in zip(
                axes, categories, vals_base, vals_defend,
                colors_base, colors_def):
            bars = ax.bar(['Baseline', 'Defended'], [vb, vd],
                          color=[cb, cd], width=0.5,
                          edgecolor='black', linewidth=0.8)
            ax.set_title(cat, fontsize=11)
            ax.set_ylim(0, 110)
            ax.set_ylabel('%')
            ax.grid(True, axis='y', alpha=0.3)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            for bar in bars:
                ax.annotate(
                    f'{bar.get_height():.1f}%',
                    xy=(bar.get_x() + bar.get_width() / 2,
                        bar.get_height()),
                    xytext=(0, 4), textcoords='offset points',
                    ha='center', fontsize=11)

        plt.tight_layout()
        p = os.path.join(DEFENSE_RESULTS_DIR,
                         f'jpeg_defense_testset_system_eval_q{q}.png')
        plt.savefig(p, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"[INFO] System eval plot saved: {p}")

        # ---- Plot B: ASR — Baseline vs JPEG q ----
        x_asr = np.arange(len(families_ordered))
        w_asr = 0.35

        asr_base = [_family_asr('no_defense', f) for f in families_ordered]
        asr_jpeg = [_family_asr(q,            f) for f in families_ordered]

        fig, ax = plt.subplots(figsize=(max(10, len(families_ordered) * 2), 6))
        fig.suptitle(
            f'Attack Success Rate: Baseline vs {q_label} (Test Set)',
            fontsize=13, fontweight='bold')

        bars_b = ax.bar(x_asr - w_asr / 2, asr_base, w_asr,
                        label='Baseline (No Defense)',
                        color='#d62728', alpha=0.85,
                        edgecolor='white', linewidth=0.5)
        bars_d = ax.bar(x_asr + w_asr / 2, asr_jpeg, w_asr,
                        label=q_label,
                        color='#1f77b4', alpha=0.85,
                        edgecolor='white', linewidth=0.5)

        for bars in (bars_b, bars_d):
            for bar in bars:
                h = bar.get_height()
                ax.annotate(f'{h:.1f}%',
                            xy=(bar.get_x() + bar.get_width() / 2, h),
                            xytext=(0, 3), textcoords='offset points',
                            ha='center', fontsize=9)

        ax.set_xticks(x_asr)
        ax.set_xticklabels(families_ordered, fontsize=11)
        ax.set_xlabel('Attack Family', fontsize=12)
        ax.set_ylabel('Attack Success Rate (%)', fontsize=12)
        ax.set_ylim(0, 115)
        ax.legend(fontsize=10)
        ax.grid(axis='y', linestyle='--', alpha=0.4)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.tight_layout()
        p = os.path.join(DEFENSE_RESULTS_DIR,
                         f'jpeg_defense_testset_asr_q{q}.png')
        plt.savefig(p, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"[INFO] ASR plot saved: {p}")

        # ---- Plot C: Stacked breakdown — Baseline vs JPEG q ----
        fig, axes = plt.subplots(
            1, 2,
            figsize=(max(14, len(families_ordered) * 3), 6),
            sharey=True)
        fig.suptitle(
            f'Per-Attack Outcome Breakdown — No Defense vs {q_label} (Test Set)',
            fontsize=13, fontweight='bold')

        for ax, cfg, title in zip(
                axes,
                ['no_defense', q],
                ['No Defense', q_label]):
            corr_rates = []
            fail_rates = []
            for family in families_ordered:
                fd    = testset_eval_results[cfg]['families'].get(
                    family, {'correct': 0, 'total': 0})
                total = fd['total'] if fd['total'] > 0 else 1
                corr_rates.append(fd['correct'] / total * 100)
                fail_rates.append(
                    (total - fd['correct']) / total * 100)

            x_s   = np.arange(len(families_ordered))
            w_stk = 0.5

            ax.bar(x_s, corr_rates, w_stk,
                   label='NN1 Correct', color='#1f77b4',
                   alpha=0.9, edgecolor='white')
            ax.bar(x_s, fail_rates, w_stk,
                   bottom=corr_rates,
                   label='NN1 Wrong (Attack Success)',
                   color='#d62728', alpha=0.9, edgecolor='white')

            for i, (cr, fr) in enumerate(zip(corr_rates, fail_rates)):
                if fr > 5:
                    ax.text(x_s[i], cr + fr / 2,
                            f'{fr:.0f}%',
                            ha='center', va='center',
                            color='white', fontweight='bold', fontsize=9)
                if cr > 5:
                    ax.text(x_s[i], cr / 2,
                            f'{cr:.0f}%',
                            ha='center', va='center',
                            color='white', fontweight='bold', fontsize=9)

            ax.set_title(title, fontsize=12, fontweight='bold')
            ax.set_xticks(x_s)
            ax.set_xticklabels(families_ordered, fontsize=10)
            ax.set_xlabel('Attack Family', fontsize=11)
            ax.set_ylim(0, 115)
            ax.legend(fontsize=9, loc='upper right')
            ax.grid(axis='y', linestyle='--', alpha=0.4)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)

        axes[0].set_ylabel(
            'Percentage of Adversarial Samples (%)', fontsize=11)
        plt.tight_layout()
        p = os.path.join(DEFENSE_RESULTS_DIR,
                         f'jpeg_defense_testset_breakdown_q{q}.png')
        plt.savefig(p, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"[INFO] Stacked breakdown plot saved: {p}")